## I. Configuration & Setup

This section initializes the analysis pipeline for soil-type-specific modeling. The pipeline processes the dataset, performing categorical regrouping and generating comparison visualizations for variety, harvest mode, and crop cycle distributions.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from pathlib import Path
import re

# ============================================================================
# CONFIGURATION
# ============================================================================

# Input file (output from your first code)
INPUT_FILE = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/final_dataset_for_analysis_with_quality_metrics2_with_field_id.csv'

# Create new output folder
OUTPUT_FOLDER = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis'
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print("="*80)
print("CATEGORICAL DATA REGROUPING AND ANALYSIS PIPELINE")
print("="*80)
print(f"Input file: {INPUT_FILE}")
print(f"Output folder: {OUTPUT_FOLDER}")

# Set Nature-style publication-quality settings
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 9,
    'axes.labelsize': 10,
    'axes.titlesize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 8,
    'figure.titlesize': 12,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'axes.linewidth': 1.0,
    'xtick.major.width': 1.0,
    'ytick.major.width': 1.0,
    'xtick.major.size': 4,
    'ytick.major.size': 4,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.edgecolor': '#333333',
    'xtick.color': '#333333',
    'ytick.color': '#333333',
    'text.color': '#333333'
})

# ============================================================================
# LOAD DATA
# ============================================================================
df = pd.read_csv(INPUT_FILE, sep=',')

print("\n" + "="*80)
print("ORIGINAL DATASET STRUCTURE")
print("="*80)
print(f"Dataset shape: {df.shape}")
print(f"Total observations: {len(df)}")

# ============================================================================
# COLUMN RENAMING (Remove SC_ prefix)
# ============================================================================
print("\n" + "="*80)
print("STEP 1: RENAMING COLUMNS")
print("="*80)

columns_to_rename = {
    "SC_pol_juice_percent": "pol_juice_percent",
    "SC_purity_percent": "purity_percent",
    "SC_fiber_percent": "fiber_percent",
    "SC_pol_cane_percent": "pol_cane_percent",
    "SC_percent_cane_percent": "percent_cane_percent",
    "SC_glucose_cane_percent": "glucose_cane_percent",
    "SC_hugot_sugar": "hugot_sugar",
    "SC_Pur_Tc": "pur_tc",
    "SC_fiber_content_percent": "fiber_content_percent",
    "SC_estimated_recoverable_pol": "estimated_recoverable_pol",
    "SC_juice_tons": "juice_tons",
    "SC_tmss": "tmss",
    "SC_tpj": "tpj",
    "SC_tpc": "tpc",
    "SC_ts": "ts",
    "SC_tsr": "tsr",
    "SC_brix_percent": "brix_percent",
    "SC_tc_per_ha_actual": "tc_per_ha_actual",
    "SC_tc_per_ha_per_week": "tc_per_ha_per_week",
    "SC_tc_per_ha_52_weeks": "tc_per_ha_52_weeks",
    "SC_variety": "variety",
    "SC_irrigation_type": "irrigation_type",
    "SC_harvest": "harvest_mode",
    "SC_harvest_mode": "harvest_mode",
    "SC_crop_cycle": "crop_cycle"
}

actual_renames = {old: new for old, new in columns_to_rename.items() if old in df.columns}
df_work = df.rename(columns=actual_renames)

print(f"Renamed {len(actual_renames)} columns:")
for old, new in actual_renames.items():
    print(f"  {old} → {new}")

# Convert categorical columns to uppercase for consistent processing
for col in ['variety', 'harvest_mode', 'irrigation_type']:
    if col in df_work.columns:
        df_work[col] = df_work[col].astype(str).str.upper().str.strip()

if 'crop_cycle' in df_work.columns:
    df_work['crop_cycle'] = df_work['crop_cycle'].astype(str).str.upper().str.strip()

# ============================================================================
# INITIAL DATA SUMMARY
# ============================================================================
print("\n" + "="*80)
print("INITIAL DATA SUMMARY (BEFORE FILTERING)")
print("="*80)
print(f"Total observations: {len(df_work)}")

if 'tc_per_ha_actual' in df_work.columns and 'brix_percent' in df_work.columns:
    print(f"\nMissing values in key metrics:")
    print(f"  tc_per_ha_actual NA: {df_work['tc_per_ha_actual'].isna().sum()}")
    print(f"  brix_percent NA: {df_work['brix_percent'].isna().sum()}")
    print(f"  Either NA: {(df_work['tc_per_ha_actual'].isna() | df_work['brix_percent'].isna()).sum()}")

if 'variety' in df_work.columns:
    print(f"  variety NA: {df_work['variety'].isna().sum()}")
if 'crop_cycle' in df_work.columns:
    print(f"  crop_cycle NA: {df_work['crop_cycle'].isna().sum()}")

# ============================================================================
# STEP 2: REMOVE OBSERVATIONS WITH NA IN KEY METRICS
# ============================================================================
print("\n" + "="*80)
print("STEP 2: REMOVING OBSERVATIONS WITH NA IN KEY METRICS")
print("="*80)

if 'tc_per_ha_actual' in df_work.columns and 'brix_percent' in df_work.columns:
    initial_count = len(df_work)
    df_work = df_work[df_work['tc_per_ha_actual'].notna() & df_work['brix_percent'].notna()].copy()
    removed_na = initial_count - len(df_work)
    
    print(f"Removed {removed_na} observations with NA in tc_per_ha_actual or brix_percent")
    print(f"Remaining observations: {len(df_work)}")
else:
    print("Key metrics not found, skipping NA removal")

# ============================================================================
# STEP 3: REMOVE SPECIAL CROP CYCLE CATEGORIES
# ============================================================================
print("\n" + "="*80)
print("STEP 3: REMOVING SPECIAL CROP CYCLE CATEGORIES")
print("="*80)

if 'crop_cycle' in df_work.columns:
    categories_to_remove = ['FRICHE', 'LESSIVAGE', 'JACHERE', 'ESSAIS']
    
    for cat in categories_to_remove:
        count = (df_work['crop_cycle'] == cat).sum()
        if count > 0:
            print(f"  Found {count} observations with crop_cycle = {cat}")
    
    initial_count = len(df_work)
    df_work = df_work[~df_work['crop_cycle'].isin(categories_to_remove)].copy()
    removed_special = initial_count - len(df_work)
    
    print(f"\nTotal removed: {removed_special} observations")
    print(f"Remaining observations: {len(df_work)}")
else:
    print("crop_cycle column not found, skipping this step")

# ============================================================================
# STEP 4: REMOVE MIXED VARIETY OBSERVATIONS
# ============================================================================
print("\n" + "="*80)
print("STEP 4: REMOVING MIXED VARIETY OBSERVATIONS")
print("="*80)

if 'variety' in df_work.columns:
    mixed_varieties = ['MIXTE', 'MIXTE D', 'MIXTE MF']
    
    for mix_var in mixed_varieties:
        count = (df_work['variety'] == mix_var).sum()
        if count > 0:
            print(f"  Found {count} observations with variety = {mix_var}")
    
    initial_count = len(df_work)
    df_work = df_work[~df_work['variety'].isin(mixed_varieties)].copy()
    removed_mixed = initial_count - len(df_work)
    
    print(f"\nTotal removed: {removed_mixed} observations")
    print(f"Remaining observations: {len(df_work)}")
else:
    print("variety column not found, skipping this step")

# ============================================================================
# VARIETY REGROUPING: Keep varieties with ≥50 observations
# ============================================================================
print("\n" + "="*80)
print("VARIETY REGROUPING (≥50 OBSERVATIONS)")
print("="*80)

if 'variety' in df_work.columns:
    df_variety_valid = df_work[df_work['variety'].notna()].copy()
    variety_counts = df_variety_valid['variety'].value_counts().sort_values(ascending=False)
    total_obs_variety = len(df_variety_valid)
    
    print(f"\nTotal observations with valid variety: {total_obs_variety}")
    
    print("\nOriginal Variety Distribution:")
    print("-" * 80)
    for variety, count in variety_counts.items():
        percentage = (count / total_obs_variety) * 100
        print(f"  {variety:<30} : {count:>6} obs ({percentage:>5.2f}%)")
    
    # Regroup varieties with < 50 observations
    min_obs_threshold = 50
    varieties_to_keep = variety_counts[variety_counts >= min_obs_threshold].index.tolist()
    
    print(f"\nVarieties with ≥ {min_obs_threshold} observations (kept):")
    for var in varieties_to_keep:
        print(f"  - {var} ({variety_counts[var]} obs)")
    
    varieties_to_regroup = variety_counts[variety_counts < min_obs_threshold].index.tolist()
    total_regrouped = variety_counts[variety_counts < min_obs_threshold].sum()
    
    print(f"\nVarieties with < {min_obs_threshold} observations (regrouped as 'OTHER VARIETIES'):")
    for var in varieties_to_regroup:
        print(f"  - {var} ({variety_counts[var]} obs)")
    print(f"\nTotal regrouped: {total_regrouped} observations")
    
    def regroup_variety(variety):
        if pd.isna(variety):
            return np.nan
        if variety in varieties_to_keep:
            return variety
        else:
            return 'OTHER VARIETIES'
    
    df_work['variety'] = df_work['variety'].apply(regroup_variety)
    
    variety_regrouped_counts = df_work['variety'].value_counts().sort_values(ascending=False)
    
    print("\nRegrouped Variety Distribution:")
    print("-" * 80)
    for variety, count in variety_regrouped_counts.items():
        percentage = (count / total_obs_variety) * 100
        print(f"  {variety:<30} : {count:>6} obs ({percentage:>5.2f}%)")

# ============================================================================
# HARVEST MODE REGROUPING: Keep MANUELLE and MECANIQUE only
# ============================================================================
print("\n" + "="*80)
print("HARVEST MODE REGROUPING (MANUELLE/MECANIQUE ONLY)")
print("="*80)

if 'harvest_mode' in df_work.columns:
    df_harvest_valid = df_work[df_work['harvest_mode'].notna()].copy()
    harvest_counts = df_harvest_valid['harvest_mode'].value_counts().sort_values(ascending=False)
    total_obs_harvest = len(df_harvest_valid)
    
    print(f"\nTotal observations with valid harvest mode: {total_obs_harvest}")
    
    print("\nOriginal Harvest Mode Distribution:")
    print("-" * 80)
    for mode, count in harvest_counts.items():
        percentage = (count / total_obs_harvest) * 100
        print(f"  {mode:<30} : {count:>6} obs ({percentage:>5.2f}%)")
    
    def regroup_harvest_mode(mode):
        if pd.isna(mode):
            return np.nan
        mode_upper = str(mode).upper().strip()
        if mode_upper in ['MANUELLE', 'MECANIQUE']:
            return mode_upper
        else:
            return np.nan
    
    df_work['harvest_mode'] = df_work['harvest_mode'].apply(regroup_harvest_mode)
    
    df_harvest_regroup_valid = df_work[df_work['harvest_mode'].notna()].copy()
    harvest_regrouped_counts = df_harvest_regroup_valid['harvest_mode'].value_counts().sort_values(ascending=False)
    total_regroup_harvest = len(df_harvest_regroup_valid)
    total_na_regroup = df_work['harvest_mode'].isna().sum()
    
    print("\nRegrouped Harvest Mode Distribution:")
    print("-" * 80)
    for mode, count in harvest_regrouped_counts.items():
        percentage = (count / total_regroup_harvest) * 100
        print(f"  {mode:<30} : {count:>6} obs ({percentage:>5.2f}%)")
    print(f"  {'NA/Unknown':<30} : {total_na_regroup:>6} obs ({(total_na_regroup/len(df_work))*100:>5.2f}%)")

# ============================================================================
# CROP CYCLE REGROUPING
# ============================================================================
print("\n" + "="*80)
print("CROP CYCLE REGROUPING")
print("="*80)

if 'crop_cycle' in df_work.columns:
    df_cycle_valid = df_work[df_work['crop_cycle'].notna()].copy()
    crop_cycle_counts = df_cycle_valid['crop_cycle'].value_counts().sort_index()
    total_obs_cycle = len(df_cycle_valid)
    
    print(f"\nTotal observations with valid crop cycle: {total_obs_cycle}")
    
    print("\nOriginal Crop Cycle Distribution:")
    print("-" * 80)
    for cycle, count in crop_cycle_counts.items():
        percentage = (count / total_obs_cycle) * 100
        print(f"  {cycle:<30} : {count:>6} obs ({percentage:>5.2f}%)")
    
    # Standardize crop cycle naming
    def standardize_crop_cycle(cycle):
        if pd.isna(cycle):
            return np.nan
        
        cycle_str = str(cycle).upper().strip()
        
        try:
            cycle_num = int(float(cycle_str))
            
            if cycle_num == 0:
                return 'Plant Cane'
            elif cycle_num == 1:
                return '1st Ratoon'
            elif cycle_num == 2:
                return '2nd Ratoon'
            elif cycle_num == 3:
                return '3rd Ratoon'
            elif cycle_num >= 4:
                return f'{cycle_num}th Ratoon'
            else:
                return np.nan
        except:
            return np.nan
    
    df_work['crop_cycle_standardized'] = df_work['crop_cycle'].apply(standardize_crop_cycle)
    
    df_cycle_std_valid = df_work[df_work['crop_cycle_standardized'].notna()].copy()
    cycle_std_counts = df_cycle_std_valid['crop_cycle_standardized'].value_counts()
    
    print("\nStandardized Crop Cycle Counts:")
    print("-" * 80)
    
    # Build ordered list
    cycle_names_ordered = ['Plant Cane']
    for i in range(1, 20):
        if i == 1:
            cycle_name = '1st Ratoon'
        elif i == 2:
            cycle_name = '2nd Ratoon'
        elif i == 3:
            cycle_name = '3rd Ratoon'
        else:
            cycle_name = f'{i}th Ratoon'
        
        if cycle_name in cycle_std_counts.index:
            cycle_names_ordered.append(cycle_name)
        else:
            break
    
    for cycle_name in cycle_names_ordered:
        if cycle_name in cycle_std_counts.index:
            count = cycle_std_counts[cycle_name]
            print(f"  {cycle_name:<30} : {count:>6} obs")
    
    # Identify cycles with ≥ 50 observations
    min_obs_threshold = 50
    cycles_to_keep = []
    last_kept_cycle_num = -1
    
    for i, cycle_name in enumerate(cycle_names_ordered):
        if cycle_name in cycle_std_counts.index:
            count = cycle_std_counts[cycle_name]
            if count >= min_obs_threshold:
                cycles_to_keep.append(cycle_name)
                last_kept_cycle_num = i
    
    print(f"\nCrop cycles with ≥ {min_obs_threshold} observations:")
    for cycle_name in cycles_to_keep:
        print(f"  - {cycle_name} ({cycle_std_counts[cycle_name]} obs)")
    
    # Determine "beyond" label
    if cycles_to_keep:
        last_cycle_name = cycles_to_keep[-1]
        if last_cycle_name == 'Plant Cane':
            last_cycle_number = 0
        else:
            match = re.search(r'(\d+)', last_cycle_name)
            if match:
                last_cycle_number = int(match.group(1))
            else:
                last_cycle_number = 0
        
        beyond_cycle_num = last_cycle_number + 1
        if beyond_cycle_num == 1:
            beyond_label = "Beyond 1st Ratoon"
        elif beyond_cycle_num == 2:
            beyond_label = "Beyond 2nd Ratoon"
        elif beyond_cycle_num == 3:
            beyond_label = "Beyond 3rd Ratoon"
        else:
            beyond_label = f"Beyond {beyond_cycle_num}th Ratoon"
    else:
        beyond_label = "Beyond Plant Cane"
    
    print(f"\nCycles to be regrouped as '{beyond_label}':")
    for i, cycle_name in enumerate(cycle_names_ordered):
        if i > last_kept_cycle_num and cycle_name in cycle_std_counts.index:
            print(f"  - {cycle_name} ({cycle_std_counts[cycle_name]} obs)")
    
    def regroup_crop_cycle(cycle_std):
        if pd.isna(cycle_std):
            return np.nan
        if cycle_std in cycles_to_keep:
            return cycle_std
        else:
            return beyond_label
    
    df_work['crop_cycle'] = df_work['crop_cycle_standardized'].apply(regroup_crop_cycle)
    df_work = df_work.drop(columns=['crop_cycle_standardized'])
    
    # Create the proper order for plotting
    cycle_order = cycles_to_keep + [beyond_label]
    
    df_cycle_regroup_valid = df_work[df_work['crop_cycle'].notna()].copy()
    
    # Create a custom sorter for proper ordering
    def get_cycle_sort_key(cycle_name):
        """Create sort key for proper cycle ordering"""
        if cycle_name == 'Plant Cane':
            return (0, 'Plant Cane')
        elif cycle_name.startswith('Beyond'):
            return (float('inf'), cycle_name)
        else:
            match = re.search(r'(\d+)', cycle_name)
            if match:
                cycle_num = int(match.group(1))
                return (cycle_num, cycle_name)
            return (float('inf'), cycle_name)
    
    # Sort the regrouped counts using the custom sorter
    crop_cycle_regrouped_counts = df_cycle_regroup_valid['crop_cycle'].value_counts()
    sorted_index = sorted(crop_cycle_regrouped_counts.index, key=get_cycle_sort_key)
    crop_cycle_regrouped_counts = crop_cycle_regrouped_counts.reindex(sorted_index)
    
    total_regroup_cycle = len(df_cycle_regroup_valid)
    
    print("\nRegrouped Crop Cycle Distribution (in proper order):")
    print("-" * 80)
    for cycle in crop_cycle_regrouped_counts.index:
        count = crop_cycle_regrouped_counts[cycle]
        percentage = (count / total_regroup_cycle) * 100
        print(f"  {cycle:<30} : {count:>6} obs ({percentage:>5.2f}%)")

# ============================================================================
# CREATE VISUALIZATIONS
# ============================================================================
print("\n" + "="*80)
print("CREATING VISUALIZATIONS")
print("="*80)

color_main = '#50B848'  # Nature green
color_accent = '#FF6B35'  # Accent orange

# Variety Distribution 
if 'variety' in df_work.columns:
    # Calculate adaptive height
    n_categories = len(variety_regrouped_counts)
    if n_categories <= 3:
        fig_height = 2.5
        bar_height = 0.6
    elif n_categories <= 6:
        fig_height = 3.5
        bar_height = 0.55
    elif n_categories <= 10:
        fig_height = 5.0
        bar_height = 0.5
    else:
        fig_height = min(8.0, 3.0 + n_categories * 0.35)
        bar_height = 0.45
    
    fig, ax = plt.subplots(figsize=(7, fig_height))
    
    variety_plot = variety_regrouped_counts.sort_values(ascending=True)
    colors = [color_accent if var == 'OTHER VARIETIES' else color_main for var in variety_plot.index]
    
    y_pos = np.arange(len(variety_plot))
    bars = ax.barh(y_pos, variety_plot.values, height=bar_height, 
                   color=colors, alpha=0.85, edgecolor='white', linewidth=0.5)
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(variety_plot.index, fontsize=10, fontweight='normal')
    ax.set_xlabel('Number of Observations', fontsize=10, fontweight='600', color='#333333')
    ax.set_title(f'Regrouped Variety Distribution (≥{min_obs_threshold} obs + Other)', 
                 fontsize=11, fontweight='bold', pad=12, color='#2C3E50')
    
    # Minimal grid
    ax.grid(True, alpha=0.15, axis='x', linestyle='-', linewidth=0.5, color='#CCCCCC')
    ax.set_axisbelow(True)
    ax.spines['left'].set_color('#CCCCCC')
    ax.spines['bottom'].set_color('#CCCCCC')
    
    # Add value labels
    max_val = variety_plot.values.max()
    for i, (variety, count) in enumerate(variety_plot.items()):
        ax.text(count + max_val*0.02, i, f'{int(count):,}', 
                va='center', ha='left', fontsize=9, fontweight='600', color='#333333')
    
    ax.set_xlim(0, max_val * 1.15)
    
    plt.tight_layout()
    
    output_path = os.path.join(OUTPUT_FOLDER, 'variety_distribution_regrouped.png')
    plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Saved: {output_path}")
    
    output_path_pdf = os.path.join(OUTPUT_FOLDER, 'variety_distribution_regrouped.pdf')
    plt.savefig(output_path_pdf, format='pdf', bbox_inches='tight', facecolor='white')
    print(f"Saved: {output_path_pdf}")
    
    plt.close()

# Harvest Mode Distribution
if 'harvest_mode' in df_work.columns:
    n_categories = len(harvest_regrouped_counts)
    if n_categories <= 3:
        fig_height = 2.5
        bar_height = 0.6
    else:
        fig_height = 3.5
        bar_height = 0.55
    
    fig, ax = plt.subplots(figsize=(6, fig_height))
    
    harvest_plot = harvest_regrouped_counts.sort_values(ascending=True)
    y_pos = np.arange(len(harvest_plot))
    
    bars = ax.barh(y_pos, harvest_plot.values, height=bar_height, 
                   color=color_main, alpha=0.85, edgecolor='white', linewidth=0.5)
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(harvest_plot.index, fontsize=10, fontweight='normal')
    ax.set_xlabel('Number of Observations', fontsize=10, fontweight='600', color='#333333')
    ax.set_title('Regrouped Harvest Mode (MANUELLE/MECANIQUE only)', 
                 fontsize=11, fontweight='bold', pad=12, color='#2C3E50')
    
    # Minimal grid
    ax.grid(True, alpha=0.15, axis='x', linestyle='-', linewidth=0.5, color='#CCCCCC')
    ax.set_axisbelow(True)
    ax.spines['left'].set_color('#CCCCCC')
    ax.spines['bottom'].set_color('#CCCCCC')
    
    # Add value labels
    max_val = harvest_plot.values.max()
    for i, count in enumerate(harvest_plot.values):
        ax.text(count + max_val*0.02, i, f'{int(count):,}', 
                va='center', ha='left', fontsize=9, fontweight='600', color='#333333')
    
    ax.set_xlim(0, max_val * 1.15)
    
    plt.tight_layout()
    
    output_path = os.path.join(OUTPUT_FOLDER, 'harvest_mode_regrouped.png')
    plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Saved: {output_path}")
    
    plt.close()

# Crop Cycle Distribution 
if 'crop_cycle' in df_work.columns:
    n_categories = len(crop_cycle_regrouped_counts)
    if n_categories <= 3:
        fig_height = 2.5
        bar_height = 0.6
    elif n_categories <= 6:
        fig_height = 3.5
        bar_height = 0.55
    elif n_categories <= 10:
        fig_height = 5.0
        bar_height = 0.5
    else:
        fig_height = min(8.0, 3.0 + n_categories * 0.35)
        bar_height = 0.45
    
    fig, ax = plt.subplots(figsize=(7, fig_height))
    
    # Reverse the order for bottom-to-top display
    crop_plot = crop_cycle_regrouped_counts[::-1]
    y_pos = np.arange(len(crop_plot))
    
    bars = ax.barh(y_pos, crop_plot.values, height=bar_height, 
                   color=color_main, alpha=0.85, edgecolor='white', linewidth=0.5)
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(crop_plot.index, fontsize=10, fontweight='normal')
    ax.set_xlabel('Number of Observations', fontsize=10, fontweight='600', color='#333333')
    ax.set_title('Regrouped Crop Cycle Distribution (in proper order)', 
                 fontsize=11, fontweight='bold', pad=12, color='#2C3E50')
    
    # Minimal grid
    ax.grid(True, alpha=0.15, axis='x', linestyle='-', linewidth=0.5, color='#CCCCCC')
    ax.set_axisbelow(True)
    ax.spines['left'].set_color('#CCCCCC')
    ax.spines['bottom'].set_color('#CCCCCC')
    
    # Add value labels
    max_val = crop_plot.values.max()
    for i, count in enumerate(crop_plot.values):
        ax.text(count + max_val*0.02, i, f'{int(count):,}', 
                va='center', ha='left', fontsize=9, fontweight='600', color='#333333')
    
    ax.set_xlim(0, max_val * 1.15)
    
    plt.tight_layout()
    
    output_path = os.path.join(OUTPUT_FOLDER, 'crop_cycle_regrouped.png')
    plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Saved: {output_path}")
    
    plt.close()

# ============================================================================
# SAVE FINAL DATASET
# ============================================================================
print("\n" + "="*80)
print("SAVING FINAL REGROUPED DATASET")
print("="*80)

output_csv = os.path.join(OUTPUT_FOLDER, 'final_dataset_regrouped_categorical_with_field_id.csv')
df_work.to_csv(output_csv, index=False, sep=',')
print(f"Saved: {output_csv}")

# ============================================================================
# SUMMARY STATISTICS
# ============================================================================
print("\n" + "="*80)
print("FINAL SUMMARY STATISTICS")
print("="*80)

print(f"\nOriginal dataset: {len(df)} observations")

CATEGORICAL DATA REGROUPING AND ANALYSIS PIPELINE
Input file: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/final_dataset_for_analysis_with_quality_metrics2_with_field_id.csv
Output folder: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis

ORIGINAL DATASET STRUCTURE
Dataset shape: (19495, 70)
Total observations: 19495

STEP 1: RENAMING COLUMNS
Renamed 15 columns:
  SC_pol_juice_percent → pol_juice_percent
  SC_purity_percent → purity_percent
  SC_fiber_percent → fiber_percent
  SC_pol_cane_percent → pol_cane_percent
  SC_glucose_cane_percent → glucose_cane_percent
  SC_hugot_sugar → hugot_sugar
  SC_estimated_recoverable_pol → estimated_recoverable_pol
  SC_brix_percent → brix_percent
  SC_tc_per_ha_actual → tc_per_ha_actual
  SC_tc_per_ha_per_week → tc_per_ha_per_week
  SC_tc_per_ha_52_weeks → tc_per_ha_52_weeks
  SC_variety → variety
  SC_irrigation_type → irrigation_type
  SC_harvest → harvest_mode
  SC_crop_cycle → crop_cycle

INITIAL DATA SU

## 2. Environment Setup & Data Loading

This section initializes the analysis environment and loads the sugarcane dataset. 


In [ ]:
"""
CATEGORICAL DATA REGROUPING AND ANALYSIS PIPELINE - SHAP STYLE
===============================================================

Author: Agricultural ML Analysis
Date: 2025
"""

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np
import os
from pathlib import Path

# ============================================================================
# FONT SETUP
# ============================================================================

available_fonts = [f.name for f in fm.fontManager.ttflist]
if 'Arial' in available_fonts:
    font_family = 'Arial'
elif 'Helvetica' in available_fonts:
    font_family = 'Helvetica'
elif 'DejaVu Sans' in available_fonts:
    font_family = 'DejaVu Sans'
elif 'Liberation Sans' in available_fonts:
    font_family = 'Liberation Sans'
else:
    font_family = 'sans-serif'

plt.rcParams.update({
    'font.family': font_family,
    'font.size': 14,
    'axes.labelsize': 14,
    'axes.titlesize': 14,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 12,
    'figure.titlesize': 16,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': False,
    'axes.axisbelow': True,
    'axes.edgecolor': '#333333',
    'text.color': '#333333',
    'axes.labelcolor': '#333333',
    'xtick.color': '#333333',
    'ytick.color': '#333333',
    'axes.linewidth': 1.2,
    'xtick.major.width': 1.0,
    'ytick.major.width': 1.0,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'text.usetex': False,
    'mathtext.default': 'regular',
})

PRIMARY_COLOR = '#18A558'  # Green for main categories
SECONDARY_COLOR = '#F18F01'  # Orange for "OTHER" categories
TERTIARY_COLOR = '#2E86AB'  # Blue for irrigation

# ============================================================================
# CONFIGURATION
# ============================================================================

# Input file
INPUT_FILE = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/final_dataset_for_analysis_with_quality_metrics2.csv'

# Create new output folder
OUTPUT_FOLDER = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis'
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print("="*80)
print("CATEGORICAL DATA REGROUPING AND ANALYSIS PIPELINE - SHAP STYLE")
print("="*80)
print(f"Input file: {INPUT_FILE}")
print(f"Output folder: {OUTPUT_FOLDER}")

# ============================================================================
# LOAD DATA
# ============================================================================
df = pd.read_csv(INPUT_FILE, sep=',')

print("\n" + "="*80)
print("ORIGINAL DATASET STRUCTURE")
print("="*80)
print(f"Dataset shape: {df.shape}")
print(f"Total observations: {len(df)}")

# ============================================================================
# COLUMN RENAMING (Remove SC_ prefix)
# ============================================================================
print("\n" + "="*80)
print("STEP 1: RENAMING COLUMNS")
print("="*80)

columns_to_rename = {
    "SC_pol_juice_percent": "pol_juice_percent",
    "SC_purity_percent": "purity_percent",
    "SC_fiber_percent": "fiber_percent",
    "SC_pol_cane_percent": "pol_cane_percent",
    "SC_percent_cane_percent": "percent_cane_percent",
    "SC_glucose_cane_percent": "glucose_cane_percent",
    "SC_hugot_sugar": "hugot_sugar",
    "SC_Pur_Tc": "pur_tc",
    "SC_fiber_content_percent": "fiber_content_percent",
    "SC_estimated_recoverable_pol": "estimated_recoverable_pol",
    "SC_juice_tons": "juice_tons",
    "SC_tmss": "tmss",
    "SC_tpj": "tpj",
    "SC_tpc": "tpc",
    "SC_ts": "ts",
    "SC_tsr": "tsr",
    "SC_brix_percent": "brix_percent",
    "SC_tc_per_ha_actual": "tc_per_ha_actual",
    "SC_tc_per_ha_per_week": "tc_per_ha_per_week",
    "SC_tc_per_ha_52_weeks": "tc_per_ha_52_weeks",
    "SC_variety": "variety",
    "SC_irrigation_type": "irrigation_type",
    "SC_harvest": "harvest_mode",
    "SC_harvest_mode": "harvest_mode",
    "SC_crop_cycle": "crop_cycle"
}

actual_renames = {old: new for old, new in columns_to_rename.items() if old in df.columns}
df_work = df.rename(columns=actual_renames)

print(f"Renamed {len(actual_renames)} columns:")
for old, new in actual_renames.items():
    print(f"  {old} → {new}")

# Convert categorical columns to uppercase for consistent processing
for col in ['variety', 'harvest_mode', 'irrigation_type']:
    if col in df_work.columns:
        df_work[col] = df_work[col].astype(str).str.upper().str.strip()

if 'crop_cycle' in df_work.columns:
    df_work['crop_cycle'] = df_work['crop_cycle'].astype(str).str.upper().str.strip()

# ============================================================================
# INITIAL DATA SUMMARY
# ============================================================================
print("\n" + "="*80)
print("INITIAL DATA SUMMARY (BEFORE FILTERING)")
print("="*80)
print(f"Total observations: {len(df_work)}")

if 'tc_per_ha_actual' in df_work.columns and 'brix_percent' in df_work.columns:
    print(f"\nMissing values in key metrics:")
    print(f"  tc_per_ha_actual NA: {df_work['tc_per_ha_actual'].isna().sum()}")
    print(f"  brix_percent NA: {df_work['brix_percent'].isna().sum()}")
    print(f"  Either NA: {(df_work['tc_per_ha_actual'].isna() | df_work['brix_percent'].isna()).sum()}")

if 'variety' in df_work.columns:
    print(f"  variety NA: {df_work['variety'].isna().sum()}")
if 'crop_cycle' in df_work.columns:
    print(f"  crop_cycle NA: {df_work['crop_cycle'].isna().sum()}")

# ============================================================================
# STEP 2: REMOVE OBSERVATIONS WITH NA IN KEY METRICS
# ============================================================================
print("\n" + "="*80)
print("STEP 2: REMOVING OBSERVATIONS WITH NA IN KEY METRICS")
print("="*80)

if 'tc_per_ha_actual' in df_work.columns and 'brix_percent' in df_work.columns:
    initial_count = len(df_work)
    df_work = df_work[df_work['tc_per_ha_actual'].notna() & df_work['brix_percent'].notna()].copy()
    removed_na = initial_count - len(df_work)
    
    print(f"Removed {removed_na} observations with NA in tc_per_ha_actual or brix_percent")
    print(f"Remaining observations: {len(df_work)}")
else:
    print("Key metrics not found, skipping NA removal")

# ============================================================================
# STEP 3: REMOVE SPECIAL CROP CYCLE CATEGORIES
# ============================================================================
print("\n" + "="*80)
print("STEP 3: REMOVING SPECIAL CROP CYCLE CATEGORIES")
print("="*80)

if 'crop_cycle' in df_work.columns:
    categories_to_remove = ['FRICHE', 'LESSIVAGE', 'JACHERE', 'ESSAIS']
    
    for cat in categories_to_remove:
        count = (df_work['crop_cycle'] == cat).sum()
        if count > 0:
            print(f"  Found {count} observations with crop_cycle = {cat}")
    
    initial_count = len(df_work)
    df_work = df_work[~df_work['crop_cycle'].isin(categories_to_remove)].copy()
    removed_special = initial_count - len(df_work)
    
    print(f"\nTotal removed: {removed_special} observations")
    print(f"Remaining observations: {len(df_work)}")
else:
    print("crop_cycle column not found, skipping this step")

# ============================================================================
# STEP 4: REMOVE MIXED VARIETY OBSERVATIONS
# ============================================================================
print("\n" + "="*80)
print("STEP 4: REMOVING MIXED VARIETY OBSERVATIONS")
print("="*80)

if 'variety' in df_work.columns:
    mixed_varieties = ['MIXTE', 'MIXTE D', 'MIXTE MF']
    
    for mix_var in mixed_varieties:
        count = (df_work['variety'] == mix_var).sum()
        if count > 0:
            print(f"  Found {count} observations with variety = {mix_var}")
    
    initial_count = len(df_work)
    df_work = df_work[~df_work['variety'].isin(mixed_varieties)].copy()
    removed_mixed = initial_count - len(df_work)
    
    print(f"\nTotal removed: {removed_mixed} observations")
    print(f"Remaining observations: {len(df_work)}")
else:
    print("variety column not found, skipping this step")

# ============================================================================
# VARIETY REGROUPING: Keep varieties with ≥50 observations
# ============================================================================
print("\n" + "="*80)
print("VARIETY REGROUPING (≥50 OBSERVATIONS)")
print("="*80)

if 'variety' in df_work.columns:
    df_variety_valid = df_work[df_work['variety'].notna()].copy()
    variety_counts = df_variety_valid['variety'].value_counts().sort_values(ascending=False)
    total_obs_variety = len(df_variety_valid)
    
    print(f"\nTotal observations with valid variety: {total_obs_variety}")
    
    print("\nOriginal Variety Distribution:")
    print("-" * 80)
    for variety, count in variety_counts.items():
        percentage = (count / total_obs_variety) * 100
        print(f"  {variety:<30} : {count:>6} obs ({percentage:>5.2f}%)")
    
    # Regroup varieties with < 50 observations
    min_obs_threshold = 50
    varieties_to_keep = variety_counts[variety_counts >= min_obs_threshold].index.tolist()
    
    print(f"\nVarieties with ≥ {min_obs_threshold} observations (kept):")
    for var in varieties_to_keep:
        print(f"  - {var} ({variety_counts[var]} obs)")
    
    varieties_to_regroup = variety_counts[variety_counts < min_obs_threshold].index.tolist()
    total_regrouped = variety_counts[variety_counts < min_obs_threshold].sum()
    
    print(f"\nVarieties with < {min_obs_threshold} observations (regrouped as 'OTHER VARIETIES'):")
    for var in varieties_to_regroup:
        print(f"  - {var} ({variety_counts[var]} obs)")
    print(f"\nTotal regrouped: {total_regrouped} observations")
    
    def regroup_variety(variety):
        if pd.isna(variety):
            return np.nan
        if variety in varieties_to_keep:
            return variety
        else:
            return 'OTHER VARIETIES'
    
    df_work['variety'] = df_work['variety'].apply(regroup_variety)
    
    variety_regrouped_counts = df_work['variety'].value_counts().sort_values(ascending=False)
    
    print("\nRegrouped Variety Distribution:")
    print("-" * 80)
    for variety, count in variety_regrouped_counts.items():
        percentage = (count / total_obs_variety) * 100
        print(f"  {variety:<30} : {count:>6} obs ({percentage:>5.2f}%)")

# ============================================================================
# HARVEST MODE REGROUPING: Keep MANUELLE and MECANIQUE only
# ============================================================================
print("\n" + "="*80)
print("HARVEST MODE REGROUPING (MANUELLE/MECANIQUE ONLY)")
print("="*80)

if 'harvest_mode' in df_work.columns:
    df_harvest_valid = df_work[df_work['harvest_mode'].notna()].copy()
    harvest_counts = df_harvest_valid['harvest_mode'].value_counts().sort_values(ascending=False)
    total_obs_harvest = len(df_harvest_valid)
    
    print(f"\nTotal observations with valid harvest mode: {total_obs_harvest}")
    
    print("\nOriginal Harvest Mode Distribution:")
    print("-" * 80)
    for mode, count in harvest_counts.items():
        percentage = (count / total_obs_harvest) * 100
        print(f"  {mode:<30} : {count:>6} obs ({percentage:>5.2f}%)")
    
    def regroup_harvest_mode(mode):
        if pd.isna(mode):
            return np.nan
        mode_upper = str(mode).upper().strip()
        if mode_upper in ['MANUELLE', 'MECANIQUE']:
            return mode_upper
        else:
            return np.nan
    
    df_work['harvest_mode'] = df_work['harvest_mode'].apply(regroup_harvest_mode)
    
    df_harvest_regroup_valid = df_work[df_work['harvest_mode'].notna()].copy()
    harvest_regrouped_counts = df_harvest_regroup_valid['harvest_mode'].value_counts().sort_values(ascending=False)
    total_regroup_harvest = len(df_harvest_regroup_valid)
    total_na_regroup = df_work['harvest_mode'].isna().sum()
    
    print("\nRegrouped Harvest Mode Distribution:")
    print("-" * 80)
    for mode, count in harvest_regrouped_counts.items():
        percentage = (count / total_regroup_harvest) * 100
        print(f"  {mode:<30} : {count:>6} obs ({percentage:>5.2f}%)")
    print(f"  {'NA/Unknown':<30} : {total_na_regroup:>6} obs ({(total_na_regroup/len(df_work))*100:>5.2f}%)")

# ============================================================================
# IRRIGATION TYPE PROCESSING
# ============================================================================
print("\n" + "="*80)
print("IRRIGATION TYPE PROCESSING")
print("="*80)

if 'irrigation_type' in df_work.columns:
    df_irrigation_valid = df_work[df_work['irrigation_type'].notna()].copy()
    irrigation_counts = df_irrigation_valid['irrigation_type'].value_counts().sort_values(ascending=False)
    total_obs_irrigation = len(df_irrigation_valid)
    
    print(f"\nTotal observations with valid irrigation type: {total_obs_irrigation}")
    
    print("\nIrrigation Type Distribution:")
    print("-" * 80)
    for irr_type, count in irrigation_counts.items():
        percentage = (count / total_obs_irrigation) * 100
        print(f"  {irr_type:<30} : {count:>6} obs ({percentage:>5.2f}%)")

# ============================================================================
# CROP CYCLE REGROUPING
# ============================================================================
print("\n" + "="*80)
print("CROP CYCLE REGROUPING")
print("="*80)

if 'crop_cycle' in df_work.columns:
    df_cycle_valid = df_work[df_work['crop_cycle'].notna()].copy()
    crop_cycle_counts = df_cycle_valid['crop_cycle'].value_counts().sort_index()
    total_obs_cycle = len(df_cycle_valid)
    
    print(f"\nTotal observations with valid crop cycle: {total_obs_cycle}")
    
    print("\nOriginal Crop Cycle Distribution:")
    print("-" * 80)
    for cycle, count in crop_cycle_counts.items():
        percentage = (count / total_obs_cycle) * 100
        print(f"  {cycle:<30} : {count:>6} obs ({percentage:>5.2f}%)")
    
    # Standardize crop cycle naming
    def standardize_crop_cycle(cycle):
        if pd.isna(cycle):
            return np.nan
        
        cycle_str = str(cycle).upper().strip()
        
        try:
            cycle_num = int(float(cycle_str))
            
            if cycle_num == 0:
                return 'Plant Cane'
            elif cycle_num == 1:
                return '1st Ratoon'
            elif cycle_num == 2:
                return '2nd Ratoon'
            elif cycle_num == 3:
                return '3rd Ratoon'
            elif cycle_num >= 4:
                return f'{cycle_num}th Ratoon'
            else:
                return np.nan
        except:
            return np.nan
    
    df_work['crop_cycle_standardized'] = df_work['crop_cycle'].apply(standardize_crop_cycle)
    
    df_cycle_std_valid = df_work[df_work['crop_cycle_standardized'].notna()].copy()
    cycle_std_counts = df_cycle_std_valid['crop_cycle_standardized'].value_counts()
    
    print("\nStandardized Crop Cycle Counts:")
    print("-" * 80)
    
    # Build ordered list
    cycle_names_ordered = ['Plant Cane']
    for i in range(1, 20):
        if i == 1:
            cycle_name = '1st Ratoon'
        elif i == 2:
            cycle_name = '2nd Ratoon'
        elif i == 3:
            cycle_name = '3rd Ratoon'
        else:
            cycle_name = f'{i}th Ratoon'
        
        if cycle_name in cycle_std_counts.index:
            cycle_names_ordered.append(cycle_name)
        else:
            break
    
    for cycle_name in cycle_names_ordered:
        if cycle_name in cycle_std_counts.index:
            count = cycle_std_counts[cycle_name]
            print(f"  {cycle_name:<30} : {count:>6} obs")
    
    # Identify cycles with ≥ 50 observations
    min_obs_threshold = 50
    cycles_to_keep = []
    last_kept_cycle_num = -1
    
    for i, cycle_name in enumerate(cycle_names_ordered):
        if cycle_name in cycle_std_counts.index:
            count = cycle_std_counts[cycle_name]
            if count >= min_obs_threshold:
                cycles_to_keep.append(cycle_name)
                last_kept_cycle_num = i
    
    print(f"\nCrop cycles with ≥ {min_obs_threshold} observations:")
    for cycle_name in cycles_to_keep:
        print(f"  - {cycle_name} ({cycle_std_counts[cycle_name]} obs)")
    
    # Determine "beyond" label
    if cycles_to_keep:
        last_cycle_name = cycles_to_keep[-1]
        if last_cycle_name == 'Plant Cane':
            last_cycle_number = 0
        else:
            import re
            match = re.search(r'(\d+)', last_cycle_name)
            if match:
                last_cycle_number = int(match.group(1))
            else:
                last_cycle_number = 0
        
        beyond_cycle_num = last_cycle_number + 1
        if beyond_cycle_num == 1:
            beyond_label = "Beyond 1st Ratoon"
        elif beyond_cycle_num == 2:
            beyond_label = "Beyond 2nd Ratoon"
        elif beyond_cycle_num == 3:
            beyond_label = "Beyond 3rd Ratoon"
        else:
            beyond_label = f"Beyond {beyond_cycle_num}th Ratoon"
    else:
        beyond_label = "Beyond Plant Cane"
    
    print(f"\nCycles to be regrouped as '{beyond_label}':")
    for i, cycle_name in enumerate(cycle_names_ordered):
        if i > last_kept_cycle_num and cycle_name in cycle_std_counts.index:
            print(f"  - {cycle_name} ({cycle_std_counts[cycle_name]} obs)")
    
    def regroup_crop_cycle(cycle_std):
        if pd.isna(cycle_std):
            return np.nan
        if cycle_std in cycles_to_keep:
            return cycle_std
        else:
            return beyond_label
    
    df_work['crop_cycle'] = df_work['crop_cycle_standardized'].apply(regroup_crop_cycle)
    df_work = df_work.drop(columns=['crop_cycle_standardized'])
    
    cycle_order = cycles_to_keep + [beyond_label]
    
    df_cycle_regroup_valid = df_work[df_work['crop_cycle'].notna()].copy()
    crop_cycle_regrouped_counts = df_cycle_regroup_valid['crop_cycle'].value_counts()
    total_regroup_cycle = len(df_cycle_regroup_valid)
    
    print("\nRegrouped Crop Cycle Distribution:")
    print("-" * 80)
    for cycle in cycle_order:
        if cycle in crop_cycle_regrouped_counts.index:
            count = crop_cycle_regrouped_counts[cycle]
            percentage = (count / total_regroup_cycle) * 100
            print(f"  {cycle:<30} : {count:>6} obs ({percentage:>5.2f}%)")

# ============================================================================
# CREATE VISUALIZATIONS 
# ============================================================================
print("\n" + "="*80)

print("="*80)

# Define consistent slim bar height for all plots
SLIM_BAR_HEIGHT = 0.5


def create_shap_style_barplot(ax, data_counts, title, panel_label, 
                              highlight_other=False, color_override=None):
    """Create horizontal bar plot with bigger fonts and slim bars"""
    # Sort data
    plot_data = data_counts.sort_values(ascending=True)
    
    # Assign colors
    if color_override:
        colors = [color_override] * len(plot_data)
    elif highlight_other:
        colors = [SECONDARY_COLOR if 'OTHER' in str(cat) else PRIMARY_COLOR 
                 for cat in plot_data.index]
    else:
        colors = [PRIMARY_COLOR] * len(plot_data)
    
    # Create slim bars 
    y_pos = np.arange(len(plot_data))
    bars = ax.barh(y_pos, plot_data.values, height=SLIM_BAR_HEIGHT, 
                   color=colors, alpha=0.8, edgecolor='white', linewidth=0.5)
    
    # Set y-axis with larger font (11)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(plot_data.index, fontsize=11, fontweight='normal')
    
    # Set x-axis with larger font (12)
    ax.set_xlabel('Number of Observations', fontweight='600', fontsize=12)
    ax.tick_params(axis='x', labelsize=13)
    
    # Title with panel label - larger font (13)
    ax.set_title(f'({panel_label}) {title}', 
                fontweight='600', fontsize=13, pad=10, loc='left', color='#333333')
    
    ax.grid(True, alpha=0.25, axis='x', linestyle=':', linewidth=0.7, zorder=0)
    ax.set_axisbelow(True)
    
    # Add value labels with larger font (9)
    max_val = plot_data.max()
    offset = max_val * 0.01
    
    for i, (cat, count) in enumerate(plot_data.items()):
        ax.text(count + offset, i, f'{count:,}', 
               va='center', ha='left', fontsize=9, fontweight='normal', color='#333333')
    
    # Set x-axis limit with some padding
    ax.set_xlim([0, max_val * 1.15])


# ============================================================================
# CREATE COMBINED 2x2 FIGURE (600 DPI)
# ============================================================================
print("\nCreating combined 2x2 figure...")

# SHAP-style compact dimensions
fig = plt.figure(figsize=(8, 7))
gs = fig.add_gridspec(2, 2, hspace=0.30, wspace=0.25)

# TOP ROW: Variety (left) and Crop Cycle (right)
ax1 = fig.add_subplot(gs[0, 0])
if 'variety' in df_work.columns:
    create_shap_style_barplot(
        ax=ax1,
        data_counts=variety_regrouped_counts,
        title='Variety Distribution',
        panel_label='A',
        highlight_other=True
    )

ax2 = fig.add_subplot(gs[0, 1])
if 'crop_cycle' in df_work.columns:
    create_shap_style_barplot(
        ax=ax2,
        data_counts=crop_cycle_regrouped_counts,
        title='Crop Cycle Distribution',
        panel_label='B',
        highlight_other=False
    )

# BOTTOM ROW: Harvest Mode (left) and Irrigation Type (right)
ax3 = fig.add_subplot(gs[1, 0])
if 'harvest_mode' in df_work.columns:
    create_shap_style_barplot(
        ax=ax3,
        data_counts=harvest_regrouped_counts,
        title='Harvest Mode Distribution',
        panel_label='C',
        highlight_other=False
    )

ax4 = fig.add_subplot(gs[1, 1])
if 'irrigation_type' in df_work.columns:
    create_shap_style_barplot(
        ax=ax4,
        data_counts=irrigation_counts,
        title='Irrigation Type Distribution',
        panel_label='D',
        highlight_other=False,
        color_override=TERTIARY_COLOR
    )

# Overall title - larger font (16)
fig.suptitle('Categorical Variable Distributions After Regrouping',
            fontsize=16, fontweight='600', y=0.98, color='#333333')

plt.tight_layout(rect=[0, 0, 1, 0.96], pad=0.5)

# Save combined figure at 600 DPI
output_combined = os.path.join(OUTPUT_FOLDER, 'categorical_distributions_combined.png')
plt.savefig(output_combined, dpi=600, bbox_inches='tight', facecolor='white')
print(f"✓ Saved combined figure (600 DPI): {output_combined}")

output_combined_pdf = os.path.join(OUTPUT_FOLDER, 'categorical_distributions_combined.pdf')
plt.savefig(output_combined_pdf, format='pdf', bbox_inches='tight', facecolor='white')
print(f"✓ Saved combined figure (PDF): {output_combined_pdf}")

plt.close()

# ============================================================================
# CREATE INDIVIDUAL PLOTS (600 DPI)
# ============================================================================
print("\nCreating individual plots (SHAP style, 600 DPI)...")

# Variety
if 'variety' in df_work.columns:
    n_var = len(variety_regrouped_counts)
    # Compact size like SHAP plots
    fig_height = 6  # Fixed compact height
    
    fig, ax = plt.subplots(figsize=(4, fig_height))
    create_shap_style_barplot(
        ax=ax,
        data_counts=variety_regrouped_counts,
        title='Variety Distribution',
        panel_label='',
        highlight_other=True
    )
    plt.tight_layout(pad=0.5)
    
    output_path = os.path.join(OUTPUT_FOLDER, 'variety_distribution_regrouped.png')
    plt.savefig(output_path, dpi=600, bbox_inches='tight', facecolor='white')
    print(f"✓ Saved: {output_path}")
    
    output_path_pdf = output_path.replace('.png', '.pdf')
    plt.savefig(output_path_pdf, format='pdf', bbox_inches='tight', facecolor='white')
    print(f"✓ Saved: {output_path_pdf}")
    
    plt.close()

# Crop Cycle
if 'crop_cycle' in df_work.columns:
    n_cycle = len(crop_cycle_regrouped_counts)
    # Compact size like SHAP plots
    fig_height = 6  # Fixed compact height
    
    fig, ax = plt.subplots(figsize=(4, fig_height))
    create_shap_style_barplot(
        ax=ax,
        data_counts=crop_cycle_regrouped_counts,
        title='Crop Cycle Distribution',
        panel_label='',
        highlight_other=False
    )
    plt.tight_layout(pad=0.5)
    
    output_path = os.path.join(OUTPUT_FOLDER, 'crop_cycle_regrouped.png')
    plt.savefig(output_path, dpi=600, bbox_inches='tight', facecolor='white')
    print(f"✓ Saved: {output_path}")
    
    output_path_pdf = output_path.replace('.png', '.pdf')
    plt.savefig(output_path_pdf, format='pdf', bbox_inches='tight', facecolor='white')
    print(f"✓ Saved: {output_path_pdf}")
    
    plt.close()

# Harvest Mode
if 'harvest_mode' in df_work.columns:
    # Compact size
    fig, ax = plt.subplots(figsize=(4, 3))
    create_shap_style_barplot(
        ax=ax,
        data_counts=harvest_regrouped_counts,
        title='Harvest Mode Distribution',
        panel_label='',
        highlight_other=False
    )
    plt.tight_layout(pad=0.5)
    
    output_path = os.path.join(OUTPUT_FOLDER, 'harvest_mode_regrouped.png')
    plt.savefig(output_path, dpi=600, bbox_inches='tight', facecolor='white')
    print(f"✓ Saved: {output_path}")
    
    output_path_pdf = output_path.replace('.png', '.pdf')
    plt.savefig(output_path_pdf, format='pdf', bbox_inches='tight', facecolor='white')
    print(f"✓ Saved: {output_path_pdf}")
    
    plt.close()

# Irrigation Type
if 'irrigation_type' in df_work.columns:
    # Compact size 
    fig, ax = plt.subplots(figsize=(4, 3))
    create_shap_style_barplot(
        ax=ax,
        data_counts=irrigation_counts,
        title='Irrigation Type Distribution',
        panel_label='',
        highlight_other=False,
        color_override=TERTIARY_COLOR
    )
    plt.tight_layout(pad=0.5)
    
    output_path = os.path.join(OUTPUT_FOLDER, 'irrigation_type_distribution.png')
    plt.savefig(output_path, dpi=600, bbox_inches='tight', facecolor='white')
    print(f"✓ Saved: {output_path}")
    
    output_path_pdf = output_path.replace('.png', '.pdf')
    plt.savefig(output_path_pdf, format='pdf', bbox_inches='tight', facecolor='white')
    print(f"✓ Saved: {output_path_pdf}")
    
    plt.close()

# ============================================================================
# SAVE FINAL DATASET
# ============================================================================
print("\n" + "="*80)
print("SAVING FINAL REGROUPED DATASET")
print("="*80)

output_csv = os.path.join(OUTPUT_FOLDER, 'final_dataset_regrouped_categorical.csv')
df_work.to_csv(output_csv, index=False, sep=',')
print(f"✓ Saved: {output_csv}")

# ============================================================================
# SUMMARY STATISTICS
# ============================================================================
print("\n" + "="*80)
print("FINAL SUMMARY STATISTICS")
print("="*80)

print(f"\nOriginal dataset: {len(df):,} observations")
print(f"Final dataset: {len(df_work):,} observations")
print(f"Total observations removed: {len(df) - len(df_work):,} ({((len(df) - len(df_work))/len(df)*100):.2f}%)")

print(f"\nFinal dataset shape: {df_work.shape}")
print(f"\nFinal categories:")
if 'variety' in df_work.columns:
    n_var = df_work['variety'].nunique()
    n_var_no_na = df_work['variety'].dropna().nunique()
    print(f"  Variety: {n_var} categories (excl. NA: {n_var_no_na})")
if 'harvest_mode' in df_work.columns:
    n_harv = df_work['harvest_mode'].nunique()
    n_harv_no_na = df_work['harvest_mode'].dropna().nunique()
    print(f"  Harvest mode: {n_harv} categories (excl. NA: {n_harv_no_na})")
if 'irrigation_type' in df_work.columns:
    n_irr = df_work['irrigation_type'].nunique()
    n_irr_no_na = df_work['irrigation_type'].dropna().nunique()
    print(f"  Irrigation type: {n_irr} categories (excl. NA: {n_irr_no_na})")
if 'crop_cycle' in df_work.columns:
    n_cycle = df_work['crop_cycle'].nunique()
    n_cycle_no_na = df_work['crop_cycle'].dropna().nunique()
    print(f"  Crop cycle: {n_cycle} categories (excl. NA: {n_cycle_no_na})")

if 'variety' in df_work.columns and 'tc_per_ha_actual' in df_work.columns:
    print("\n" + "-"*80)
    print("YIELD STATISTICS BY VARIETY")
    print("-"*80)
    yield_stats = df_work.groupby('variety')['tc_per_ha_actual'].agg([
        ('Count', 'count'),
        ('Mean', 'mean'),
        ('Std', 'std'),
        ('Min', 'min'),
        ('Max', 'max')
    ]).round(2).sort_values('Mean', ascending=False)
    print(yield_stats.to_string())
    
    # Save to CSV
    yield_stats_path = os.path.join(OUTPUT_FOLDER, 'yield_stats_by_variety.csv')
    yield_stats.to_csv(yield_stats_path)
    print(f"\n✓ Saved yield statistics: {yield_stats_path}")

print("\n" + "="*80)
print("PIPELINE COMPLETE!")
print("="*80)
print(f"\nAll outputs saved to: {OUTPUT_FOLDER}")
print("="*80)

CATEGORICAL DATA REGROUPING AND ANALYSIS PIPELINE - ENHANCED VIF STYLE
Input file: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/final_dataset_for_analysis_with_quality_metrics2.csv
Output folder: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis_Enhanced

ORIGINAL DATASET STRUCTURE
Dataset shape: (19495, 67)
Total observations: 19495

STEP 1: RENAMING COLUMNS
Renamed 15 columns:
  SC_pol_juice_percent → pol_juice_percent
  SC_purity_percent → purity_percent
  SC_fiber_percent → fiber_percent
  SC_pol_cane_percent → pol_cane_percent
  SC_glucose_cane_percent → glucose_cane_percent
  SC_hugot_sugar → hugot_sugar
  SC_estimated_recoverable_pol → estimated_recoverable_pol
  SC_brix_percent → brix_percent
  SC_tc_per_ha_actual → tc_per_ha_actual
  SC_tc_per_ha_per_week → tc_per_ha_per_week
  SC_tc_per_ha_52_weeks → tc_per_ha_52_weeks
  SC_variety → variety
  SC_irrigation_type → irrigation_type
  SC_harvest → harvest_mode
  SC_crop_cycle → crop_cycle


C:\Users\U078-N800\AppData\Local\Temp\ipykernel_10144\813786929.py:617: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 1, 0.96], pad=0.5)


✓ Saved combined figure (600 DPI): C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis_Enhanced\categorical_distributions_combined.png
✓ Saved combined figure (PDF): C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis_Enhanced\categorical_distributions_combined.pdf

Creating individual plots (EXACT VIF style, 600 DPI)...
✓ Saved: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis_Enhanced\variety_distribution_regrouped.png
✓ Saved: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis_Enhanced\variety_distribution_regrouped.pdf
✓ Saved: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis_Enhanced\crop_cycle_regrouped.png
✓ Saved: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis_Enhanced\crop_cycle_regrouped.pdf
✓ Saved: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis_Enhanced\harvest_mode_regrouped.png
✓ Sav

# CATEGORICAL DATA REGROUPING AND ANALYSIS PIPELINE - BILINGUAL VERSION

In [ ]:
"""
CATEGORICAL DATA REGROUPING AND ANALYSIS PIPELINE - BILINGUAL VERSION
=======================================================================

Author: Agricultural ML Analysis
Date: 2025
"""

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np
import os
from pathlib import Path

# ============================================================================
# FONT SETUP
# ============================================================================

available_fonts = [f.name for f in fm.fontManager.ttflist]
if 'Arial' in available_fonts:
    font_family = 'Arial'
elif 'Helvetica' in available_fonts:
    font_family = 'Helvetica'
elif 'DejaVu Sans' in available_fonts:
    font_family = 'DejaVu Sans'
elif 'Liberation Sans' in available_fonts:
    font_family = 'Liberation Sans'
else:
    font_family = 'sans-serif'

plt.rcParams.update({
    'font.family': font_family,
    'font.size': 14,
    'axes.labelsize': 14,
    'axes.titlesize': 14,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 12,
    'figure.titlesize': 16,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': False,
    'axes.axisbelow': True,
    'axes.edgecolor': '#333333',
    'text.color': '#333333',
    'axes.labelcolor': '#333333',
    'xtick.color': '#333333',
    'ytick.color': '#333333',
    'axes.linewidth': 1.2,
    'xtick.major.width': 1.0,
    'ytick.major.width': 1.0,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'text.usetex': False,
    'mathtext.default': 'regular',
})

# Color scheme - Professional
PRIMARY_COLOR = '#18A558'  # Green for main categories
SECONDARY_COLOR = '#F18F01'  # Orange for "OTHER" categories
TERTIARY_COLOR = '#2E86AB'  # Blue for irrigation

# Irrigation type colors for stacked bars
IRRIGATION_COLORS = {
    'GAG': '#4A90E2',  # Blue for GAG (Drip/Goutte à goutte)
    'GRV': '#E63946'   # Red for GRV (Gravity/Gravitaire)
}

# Define consistent slim bar height
SLIM_BAR_HEIGHT = 0.5

# ============================================================================
# BILINGUAL TRANSLATIONS
# ============================================================================

TRANSLATIONS = {
    'en': {
        'title_combined': 'Categorical Variable Distributions After Regrouping',
        'variety_title': 'Variety Distribution',
        'crop_cycle_title': 'Crop Cycle Distribution',
        'harvest_title': 'Harvest Mode Distribution',
        'irrigation_title': 'Irrigation Type Distribution',
        'xlabel': 'Number of Observations',
        'plant_cane': 'Plant Cane',
        'ratoon': 'Ratoon',
        'beyond': 'Beyond',
        'other_varieties': 'OTHER VARIETIES',
        'irrigation_labels': {
            'GAG': 'GAG\n(Drip irrigation)',
            'GRV': 'GRV\n(Gravity irrigation)'
        },
        'harvest_labels': {
            'MANUELLE': 'Manual',
            'MECANIQUE': 'Mechanized'
        }
    },
    'fr': {
        'title_combined': 'Distribution des Variables Catégorielles Après Regroupement',
        'variety_title': 'Distribution des Variétés',
        'crop_cycle_title': 'Distribution des Cycles de Culture',
        'harvest_title': 'Distribution des Modes de Récolte',
        'irrigation_title': 'Distribution des Types d\'Irrigation',
        'xlabel': 'Nombre d\'Observations',
        'plant_cane': 'Canne Vierge',
        'ratoon': 'Repousse',
        'beyond': 'Au-delà',
        'other_varieties': 'AUTRES VARIÉTÉS',
        'irrigation_labels': {
            'GAG': 'GAG\n(Irrigation goutte à goutte)',
            'GRV': 'GRV\n(Irrigation gravitaire)'
        },
        'harvest_labels': {
            'MANUELLE': 'Manuelle',
            'MECANIQUE': 'Mécanique'
        }
    }
}

def translate_category(category, lang='en'):
    """Translate category labels based on language"""
    trans = TRANSLATIONS[lang]
    
    # Handle crop cycle translations
    if category == 'Plant Cane':
        return trans['plant_cane']
    elif 'Ratoon' in str(category):
        if 'Beyond' in str(category):
            # e.g., "Beyond 3rd Ratoon" -> "Au-delà 3ème Repousse"
            parts = category.split()
            if lang == 'fr':
                num = parts[1]  # e.g., "3rd"
                # Convert ordinals
                ordinal_map = {'1st': '1ère', '2nd': '2ème', '3rd': '3ème', 
                              '4th': '4ème', '5th': '5ème', '6th': '6ème',
                              '7th': '7ème', '8th': '8ème', '9th': '9ème', '10th': '10ème'}
                num_fr = ordinal_map.get(num, num)
                return f"{trans['beyond']} {num_fr} {trans['ratoon']}"
            else:
                return category
        else:
            # e.g., "1st Ratoon" -> "1ère Repousse"
            if lang == 'fr':
                num = category.split()[0]
                ordinal_map = {'1st': '1ère', '2nd': '2ème', '3rd': '3ème', 
                              '4th': '4ème', '5th': '5ème', '6th': '6ème',
                              '7th': '7ème', '8th': '8ème', '9th': '9ème', '10th': '10ème'}
                num_fr = ordinal_map.get(num, num)
                return f"{num_fr} {trans['ratoon']}"
            else:
                return category
    
    # Handle other varieties
    elif category == 'OTHER VARIETIES':
        return trans['other_varieties']
    
    # Handle harvest mode
    elif category in ['MANUELLE', 'MECANIQUE']:
        return trans['harvest_labels'].get(category, category)
    
    # Return as-is for other categories (variety names, irrigation types)
    return category

# ============================================================================
# CONFIGURATION
# ============================================================================

# Input file
INPUT_FILE = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/final_dataset_for_analysis_with_quality_metrics2.csv'

# Create new output folder
OUTPUT_FOLDER = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis_Bilingual'
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print("="*80)
print("CATEGORICAL DATA REGROUPING - BILINGUAL VERSION")
print("="*80)
print(f"Input file: {INPUT_FILE}")
print(f"Output folder: {OUTPUT_FOLDER}")

# ============================================================================
# LOAD DATA
# ============================================================================
df = pd.read_csv(INPUT_FILE, sep=',')

print("\n" + "="*80)
print("ORIGINAL DATASET STRUCTURE")
print("="*80)
print(f"Dataset shape: {df.shape}")
print(f"Total observations: {len(df)}")

# ============================================================================
# COLUMN RENAMING (Remove SC_ prefix)
# ============================================================================
print("\n" + "="*80)
print("STEP 1: RENAMING COLUMNS")
print("="*80)

columns_to_rename = {
    "SC_pol_juice_percent": "pol_juice_percent",
    "SC_purity_percent": "purity_percent",
    "SC_fiber_percent": "fiber_percent",
    "SC_pol_cane_percent": "pol_cane_percent",
    "SC_percent_cane_percent": "percent_cane_percent",
    "SC_glucose_cane_percent": "glucose_cane_percent",
    "SC_hugot_sugar": "hugot_sugar",
    "SC_Pur_Tc": "pur_tc",
    "SC_fiber_content_percent": "fiber_content_percent",
    "SC_estimated_recoverable_pol": "estimated_recoverable_pol",
    "SC_juice_tons": "juice_tons",
    "SC_tmss": "tmss",
    "SC_tpj": "tpj",
    "SC_tpc": "tpc",
    "SC_ts": "ts",
    "SC_tsr": "tsr",
    "SC_brix_percent": "brix_percent",
    "SC_tc_per_ha_actual": "tc_per_ha_actual",
    "SC_tc_per_ha_per_week": "tc_per_ha_per_week",
    "SC_tc_per_ha_52_weeks": "tc_per_ha_52_weeks",
    "SC_variety": "variety",
    "SC_irrigation_type": "irrigation_type",
    "SC_harvest": "harvest_mode",
    "SC_harvest_mode": "harvest_mode",
    "SC_crop_cycle": "crop_cycle"
}

actual_renames = {old: new for old, new in columns_to_rename.items() if old in df.columns}
df_work = df.rename(columns=actual_renames)

print(f"Renamed {len(actual_renames)} columns")

# Convert categorical columns to uppercase for consistent processing
for col in ['variety', 'harvest_mode', 'irrigation_type']:
    if col in df_work.columns:
        df_work[col] = df_work[col].astype(str).str.upper().str.strip()

if 'crop_cycle' in df_work.columns:
    df_work['crop_cycle'] = df_work['crop_cycle'].astype(str).str.upper().str.strip()

# ============================================================================
# STEP 2: REMOVE OBSERVATIONS WITH NA IN KEY METRICS
# ============================================================================
print("\n" + "="*80)
print("STEP 2: REMOVING OBSERVATIONS WITH NA IN KEY METRICS")
print("="*80)

if 'tc_per_ha_actual' in df_work.columns and 'brix_percent' in df_work.columns:
    initial_count = len(df_work)
    df_work = df_work[df_work['tc_per_ha_actual'].notna() & df_work['brix_percent'].notna()].copy()
    removed_na = initial_count - len(df_work)
    
    print(f"Removed {removed_na} observations with NA in tc_per_ha_actual or brix_percent")
    print(f"Remaining observations: {len(df_work)}")

# ============================================================================
# STEP 3: REMOVE SPECIAL CROP CYCLE CATEGORIES
# ============================================================================
print("\n" + "="*80)
print("STEP 3: REMOVING SPECIAL CROP CYCLE CATEGORIES")
print("="*80)

if 'crop_cycle' in df_work.columns:
    categories_to_remove = ['FRICHE', 'LESSIVAGE', 'JACHERE', 'ESSAIS']
    
    initial_count = len(df_work)
    df_work = df_work[~df_work['crop_cycle'].isin(categories_to_remove)].copy()
    removed_special = initial_count - len(df_work)
    
    print(f"Total removed: {removed_special} observations")
    print(f"Remaining observations: {len(df_work)}")

# ============================================================================
# STEP 4: REMOVE MIXED VARIETY OBSERVATIONS
# ============================================================================
print("\n" + "="*80)
print("STEP 4: REMOVING MIXED VARIETY OBSERVATIONS")
print("="*80)

if 'variety' in df_work.columns:
    mixed_varieties = ['MIXTE', 'MIXTE D', 'MIXTE MF']
    
    initial_count = len(df_work)
    df_work = df_work[~df_work['variety'].isin(mixed_varieties)].copy()
    removed_mixed = initial_count - len(df_work)
    
    print(f"Total removed: {removed_mixed} observations")
    print(f"Remaining observations: {len(df_work)}")

# ============================================================================
# VARIETY REGROUPING: Keep varieties with ≥50 observations
# ============================================================================
print("\n" + "="*80)
print("VARIETY REGROUPING (≥50 OBSERVATIONS)")
print("="*80)

if 'variety' in df_work.columns:
    df_variety_valid = df_work[df_work['variety'].notna()].copy()
    variety_counts = df_variety_valid['variety'].value_counts().sort_values(ascending=False)
    total_obs_variety = len(df_variety_valid)
    
    print(f"\nTotal observations with valid variety: {total_obs_variety}")
    
    # Regroup varieties with < 50 observations
    min_obs_threshold = 50
    varieties_to_keep = variety_counts[variety_counts >= min_obs_threshold].index.tolist()
    
    print(f"\nVarieties with ≥ {min_obs_threshold} observations: {len(varieties_to_keep)}")
    
    def regroup_variety(variety):
        if pd.isna(variety):
            return np.nan
        if variety in varieties_to_keep:
            return variety
        else:
            return 'OTHER VARIETIES'
    
    df_work['variety'] = df_work['variety'].apply(regroup_variety)
    
    variety_regrouped_counts = df_work['variety'].value_counts().sort_values(ascending=False)
    
    print("\nRegrouped Variety Distribution:")
    print("-" * 80)
    for variety, count in variety_regrouped_counts.items():
        percentage = (count / total_obs_variety) * 100
        print(f"  {variety:<30} : {count:>6} obs ({percentage:>5.2f}%)")

# ============================================================================
# HARVEST MODE REGROUPING: Keep MANUELLE and MECANIQUE only
# ============================================================================
print("\n" + "="*80)
print("HARVEST MODE REGROUPING (MANUELLE/MECANIQUE ONLY)")
print("="*80)

if 'harvest_mode' in df_work.columns:
    df_harvest_valid = df_work[df_work['harvest_mode'].notna()].copy()
    harvest_counts = df_harvest_valid['harvest_mode'].value_counts().sort_values(ascending=False)
    
    def regroup_harvest_mode(mode):
        if pd.isna(mode):
            return np.nan
        mode_upper = str(mode).upper().strip()
        if mode_upper in ['MANUELLE', 'MECANIQUE']:
            return mode_upper
        else:
            return np.nan
    
    df_work['harvest_mode'] = df_work['harvest_mode'].apply(regroup_harvest_mode)
    
    df_harvest_regroup_valid = df_work[df_work['harvest_mode'].notna()].copy()
    harvest_regrouped_counts = df_harvest_regroup_valid['harvest_mode'].value_counts().sort_values(ascending=False)
    
    print("\nRegrouped Harvest Mode Distribution:")
    print("-" * 80)
    for mode, count in harvest_regrouped_counts.items():
        print(f"  {mode:<30} : {count:>6} obs")

# ============================================================================
# IRRIGATION TYPE PROCESSING
# ============================================================================
print("\n" + "="*80)
print("IRRIGATION TYPE PROCESSING")
print("="*80)

if 'irrigation_type' in df_work.columns:
    df_irrigation_valid = df_work[df_work['irrigation_type'].notna()].copy()
    irrigation_counts = df_irrigation_valid['irrigation_type'].value_counts().sort_values(ascending=False)
    
    print("\nIrrigation Type Distribution:")
    print("-" * 80)
    for irr_type, count in irrigation_counts.items():
        print(f"  {irr_type:<30} : {count:>6} obs")

# ============================================================================
# CROP CYCLE REGROUPING
# ============================================================================
print("\n" + "="*80)
print("CROP CYCLE REGROUPING")
print("="*80)

if 'crop_cycle' in df_work.columns:
    df_cycle_valid = df_work[df_work['crop_cycle'].notna()].copy()
    crop_cycle_counts = df_cycle_valid['crop_cycle'].value_counts().sort_index()
    
    # Standardize crop cycle naming
    def standardize_crop_cycle(cycle):
        if pd.isna(cycle):
            return np.nan
        
        cycle_str = str(cycle).upper().strip()
        
        try:
            cycle_num = int(float(cycle_str))
            
            if cycle_num == 0:
                return 'Plant Cane'
            elif cycle_num == 1:
                return '1st Ratoon'
            elif cycle_num == 2:
                return '2nd Ratoon'
            elif cycle_num == 3:
                return '3rd Ratoon'
            elif cycle_num >= 4:
                return f'{cycle_num}th Ratoon'
            else:
                return np.nan
        except:
            return np.nan
    
    df_work['crop_cycle_standardized'] = df_work['crop_cycle'].apply(standardize_crop_cycle)
    
    df_cycle_std_valid = df_work[df_work['crop_cycle_standardized'].notna()].copy()
    cycle_std_counts = df_cycle_std_valid['crop_cycle_standardized'].value_counts()
    
    # Build ordered list
    cycle_names_ordered = ['Plant Cane']
    for i in range(1, 20):
        if i == 1:
            cycle_name = '1st Ratoon'
        elif i == 2:
            cycle_name = '2nd Ratoon'
        elif i == 3:
            cycle_name = '3rd Ratoon'
        else:
            cycle_name = f'{i}th Ratoon'
        
        if cycle_name in cycle_std_counts.index:
            cycle_names_ordered.append(cycle_name)
        else:
            break
    
    # Identify cycles with ≥ 50 observations
    min_obs_threshold = 50
    cycles_to_keep = []
    last_kept_cycle_num = -1
    
    for i, cycle_name in enumerate(cycle_names_ordered):
        if cycle_name in cycle_std_counts.index:
            count = cycle_std_counts[cycle_name]
            if count >= min_obs_threshold:
                cycles_to_keep.append(cycle_name)
                last_kept_cycle_num = i
    
    print(f"\nCrop cycles with ≥ {min_obs_threshold} observations: {len(cycles_to_keep)}")
    
    # Determine "beyond" label
    if cycles_to_keep:
        last_cycle_name = cycles_to_keep[-1]
        if last_cycle_name == 'Plant Cane':
            last_cycle_number = 0
        else:
            import re
            match = re.search(r'(\d+)', last_cycle_name)
            if match:
                last_cycle_number = int(match.group(1))
            else:
                last_cycle_number = 0
        
        beyond_cycle_num = last_cycle_number + 1
        if beyond_cycle_num == 1:
            beyond_label = "Beyond 1st Ratoon"
        elif beyond_cycle_num == 2:
            beyond_label = "Beyond 2nd Ratoon"
        elif beyond_cycle_num == 3:
            beyond_label = "Beyond 3rd Ratoon"
        else:
            beyond_label = f"Beyond {beyond_cycle_num}th Ratoon"
    else:
        beyond_label = "Beyond Plant Cane"
    
    def regroup_crop_cycle(cycle_std):
        if pd.isna(cycle_std):
            return np.nan
        if cycle_std in cycles_to_keep:
            return cycle_std
        else:
            return beyond_label
    
    df_work['crop_cycle'] = df_work['crop_cycle_standardized'].apply(regroup_crop_cycle)
    df_work = df_work.drop(columns=['crop_cycle_standardized'])
    
    cycle_order = cycles_to_keep + [beyond_label]
    
    df_cycle_regroup_valid = df_work[df_work['crop_cycle'].notna()].copy()
    crop_cycle_regrouped_counts = df_cycle_regroup_valid['crop_cycle'].value_counts()
    
    print("\nRegrouped Crop Cycle Distribution:")
    print("-" * 80)
    for cycle in cycle_order:
        if cycle in crop_cycle_regrouped_counts.index:
            count = crop_cycle_regrouped_counts[cycle]
            print(f"  {cycle:<30} : {count:>6} obs")

# ============================================================================
# PLOTTING FUNCTIONS
# ============================================================================

def create_vif_style_barplot(ax, data_counts, title, panel_label, 
                              highlight_other=False, color_override=None,
                              category_col=None, stack_by_irrigation=False, lang='en'):
    """Create EXACT VIF-style horizontal bar plot with optional stacking by irrigation_type"""
    
    trans = TRANSLATIONS[lang]
    
    if stack_by_irrigation and category_col is not None and 'irrigation_type' in df_work.columns:
        # Create stacked bars by irrigation type
        df_filtered = df_work[df_work[category_col].notna()].copy()
        
        # Create cross-tabulation
        cross_tab = pd.crosstab(df_filtered[category_col], df_filtered['irrigation_type'])
        
        # Ensure we have GAG and GRV columns (add if missing)
        for irr_type in ['GAG', 'GRV']:
            if irr_type not in cross_tab.columns:
                cross_tab[irr_type] = 0
        
        # Sort cross_tab based on total counts
        cross_tab['Total'] = cross_tab.sum(axis=1)
        cross_tab = cross_tab.sort_values('Total', ascending=True)
        cross_tab = cross_tab.drop('Total', axis=1)
        
        # Handle special ordering for crop_cycle
        if category_col == 'crop_cycle':
            # Define desired order (reversed - Plant Cane on top)
            cycle_order = ['Plant Cane', '1st Ratoon', '2nd Ratoon', '3rd Ratoon', 
                          '4th Ratoon', '5th Ratoon', '6th Ratoon', '7th Ratoon',
                          '8th Ratoon', '9th Ratoon', '10th Ratoon']
            # Add "Beyond" categories
            beyond_cats = [idx for idx in cross_tab.index if 'Beyond' in str(idx)]
            cycle_order.extend(beyond_cats)
            
            # Reorder cross_tab according to cycle_order and REVERSE
            ordered_index = [cat for cat in cycle_order if cat in cross_tab.index]
            ordered_index.reverse()  # REVERSE so Plant Cane is on top
            cross_tab = cross_tab.reindex(ordered_index)
        
        # Translate y-axis labels
        translated_labels = [translate_category(cat, lang) for cat in cross_tab.index]
        
        # Create y positions
        y_pos = np.arange(len(cross_tab))
        
        # Plot stacked bars
        left_pos = np.zeros(len(cross_tab))
        
        for irr_type in ['GAG', 'GRV']:
            if irr_type in cross_tab.columns:
                ax.barh(y_pos, cross_tab[irr_type], left=left_pos, 
                       height=SLIM_BAR_HEIGHT, 
                       color=IRRIGATION_COLORS[irr_type], 
                       alpha=0.8, 
                       edgecolor='white', 
                       linewidth=0.5,
                       label=trans['irrigation_labels'][irr_type])
                left_pos += cross_tab[irr_type].values
        
        # Set y-axis with translated labels
        ax.set_yticks(y_pos)
        ax.set_yticklabels(translated_labels, fontsize=11, fontweight='normal')
        
        # Add total count labels
        totals = cross_tab.sum(axis=1)
        max_val = totals.max()
        offset = max_val * 0.01
        
        for i, total in enumerate(totals):
            ax.text(total + offset, i, f'{int(total):,}', 
                   va='center', ha='left', fontsize=9, fontweight='normal', color='#333333')
        
        # Add legend with smaller font to avoid overlap
        ax.legend(loc='lower right', fontsize=9, frameon=True, 
                 fancybox=False, shadow=False, edgecolor='#333333')
        
        # Set x-axis limit
        ax.set_xlim([0, max_val * 1.15])
        
        # Set y-axis limits
        ax.set_ylim(-0.5, len(cross_tab) - 0.5)
        
    else:
        # Original non-stacked version
        plot_data = data_counts.sort_values(ascending=True)
        
        # Handle special ordering for crop_cycle
        if category_col == 'crop_cycle':
            # Define desired order (reversed - Plant Cane on top)
            cycle_order = ['Plant Cane', '1st Ratoon', '2nd Ratoon', '3rd Ratoon', 
                          '4th Ratoon', '5th Ratoon', '6th Ratoon', '7th Ratoon',
                          '8th Ratoon', '9th Ratoon', '10th Ratoon']
            # Add "Beyond" categories
            beyond_cats = [idx for idx in plot_data.index if 'Beyond' in str(idx)]
            cycle_order.extend(beyond_cats)
            
            # Reorder plot_data according to cycle_order and REVERSE
            ordered_index = [cat for cat in cycle_order if cat in plot_data.index]
            ordered_index.reverse()  # REVERSE so Plant Cane is on top
            plot_data = plot_data.reindex(ordered_index)
        
        # Translate y-axis labels
        translated_labels = [translate_category(cat, lang) for cat in plot_data.index]
        
        # Assign colors
        if color_override:
            colors = [color_override] * len(plot_data)
        elif highlight_other:
            colors = [SECONDARY_COLOR if 'OTHER' in str(cat) else PRIMARY_COLOR 
                     for cat in plot_data.index]
        else:
            colors = [PRIMARY_COLOR] * len(plot_data)
        
        # Create slim bars
        y_pos = np.arange(len(plot_data))
        bars = ax.barh(y_pos, plot_data.values, height=SLIM_BAR_HEIGHT, 
                       color=colors, alpha=0.8, edgecolor='white', linewidth=0.5)
        
        # Set y-axis with font size 11 and translated labels
        ax.set_yticks(y_pos)
        ax.set_yticklabels(translated_labels, fontsize=11, fontweight='normal')
        
        # Add value labels with font size 9
        max_val = plot_data.max()
        offset = max_val * 0.01
        
        for i, (cat, count) in enumerate(plot_data.items()):
            ax.text(count + offset, i, f'{count:,}', 
                   va='center', ha='left', fontsize=9, fontweight='normal', color='#333333')
        
        # Set x-axis limit with padding
        ax.set_xlim([0, max_val * 1.15])
        
        # Set y-axis limits
        ax.set_ylim(-0.5, len(plot_data) - 0.5)
    
    # Common settings for both versions
    # Set x-axis with font size 12, fontweight 600 
    ax.set_xlabel(trans['xlabel'], fontweight='600', fontsize=12)
    ax.tick_params(axis='x', labelsize=13)
    
    # Title with panel label - font size 13, fontweight 600 
    if panel_label:
        ax.set_title(f'{panel_label}. {title}', 
                    fontweight='600', fontsize=13, pad=10, loc='left', color='#333333')
    else:
        ax.set_title(title, 
                    fontweight='600', fontsize=13, pad=10, loc='left', color='#333333')
    
    ax.grid(True, alpha=0.25, axis='x', linestyle=':', linewidth=0.7, zorder=0)
    ax.set_axisbelow(True)


def create_all_plots_for_language(lang='en'):
    """Create all plots (combined and individual) for specified language"""
    
    trans = TRANSLATIONS[lang]
    lang_suffix = f'_{lang}'
    
    print(f"\n{'='*80}")
    print(f"CREATING PLOTS IN {'ENGLISH' if lang == 'en' else 'FRENCH'}")
    print(f"{'='*80}")
    
    # ========================================================================
    # CREATE COMBINED 2x2 FIGURE (600 DPI) 
    # ========================================================================
    print(f"\nCreating combined 2x2 figure ({lang.upper()})...")
    
    # compact dimensions (8" x 7")
    fig = plt.figure(figsize=(8, 7))
    gs = fig.add_gridspec(2, 2, hspace=0.30, wspace=0.25)
    
    # TOP ROW: Variety (left) and Crop Cycle (right)
    ax1 = fig.add_subplot(gs[0, 0])
    if 'variety' in df_work.columns:
        create_vif_style_barplot(
            ax=ax1,
            data_counts=variety_regrouped_counts,
            title=trans['variety_title'],
            panel_label='a',
            highlight_other=True,
            category_col='variety',
            stack_by_irrigation=True,
            lang=lang
        )
    
    ax2 = fig.add_subplot(gs[0, 1])
    if 'crop_cycle' in df_work.columns:
        create_vif_style_barplot(
            ax=ax2,
            data_counts=crop_cycle_regrouped_counts,
            title=trans['crop_cycle_title'],
            panel_label='b',
            highlight_other=False,
            category_col='crop_cycle',
            stack_by_irrigation=True,
            lang=lang
        )
    
    # BOTTOM ROW: Harvest Mode (left) and Irrigation Type (right)
    ax3 = fig.add_subplot(gs[1, 0])
    if 'harvest_mode' in df_work.columns:
        create_vif_style_barplot(
            ax=ax3,
            data_counts=harvest_regrouped_counts,
            title=trans['harvest_title'],
            panel_label='c',
            highlight_other=False,
            category_col='harvest_mode',
            stack_by_irrigation=True,
            lang=lang
        )
    
    ax4 = fig.add_subplot(gs[1, 1])
    if 'irrigation_type' in df_work.columns:
        create_vif_style_barplot(
            ax=ax4,
            data_counts=irrigation_counts,
            title=trans['irrigation_title'],
            panel_label='d',
            highlight_other=False,
            color_override=TERTIARY_COLOR,
            category_col='irrigation_type',
            stack_by_irrigation=False,  # Don't stack irrigation by itself
            lang=lang
        )
    
    # Overall title - font size 16, fontweight 600 
    fig.suptitle(trans['title_combined'],
                fontsize=16, fontweight='600', y=0.98, color='#333333')
    
    # EXACT VIF TIGHT LAYOUT
    plt.tight_layout(rect=[0, 0, 1, 0.96], pad=0.5)
    
    # Save combined figure at 600 DPI
    output_combined = os.path.join(OUTPUT_FOLDER, f'categorical_distributions_combined{lang_suffix}.png')
    plt.savefig(output_combined, dpi=600, bbox_inches='tight', facecolor='white')
    print(f"✓ Saved combined figure (600 DPI): {output_combined}")
    
    output_combined_pdf = os.path.join(OUTPUT_FOLDER, f'categorical_distributions_combined{lang_suffix}.pdf')
    plt.savefig(output_combined_pdf, format='pdf', bbox_inches='tight', facecolor='white')
    print(f"✓ Saved combined figure (PDF): {output_combined_pdf}")
    
    plt.close()
    
    # ========================================================================
    # CREATE INDIVIDUAL PLOTS (600 DPI)
    # ========================================================================
    print(f"\nCreating individual plots ({lang.upper()})...")
    
    # Variety
    if 'variety' in df_work.columns:
        fig, ax = plt.subplots(figsize=(4, 6))
        create_vif_style_barplot(
            ax=ax,
            data_counts=variety_regrouped_counts,
            title=trans['variety_title'],
            panel_label='a',
            highlight_other=True,
            category_col='variety',
            stack_by_irrigation=True,
            lang=lang
        )
        plt.tight_layout(pad=0.5)
        
        output_path = os.path.join(OUTPUT_FOLDER, f'variety_distribution{lang_suffix}.png')
        plt.savefig(output_path, dpi=600, bbox_inches='tight', facecolor='white')
        print(f"✓ Saved: {output_path}")
        
        output_path_pdf = output_path.replace('.png', '.pdf')
        plt.savefig(output_path_pdf, format='pdf', bbox_inches='tight', facecolor='white')
        print(f"✓ Saved: {output_path_pdf}")
        
        plt.close()
    
    # Crop Cycle
    if 'crop_cycle' in df_work.columns:
        fig, ax = plt.subplots(figsize=(4, 6))
        create_vif_style_barplot(
            ax=ax,
            data_counts=crop_cycle_regrouped_counts,
            title=trans['crop_cycle_title'],
            panel_label='b',
            highlight_other=False,
            category_col='crop_cycle',
            stack_by_irrigation=True,
            lang=lang
        )
        plt.tight_layout(pad=0.5)
        
        output_path = os.path.join(OUTPUT_FOLDER, f'crop_cycle{lang_suffix}.png')
        plt.savefig(output_path, dpi=600, bbox_inches='tight', facecolor='white')
        print(f"✓ Saved: {output_path}")
        
        output_path_pdf = output_path.replace('.png', '.pdf')
        plt.savefig(output_path_pdf, format='pdf', bbox_inches='tight', facecolor='white')
        print(f"✓ Saved: {output_path_pdf}")
        
        plt.close()
    
    # Harvest Mode
    if 'harvest_mode' in df_work.columns:
        fig, ax = plt.subplots(figsize=(4, 2.5))
        create_vif_style_barplot(
            ax=ax,
            data_counts=harvest_regrouped_counts,
            title=trans['harvest_title'],
            panel_label='c',
            highlight_other=False,
            category_col='harvest_mode',
            stack_by_irrigation=True,
            lang=lang
        )
        plt.tight_layout(pad=0.5)
        
        output_path = os.path.join(OUTPUT_FOLDER, f'harvest_mode{lang_suffix}.png')
        plt.savefig(output_path, dpi=600, bbox_inches='tight', facecolor='white')
        print(f"✓ Saved: {output_path}")
        
        output_path_pdf = output_path.replace('.png', '.pdf')
        plt.savefig(output_path_pdf, format='pdf', bbox_inches='tight', facecolor='white')
        print(f"✓ Saved: {output_path_pdf}")
        
        plt.close()
    
    # Irrigation Type
    if 'irrigation_type' in df_work.columns:
        fig, ax = plt.subplots(figsize=(4, 2.5))
        create_vif_style_barplot(
            ax=ax,
            data_counts=irrigation_counts,
            title=trans['irrigation_title'],
            panel_label='d',
            highlight_other=False,
            color_override=TERTIARY_COLOR,
            category_col='irrigation_type',
            stack_by_irrigation=False,
            lang=lang
        )
        plt.tight_layout(pad=0.5)
        
        output_path = os.path.join(OUTPUT_FOLDER, f'irrigation_type{lang_suffix}.png')
        plt.savefig(output_path, dpi=600, bbox_inches='tight', facecolor='white')
        print(f"✓ Saved: {output_path}")
        
        output_path_pdf = output_path.replace('.png', '.pdf')
        plt.savefig(output_path_pdf, format='pdf', bbox_inches='tight', facecolor='white')
        print(f"✓ Saved: {output_path_pdf}")
        
        plt.close()


# ============================================================================
# CREATE VISUALIZATIONS FOR BOTH LANGUAGES
# ============================================================================
print("\n" + "="*80)
print("CREATING BILINGUAL VISUALIZATIONS (600 DPI)")
print("="*80)

# Create plots for both languages
create_all_plots_for_language('en')
create_all_plots_for_language('fr')

# ============================================================================
# SAVE FINAL DATASET
# ============================================================================
print("\n" + "="*80)
print("SAVING FINAL REGROUPED DATASET")
print("="*80)

output_csv = os.path.join(OUTPUT_FOLDER, 'final_dataset_regrouped_categorical.csv')
df_work.to_csv(output_csv, index=False, sep=',')
print(f"✓ Saved: {output_csv}")

# ============================================================================
# SUMMARY STATISTICS
# ============================================================================
print("\n" + "="*80)
print("FINAL SUMMARY STATISTICS")
print("="*80)

print(f"\nOriginal dataset: {len(df):,} observations")
print(f"Final dataset: {len(df_work):,} observations")
print(f"Total observations removed: {len(df) - len(df_work):,} ({((len(df) - len(df_work))/len(df)*100):.2f}%)")

print(f"\nFinal dataset shape: {df_work.shape}")
print(f"\nFinal categories:")
if 'variety' in df_work.columns:
    n_var = df_work['variety'].nunique()
    n_var_no_na = df_work['variety'].dropna().nunique()
    print(f"  Variety: {n_var} categories (excl. NA: {n_var_no_na})")
if 'harvest_mode' in df_work.columns:
    n_harv = df_work['harvest_mode'].nunique()
    n_harv_no_na = df_work['harvest_mode'].dropna().nunique()
    print(f"  Harvest mode: {n_harv} categories (excl. NA: {n_harv_no_na})")
if 'irrigation_type' in df_work.columns:
    n_irr = df_work['irrigation_type'].nunique()
    n_irr_no_na = df_work['irrigation_type'].dropna().nunique()
    print(f"  Irrigation type: {n_irr} categories (excl. NA: {n_irr_no_na})")
if 'crop_cycle' in df_work.columns:
    n_cycle = df_work['crop_cycle'].nunique()
    n_cycle_no_na = df_work['crop_cycle'].dropna().nunique()
    print(f"  Crop cycle: {n_cycle} categories (excl. NA: {n_cycle_no_na})")

if 'variety' in df_work.columns and 'tc_per_ha_actual' in df_work.columns:
    print("\n" + "-"*80)
    print("YIELD STATISTICS BY VARIETY")
    print("-"*80)
    yield_stats = df_work.groupby('variety')['tc_per_ha_actual'].agg([
        ('Count', 'count'),
        ('Mean', 'mean'),
        ('Std', 'std'),
        ('Min', 'min'),
        ('Max', 'max')
    ]).round(2).sort_values('Mean', ascending=False)
    print(yield_stats.to_string())
    
    # Save to CSV
    yield_stats_path = os.path.join(OUTPUT_FOLDER, 'yield_stats_by_variety.csv')
    yield_stats.to_csv(yield_stats_path)
    print(f"\n✓ Saved yield statistics: {yield_stats_path}")

print("\n" + "="*80)
print("PIPELINE COMPLETE!")
print("="*80)
print(f"\nAll outputs saved to: {OUTPUT_FOLDER}")
print("="*80)

CATEGORICAL DATA REGROUPING - BILINGUAL VERSION
Input file: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/final_dataset_for_analysis_with_quality_metrics2.csv
Output folder: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis_Bilingual

ORIGINAL DATASET STRUCTURE
Dataset shape: (19495, 67)
Total observations: 19495

STEP 1: RENAMING COLUMNS
Renamed 15 columns

STEP 2: REMOVING OBSERVATIONS WITH NA IN KEY METRICS
Removed 5613 observations with NA in tc_per_ha_actual or brix_percent
Remaining observations: 13882

STEP 3: REMOVING SPECIAL CROP CYCLE CATEGORIES
Total removed: 22 observations
Remaining observations: 13860

STEP 4: REMOVING MIXED VARIETY OBSERVATIONS
Total removed: 484 observations
Remaining observations: 13376

VARIETY REGROUPING (≥50 OBSERVATIONS)

Total observations with valid variety: 13376

Varieties with ≥ 50 observations: 15

Regrouped Variety Distribution:
---------------------------------------------------------------------------

C:\Users\U078-N800\AppData\Local\Temp\ipykernel_10144\508476689.py:740: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 1, 0.96], pad=0.5)


✓ Saved combined figure (600 DPI): C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis_Bilingual\categorical_distributions_combined_en.png
✓ Saved combined figure (PDF): C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis_Bilingual\categorical_distributions_combined_en.pdf

Creating individual plots (EN)...
✓ Saved: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis_Bilingual\variety_distribution_en.png
✓ Saved: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis_Bilingual\variety_distribution_en.pdf
✓ Saved: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis_Bilingual\crop_cycle_en.png
✓ Saved: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis_Bilingual\crop_cycle_en.pdf
✓ Saved: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis_Bilingual\harvest_mode_en.png
✓ Saved: C:/Users/U078-N800/Desktop/Ml_Internship

C:\Users\U078-N800\AppData\Local\Temp\ipykernel_10144\508476689.py:740: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 1, 0.96], pad=0.5)


✓ Saved combined figure (600 DPI): C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis_Bilingual\categorical_distributions_combined_fr.png
✓ Saved combined figure (PDF): C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis_Bilingual\categorical_distributions_combined_fr.pdf

Creating individual plots (FR)...
✓ Saved: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis_Bilingual\variety_distribution_fr.png
✓ Saved: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis_Bilingual\variety_distribution_fr.pdf
✓ Saved: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis_Bilingual\crop_cycle_fr.png
✓ Saved: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis_Bilingual\crop_cycle_fr.pdf
✓ Saved: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis_Bilingual\harvest_mode_fr.png
✓ Saved: C:/Users/U078-N800/Desktop/Ml_Internship

# Calculates TPER (t/ha) = estimated_recoverable_pol (%) × tc_per_ha_actual (t/ha) / 100

In [1]:
"""
ADD TOTAL RECOVERABLE SUGAR (TPER) COLUMN
=========================================
Calculates TPER (t/ha) = estimated_recoverable_pol (%) × tc_per_ha_actual (t/ha) / 100

Author: Agricultural ML Analysis
Date: 2025
"""

import pandas as pd
import os

# ============================================================================
# CONFIGURATION
# ============================================================================

# Input file 
INPUT_FILE = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis/final_dataset_regrouped_categorical.csv'

print("="*80)
print("ADDING TOTAL RECOVERABLE SUGAR (TPER) COLUMN")
print("="*80)
print(f"Input file: {INPUT_FILE}")

# ============================================================================
# LOAD DATA
# ============================================================================
df = pd.read_csv(INPUT_FILE, sep=',')

print(f"\nOriginal dataset shape: {df.shape}")
print(f"Total observations: {len(df):,}")

# ============================================================================
# CALCULATE TOTAL RECOVERABLE SUGAR (TPER)
# ============================================================================
print("\n" + "="*80)
print("CALCULATING TOTAL RECOVERABLE SUGAR (TPER in t/ha)")
print("="*80)

# Check if required columns exist
if 'estimated_recoverable_pol' not in df.columns:
    print("ERROR: 'estimated_recoverable_pol' column not found in dataset!")
    exit(1)

if 'tc_per_ha_actual' not in df.columns:
    print("ERROR: 'tc_per_ha_actual' column not found in dataset!")
    exit(1)

# Calculate TPER: (estimated_recoverable_pol / 100) × tc_per_ha_actual
# estimated_recoverable_pol is in %, tc_per_ha_actual is in t/ha
# Result is in t/ha
df['total_recoverable_sugar_t_per_ha'] = (df['estimated_recoverable_pol'] / 100.0) * df['tc_per_ha_actual']

print(f"\n✓ Created column: 'total_recoverable_sugar_t_per_ha'")
print(f"  Formula: (estimated_recoverable_pol / 100) × tc_per_ha_actual")

# ============================================================================
# STATISTICS
# ============================================================================
print("\n" + "="*80)
print("STATISTICS FOR NEW COLUMN")
print("="*80)

print(f"\nTotal observations: {len(df):,}")
print(f"Non-NA values: {df['total_recoverable_sugar_t_per_ha'].notna().sum():,}")
print(f"NA values: {df['total_recoverable_sugar_t_per_ha'].isna().sum():,}")

# Statistics for non-NA values
valid_trs = df['total_recoverable_sugar_t_per_ha'].dropna()

if len(valid_trs) > 0:
    print(f"\nDescriptive Statistics (Total Recoverable Sugar in t/ha):")
    print(f"  Mean:   {valid_trs.mean():.2f} t/ha")
    print(f"  Median: {valid_trs.median():.2f} t/ha")
    print(f"  Std:    {valid_trs.std():.2f} t/ha")
    print(f"  Min:    {valid_trs.min():.2f} t/ha")
    print(f"  Max:    {valid_trs.max():.2f} t/ha")
    
    print(f"\nSample calculations (first 5 non-NA rows):")
    print("-"*80)
    sample_df = df[df['total_recoverable_sugar_t_per_ha'].notna()][['estimated_recoverable_pol', 'tc_per_ha_actual', 'total_recoverable_sugar_t_per_ha']].head(5)
    print(sample_df.to_string(index=False))

# ============================================================================
# SAVE UPDATED DATASET (OVERWRITE ORIGINAL)
# ============================================================================
print("\n" + "="*80)
print("SAVING UPDATED DATASET")
print("="*80)

# Save to same file (overwrite)
df.to_csv(INPUT_FILE, index=False, sep=',')
print(f"✓ Dataset saved (overwritten): {INPUT_FILE}")
print(f"  New shape: {df.shape}")
print(f"  Total columns: {len(df.columns)}")

print("\n" + "="*80)
print("COMPLETE!")
print("="*80)
print(f"\nThe column 'total_recoverable_sugar_t_per_ha' has been added to:")
print(f"  {INPUT_FILE}")
print("\nFormula: TPER (t/ha) = (estimated_recoverable_pol / 100) × tc_per_ha_actual")
print("="*80)

ADDING TOTAL RECOVERABLE SUGAR (TPER) COLUMN
Input file: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis/final_dataset_regrouped_categorical.csv

Original dataset shape: (13376, 68)
Total observations: 13,376

CALCULATING TOTAL RECOVERABLE SUGAR (TPER in t/ha)

✓ Created column: 'total_recoverable_sugar_t_per_ha'
  Formula: (estimated_recoverable_pol / 100) × tc_per_ha_actual

STATISTICS FOR NEW COLUMN

Total observations: 13,376
Non-NA values: 13,376
NA values: 0

Descriptive Statistics (Total Recoverable Sugar in t/ha):
  Mean:   13.95 t/ha
  Median: 14.00 t/ha
  Std:    2.74 t/ha
  Min:    2.08 t/ha
  Max:    25.88 t/ha

Sample calculations (first 5 non-NA rows):
--------------------------------------------------------------------------------
 estimated_recoverable_pol  tc_per_ha_actual  total_recoverable_sugar_t_per_ha
                 11.182208         66.459740                          7.431666
                 11.627661         67.548052             

## 3. Comparative Analysis: Boxplot Visualizations

This section generates publication-quality boxplots to examine the distribution of sugar content metrics and yield parameters across all categorical variables (variety, harvest mode, crop cycle, and irrigation type).

**Analysis Objectives:**
- Compare sugar content metrics (Brix, Pol, Purity, Glucose, Hugot sugar) across categories
- Evaluate yield performance (cane yield, fiber content) by management practices
- Identify significant differences between varieties, harvest methods, and crop cycles
- Visualize data distributions with statistical summaries (medians, quartiles, outliers)

**Output:** 2×2 subplot layouts for each response variable, saved in PNG (300 DPI) and PDF formats.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

# ============================================================================
# CONFIGURATION
# ============================================================================

# Input file (output from your regrouping code)
INPUT_FILE = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis/final_dataset_regrouped_categorical.csv'

# Output folder
OUTPUT_FOLDER = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis/Nature_Style_Plots'
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print("="*80)
print("BOXPLOTS FOR SUGAR CONTENT AND YIELD PARAMETERS")
print("="*80)
print(f"Input file: {INPUT_FILE}")
print(f"Output folder: {OUTPUT_FOLDER}")

# ============================================================================
# 
# ============================================================================

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 9,
    'axes.labelsize': 10,
    'axes.titlesize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 8,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'axes.linewidth': 1,
    'xtick.major.width': 1,
    'ytick.major.width': 1,
    'xtick.major.size': 4,
    'ytick.major.size': 4,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white'
})

# Set seaborn style
sns.set_style("ticks", {
    'axes.grid': False,
    'axes.spines.top': False,
    'axes.spines.right': False
})

# ============================================================================
# LOAD DATA
# ============================================================================

df = pd.read_csv(INPUT_FILE, sep=',')

print("\nDataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# ============================================================================
# DEFINE VARIABLES
# ============================================================================

# Define categorical variables
categorical_vars = {
    'variety': 'Variety',
    'harvest_mode': 'Harvest Mode',
    'crop_cycle': 'Crop Cycle',
    'irrigation_type': 'Irrigation Type'
}

# Sugar content metrics (INCLUDING TOTAL RECOVERABLE SUGAR)
SUGAR_CONTENT_METRICS = {
    "brix_percent": "Brix (%)",
    "pol_juice_percent": "Pol juice (%)",
    "pol_cane_percent": "Pol cane (%)",
    "purity_percent": "Purity (%)",
    "glucose_cane_percent": "Glucose (%)",
    "hugot_sugar": "Hugot sugar (%)",
    "total_recoverable_sugar_t_per_ha": "TPER (t ha⁻¹)"
}

# Yield parameters
YIELD_PARAMETERS = {
    "tc_per_ha_actual": "Cane yield (t ha⁻¹)",
    "tc_per_ha_per_week": "Cane yield per week (t ha⁻¹/week)",
    "tc_per_ha_52_weeks": "Cane yield 52 weeks (t ha⁻¹)",
    "fiber_percent": "Fiber in Cane (%)"
}

# Combine all response variables
response_vars = {**SUGAR_CONTENT_METRICS, **YIELD_PARAMETERS}

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def build_crop_cycle_order(df):
    """Build ordered list of crop cycles from the data"""
    if 'crop_cycle' not in df.columns:
        return []
    
    # Get unique crop cycles
    unique_cycles = df['crop_cycle'].dropna().unique()
    
    # Define the proper order
    order = ['Plant Cane']
    
    # Add ratoons in order (1st through as many as exist)
    for i in range(1, 20):
        if i == 1:
            cycle_name = '1st Ratoon'
        elif i == 2:
            cycle_name = '2nd Ratoon'
        elif i == 3:
            cycle_name = '3rd Ratoon'
        else:
            cycle_name = f'{i}th Ratoon'
        
        if cycle_name in unique_cycles:
            order.append(cycle_name)
    
    # Add "Beyond" categories if they exist
    beyond_cycles = [c for c in unique_cycles if c.startswith('Beyond')]
    order.extend(sorted(beyond_cycles))
    
    return order

crop_cycle_order = build_crop_cycle_order(df)
print(f"\nCrop cycle order detected: {crop_cycle_order}")

# Use viridis color palette for all plots
max_categories = max([df[var].nunique() for var in categorical_vars.keys() if var in df.columns])
viridis_colors = sns.color_palette("viridis", n_colors=max_categories)

# Consistent boxplot width
BOXPLOT_WIDTH = 0.5

# ============================================================================
# CREATE BOXPLOTS FOR EACH RESPONSE VARIABLE
# ============================================================================

print("\n" + "="*80)
print("CREATING BOXPLOTS")
print("="*80)

for response_var, response_label in response_vars.items():
    
    # Check if response variable exists
    if response_var not in df.columns:
        print(f"\nWarning: {response_var} not found in dataset. Skipping...")
        continue
    
    print(f"\nProcessing: {response_label}")
    
    # Create figure with 2x2 subplots
    fig, axes = plt.subplots(2, 2, figsize=(9, 8))
    axes = axes.flatten()
    
    # Plot each categorical variable
    for idx, (var, title) in enumerate(categorical_vars.items()):
        ax = axes[idx]
        
        # Check if variable exists in dataframe
        if var not in df.columns:
            ax.text(0.5, 0.5, f'{var}\nnot found', 
                    ha='center', va='center', fontsize=10)
            ax.set_title(title, fontweight='bold', pad=10)
            continue
        
        # Remove NA values for this plot
        df_plot = df[[var, response_var]].dropna()
        
        # Check if we have data to plot
        if len(df_plot) == 0:
            ax.text(0.5, 0.5, 'No data\navailable', 
                    ha='center', va='center', fontsize=10)
            ax.set_title(title, fontweight='bold', pad=10)
            continue
        
        # For crop_cycle, exclude FRICHE, JACHERE, LESSIVAGE (if they still exist)
        if var == 'crop_cycle':
            df_plot = df_plot[~df_plot[var].isin(['FRICHE', 'JACHERE', 'LESSIVAGE', 'ESSAIS'])]
        
        # Get value counts
        value_counts = df_plot[var].value_counts()
        
        # Determine order based on variable
        if var == 'crop_cycle':
            # Use dynamic order for crop cycle
            order = [cat for cat in crop_cycle_order if cat in df_plot[var].unique()]
        elif var == 'variety':
            # Order by frequency for variety (top varieties first)
            order = value_counts.index.tolist()
            # Move 'OTHER VARIETIES' to the end if it exists
            if 'OTHER VARIETIES' in order:
                order.remove('OTHER VARIETIES')
                order.append('OTHER VARIETIES')
        else:
            # Order by frequency for other variables
            order = value_counts.index.tolist()
        
        # Skip if no categories to plot
        if len(order) == 0:
            ax.text(0.5, 0.5, 'No categories\nto display', 
                    ha='center', va='center', fontsize=10)
            ax.set_title(title, fontweight='bold', pad=10)
            continue
        
        # Get colors for this variable (take first N colors from viridis)
        n_cats = len(order)
        colors = viridis_colors[:n_cats]
        
        # Create boxplot
        bp = sns.boxplot(
            data=df_plot,
            y=var,
            x=response_var,
            hue=var,
            order=order,
            palette=colors,
            width=BOXPLOT_WIDTH,
            linewidth=1.2,
            fliersize=2,
            legend=False,
            flierprops={
                'marker': 'o',
                'markerfacecolor': 'gray',
                'markeredgecolor': 'gray',
                'alpha': 0.4,
                'markersize': 3
            },
            boxprops={'edgecolor': 'black', 'linewidth': 1.2},
            whiskerprops={'color': 'black', 'linewidth': 1.2},
            capprops={'color': 'black', 'linewidth': 1.2},
            medianprops={'color': 'darkred', 'linewidth': 1.8},
            ax=ax
        )
        
        # Customize axes
        ax.set_xlabel(response_label, fontweight='normal', fontsize=10)
        ax.set_ylabel('')
        ax.set_title(title, fontweight='bold', pad=12, fontsize=11)
        
        # Add grid for better readability
        ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.8)
        ax.set_axisbelow(True)
        
        # Set dynamic x-axis range based on actual data with some padding
        x_min = df_plot[response_var].min()
        x_max = df_plot[response_var].max()
        x_range = x_max - x_min
        padding = x_range * 0.05  # 5% padding on each side
        ax.set_xlim(x_min - padding, x_max + padding)
        
        # Add sample size annotations on top of each boxplot center
        y_positions = range(len(order))
        for i, category in enumerate(order):
            count = value_counts[category]
            # Get the maximum value for this category to position text above
            cat_data = df_plot[df_plot[var] == category][response_var]
            x_pos = cat_data.quantile(0.75)  # Position at 75th percentile (top of box)
            
            # Shift position to the right and higher
            x_offset = (x_max - x_min) * 0.09  # 9% to the right
            y_offset = 0.25  # Move up slightly
            
            # Add text on top right of boxplot
            ax.text(x_pos + x_offset, i + y_offset, f'n={count}', 
                    va='center', ha='center', fontsize=7, 
                    color='black', style='italic',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', 
                             edgecolor='none', alpha=0.3))
        
        # Adjust tick parameters
        ax.tick_params(axis='both', which='major', labelsize=9)
        
        # Ensure y-axis labels don't overlap - adjust font size if many categories
        label_fontsize = 8.5 if len(order) <= 10 else 7
        ax.set_yticks(range(len(order)))
        ax.set_yticklabels(order, fontsize=label_fontsize)
    
    # Overall title based on response variable category
    if response_var in SUGAR_CONTENT_METRICS:
        fig.suptitle(f'Sugarcane {response_label} Distribution by Categorical Variables', 
                     fontsize=13, fontweight='bold', y=0.995)
        output_name = f'{response_var}_boxplots_nature_style'
    elif response_var in YIELD_PARAMETERS:
        fig.suptitle(f'Sugarcane {response_label} Distribution by Categorical Variables', 
                     fontsize=13, fontweight='bold', y=0.995)
        output_name = f'{response_var}_boxplots_nature_style'
    
    # Adjust layout to prevent overlap
    plt.tight_layout(rect=[0, 0, 1, 0.99], h_pad=2.5, w_pad=2)
    
    # Save the figure
    output_path = os.path.join(OUTPUT_FOLDER, f'{output_name}.png')
    plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"  Saved PNG: {output_path}")
    
    output_path_pdf = os.path.join(OUTPUT_FOLDER, f'{output_name}.pdf')
    plt.savefig(output_path_pdf, format='pdf', bbox_inches='tight', facecolor='white')
    print(f"  Saved PDF: {output_path_pdf}")
    
    plt.close()

# ============================================================================
# SUMMARY STATISTICS
# ============================================================================

print("\n" + "="*80)
print("SUMMARY STATISTICS: Observations per Category")
print("="*80)

for response_var, response_label in response_vars.items():
    if response_var not in df.columns:
        continue
        
    print(f"\n{'='*80}")
    print(f"RESPONSE VARIABLE: {response_label}")
    print(f"{'='*80}")
    
    for var, title in categorical_vars.items():
        if var in df.columns:
            print(f"\n{title}:")
            print("-" * 60)
            df_valid = df[[var, response_var]].dropna()
            
            # For crop_cycle, exclude special categories
            if var == 'crop_cycle':
                df_valid = df_valid[~df_valid[var].isin(['FRICHE', 'JACHERE', 'LESSIVAGE', 'ESSAIS'])]
            
            if len(df_valid) == 0:
                print("  No data available")
                continue
            
            counts = df_valid[var].value_counts().sort_values(ascending=False)
            
            # Sort by proper order for crop_cycle
            if var == 'crop_cycle':
                ordered_counts = []
                for cat in crop_cycle_order:
                    if cat in counts.index:
                        ordered_counts.append((cat, counts[cat]))
                if ordered_counts:
                    counts = pd.Series(dict(ordered_counts))
            
            for category, count in counts.items():
                mean_value = df_valid[df_valid[var] == category][response_var].mean()
                std_value = df_valid[df_valid[var] == category][response_var].std()
                print(f"  {category:<30} : {count:>5} obs (mean: {mean_value:>6.2f} ± {std_value:>5.2f})")

# ============================================================================
# CREATE SUMMARY REPORT
# ============================================================================

print("\n" + "="*80)
print("CREATING SUMMARY REPORT")
print("="*80)

summary_lines = []
summary_lines.append("="*80)
summary_lines.append("BOXPLOT ANALYSIS SUMMARY")
summary_lines.append("="*80)
summary_lines.append(f"Input file: {INPUT_FILE}")
summary_lines.append(f"Output folder: {OUTPUT_FOLDER}")
summary_lines.append(f"Dataset shape: {df.shape}")
summary_lines.append(f"Total observations: {len(df)}")
summary_lines.append("")

summary_lines.append("="*80)
summary_lines.append("VARIABLES ANALYZED")
summary_lines.append("="*80)
summary_lines.append("\nSugar Content Metrics:")
for var, label in SUGAR_CONTENT_METRICS.items():
    status = "[YES] Available" if var in df.columns else "[NO] Not found"
    summary_lines.append(f"  {label:<45} : {status}")

summary_lines.append("\nYield Parameters:")
for var, label in YIELD_PARAMETERS.items():
    status = "[YES] Available" if var in df.columns else "[NO] Not found"
    summary_lines.append(f"  {label:<45} : {status}")

summary_lines.append("\n" + "="*80)
summary_lines.append("CATEGORICAL VARIABLES")
summary_lines.append("="*80)
for var, label in categorical_vars.items():
    if var in df.columns:
        n_cats = df[var].nunique()
        n_valid = df[var].notna().sum()
        summary_lines.append(f"{label}:")
        summary_lines.append(f"  Categories: {n_cats} (including NA)")
        summary_lines.append(f"  Valid observations: {n_valid}")
        summary_lines.append("")

summary_lines.append("="*80)
summary_lines.append("OUTPUT FILES")
summary_lines.append("="*80)
summary_lines.append(f"\nTotal plots created: {len([v for v in response_vars.keys() if v in df.columns])}")
summary_lines.append("\nEach plot includes:")
summary_lines.append("  - 2x2 subplot layout (4 categorical variables)")
summary_lines.append("  - PNG format (300 DPI)")
summary_lines.append("  - PDF format (vector graphics)")
summary_lines.append("")

summary_lines.append("="*80)
summary_lines.append("ANALYSIS COMPLETE")
summary_lines.append("="*80)

# Save summary to file
summary_path = os.path.join(OUTPUT_FOLDER, 'analysis_summary.txt')
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(summary_lines))

print('\n'.join(summary_lines))
print(f"\nSummary report saved to: {summary_path}")

print("\n" + "="*80)
print("ALL BOXPLOT VISUALIZATIONS COMPLETE!")
print("="*80)
print(f"\nAll outputs saved to: {OUTPUT_FOLDER}")

NATURE STYLE BOXPLOTS FOR SUGAR CONTENT AND YIELD PARAMETERS
Input file: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis/final_dataset_regrouped_categorical.csv
Output folder: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis/Nature_Style_Plots

Dataset loaded successfully!
Shape: (13376, 68)
Columns: ['tc_per_ha_actual', 'brix_percent', 'variety', 'irrigation_type', 'crop_cycle', 'harvest_mode', 'campaign_year', 'pH_complete', 'Exchangeable_Calcium_meq_100g_complete', 'Exchangeable_Magnesium_meq_100g_complete', 'Exchangeable_Potassium_meq_100g_complete', 'Clay_percent_complete', 'Fine_Silt_percent_complete', 'Coarse_Silt_percent_complete', 'Fine_Sand_percent_complete', 'Coarse_Sand_percent_complete', 'Total_Nitrogen_per_mille_complete', 'Organic_Carbon_per_mille_complete', 'Mg_K_Ratio_complete', 'Mg_Ca_Ratio_complete', 'SC_drying_duration_actual_days', 'growing_season_rainfall_mm', 'temp_mini_mean', 'temp_maxi_mean', 'temp_moy_me

C:\Users\U078-N800\AppData\Local\Temp\ipykernel_25096\2962144306.py:302: UserWarning: Glyph 8315 (\N{SUPERSCRIPT MINUS}) missing from font(s) Arial.
  plt.tight_layout(rect=[0, 0, 1, 0.99], h_pad=2.5, w_pad=2)
C:\Users\U078-N800\AppData\Local\Temp\ipykernel_25096\2962144306.py:306: UserWarning: Glyph 8315 (\N{SUPERSCRIPT MINUS}) missing from font(s) Arial.
  plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')


  Saved PNG: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis/Nature_Style_Plots\total_recoverable_sugar_t_per_ha_boxplots_nature_style.png


C:\Users\U078-N800\AppData\Local\Temp\ipykernel_25096\2962144306.py:310: UserWarning: Glyph 8315 (\N{SUPERSCRIPT MINUS}) missing from font(s) Arial.
  plt.savefig(output_path_pdf, format='pdf', bbox_inches='tight', facecolor='white')
C:\Users\U078-N800\AppData\Local\Temp\ipykernel_25096\2962144306.py:310: UserWarning: Glyph 8315 (\N{SUPERSCRIPT MINUS}) missing from font(s) Arial.
  plt.savefig(output_path_pdf, format='pdf', bbox_inches='tight', facecolor='white')


  Saved PDF: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis/Nature_Style_Plots\total_recoverable_sugar_t_per_ha_boxplots_nature_style.pdf

Processing: Cane yield (t ha⁻¹)


...

  Saved PDF: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis/Nature_Style_Plots\tc_per_ha_per_week_boxplots_nature_style.pdf

Processing: Cane yield 52 weeks (t ha⁻¹)


C:\Users\U078-N800\AppData\Local\Temp\ipykernel_25096\2962144306.py:302: UserWarning: Glyph 8315 (\N{SUPERSCRIPT MINUS}) missing from font(s) Arial.
  plt.tight_layout(rect=[0, 0, 1, 0.99], h_pad=2.5, w_pad=2)
C:\Users\U078-N800\AppData\Local\Temp\ipykernel_25096\2962144306.py:306: UserWarning: Glyph 8315 (\N{SUPERSCRIPT MINUS}) missing from font(s) Arial.
  plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')


  Saved PNG: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis/Nature_Style_Plots\tc_per_ha_52_weeks_boxplots_nature_style.png


C:\Users\U078-N800\AppData\Local\Temp\ipykernel_25096\2962144306.py:310: UserWarning: Glyph 8315 (\N{SUPERSCRIPT MINUS}) missing from font(s) Arial.
  plt.savefig(output_path_pdf, format='pdf', bbox_inches='tight', facecolor='white')
C:\Users\U078-N800\AppData\Local\Temp\ipykernel_25096\2962144306.py:310: UserWarning: Glyph 8315 (\N{SUPERSCRIPT MINUS}) missing from font(s) Arial.
  plt.savefig(output_path_pdf, format='pdf', bbox_inches='tight', facecolor='white')


  Saved PDF: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis/Nature_Style_Plots\tc_per_ha_52_weeks_boxplots_nature_style.pdf

Processing: Fiber in Cane (%)
  Saved PNG: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis/Nature_Style_Plots\fiber_percent_boxplots_nature_style.png
  Saved PDF: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis/Nature_Style_Plots\fiber_percent_boxplots_nature_style.pdf

SUMMARY STATISTICS: Observations per Category

RESPONSE VARIABLE: Brix (%)

Variety:
------------------------------------------------------------
  N 14                           :  4559 obs (mean:  19.03 ±  1.34)
  R 579                          :  1958 obs (mean:  19.75 ±  1.09)
  SP 701284                      :  1731 obs (mean:  20.64 ±  1.29)
  CP 742005                      :  1307 obs (mean:  19.18 ±  1.38)
  CP 722086                      :  1281 obs (mean:  19.25 ±  1.36)
  FR 8783                     

In [ ]:
all_selected_features = [
    'pH_complete', 'Exchangeable_Calcium_meq_100g_complete', 'Exchangeable_Magnesium_meq_100g_complete', 'Exchangeable_Potassium_meq_100g_complete', 'Exchangeable_Sodium_meq_100g_complete',
    'Available_P2O5_mg_kg_complete', 'Total_Nitrogen_per_mille_complete', 'Organic_Carbon_per_mille_complete', 'Electrical_Conductivity_mmho_cm_complete', 'Mg_K_Ratio_complete', 'Mg_Ca_Ratio_complete',
    'Clay_percent_complete', 'Coarse_Silt_percent_complete', 'Coarse_Sand_percent_complete', 'growing_season_rainfall_mm', 'evaporation_total', 'CDD', 'SDII',
    'temp_mini_mean', 'temp_maxi_mean', 'temp_moy_mean', 'ampl_mean', 'h_mini_mean', 'h_maxi_mean', 'h_moy_mean', 'h_6h_mean', 'h_12h_mean', 'h_16h_mean', 'h_24h_mean',
    'soil_temp_6h_mean', 'soil_temp_16h_mean', 'radiation_ERA_total', 'HD'
]

In [ ]:
all_selected_features = [
    'pH_complete', 'Exchangeable_Calcium_meq_100g_complete', 'Exchangeable_Potassium_meq_100g_complete', 'Exchangeable_Sodium_meq_100g_complete',
    'Available_P2O5_mg_kg_complete', 'Total_Nitrogen_per_mille_complete', 'Electrical_Conductivity_mmho_cm_complete', 'Mg_K_Ratio_complete',
    'Coarse_Silt_percent_complete', 'Coarse_Sand_percent_complete', 'growing_season_rainfall_mm', 'CDD', 'SDII',
    'temp_mini_mean', 'temp_maxi_mean', 'h_mini_mean', 'h_maxi_mean',
    'soil_temp_6h_mean', 'radiation_ERA_total', 'CWD', 'Rx1', 'R20', 'TNn', 'TNx'
    ]

## 4. Machine Learning Pipeline: K-Fold Cross-Validation

This section implements a comprehensive machine learning pipeline to predict sugar content and yield parameters using environmental, soil, and agronomic predictors.

**Pipeline Components:**

**I. Model Selection & Training**
- Three regression algorithms: Linear Regression, Random Forest, XGBoost
- Hyperparameter optimization using Optuna (Bayesian optimization)
- 5-fold cross-validation for robust performance estimation

**II. Performance Metrics**
- **R²**: Coefficient of determination (model fit quality)
- **RMSE**: Root Mean Square Error (prediction accuracy)
- **MAE**: Mean Absolute Error (average prediction deviation)
- **RPD**: Ratio of Performance to Deviation (model reliability)
- **RPIQ**: Ratio of Performance to Inter-Quartile range (outlier robustness)

**III. Feature Importance Analysis**
- Identifies key environmental and soil drivers
- Ranks predictors by their contribution to model performance

**Output:** Trained models, performance summaries, feature importance rankings, and fold-by-fold results for each response variable.

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, cross_validate
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, VotingRegressor, BaggingRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
import optuna
import joblib
import os
import json
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# DEFINE AVAILABLE RESPONSE VARIABLES
# ============================================================================

SUGAR_CONTENT_METRICS = {
    "brix_percent": "Brix (%)",
    "pol_juice_percent": "Pol juice (%)",
    "pol_cane_percent": "Pol cane (%)",
    "purity_percent": "Purity (%)",
    "glucose_cane_percent": "Glucose (%)",
    "hugot_sugar": "Hugot sugar (%)",
    "total_recoverable_sugar_t_per_ha": "TPER (t ha⁻¹)"
}

YIELD_PARAMETERS = {
    "tc_per_ha_actual": "Cane yield (t ha⁻¹)",
    "tc_per_ha_per_week": "Cane yield per week (t ha⁻¹/week)",
    "tc_per_ha_52_weeks": "Cane yield 52 weeks (t ha⁻¹)",
    "fiber_percent": "Fiber in Cane (%)",
}

ALL_RESPONSE_VARIABLES = {**SUGAR_CONTENT_METRICS, **YIELD_PARAMETERS}

# ============================================================================
# MODIFIED DATA LOADING (WITH RESPONSE VARIABLE PARAMETER)
# ============================================================================

def load_and_prepare_multiyear_data(file_path, response_variable, custom_predictors=None, categorical_vars=None):
    """Load CSV data and prepare it for prediction with specified response variable"""
    print(f"Starting Analysis for Response Variable: {response_variable}")
    print(f"Display name: {ALL_RESPONSE_VARIABLES.get(response_variable, response_variable)}")
    print("="*80)

    df = pd.read_csv(file_path)
    
    # Check if response variable exists
    if response_variable not in df.columns:
        raise ValueError(f"Response variable '{response_variable}' not found in dataset!")
    
    # Drop ALL other response variables (keep only target)
    all_response_cols = list(ALL_RESPONSE_VARIABLES.keys())
    cols_to_drop = [col for col in all_response_cols 
                    if col != response_variable and col in df.columns]
    
    if cols_to_drop:
        print(f"Dropping other response variables: {cols_to_drop}")
        df = df.drop(columns=cols_to_drop)

    if custom_predictors is not None:
        print(f"Using custom predictors: {custom_predictors}")
        
        year_column = None
        for col in ['campaign_year', 'Harvest_Year', 'year']:
            if col in df.columns:
                year_column = col
                break

        columns_to_keep = [response_variable] + custom_predictors
        if year_column:
            columns_to_keep.append(year_column)
        if categorical_vars:
            columns_to_keep.extend(categorical_vars)

        missing_predictors = [col for col in custom_predictors if col not in df.columns]
        if missing_predictors:
            raise ValueError(f"Custom predictors not found: {missing_predictors}")
        
        if categorical_vars:
            missing_categorical = [col for col in categorical_vars if col not in df.columns]
            if missing_categorical:
                raise ValueError(f"Categorical variables not found: {missing_categorical}")

        df = df[columns_to_keep]
        print(f"Remaining columns: {list(df.columns)}")

    initial_count = len(df)
    df = df.dropna(subset=[response_variable])
    final_count = len(df)
    
    if initial_count - final_count > 0:
        print(f"Dropped {initial_count - final_count} observations with missing {response_variable}")

    year_column = None
    for col in ['campaign_year', 'Harvest_Year', 'year']:
        if col in df.columns:
            year_column = col
            break

    samples = df.copy()
    encoders = {}

    if year_column:
        unique_years = sorted(samples[year_column].unique())
        print(f"Years available: {unique_years}")
        year_counts = samples[year_column].value_counts().sort_index()
        for year, count in year_counts.items():
            print(f"   {year}: {count} samples")
        
        year_encoder = LabelEncoder()
        samples[f'{year_column}_encoded'] = year_encoder.fit_transform(samples[year_column])
        encoders[year_column] = year_encoder
    else:
        unique_years = ["All data"]
        print("No year column found")

    # Encode categorical variables
    if categorical_vars:
        print(f"\nEncoding categorical variables: {categorical_vars}")
        for cat_var in categorical_vars:
            if cat_var in samples.columns:
                cat_encoder = LabelEncoder()
                samples[f'{cat_var}_encoded'] = cat_encoder.fit_transform(samples[cat_var].astype(str))
                encoders[cat_var] = cat_encoder
                print(f"   {cat_var}: {len(cat_encoder.classes_)} unique values")
                print(f"      Classes: {list(cat_encoder.classes_)}")

    numeric_cols = samples.select_dtypes(include=[np.number]).columns.tolist()
    for col in numeric_cols:
        if samples[col].isnull().any():
            samples[col] = samples[col].fillna(samples[col].median())

    print(f"Total dataset shape: {samples.shape}")
    print(f"{response_variable} statistics:\n{samples[response_variable].describe()}")

    return samples, encoders, unique_years, year_column

def prepare_multiyear_features_and_target(samples, encoders, year_column, response_variable, 
                                         custom_predictors=None, categorical_vars=None):
    """Prepare features and target for modeling"""
    TARGET = response_variable

    if custom_predictors is not None:
        available_predictors = [col for col in custom_predictors if col in samples.columns]
        FEATURES = available_predictors.copy()
        
        # Add encoded categorical variables
        if categorical_vars:
            for cat_var in categorical_vars:
                encoded_col = f'{cat_var}_encoded'
                if encoded_col in samples.columns:
                    FEATURES.append(encoded_col)
        
        print(f"Using custom predictors: {available_predictors}")
        if categorical_vars:
            print(f"Added encoded categorical variables: {[f'{cv}_encoded' for cv in categorical_vars]}")
    else:
        numeric_cols = samples.select_dtypes(include=[np.number]).columns.tolist()
        exclude_cols = [response_variable]
        if year_column:
            exclude_cols.extend([year_column, f'{year_column}_encoded'])
        if categorical_vars:
            exclude_cols.extend(categorical_vars)
        # Exclude all other response variables
        all_response_cols = list(ALL_RESPONSE_VARIABLES.keys())
        exclude_cols.extend([col for col in all_response_cols if col != response_variable])
        FEATURES = [col for col in numeric_cols if col not in exclude_cols]

    print(f"Total features: {len(FEATURES)}")
    return FEATURES, TARGET

def get_model_configurations():
    """Define all models for prediction"""
    models = {
        'LinearRegression': {
            'model': LinearRegression(),
            'params': {},
            'complexity': 'Simple'
        },
        'RandomForest': {
            'model': RandomForestRegressor(random_state=42),
            'params': {
                'n_estimators': [100, 200, 300, 500],
                'max_depth': [10, 15, 20, 25, None],
                'min_samples_split': [2, 5, 10],
                'min_samples_leaf': [1, 2, 4],
                'max_features': ['sqrt', 'log2', None]
            },
            'complexity': 'Moderate'
        },
        'XGBoost': {
            'model': xgb.XGBRegressor(random_state=42),
            'params': {
                'n_estimators': [100, 200, 400, 600],
                'max_depth': [3, 5, 7, 9],
                'learning_rate': [0.05, 0.1, 0.15, 0.2],
                'subsample': [0.8, 0.9, 1.0],
                'colsample_bytree': [0.8, 0.9, 1.0],
                'reg_alpha': [0, 0.1, 0.5],
                'reg_lambda': [1, 1.5, 2]
            },
            'complexity': 'Complex'
        }
    }
    return models

def optimize_model_optuna(model_name, model_config, X, y, n_trials=50):
    """Optimize model hyperparameters using Optuna with inner CV"""
    def objective(trial):
        params = {}
        for param_name, param_values in model_config['params'].items():
            if isinstance(param_values, list):
                if all(isinstance(v, (int, float)) for v in param_values):
                    if all(isinstance(v, int) for v in param_values):
                        params[param_name] = trial.suggest_int(param_name, min(param_values), max(param_values))
                    else:
                        params[param_name] = trial.suggest_float(param_name, min(param_values), max(param_values))
                else:
                    params[param_name] = trial.suggest_categorical(param_name, param_values)

        model = model_config['model'].__class__(**{**model_config['model'].get_params(), **params})
        
        inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)
        scores = []
        
        for train_idx, val_idx in inner_cv.split(X):
            try:
                X_train_inner = X[train_idx].copy()
                X_val_inner = X[val_idx].copy()
                y_train_inner = y.iloc[train_idx].copy()
                y_val_inner = y.iloc[val_idx].copy()
                
                scaler = StandardScaler()
                X_train_scaled = scaler.fit_transform(X_train_inner)
                X_val_scaled = scaler.transform(X_val_inner)
                
                model.fit(X_train_scaled, y_train_inner)
                y_pred = model.predict(X_val_scaled)
                rmse = np.sqrt(mean_squared_error(y_val_inner, y_pred))
                scores.append(rmse)
                
                del X_train_inner, X_val_inner, X_train_scaled, X_val_scaled, y_pred
                
            except (MemoryError, np.core._exceptions._ArrayMemoryError) as e:
                print(f"   Memory error in fold, skipping trial with penalty")
                return 1e6
        
        if len(scores) == 0:
            return 1e6
            
        return np.mean(scores)

    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=n_trials, catch=(MemoryError,))
    
    return study.best_params, study.best_value

def calculate_rpd_rpiq(y_true, y_pred):
    """Calculate RPD and RPIQ metrics"""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    sd = np.std(y_true)
    q75, q25 = np.percentile(y_true, [75, 25])
    iqr = q75 - q25
    
    rpd = sd / rmse if rmse > 0 else 0
    rpiq = iqr / rmse if rmse > 0 else 0
    
    return rpd, rpiq

def perform_kfold_cv(model, X, y, n_splits=5):
    """Perform k-fold cross-validation with proper preprocessing in each fold"""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    fold_results = []
    all_predictions = []
    all_actuals = []
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)
        
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_val_scaled)
        
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        r2 = r2_score(y_val, y_pred)
        mae = mean_absolute_error(y_val, y_pred)
        rpd, rpiq = calculate_rpd_rpiq(y_val, y_pred)
        
        fold_results.append({
            'fold': fold,
            'rmse': rmse,
            'r2': r2,
            'mae': mae,
            'rpd': rpd,
            'rpiq': rpiq
        })
        
        all_predictions.extend(y_pred)
        all_actuals.extend(y_val)
    
    overall_rmse = np.sqrt(mean_squared_error(all_actuals, all_predictions))
    overall_r2 = r2_score(all_actuals, all_predictions)
    overall_mae = mean_absolute_error(all_actuals, all_predictions)
    overall_rpd, overall_rpiq = calculate_rpd_rpiq(all_actuals, all_predictions)
    
    fold_rmse = [f['rmse'] for f in fold_results]
    fold_r2 = [f['r2'] for f in fold_results]
    fold_mae = [f['mae'] for f in fold_results]
    fold_rpd = [f['rpd'] for f in fold_results]
    fold_rpiq = [f['rpiq'] for f in fold_results]
    
    return {
        'fold_results': fold_results,
        'mean_rmse': np.mean(fold_rmse),
        'std_rmse': np.std(fold_rmse),
        'mean_r2': np.mean(fold_r2),
        'std_r2': np.std(fold_r2),
        'mean_mae': np.mean(fold_mae),
        'std_mae': np.std(fold_mae),
        'mean_rpd': np.mean(fold_rpd),
        'std_rpd': np.std(fold_rpd),
        'mean_rpiq': np.mean(fold_rpiq),
        'std_rpiq': np.std(fold_rpiq),
        'overall_rmse': overall_rmse,
        'overall_r2': overall_r2,
        'overall_mae': overall_mae,
        'overall_rpd': overall_rpd,
        'overall_rpiq': overall_rpiq
    }

def train_and_evaluate_models_kfold(X, y, models, optimize_hyperparams=True, n_trials=50, n_folds=5):
    """Train and evaluate all models using k-fold CV"""
    results = {}
    
    for model_name, model_config in models.items():
        print(f"\nTraining {model_name} with {n_folds}-fold CV...")
        
        try:
            if optimize_hyperparams and model_config['params']:
                print(f"   Optimizing hyperparameters ({n_trials} trials)...")
                best_params, best_cv_score = optimize_model_optuna(
                    model_name, model_config, X.values, y, n_trials=n_trials
                )
                model = model_config['model'].__class__(**{**model_config['model'].get_params(), **best_params})
            else:
                model = model_config['model']
                best_params = model_config['model'].get_params()
            
            cv_results = perform_kfold_cv(model, X, y, n_splits=n_folds)
            
            scaler = StandardScaler()
            X_scaled = scaler.fit_transform(X)
            model.fit(X_scaled, y)
            
            results[model_name] = {
                'model': model,
                'scaler': scaler,
                'best_params': best_params,
                'complexity': model_config['complexity'],
                'cv_results': cv_results,
                'mean_rmse': cv_results['mean_rmse'],
                'std_rmse': cv_results['std_rmse'],
                'mean_r2': cv_results['mean_r2'],
                'std_r2': cv_results['std_r2'],
                'mean_mae': cv_results['mean_mae'],
                'std_mae': cv_results['std_mae'],
                'mean_rpd': cv_results['mean_rpd'],
                'std_rpd': cv_results['std_rpd'],
                'mean_rpiq': cv_results['mean_rpiq'],
                'std_rpiq': cv_results['std_rpiq'],
                'overall_rmse': cv_results['overall_rmse'],
                'overall_r2': cv_results['overall_r2'],
                'overall_mae': cv_results['overall_mae'],
                'overall_rpd': cv_results['overall_rpd'],
                'overall_rpiq': cv_results['overall_rpiq']
            }
            
            print(f"   Mean CV R²: {cv_results['mean_r2']:.4f} ± {cv_results['std_r2']:.4f}")
            print(f"   Mean CV RMSE: {cv_results['mean_rmse']:.4f} ± {cv_results['std_rmse']:.4f}")
            print(f"   Mean CV RPD: {cv_results['mean_rpd']:.4f} ± {cv_results['std_rpd']:.4f}")
            print(f"   Mean CV RPIQ: {cv_results['mean_rpiq']:.4f} ± {cv_results['std_rpiq']:.4f}")
            print(f"   Overall R²: {cv_results['overall_r2']:.4f}")
            
        except Exception as e:
            print(f"   Error training {model_name}: {str(e)}")
            continue
    
    return results

def create_comprehensive_summary(results):
    """Create summary dataframe of all models"""
    summary_data = []
    
    for model_name, result in results.items():
        summary_data.append({
            'Model': model_name,
            'Complexity': result['complexity'],
            'Mean_CV_R2': result['mean_r2'],
            'Std_CV_R2': result['std_r2'],
            'Mean_CV_RMSE': result['mean_rmse'],
            'Std_CV_RMSE': result['std_rmse'],
            'Mean_CV_MAE': result['mean_mae'],
            'Std_CV_MAE': result['std_mae'],
            'Mean_CV_RPD': result['mean_rpd'],
            'Std_CV_RPD': result['std_rpd'],
            'Mean_CV_RPIQ': result['mean_rpiq'],
            'Std_CV_RPIQ': result['std_rpiq'],
            'Overall_R2': result['overall_r2'],
            'Overall_RMSE': result['overall_rmse'],
            'Overall_MAE': result['overall_mae'],
            'Overall_RPD': result['overall_rpd'],
            'Overall_RPIQ': result['overall_rpiq']
        })
    
    summary_df = pd.DataFrame(summary_data)
    summary_df = summary_df.sort_values(['Overall_R2', 'Overall_RMSE'], ascending=[False, True])
    
    return summary_df

def analyze_feature_importance(results, feature_cols):
    """Analyze feature importance from tree-based models"""
    print("\n=== FEATURE IMPORTANCE ANALYSIS ===")
    tree_models = ['RandomForest', 'ExtraTrees', 'GradientBoosting', 'XGBoost', 'LightGBM', 'CatBoost']
    
    feature_importance_summary = {}
    
    for model_name in tree_models:
        if model_name in results:
            model = results[model_name]['model']
            if hasattr(model, 'feature_importances_'):
                importances = model.feature_importances_
                feature_importance = list(zip(feature_cols, importances))
                feature_importance.sort(key=lambda x: x[1], reverse=True)
                feature_importance_summary[model_name] = feature_importance
                
                print(f"\nTop 15 features ({model_name}):")
                for i, (feature, importance) in enumerate(feature_importance[:15]):
                    print(f"  {i+1:2d}. {feature:<40}: {importance:.4f}")
    
    if feature_importance_summary:
        print(f"\n=== AVERAGE FEATURE IMPORTANCE ===")
        feature_avg_importance = {}
        for feature in feature_cols:
            importances = []
            for model_name, feature_importance in feature_importance_summary.items():
                for feat, imp in feature_importance:
                    if feat == feature:
                        importances.append(imp)
                        break
            if importances:
                feature_avg_importance[feature] = np.mean(importances)
        
        sorted_avg_importance = sorted(feature_avg_importance.items(), key=lambda x: x[1], reverse=True)
        
        print("Top 15 features by average importance:")
        for i, (feature, avg_importance) in enumerate(sorted_avg_importance[:15]):
            print(f"  {i+1:2d}. {feature:<40}: {avg_importance:.4f}")
        
        return sorted_avg_importance
    
    return None

def save_kfold_results(results, best_model_name, unique_years, feature_importance, 
                      year_column, encoders, response_variable, output_dir):
    """Save all models and results"""
    os.makedirs(output_dir, exist_ok=True)
    
    models_dir = os.path.join(output_dir, 'models')
    analysis_dir = os.path.join(output_dir, 'analysis')
    os.makedirs(models_dir, exist_ok=True)
    os.makedirs(analysis_dir, exist_ok=True)
    
    for model_name, result in results.items():
        model_filename = f'{model_name.replace(" ", "_")}_{response_variable}.pkl'
        joblib.dump({
            'model': result['model'],
            'scaler': result['scaler'],
            'encoders': encoders
        }, os.path.join(models_dir, model_filename))
    
    summary_df = create_comprehensive_summary(results)
    summary_df.to_csv(os.path.join(output_dir, f'kfold_cv_summary_{response_variable}.csv'), index=False)
    
    fold_details = []
    for model_name, result in results.items():
        for fold_result in result['cv_results']['fold_results']:
            fold_details.append({
                'Model': model_name,
                'Fold': fold_result['fold'],
                'RMSE': fold_result['rmse'],
                'R2': fold_result['r2'],
                'MAE': fold_result['mae'],
                'RPD': fold_result['rpd'],
                'RPIQ': fold_result['rpiq']
            })
    
    fold_df = pd.DataFrame(fold_details)
    fold_df.to_csv(os.path.join(output_dir, f'fold_by_fold_results_{response_variable}.csv'), index=False)
    
    if feature_importance:
        feature_importance_df = pd.DataFrame(feature_importance, columns=['Feature', 'Importance'])
        feature_importance_df.to_csv(os.path.join(analysis_dir, f'feature_importance_{response_variable}.csv'), index=False)
    
    if encoders:
        encoder_info = {}
        for var_name, encoder in encoders.items():
            encoder_info[var_name] = {
                'classes': [str(c) for c in encoder.classes_]
            }
        with open(os.path.join(output_dir, f'encoders_info_{response_variable}.json'), 'w') as f:
            json.dump(encoder_info, f, indent=4)
    
    best_model_info = {
        'best_model_name': best_model_name,
        'mean_cv_r2': float(results[best_model_name]['mean_r2']),
        'std_cv_r2': float(results[best_model_name]['std_r2']),
        'mean_cv_rmse': float(results[best_model_name]['mean_rmse']),
        'std_cv_rmse': float(results[best_model_name]['std_rmse']),
        'mean_cv_rpd': float(results[best_model_name]['mean_rpd']),
        'std_cv_rpd': float(results[best_model_name]['std_rpd']),
        'mean_cv_rpiq': float(results[best_model_name]['mean_rpiq']),
        'std_cv_rpiq': float(results[best_model_name]['std_rpiq']),
        'overall_r2': float(results[best_model_name]['overall_r2']),
        'overall_rmse': float(results[best_model_name]['overall_rmse']),
        'overall_rpd': float(results[best_model_name]['overall_rpd']),
        'overall_rpiq': float(results[best_model_name]['overall_rpiq']),
        'years_in_dataset': [int(y) if hasattr(y, 'item') else y for y in unique_years],
        'year_column': year_column,
        'target_variable': response_variable
    }
    
    with open(os.path.join(output_dir, f'best_model_info_{response_variable}.json'), 'w') as f:
        json.dump(best_model_info, f, indent=4)
    
    print(f"\nAll models and results saved to: {output_dir}")
    print(f"Best model: {best_model_name}")

def main(file_path, response_variable, custom_predictors=None, categorical_vars=None, 
         base_output_dir=None, n_folds=5):
    """Main function for k-fold CV pipeline with flexible response variable"""
    
    # Validate response variable
    if response_variable not in ALL_RESPONSE_VARIABLES:
        raise ValueError(f"Response variable '{response_variable}' not recognized!\n"
                        f"Available options: {list(ALL_RESPONSE_VARIABLES.keys())}")
    
    # Create response-specific output directory
    if base_output_dir is None:
        base_output_dir = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis'
    
    output_dir = os.path.join(base_output_dir, f'kfold_models_{response_variable}')
    
    if categorical_vars:
        output_dir += '_with_categorical'
    elif custom_predictors:
        output_dir += '_custom'
    
    os.makedirs(output_dir, exist_ok=True)
    
    print("\n" + "="*80)
    print(f"K-FOLD CV PIPELINE FOR: {response_variable}")
    print(f"Display name: {ALL_RESPONSE_VARIABLES[response_variable]}")
    print("="*80)
    print(f"Output directory: {output_dir}\n")
    
    print("Loading data...")
    samples, encoders, unique_years, year_column = load_and_prepare_multiyear_data(
        file_path, response_variable, custom_predictors, categorical_vars
    )
    
    FEATURES, TARGET = prepare_multiyear_features_and_target(
        samples, encoders, year_column, response_variable, custom_predictors, categorical_vars
    )
    
    print(f"\nDataset shape: {samples.shape}")
    print(f"Features: {len(FEATURES)}")
    print(f"Target range: {samples[TARGET].min():.2f} - {samples[TARGET].max():.2f}")
    
    X = samples[FEATURES].copy()
    y = samples[TARGET].copy()
    
    print("\nInitializing models...")
    models = get_model_configurations()
    
    print(f"\nTraining models with {n_folds}-fold cross-validation...")
    results = train_and_evaluate_models_kfold(
        X, y, models, optimize_hyperparams=True, n_trials=30, n_folds=n_folds
    )
    
    best_model_name = max(results.keys(), key=lambda x: results[x]['overall_r2'])
    
    print(f"\n=== BEST MODEL: {best_model_name} ===")
    print(f"Overall R²: {results[best_model_name]['overall_r2']:.4f}")
    print(f"Overall RMSE: {results[best_model_name]['overall_rmse']:.4f}")
    print(f"Overall RPD: {results[best_model_name]['overall_rpd']:.4f}")
    print(f"Overall RPIQ: {results[best_model_name]['overall_rpiq']:.4f}")
    print(f"Mean CV R²: {results[best_model_name]['mean_r2']:.4f} ± {results[best_model_name]['std_r2']:.4f}")
    print(f"Mean CV RMSE: {results[best_model_name]['mean_rmse']:.4f} ± {results[best_model_name]['std_rmse']:.4f}")
    print(f"Mean CV RPD: {results[best_model_name]['mean_rpd']:.4f} ± {results[best_model_name]['std_rpd']:.4f}")
    print(f"Mean CV RPIQ: {results[best_model_name]['mean_rpiq']:.4f} ± {results[best_model_name]['std_rpiq']:.4f}")
    
    feature_importance = analyze_feature_importance(results, FEATURES)
    
    print("\n=== MODEL SUMMARY ===")
    summary_df = create_comprehensive_summary(results)
    print(summary_df.to_string(index=False))
    
    save_kfold_results(results, best_model_name, unique_years, feature_importance, 
                      year_column, encoders, response_variable, output_dir)
    
    print("\n=== K-FOLD CV ANALYSIS COMPLETE ===")
    print(f"Results saved to: {output_dir}")
    print(f"Response variable: {response_variable} - {ALL_RESPONSE_VARIABLES[response_variable]}")
    
    return {
        'results': results,
        'best_model_name': best_model_name,
        'output_dir': output_dir,
        'response_variable': response_variable
    }

# ============================================================================
# USAGE EXAMPLES
# ============================================================================

if __name__ == "__main__":
    
    # Same predictors as in your original code
    custom_features = [
    'pH_complete', 'Exchangeable_Calcium_meq_100g_complete', 'Exchangeable_Potassium_meq_100g_complete', 'Exchangeable_Sodium_meq_100g_complete',
    'Available_P2O5_mg_kg_complete', 'Total_Nitrogen_per_mille_complete', 'Electrical_Conductivity_mmho_cm_complete', 'Mg_K_Ratio_complete',
    'Coarse_Silt_percent_complete', 'Coarse_Sand_percent_complete', 'growing_season_rainfall_mm', 'CDD', 'SDII',
    'temp_mini_mean', 'temp_maxi_mean', 'h_mini_mean', 'h_maxi_mean',
    'soil_temp_6h_mean', 'radiation_ERA_total', 'CWD', 'Rx1', 'R20', 'TNn', 'TNx'
    ]
    
    categorical_features = ['variety', 'crop_cycle', 'irrigation_type', 'harvest_mode']
    
    # File path
    data_file = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis/final_dataset_regrouped_categorical.csv'
    
    # Base output directory (subfolders will be created for each response)
    base_output = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis'
    
    print("="*80)
    print("FLEXIBLE K-FOLD CV PIPELINE - MULTIPLE RESPONSE VARIABLES")
    print("="*80)
    print("\nAvailable response variables:")
    print("\nSUGAR CONTENT METRICS:")
    for key, label in SUGAR_CONTENT_METRICS.items():
        print(f"  - {key}: {label}")
    print("\nYIELD PARAMETERS:")
    for key, label in YIELD_PARAMETERS.items():
        print(f"  - {key}: {label}")
    print("="*80 + "\n")
    
    try:
        # Example 1: Run for brix_percent (original)
        """print("\n### RUNNING PIPELINE FOR: brix_percent ###\n")
        results_brix = main(
            file_path=data_file,
            response_variable='brix_percent',
            custom_predictors=custom_features,
            categorical_vars=categorical_features,
            base_output_dir=base_output,
            n_folds=5
        )
        
        # Example 2: Run for pol_cane_percent
        print("\n\n### RUNNING PIPELINE FOR: pol_cane_percent ###\n")
        results_pol = main(
            file_path=data_file,
            response_variable='hugot_sugar',
            custom_predictors=custom_features,
            categorical_vars=categorical_features,
            base_output_dir=base_output,
            n_folds=5
        )
        
        # Example 3: Run for tc_per_ha_actual
        print("\n\n### RUNNING PIPELINE FOR: tc_per_ha_actual ###\n")
        results_yield = main(
            file_path=data_file,
            response_variable='tc_per_ha_actual',
            custom_predictors=custom_features,
            categorical_vars=categorical_features,
            base_output_dir=base_output,
            n_folds=5
        )

        # Example 4: Run for purity
        print("\n\n### RUNNING PIPELINE FOR: purity_percent ###\n")
        results_yield = main(
            file_path=data_file,
            response_variable='purity_percent',
            custom_predictors=custom_features,
            categorical_vars=categorical_features,
            base_output_dir=base_output,
            n_folds=5
        )
        """
        # Example 5: Run for TPER
        print("\n\n### RUNNING PIPELINE FOR: total_recoverable_sugar_t_per_ha ###\n")
        results_tper = main(
            file_path=data_file,
            response_variable='total_recoverable_sugar_t_per_ha',
            custom_predictors=custom_features,
            categorical_vars=categorical_features,
            base_output_dir=base_output,
            n_folds=5
        )
        

        print("\n\n### RUNNING PIPELINE FOR: tc_per_ha_actual ###\n")       
        print("\n" + "="*80)
        print("ALL PIPELINES COMPLETED SUCCESSFULLY!")
        print("="*80)
        print("\nSummary of Results:")
        print(f"\n1. brix_percent ({SUGAR_CONTENT_METRICS['brix_percent']}):")
        #print(f"   Best Model: {results_brix['best_model_name']}")
        #print(f"   Output: {results_brix['output_dir']}")
        
        print(f"\n2. pol_cane_percent ({SUGAR_CONTENT_METRICS['pol_cane_percent']}):")
        #print(f"   Best Model: {results_pol['best_model_name']}")
        #print(f"   Output: {results_pol['output_dir']}")
        
        print(f"\n3. tc_per_ha_actual ({YIELD_PARAMETERS['tc_per_ha_actual']}):")
        #print(f"   Best Model: {results_yield['best_model_name']}")
        #print(f"   Output: {results_yield['output_dir']}")
        
        print("\n" + "="*80)
        print("TO RUN FOR A DIFFERENT RESPONSE VARIABLE:")
        print("="*80)
        print("\nSimply call:")
        print("  main(file_path=data_file,")
        print("       response_variable='your_choice',  # e.g., 'purity_percent'")
        print("       custom_predictors=custom_features,")
        print("       categorical_vars=categorical_features,")
        print("       base_output_dir=base_output,")
        print("       n_folds=5)")
        print("\nAvailable choices:")
        for key in ALL_RESPONSE_VARIABLES.keys():
            print(f"  - '{key}'")
        print("="*80)
        
    except Exception as e:
        print(f"\nError: {e}")
        import traceback
        traceback.print_exc()

FLEXIBLE K-FOLD CV PIPELINE - MULTIPLE RESPONSE VARIABLES

Available response variables:

SUGAR CONTENT METRICS:
  - brix_percent: Brix (%)
  - pol_juice_percent: Pol juice (%)
  - pol_cane_percent: Pol cane (%)
  - purity_percent: Purity (%)
  - glucose_cane_percent: Glucose (%)
  - hugot_sugar: Hugot sugar (%)
  - total_recoverable_sugar_t_per_ha: TRS (t ha⁻¹)

YIELD PARAMETERS:
  - tc_per_ha_actual: Cane yield (t ha⁻¹)
  - tc_per_ha_per_week: Cane yield per week (t ha⁻¹/week)
  - tc_per_ha_52_weeks: Cane yield 52 weeks (t ha⁻¹)
  - fiber_percent: Fiber in Cane (%)



### RUNNING PIPELINE FOR: purity_percent ###


K-FOLD CV PIPELINE FOR: total_recoverable_sugar_t_per_ha
Display name: TRS (t ha⁻¹)
Output directory: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis\kfold_models_total_recoverable_sugar_t_per_ha_with_categorical

Loading data...
Starting Analysis for Response Variable: total_recoverable_sugar_t_per_ha
Display name: TRS (t ha⁻¹)
Dropping other r

[I 2026-01-06 08:46:26,147] A new study created in memory with name: no-name-0089c43f-ccae-441b-aa5a-156ad34c1dd9


   Mean CV R²: 0.2666 ± 0.0134
   Mean CV RMSE: 2.3425 ± 0.0205
   Mean CV RPD: 1.1678 ± 0.0107
   Mean CV RPIQ: 1.5150 ± 0.0229
   Overall R²: 0.2670

Training RandomForest with 5-fold CV...
   Optimizing hyperparameters (30 trials)...


[I 2026-01-06 08:46:47,437] Trial 0 finished with value: 1.932924805237539 and parameters: {'n_estimators': 250, 'max_depth': 10, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 'log2'}. Best is trial 0 with value: 1.932924805237539.
[I 2026-01-06 08:47:27,935] Trial 1 finished with value: 1.9235523526206946 and parameters: {'n_estimators': 488, 'max_depth': 10, 'min_samples_split': 6, 'min_samples_leaf': 2, 'max_features': 'log2'}. Best is trial 1 with value: 1.9235523526206946.
[I 2026-01-06 08:47:55,934] Trial 2 finished with value: 1.6944821865660467 and parameters: {'n_estimators': 217, 'max_depth': 20, 'min_samples_split': 7, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 2 with value: 1.6944821865660467.
[I 2026-01-06 08:51:31,513] Trial 3 finished with value: 1.86571688659072 and parameters: {'n_estimators': 480, 'max_depth': 10, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': None}. Best is trial 2 with value: 1.6944821865660467.
[I

   Mean CV R²: 0.6355 ± 0.0069
   Mean CV RMSE: 1.6515 ± 0.0253
   Mean CV RPD: 1.6566 ± 0.0156
   Mean CV RPIQ: 2.1491 ± 0.0369
   Overall R²: 0.6356

Training XGBoost with 5-fold CV...
   Optimizing hyperparameters (30 trials)...


[I 2026-01-06 09:30:57,017] Trial 0 finished with value: 1.6253653736605946 and parameters: {'n_estimators': 287, 'max_depth': 9, 'learning_rate': 0.15979909127171077, 'subsample': 0.9197316968394074, 'colsample_bytree': 0.8312037280884873, 'reg_alpha': 0.07799726016810132, 'reg_lambda': 1.0580836121681996}. Best is trial 0 with value: 1.6253653736605946.
[I 2026-01-06 09:31:24,089] Trial 1 finished with value: 1.6233635271309217 and parameters: {'n_estimators': 533, 'max_depth': 7, 'learning_rate': 0.15621088666940686, 'subsample': 0.8041168988591605, 'colsample_bytree': 0.9939819704323989, 'reg_alpha': 0.41622132040021087, 'reg_lambda': 1.2123391106782762}. Best is trial 1 with value: 1.6233635271309217.
[I 2026-01-06 09:31:26,354] Trial 2 finished with value: 1.7447757871732623 and parameters: {'n_estimators': 191, 'max_depth': 4, 'learning_rate': 0.09563633644393066, 'subsample': 0.9049512863264476, 'colsample_bytree': 0.8863890037284232, 'reg_alpha': 0.14561457009902096, 'reg_lamb

   Mean CV R²: 0.6771 ± 0.0066
   Mean CV RMSE: 1.5545 ± 0.0291
   Mean CV RPD: 1.7601 ± 0.0181
   Mean CV RPIQ: 2.2832 ± 0.0221
   Overall R²: 0.6771

=== BEST MODEL: XGBoost ===
Overall R²: 0.6771
Overall RMSE: 1.5548
Overall RPD: 1.7598
Overall RPIQ: 2.2867
Mean CV R²: 0.6771 ± 0.0066
Mean CV RMSE: 1.5545 ± 0.0291
Mean CV RPD: 1.7601 ± 0.0181
Mean CV RPIQ: 2.2832 ± 0.0221

=== FEATURE IMPORTANCE ANALYSIS ===

Top 15 features (RandomForest):
   1. radiation_ERA_total                     : 0.1641
   2. soil_temp_6h_mean                       : 0.0810
   3. variety_encoded                         : 0.0792
   4. CDD                                     : 0.0647
   5. crop_cycle_encoded                      : 0.0598
   6. temp_maxi_mean                          : 0.0454
   7. Exchangeable_Calcium_meq_100g_complete  : 0.0417
   8. Electrical_Conductivity_mmho_cm_complete: 0.0387
   9. Available_P2O5_mg_kg_complete           : 0.0379
  10. temp_mini_mean                          : 0.0364
  

## 5. Temporal Decline Analysis: Understanding Changes Over Time

This section investigates how features and performance have evolved across years to explain observed declines in sugarcane quality or yield. The analysis includes temporal trend detection, period-to-period comparisons, SHAP-based attribution of prediction changes, and counterfactual scenarios showing what outcomes would be if historical conditions still prevailed.TEMPORAL DECLINE ANALYSIS MODULE

In [ ]:
"""
TEMPORAL DECLINE ANALYSIS MODULE
=================================
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
import shap
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 1. TEMPORAL FEATURE EVOLUTION ANALYSIS
# ============================================================================

def analyze_feature_trends_over_time(df, features, year_column='campaign_year', 
                                     response_variable=None):
    """
    Analyze how features have changed over time
    
    Returns:
    - Significant trends (increasing/decreasing)
    - Change rates
    - Statistical significance
    """
    print("\n" + "="*80)
    print("TEMPORAL FEATURE EVOLUTION ANALYSIS")
    print("="*80)
    
    trends = []
    
    for feature in features:
        if feature not in df.columns:
            continue
            
        # Group by year and calculate mean
        yearly_means = df.groupby(year_column)[feature].agg(['mean', 'std', 'count'])
        
        # Linear regression to detect trend
        years = yearly_means.index.values
        values = yearly_means['mean'].values
        
        if len(years) < 3:  # Need at least 3 years
            continue
            
        slope, intercept, r_value, p_value, std_err = stats.linregress(years, values)
        
        # Calculate percent change from first to last year
        first_year_val = values[0]
        last_year_val = values[-1]
        percent_change = ((last_year_val - first_year_val) / first_year_val * 100) if first_year_val != 0 else 0
        
        trends.append({
            'feature': feature,
            'slope': slope,
            'r_squared': r_value**2,
            'p_value': p_value,
            'significant': p_value < 0.05,
            'direction': 'increasing' if slope > 0 else 'decreasing',
            'percent_change': percent_change,
            'first_year_mean': first_year_val,
            'last_year_mean': last_year_val,
            'years_analyzed': f"{years[0]}-{years[-1]}"
        })
    
    trends_df = pd.DataFrame(trends)
    trends_df = trends_df.sort_values('percent_change', key=abs, ascending=False)
    
    # Show significant changes
    significant_changes = trends_df[trends_df['significant']]
    
    print(f"\n{len(significant_changes)} features with SIGNIFICANT changes over time:")
    print("\nTop 20 features with largest changes:")
    for idx, row in significant_changes.head(20).iterrows():
        direction_symbol = "↑" if row['slope'] > 0 else "↓"
        print(f"{direction_symbol} {row['feature']:<45} | "
              f"Change: {row['percent_change']:+7.2f}% | "
              f"R²: {row['r_squared']:.3f} | "
              f"p-value: {row['p_value']:.4f}")
    
    if response_variable:
        print(f"\n\n{response_variable.upper()} TREND:")
        response_trend = trends_df[trends_df['feature'] == response_variable]
        if not response_trend.empty:
            row = response_trend.iloc[0]
            print(f"  Direction: {row['direction']}")
            print(f"  Change: {row['percent_change']:+.2f}%")
            print(f"  Significance: {'YES' if row['significant'] else 'NO'} (p={row['p_value']:.4f})")
    
    return trends_df

# ============================================================================
# 2. YEAR-OVER-YEAR PERFORMANCE COMPARISON
# ============================================================================

def compare_periods(df, year_column='campaign_year', response_variable='brix_percent',
                   early_years=None, recent_years=None):
    """
    Compare early vs recent periods to identify what changed
    """
    print("\n" + "="*80)
    print("PERIOD COMPARISON ANALYSIS")
    print("="*80)
    
    years = sorted(df[year_column].unique())
    
    # Auto-detect periods if not provided
    if early_years is None:
        early_years = years[:len(years)//2]
    if recent_years is None:
        recent_years = years[len(years)//2:]
    
    print(f"\nEarly Period: {early_years}")
    print(f"Recent Period: {recent_years}")
    
    df_early = df[df[year_column].isin(early_years)]
    df_recent = df[df[year_column].isin(recent_years)]
    
    # Compare response variable
    print(f"\n{response_variable.upper()} Comparison:")
    early_mean = df_early[response_variable].mean()
    recent_mean = df_recent[response_variable].mean()
    change = recent_mean - early_mean
    percent_change = (change / early_mean * 100)
    
    print(f"  Early period mean:  {early_mean:.2f}")
    print(f"  Recent period mean: {recent_mean:.2f}")
    print(f"  Change: {change:+.2f} ({percent_change:+.2f}%)")
    
    # Statistical test
    t_stat, p_val = stats.ttest_ind(df_early[response_variable], df_recent[response_variable])
    print(f"  Statistically significant: {'YES' if p_val < 0.05 else 'NO'} (p={p_val:.4f})")
    
    # Compare all numeric features
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    comparisons = []
    
    for col in numeric_cols:
        if col == year_column or col not in df_early.columns or col not in df_recent.columns:
            continue
            
        early_val = df_early[col].mean()
        recent_val = df_recent[col].mean()
        
        if early_val != 0:
            change = recent_val - early_val
            pct_change = (change / early_val * 100)
            
            # T-test
            try:
                t_stat, p_val = stats.ttest_ind(df_early[col].dropna(), df_recent[col].dropna())
            except:
                p_val = 1.0
            
            comparisons.append({
                'feature': col,
                'early_mean': early_val,
                'recent_mean': recent_val,
                'absolute_change': change,
                'percent_change': pct_change,
                'p_value': p_val,
                'significant': p_val < 0.05
            })
    
    comparison_df = pd.DataFrame(comparisons)
    comparison_df = comparison_df.sort_values('percent_change', key=abs, ascending=False)
    
    significant_changes = comparison_df[comparison_df['significant']]
    print(f"\n\n{len(significant_changes)} features with SIGNIFICANT differences between periods:")
    print("\nTop 20 features with largest changes:")
    for idx, row in significant_changes.head(20).iterrows():
        direction = "↑" if row['absolute_change'] > 0 else "↓"
        print(f"{direction} {row['feature']:<45} | "
              f"{row['percent_change']:+7.2f}% | "
              f"p={row['p_value']:.4f}")
    
    return comparison_df, df_early, df_recent

# ============================================================================
# 3. SHAP ANALYSIS FOR TEMPORAL EXPLANATION
# ============================================================================

def analyze_predictions_with_shap(model, scaler, X, y, df, year_column='campaign_year',
                                 feature_names=None, response_variable='brix_percent'):
    """
    Use SHAP to explain predictions and see how contributions changed over time
    """
    print("\n" + "="*80)
    print("SHAP ANALYSIS - EXPLAINING PREDICTIONS OVER TIME")
    print("="*80)
    
    # Scale data
    X_scaled = scaler.transform(X)
    
    # Create SHAP explainer (using sample for speed)
    sample_size = min(1000, len(X))
    sample_idx = np.random.choice(len(X), sample_size, replace=False)
    
    print(f"\nCreating SHAP explainer (using {sample_size} samples)...")
    
    try:
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_scaled[sample_idx])
    except:
        # Fallback to KernelExplainer if TreeExplainer doesn't work
        print("Using KernelExplainer (this may take longer)...")
        explainer = shap.KernelExplainer(model.predict, X_scaled[sample_idx[:50]])
        shap_values = explainer.shap_values(X_scaled[sample_idx])
    
    # Create DataFrame with SHAP values
    shap_df = pd.DataFrame(shap_values, columns=feature_names if feature_names else X.columns)
    shap_df['year'] = df.iloc[sample_idx][year_column].values
    shap_df['actual'] = y.iloc[sample_idx].values
    shap_df['predicted'] = model.predict(X_scaled[sample_idx])
    
    # Analyze SHAP values by period
    years = sorted(shap_df['year'].unique())
    early_years = years[:len(years)//2]
    recent_years = years[len(years)//2:]
    
    shap_early = shap_df[shap_df['year'].isin(early_years)]
    shap_recent = shap_df[shap_df['year'].isin(recent_years)]
    
    # Compare feature contributions
    print(f"\nComparing feature contributions (SHAP values):")
    print(f"Early period: {early_years}")
    print(f"Recent period: {recent_years}")
    
    contribution_changes = []
    for col in feature_names if feature_names else X.columns:
        early_contrib = shap_early[col].abs().mean()
        recent_contrib = shap_recent[col].abs().mean()
        change = recent_contrib - early_contrib
        
        contribution_changes.append({
            'feature': col,
            'early_contribution': early_contrib,
            'recent_contribution': recent_contrib,
            'change': change,
            'percent_change': (change / early_contrib * 100) if early_contrib != 0 else 0
        })
    
    contrib_df = pd.DataFrame(contribution_changes)
    contrib_df = contrib_df.sort_values('change', key=abs, ascending=False)
    
    print("\nTop 15 features with LARGEST CHANGES in contribution:")
    for idx, row in contrib_df.head(15).iterrows():
        direction = "↑" if row['change'] > 0 else "↓"
        print(f"{direction} {row['feature']:<45} | "
              f"Change: {row['percent_change']:+7.2f}%")
    
    return shap_df, contrib_df, explainer, shap_values

# ============================================================================
# 4. COUNTERFACTUAL ANALYSIS
# ============================================================================

def counterfactual_analysis(model, scaler, df, features, year_column='campaign_year',
                           response_variable='brix_percent', baseline_year=None, target_year=None):
    """
    What-if analysis: What would predictions be if we used old conditions?
    """
    print("\n" + "="*80)
    print("COUNTERFACTUAL ANALYSIS")
    print("="*80)
    
    years = sorted(df[year_column].unique())
    
    if baseline_year is None:
        baseline_year = years[0]
    if target_year is None:
        target_year = years[-1]
    
    print(f"\nBaseline year: {baseline_year}")
    print(f"Target year: {target_year}")
    
    # Get baseline conditions
    baseline_data = df[df[year_column] == baseline_year]
    target_data = df[df[year_column] == target_year]
    
    print(f"\nActual {response_variable}:")
    print(f"  {baseline_year}: {baseline_data[response_variable].mean():.2f}")
    print(f"  {target_year}: {target_data[response_variable].mean():.2f}")
    
    # Create counterfactual: target year with baseline features
    counterfactual_data = target_data[features].copy()
    
    # Replace features with baseline means
    feature_changes = []
    for feature in features:
        baseline_mean = baseline_data[feature].mean()
        target_mean = target_data[feature].mean()
        
        if not np.isnan(baseline_mean) and not np.isnan(target_mean):
            counterfactual_data[feature] = baseline_mean
            
            feature_changes.append({
                'feature': feature,
                'baseline_value': baseline_mean,
                'target_value': target_mean,
                'change': target_mean - baseline_mean,
                'percent_change': ((target_mean - baseline_mean) / baseline_mean * 100) if baseline_mean != 0 else 0
            })
    
    # Make predictions
    X_target_scaled = scaler.transform(target_data[features])
    X_counterfactual_scaled = scaler.transform(counterfactual_data)
    
    pred_target = model.predict(X_target_scaled).mean()
    pred_counterfactual = model.predict(X_counterfactual_scaled).mean()
    
    print(f"\nPredicted {response_variable} (model):")
    print(f"  {target_year} with current conditions: {pred_target:.2f}")
    print(f"  {target_year} with {baseline_year} conditions: {pred_counterfactual:.2f}")
    print(f"  Difference: {pred_counterfactual - pred_target:+.2f}")
    print(f"  → If we had {baseline_year} conditions, we would have {pred_counterfactual - pred_target:+.2f} more/less")
    
    # Identify most important changes
    change_df = pd.DataFrame(feature_changes)
    change_df = change_df.sort_values('percent_change', key=abs, ascending=False)
    
    print(f"\nTop 15 feature changes from {baseline_year} to {target_year}:")
    for idx, row in change_df.head(15).iterrows():
        direction = "↑" if row['change'] > 0 else "↓"
        print(f"{direction} {row['feature']:<45} | "
              f"{row['percent_change']:+7.2f}%")
    
    return change_df, pred_target, pred_counterfactual

# ============================================================================
# 5. INTEGRATED DECLINE ANALYSIS REPORT
# ============================================================================

def generate_decline_analysis_report(df, model, scaler, features, year_column='campaign_year',
                                    response_variable='brix_percent', output_dir='decline_analysis'):
    """
    Complete analysis to answer: "What has changed that explains the decline?"
    """
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    print("\n" + "="*80)
    print(f"COMPREHENSIVE DECLINE ANALYSIS FOR: {response_variable}")
    print("="*80)
    
    X = df[features].copy()
    y = df[response_variable].copy()
    
    # 1. Temporal trends
    trends_df = analyze_feature_trends_over_time(df, features, year_column, response_variable)
    trends_df.to_csv(f"{output_dir}/temporal_trends_{response_variable}.csv", index=False)
    
    # 2. Period comparison
    comparison_df, df_early, df_recent = compare_periods(df, year_column, response_variable)
    comparison_df.to_csv(f"{output_dir}/period_comparison_{response_variable}.csv", index=False)
    
    # 3. SHAP analysis
    try:
        shap_df, contrib_df, explainer, shap_values = analyze_predictions_with_shap(
            model, scaler, X, y, df, year_column, features, response_variable
        )
        shap_df.to_csv(f"{output_dir}/shap_analysis_{response_variable}.csv", index=False)
        contrib_df.to_csv(f"{output_dir}/shap_contributions_{response_variable}.csv", index=False)
    except Exception as e:
        print(f"SHAP analysis failed: {e}")
        shap_df = None
    
    # 4. Counterfactual analysis
    years = sorted(df[year_column].unique())
    change_df, pred_current, pred_counterfactual = counterfactual_analysis(
        model, scaler, df, features, year_column, response_variable,
        baseline_year=years[0], target_year=years[-1]
    )
    change_df.to_csv(f"{output_dir}/counterfactual_analysis_{response_variable}.csv", index=False)
    
    # Generate summary report
    print("\n" + "="*80)
    print("GENERATING SUMMARY REPORT")
    print("="*80)
    
    report_lines = []
    report_lines.append(f"DECLINE ANALYSIS REPORT: {response_variable}")
    report_lines.append("="*80)
    report_lines.append(f"\nGenerated: {pd.Timestamp.now()}")
    report_lines.append(f"Dataset: {len(df)} observations from {years[0]} to {years[-1]}")
    
    # Key findings
    report_lines.append("\n\n1. RESPONSE VARIABLE TREND:")
    response_trend = trends_df[trends_df['feature'] == response_variable]
    if not response_trend.empty:
        row = response_trend.iloc[0]
        report_lines.append(f"   - Direction: {row['direction']}")
        report_lines.append(f"   - Overall change: {row['percent_change']:+.2f}%")
        report_lines.append(f"   - First year: {row['first_year_mean']:.2f}")
        report_lines.append(f"   - Last year: {row['last_year_mean']:.2f}")
    
    report_lines.append("\n\n2. TOP 10 FEATURES WITH SIGNIFICANT CHANGES:")
    significant = trends_df[trends_df['significant']].head(10)
    for idx, row in significant.iterrows():
        report_lines.append(f"   {row['direction'][:3].upper()} {row['feature']:<40} {row['percent_change']:+7.2f}%")
    
    report_lines.append("\n\n3. COUNTERFACTUAL RESULT:")
    report_lines.append(f"   If we had {years[0]} conditions in {years[-1]}:")
    report_lines.append(f"   Predicted {response_variable} would be: {pred_counterfactual:.2f}")
    report_lines.append(f"   vs Actual prediction: {pred_current:.2f}")
    report_lines.append(f"   Potential gain/loss: {pred_counterfactual - pred_current:+.2f}")
    
    if shap_df is not None:
        report_lines.append("\n\n4. FEATURES WITH CHANGING IMPORTANCE:")
        top_contrib = contrib_df.head(10)
        for idx, row in top_contrib.iterrows():
            report_lines.append(f"   {row['feature']:<40} {row['percent_change']:+7.2f}%")
    
    # Save report
    report_text = "\n".join(report_lines)
    with open(f"{output_dir}/SUMMARY_REPORT_{response_variable}.txt", "w") as f:
        f.write(report_text)
    
    print(report_text)
    print(f"\n\nAll analysis saved to: {output_dir}/")
    
    return {
        'trends': trends_df,
        'comparison': comparison_df,
        'shap': shap_df,
        'contribution_changes': contrib_df if shap_df is not None else None,
        'counterfactual': change_df,
        'report_text': report_text
    }

# ============================================================================
# 6. VISUALIZATION FUNCTIONS
# ============================================================================

def plot_temporal_evolution(df, features, year_column='campaign_year', 
                           response_variable='brix_percent', output_dir='decline_analysis'):
    """
    Create visualizations of temporal changes
    """
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    # Plot response variable over time
    fig, ax = plt.subplots(figsize=(12, 6))
    yearly_stats = df.groupby(year_column)[response_variable].agg(['mean', 'std', 'count'])
    
    ax.errorbar(yearly_stats.index, yearly_stats['mean'], 
                yerr=yearly_stats['std'], marker='o', capsize=5, capthick=2)
    ax.set_xlabel('Year', fontsize=12)
    ax.set_ylabel(response_variable, fontsize=12)
    ax.set_title(f'{response_variable} Evolution Over Time', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Add trend line
    z = np.polyfit(yearly_stats.index, yearly_stats['mean'], 1)
    p = np.poly1d(z)
    ax.plot(yearly_stats.index, p(yearly_stats.index), "r--", alpha=0.8, label='Trend')
    ax.legend()
    
    plt.tight_layout()
    plt.savefig(f"{output_dir}/temporal_evolution_{response_variable}.png", dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Visualization saved: temporal_evolution_{response_variable}.png")

In [27]:
"""
INTEGRATION SCRIPT: Using Temporal Analysis with Trained Models
================================================================

This script shows how to load your trained models and run temporal analysis
to answer: "What has changed in recent years that explains the decline?"

KEY FEATURES:
-------------
1. Model Selection: Choose which model to use (RandomForest, XGBoost, etc.)
2. Output Organization: Results saved with model name for easy comparison
3. Flexible Analysis: Quick trends or full SHAP analysis

QUICK START:
------------
1. Update DATA_FILE and BASE_MODEL_DIR paths below
2. Set SELECTED_MODEL to your preferred model (e.g., 'RandomForest')
   - Or set to None to use the best performing model
3. Run: python use_temporal_analysis.py

COMPARING MODELS:
-----------------
To compare different models:
1. Run with SELECTED_MODEL = 'RandomForest'
2. Run again with SELECTED_MODEL = 'XGBoost'
3. Compare outputs in separate folders:
   - decline_analysis_brix_percent_RandomForest/
   - decline_analysis_brix_percent_XGBoost/

Usage:
    python use_temporal_analysis.py
"""

import pandas as pd
import joblib
import json
import os
import sys

# ============================================================================
# CONFIGURATION
# ============================================================================

# Paths (ADJUST THESE TO YOUR PATHS)
DATA_FILE = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis/final_dataset_regrouped_categorical.csv'
BASE_MODEL_DIR = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis'

# Response variables to analyze
RESPONSE_VARIABLES = ['brix_percent',
                      'pol_juice_percent',
                      'pol_cane_percent',
                      'purity_percent',
                      'glucose_cane_percent',
                      'hugot_sugar',
                      'tc_per_ha_actual',
                      'tc_per_ha_per_week',
                      'tc_per_ha_52_weeks',
                      'fiber_percent']

# ============================================================================
# MODEL SELECTION CONFIGURATION
# ============================================================================

# Choose which model to use for analysis (leave None to use best model)
# Options: 'RandomForest', 'XGBoost', 'LightGBM', 'CatBoost', 'ExtraTrees', 
#          'GradientBoosting', 'LinearRegression', 'KNN', 'DecisionTree', 
#          'BaggingRegressor', or None (uses best model)
SELECTED_MODEL = 'XGBoost'  # <-- CHANGE THIS TO SELECT DIFFERENT MODEL

# Available models in your trained models directory
AVAILABLE_MODELS = [
    'LinearRegression',
    'KNN',
    'DecisionTree',
    'RandomForest',
    'ExtraTrees',
    'GradientBoosting',
    'BaggingRegressor',
    'XGBoost',
    'LightGBM',
    'CatBoost'
]

# ============================================================================
# MAIN ANALYSIS FUNCTION
# ============================================================================

def list_available_models(response_variable, model_dir_base):
    """
    List all available trained models for a response variable
    
    Parameters:
    -----------
    response_variable : str
        The response variable
    model_dir_base : str
        Base directory containing model folders
    
    Returns:
    --------
    list : Available model names
    """
    model_dir = os.path.join(model_dir_base, f'kfold_models_{response_variable}_custom')
    
    if not os.path.exists(model_dir):
        # Try without categorical suffix
        model_dir = os.path.join(model_dir_base, f'kfold_models_{response_variable}_custom')
        if not os.path.exists(model_dir):
            model_dir = os.path.join(model_dir_base, f'kfold_models_{response_variable}')
    
    models_path = os.path.join(model_dir, 'models')
    
    if not os.path.exists(models_path):
        print(f"No models directory found at: {models_path}")
        return []
    
    available_models = []
    for f in os.listdir(models_path):
        if f.endswith('.pkl'):
            model_name = f.replace(f'_{response_variable}.pkl', '').replace('_', ' ')
            available_models.append(model_name)
    
    return sorted(available_models)

def analyze_decline_for_response_variable(response_variable, data_file, model_dir_base, selected_model=None):
    """
    Complete decline analysis for a specific response variable
    
    Parameters:
    -----------
    response_variable : str
        The response variable to analyze
    data_file : str
        Path to the data CSV file
    model_dir_base : str
        Base directory containing model folders
    selected_model : str or None
        Model name to use (e.g., 'RandomForest', 'XGBoost'). 
        If None, uses the best model.
    """
    print("\n" + "="*100)
    print(f"ANALYZING DECLINE FOR: {response_variable}")
    print("="*100)
    
    # 1. Load the trained model
    model_dir = os.path.join(model_dir_base, f'kfold_models_{response_variable}_custom')
    
    # Check if model directory exists
    if not os.path.exists(model_dir):
        print(f"Model directory not found: {model_dir}")
        print("Please train the model first!")
        return None
    
    # Load best model info
    info_file = os.path.join(model_dir, f'best_model_info_{response_variable}.json')
    with open(info_file, 'r') as f:
        model_info = json.load(f)
    
    best_model_name = model_info['best_model_name']
    
    # Determine which model to use
    if selected_model is not None:
        model_name_to_use = selected_model
        print(f"\nUsing SELECTED model: {model_name_to_use}")
        
        # Check if selected model exists
        model_file_check = os.path.join(model_dir, 'models', f'{model_name_to_use.replace(" ", "_")}_{response_variable}.pkl')
        if not os.path.exists(model_file_check):
            print(f"WARNING: Selected model '{model_name_to_use}' not found!")
            models_path = os.path.join(model_dir, 'models')
            if os.path.exists(models_path):
                print(f"Available models in {models_path}:")
                available_models = [f.replace(f'_{response_variable}.pkl', '').replace('_', ' ') 
                                  for f in os.listdir(models_path) 
                                  if f.endswith('.pkl')]
                for m in sorted(available_models):
                    print(f"  - {m}")
            print(f"\nFalling back to best model: {best_model_name}")
            model_name_to_use = best_model_name
    else:
        model_name_to_use = best_model_name
        print(f"\nUsing BEST model: {model_name_to_use}")
    
    print(f"Best model performance -> R²: {model_info['overall_r2']:.4f}, RMSE: {model_info['overall_rmse']:.4f}")
    
    # Load the selected model
    model_file = os.path.join(model_dir, 'models', f'{model_name_to_use.replace(" ", "_")}_{response_variable}.pkl')
    print(f"Loading model from: {model_file}")
    model_data = joblib.load(model_file)
    
    model = model_data['model']
    scaler = model_data['scaler']
    encoders = model_data['encoders']
    
    print(f"Model loaded successfully!")
    
    # 2. Load and prepare data
    print("\nLoading dataset...")
    df = pd.read_csv(data_file)
    
    # Identify year column
    year_column = None
    for col in ['campaign_year', 'Harvest_Year', 'year']:
        if col in df.columns:
            year_column = col
            break
    
    if year_column is None:
        print("ERROR: No year column found!")
        return None
    
    print(f"Year column: {year_column}")
    print(f"Years available: {sorted(df[year_column].unique())}")
    
    # Drop rows with missing response variable
    df = df.dropna(subset=[response_variable])
    print(f"Total observations: {len(df)}")
    
    # Encode categorical variables (UPDATE THIS IF YOU HAVE DIFFERENT CATEGORICAL VARS)
    categorical_vars = []
    # NOTE: Set to empty list [] if you trained without categorical variables
    # categorical_vars = []  # Use this for models trained with '_custom' suffix
    
    for cat_var in categorical_vars:
        if cat_var in df.columns and cat_var in encoders:
            df[f'{cat_var}_encoded'] = encoders[cat_var].transform(df[cat_var].astype(str))
    
    # Get feature list (same as used in training)
    # You need to reconstruct this based on your training configuration
    custom_features = [
        'ampl_mean', 'TNn', 'TNx', 'Coarse_Silt_percent_complete',
        'Total_Nitrogen_per_mille_complete', 'pH_complete', 'temp_moy_mean', 
        'Exchangeable_Calcium_meq_100g_complete', 'h_maxi_mean', 'CDD', 
        'radiation_ERA_total', 'Rx1', 'R20', 'SDII',
        'h_mini_mean', 'HD', 'Available_P2O5_mg_kg_complete', 
        'Electrical_Conductivity_mmho_cm_complete', 
        'Exchangeable_Sodium_meq_100g_complete',
        'R99p', 'growing_season_rainfall_mm', 'soil_temp_6h_mean', 
        'CWD', 'Mg_K_Ratio_complete'
    ]
    
    # Add encoded categorical features
    for cat_var in categorical_vars:
        if f'{cat_var}_encoded' in df.columns:
            custom_features.append(f'{cat_var}_encoded')
    
    # Filter to available features
    features = [f for f in custom_features if f in df.columns]
    print(f"\nUsing {len(features)} features")
    
    # 3. Create output directory with model name
    model_name_clean = model_name_to_use.replace(" ", "_")
    output_dir = os.path.join(model_dir, f'decline_analysis_{response_variable}_{model_name_clean}')
    os.makedirs(output_dir, exist_ok=True)
    print(f"\nOutput directory: {output_dir}")
    
    # 4. Run comprehensive decline analysis
    print("\n" + "="*100)
    print("RUNNING COMPREHENSIVE DECLINE ANALYSIS")
    print("="*100)
    
    results = generate_decline_analysis_report(
        df=df,
        model=model,
        scaler=scaler,
        features=features,
        year_column=year_column,
        response_variable=response_variable,
        output_dir=output_dir
    )
    
    # 5. Create visualizations
    print("\nGenerating visualizations...")
    plot_temporal_evolution(df, features, year_column, response_variable, output_dir)
    
    print("\n" + "="*100)
    print(f"ANALYSIS COMPLETE FOR: {response_variable}")
    print("="*100)
    print(f"\nAll results saved to: {output_dir}")
    print("\nGenerated files:")
    print(f"  - SUMMARY_REPORT_{response_variable}.txt")
    print(f"  - temporal_trends_{response_variable}.csv")
    print(f"  - period_comparison_{response_variable}.csv")
    print(f"  - counterfactual_analysis_{response_variable}.csv")
    print(f"  - shap_analysis_{response_variable}.csv")
    print(f"  - temporal_evolution_{response_variable}.png")
    
    return results

# ============================================================================
# SPECIFIC ANALYSIS FUNCTIONS
# ============================================================================

def quick_trend_analysis(response_variable, data_file, model_dir_base):
    """
    Quick analysis: Just show the trends without full SHAP analysis
    """
    print(f"\n{'='*80}")
    print(f"QUICK TREND ANALYSIS: {response_variable}")
    print(f"{'='*80}")
    
    # Load data
    df = pd.read_csv(data_file)
    df = df.dropna(subset=[response_variable])
    
    # Find year column
    year_column = None
    for col in ['campaign_year', 'Harvest_Year', 'year']:
        if col in df.columns:
            year_column = col
            break
    
    # Get features
    custom_features = [
        'ampl_mean', 'TNn', 'TNx', 'Coarse_Silt_percent_complete',
        'Total_Nitrogen_per_mille_complete', 'pH_complete', 'temp_moy_mean', 
        'Exchangeable_Calcium_meq_100g_complete', 'h_maxi_mean', 'CDD', 
        'radiation_ERA_total', 'Rx1', 'R20', 'SDII',
        'h_mini_mean', 'HD', 'Available_P2O5_mg_kg_complete', 
        'Electrical_Conductivity_mmho_cm_complete', 
        'Exchangeable_Sodium_meq_100g_complete',
        'R99p', 'growing_season_rainfall_mm', 'soil_temp_6h_mean', 
        'CWD', 'Mg_K_Ratio_complete'
    ]
    
    features = [f for f in custom_features if f in df.columns]
    
    # Run trend analysis
    trends_df = analyze_feature_trends_over_time(df, features, year_column, response_variable)
    
    # Run period comparison
    comparison_df, df_early, df_recent = compare_periods(df, year_column, response_variable)
    
    return trends_df, comparison_df

def compare_multiple_response_variables(response_variables, data_file):
    """
    Compare trends across multiple response variables
    """
    print("\n" + "="*100)
    print("COMPARING MULTIPLE RESPONSE VARIABLES")
    print("="*100)
    
    df = pd.read_csv(data_file)
    
    # Find year column
    year_column = None
    for col in ['campaign_year', 'Harvest_Year', 'year']:
        if col in df.columns:
            year_column = col
            break
    
    summary = []
    
    for response_var in response_variables:
        if response_var not in df.columns:
            print(f"Skipping {response_var} - not found in dataset")
            continue
        
        df_clean = df.dropna(subset=[response_var])
        
        # Calculate trend
        yearly_means = df_clean.groupby(year_column)[response_var].mean()
        years = yearly_means.index.values
        values = yearly_means.values
        
        if len(years) >= 3:
            from scipy import stats
            slope, intercept, r_value, p_value, std_err = stats.linregress(years, values)
            
            first_val = values[0]
            last_val = values[-1]
            percent_change = ((last_val - first_val) / first_val * 100) if first_val != 0 else 0
            
            summary.append({
                'response_variable': response_var,
                'first_year_mean': first_val,
                'last_year_mean': last_val,
                'absolute_change': last_val - first_val,
                'percent_change': percent_change,
                'trend_slope': slope,
                'r_squared': r_value**2,
                'p_value': p_value,
                'significant': p_value < 0.05,
                'direction': 'increasing' if slope > 0 else 'decreasing'
            })
    
    summary_df = pd.DataFrame(summary)
    print("\nSUMMARY OF TRENDS ACROSS RESPONSE VARIABLES:")
    print(summary_df.to_string(index=False))
    
    return summary_df

# ============================================================================
# EXAMPLE USAGE
# ============================================================================

if __name__ == "__main__":
    
    print("""
    ╔═══════════════════════════════════════════════════════════════════════════╗
    ║                   TEMPORAL DECLINE ANALYSIS TOOL                          ║
    ║                                                                           ║
    ║  Purpose: Identify what has changed over time that explains declines     ║
    ║           in sugar content, yields, or other response variables          ║
    ╚═══════════════════════════════════════════════════════════════════════════╝
    """)
    
    # Display model selection configuration
    print(f"\n{'='*100}")
    print("MODEL SELECTION CONFIGURATION")
    print(f"{'='*100}")
    if SELECTED_MODEL is not None:
        print(f"Selected Model: {SELECTED_MODEL}")
        print(f"Note: To use the best model instead, set SELECTED_MODEL = None at line 53")
    else:
        print(f"Using: Best performing model for each response variable")
    print(f"\nAvailable model types: {', '.join(AVAILABLE_MODELS)}")
    
    # Show available models for first response variable as example
    print(f"\n--- Checking available models for '{RESPONSE_VARIABLES[0]}' ---")
    example_models = list_available_models(RESPONSE_VARIABLES[0], BASE_MODEL_DIR)
    if example_models:
        print(f"Found {len(example_models)} trained models:")
        for m in example_models:
            marker = " ← SELECTED" if m == SELECTED_MODEL else ""
            print(f"  • {m}{marker}")
    else:
        print("No models found. Please train models first!")
    print(f"{'='*100}\n")
    
    # ========================================================================
    # OPTION 1: Full Analysis for One Response Variable
    # ========================================================================
    
    print("\n### OPTION 1: Full Comprehensive Analysis ###")
    print("This will perform complete analysis including SHAP, counterfactuals, etc.")
    
    # Example: Analyze brix_percent
    response_to_analyze = 'brix_percent'
    
    results = analyze_decline_for_response_variable(
        response_variable=response_to_analyze,
        data_file=DATA_FILE,
        model_dir_base=BASE_MODEL_DIR,
        selected_model=SELECTED_MODEL  # Pass the selected model
    )
    
    # ========================================================================
    # OPTION 2: Quick Trend Analysis (Faster, No SHAP)
    # ========================================================================
    
    print("\n\n### OPTION 2: Quick Trend Analysis ###")
    print("This is faster - just trends and period comparisons, no SHAP")
    
    trends_df, comparison_df = quick_trend_analysis(
        response_variable='pol_cane_percent',
        data_file=DATA_FILE,
        model_dir_base=BASE_MODEL_DIR
    )
    
    # ========================================================================
    # OPTION 3: Compare All Response Variables
    # ========================================================================
    
    print("\n\n### OPTION 3: Compare All Response Variables ###")
    
    comparison_summary = compare_multiple_response_variables(
        response_variables=RESPONSE_VARIABLES,
        data_file=DATA_FILE
    )
    
    # ========================================================================
    # OPTION 4: Batch Analysis for All Response Variables
    # ========================================================================
    
    print("\n\n### OPTION 4: Batch Analysis (All Response Variables) ###")
    print("WARNING: This will take a long time!")
    if SELECTED_MODEL:
        print(f"Using model: {SELECTED_MODEL} for all response variables")
    
    run_batch = input("\nRun full analysis for all response variables? (yes/no): ")
    
    if run_batch.lower() == 'yes':
        for response_var in RESPONSE_VARIABLES:
            try:
                print(f"\n\n{'='*100}")
                print(f"Processing: {response_var}")
                print(f"{'='*100}")
                
                results = analyze_decline_for_response_variable(
                    response_variable=response_var,
                    data_file=DATA_FILE,
                    model_dir_base=BASE_MODEL_DIR,
                    selected_model=SELECTED_MODEL  # Pass the selected model
                )
                
                print(f"\n✓ Completed: {response_var}")
                
            except Exception as e:
                print(f"\n✗ Error analyzing {response_var}: {e}")
                continue
        
        print("\n\n" + "="*100)
        print("BATCH ANALYSIS COMPLETE!")
        print("="*100)
    
    print("\n\nAnalysis complete! Check the output directories for results.")


    ╔═══════════════════════════════════════════════════════════════════════════╗
    ║                   TEMPORAL DECLINE ANALYSIS TOOL                          ║
    ║                                                                           ║
    ║  Purpose: Identify what has changed over time that explains declines     ║
    ║           in sugar content, yields, or other response variables          ║
    ╚═══════════════════════════════════════════════════════════════════════════╝
    

MODEL SELECTION CONFIGURATION
Selected Model: XGBoost
Note: To use the best model instead, set SELECTED_MODEL = None at line 53

Available model types: LinearRegression, KNN, DecisionTree, RandomForest, ExtraTrees, GradientBoosting, BaggingRegressor, XGBoost, LightGBM, CatBoost

--- Checking available models for 'brix_percent' ---
Found 10 trained models:
  • BaggingRegressor
  • CatBoost
  • DecisionTree
  • ExtraTrees
  • GradientBoosting
  • KNN
  • LightGBM
  • LinearRegression
  • RandomForest

COMBINED TEMPORAL TRENDS VISUALIZATION
======================================

In [ ]:
"""
Plot: SHAP + Temporal Change (TWO SECTIONS - ALL TOP FEATURES)
============================================================================================

Two separate sections showing climate and soil features with distinct spacing.
Shows ALL top features (regardless of significance), but only annotates significant changes.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 
# ============================================================================

plt.rcParams.update({
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 9,
    'axes.labelsize': 10,
    'axes.titlesize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'axes.linewidth': 1,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'xtick.major.width': 1,
    'ytick.major.width': 1,
})

# Color scheme
SHAP_COLOR = '#E64B35'

TEMPORAL_COLORS = {
    'increase': '#DC3545',
    'decrease': '#28A745',
    'stable': '#6C757D',
}

# ============================================================================
# FEATURE CATEGORIZATION
# ============================================================================

CLIMATE_FEATURES = [
    'growing_season_rainfall_mm',
    'CDD', 'CWD',
    'SDII',
    'Rx1', 'R20',
    'TNn', 'TNx',
    'temp_mini_mean', 'temp_maxi_mean',
    'h_mini_mean', 'h_maxi_mean',
    'soil_temp_6h_mean',
    'radiation_ERA_total'
]


SOIL_FEATURES = [
    'pH_complete',
    'Exchangeable_Calcium_meq_100g_complete',
    'Exchangeable_Potassium_meq_100g_complete',
    'Exchangeable_Sodium_meq_100g_complete',
    'Available_P2O5_mg_kg_complete',
    'Total_Nitrogen_per_mille_complete',
    'Electrical_Conductivity_mmho_cm_complete',
    'Mg_K_Ratio_complete',
    'Coarse_Silt_percent_complete',
    'Coarse_Sand_percent_complete'
]

def categorize_feature(feature_name):
    """Determine if feature is climate or soil"""
    # Check exact match first
    if feature_name in CLIMATE_FEATURES:
        return 'climate'
    elif feature_name in SOIL_FEATURES:
        return 'soil'
    
    # Check partial matches (for variations in naming)
    feature_lower = feature_name.lower()
    
    # Climate indicators
    climate_keywords = ['temp', 'rain', 'radiation', 'cdd', 'cwd', 'tnn', 'tnx',
                       'ampl', 'rx1', 'r20', 'r99p', 'sdii', 'hd', 'h_maxi', 'h_mini']
    if any(kw in feature_lower for kw in climate_keywords):
        return 'climate'
    
    # Soil indicators
    soil_keywords = ['silt', 'nitrogen', 'ph', 'calcium', 'p2o5', 'conductivity',
                    'sodium', 'mg_k', 'soil']
    if any(kw in feature_lower for kw in soil_keywords):
        return 'soil'
    
    return 'other'

def format_feature_label(feature):
    """Format feature name with proper scientific notation and line breaks"""
    
    # English feature mapping (using short_en labels)
    feature_labels_en = {
        # Climate features
        'ampl_mean': 'Temp Ampl\n(°C)',
        'TNn': 'TNn\n(°C)',
        'TNx': 'TNx\n(°C)',
        'temp_moy_mean': 'Mean Temp\n(°C)',
        'temp_maxi_mean': 'T° max\n(°C)',
        'temp_mini_mean': 'T° min\n(°C)',
        'h_maxi_mean': 'RH max\n(%)',
        'h_mini_mean': 'RH min\n(%)',
        'h_moy_mean': 'Mean Humid\n(%)',
        'CDD': 'CDD\n(days)',
        'CWD': 'CWD\n(days)',
        'radiation_ERA_total': 'Solar rad.\n(cal/cm²)',
        'Rx1': 'Rx1\n(mm)',
        'R20': 'R20\n(days)',
        'R10': 'R10\n(days)',
        'SDII': 'SDII\n(mm/day)',
        'HD': 'Hot Days\n(days)',
        'R99p': 'R99p\n(mm)',
        'R95p': 'R95p\n(mm)',
        'growing_season_rainfall_mm': 'Growing season\nrain (mm)',
        'soil_temp_6h_mean': 'Soil T° 6h\n(°C)',
        'soil_temp_16h_mean': 'Soil T 16h\n(°C)',
        'precipitation_total': 'Precip\n(mm)',
        'evaporation_total': 'Evaporation\n(mm)',
        'TXn': 'TXn\n(°C)',
        'TXx': 'TXx\n(°C)',
        'Rx5': 'Max 5D\n(mm)',
        
        # Soil features
        'Coarse_Silt_percent_complete': 'Coarse silt\n(%)',
        'Fine_Silt_percent_complete': 'Fine Silt\n(%)',
        'Total_Nitrogen_per_mille_complete': 'N total\n(‰)',
        'pH_complete': 'pH',
        'Exchangeable_Calcium_meq_100g_complete': 'Ca exch.\n(meq/100g)',
        'Exchangeable_Magnesium_meq_100g_complete': 'Mg\n(meq/100g)',
        'Exchangeable_Potassium_meq_100g_complete': 'K exch.\n(meq/100g)',
        'Exchangeable_Sodium_meq_100g_complete': 'Na exch.\n(meq/100g)',
        'Available_P2O5_mg_kg_complete': 'P₂O₅ avail.\n(mg/kg)',
        'Electrical_Conductivity_mmho_cm_complete': 'EC\n(mmho/cm)',
        'Mg_K_Ratio_complete': 'Mg/K ratio',
        'Mg_Ca_Ratio_complete': 'Mg/Ca',
        'Organic_Carbon_per_mille_complete': 'Org C\n(‰)',
        'Clay_percent_complete': 'Clay\n(%)',
        'Fine_Sand_percent_complete': 'Fine Sand\n(%)',
        'Coarse_Sand_percent_complete': 'Coarse sand\n(%)',
    }
    
    # Return mapped label if exists
    if feature in feature_labels_en:
        return feature_labels_en[feature]
    
    # Fallback: generic formatting
    label = feature.replace('_', ' ').replace('complete', '').strip()
    
    # Dictionary of scientific terms that should maintain specific formatting
    scientific_terms = {
        'ph': 'pH',
        'p2o5': 'P₂O₅',
        'mg k ratio': 'Mg/K ratio',
        'meq 100g': 'meq/100g',
        'mg kg': 'mg/kg',
        'mmho cm': 'mmho/cm',
        'per mille': '‰',
        'tnn': 'TNn',
        'tnx': 'TNx',
        'cdd': 'CDD',
        'cwd': 'CWD',
        'sdii': 'SDII',
        'hd': 'HD',
        'rx1': 'Rx1',
        'r20': 'R20',
        'r99p': 'R99p',
        'era': 'ERA'
    }
    
    # Split and capitalize
    words = label.split()
    formatted_words = []
    
    for word in words:
        word_lower = word.lower()
        if word_lower in scientific_terms:
            formatted_words.append(scientific_terms[word_lower])
        else:
            formatted_words.append(word.capitalize())
    
    label = ' '.join(formatted_words)
    
    # Add line breaks for long labels (>25 characters)
    if len(label) > 25:
        words = label.split()
        mid_point = len(words) // 2
        label = ' '.join(words[:mid_point]) + '\n' + ' '.join(words[mid_point:])
    
    return label


"""
# FRENCH VERSION - Uncomment to use French labels
def format_feature_label(feature):
    '''Format feature name with proper scientific notation and line breaks (French)'''
    
    # French feature mapping (using short_fr labels)
    feature_labels_fr = {
        # Climate features
        'ampl_mean': 'Ampl temp\n(°C)',
        'TNn': 'TNn\n(°C)',
        'TNx': 'TNx\n(°C)',
        'temp_moy_mean': 'Temp moyenne\n(°C)',
        'temp_maxi_mean': 'T° max\n(°C)',
        'temp_mini_mean': 'T° min\n(°C)',
        'h_maxi_mean': 'HR max\n(%)',
        'h_mini_mean': 'HR min\n(%)',
        'h_moy_mean': 'Humid moy\n(%)',
        'CDD': 'CDD\n(jours)',
        'CWD': 'CWD\n(jours)',
        'radiation_ERA_total': 'Rayonnement\n(cal/cm²)',
        'Rx1': 'Rx1\n(mm)',
        'R20': 'R20\n(jours)',
        'R10': 'R10\n(jours)',
        'SDII': 'SDII\n(mm/jour)',
        'HD': 'Jours chauds\n(jours)',
        'R99p': 'R99p\n(mm)',
        'R95p': 'R95p\n(mm)',
        'growing_season_rainfall_mm': 'Pluie saison\n(mm)',
        'soil_temp_6h_mean': 'T° sol 6h\n(°C)',
        'soil_temp_16h_mean': 'Temp sol 16h\n(°C)',
        'precipitation_total': 'Précip\n(mm)',
        'evaporation_total': 'Évaporation\n(mm)',
        'TXn': 'TXn\n(°C)',
        'TXx': 'TXx\n(°C)',
        'Rx5': 'Max 5J\n(mm)',
        
        # Soil features
        'Coarse_Silt_percent_complete': 'Limon grossier\n(%)',
        'Fine_Silt_percent_complete': 'Limon fin\n(%)',
        'Total_Nitrogen_per_mille_complete': 'N total\n(‰)',
        'pH_complete': 'pH',
        'Exchangeable_Calcium_meq_100g_complete': 'Ca éch.\n(meq/100g)',
        'Exchangeable_Magnesium_meq_100g_complete': 'Mg\n(meq/100g)',
        'Exchangeable_Potassium_meq_100g_complete': 'K éch.\n(meq/100g)',
        'Exchangeable_Sodium_meq_100g_complete': 'Na éch.\n(meq/100g)',
        'Available_P2O5_mg_kg_complete': 'P₂O₅ disp.\n(mg/kg)',
        'Electrical_Conductivity_mmho_cm_complete': 'CE\n(mmho/cm)',
        'Mg_K_Ratio_complete': 'Rapport Mg/K',
        'Mg_Ca_Ratio_complete': 'Mg/Ca',
        'Organic_Carbon_per_mille_complete': 'C organique\n(‰)',
        'Clay_percent_complete': 'Argile\n(%)',
        'Fine_Sand_percent_complete': 'Sable fin\n(%)',
        'Coarse_Sand_percent_complete': 'Sable grossier\n(%)',
    }
    
    # Return mapped label if exists
    if feature in feature_labels_fr:
        return feature_labels_fr[feature]
    
    # Fallback: generic formatting
    label = feature.replace('_', ' ').replace('complete', '').strip()
    
    # Dictionary of scientific terms that should maintain specific formatting
    scientific_terms = {
        'ph': 'pH',
        'p2o5': 'P₂O₅',
        'mg k ratio': 'Rapport Mg/K',
        'meq 100g': 'meq/100g',
        'mg kg': 'mg/kg',
        'mmho cm': 'mmho/cm',
        'per mille': '‰',
        'tnn': 'TNn',
        'tnx': 'TNx',
        'cdd': 'CDD',
        'cwd': 'CWD',
        'sdii': 'SDII',
        'hd': 'HD',
        'rx1': 'Rx1',
        'r20': 'R20',
        'r99p': 'R99p',
        'era': 'ERA'
    }
    
    # Split and capitalize
    words = label.split()
    formatted_words = []
    
    for word in words:
        word_lower = word.lower()
        if word_lower in scientific_terms:
            formatted_words.append(scientific_terms[word_lower])
        else:
            formatted_words.append(word.capitalize())
    
    label = ' '.join(formatted_words)
    
    # Add line breaks for long labels (>25 characters)
    if len(label) > 25:
        words = label.split()
        mid_point = len(words) // 2
        label = ' '.join(words[:mid_point]) + '\n' + ' '.join(words[mid_point:])
    
    return label
"""

# ============================================================================
# MAIN PLOTTING FUNCTION
# ============================================================================

def create_enhanced_two_section_figure(response_variable, model_name, base_model_dir, 
                                       data_file, n_features_per_category=5):
    """
    Create enhanced two-section figure with separate climate/soil sections
    Shows ALL top features (regardless of significance)
    
    Parameters:
    -----------
    response_variable : str
        Response variable name
    model_name : str
        Model name
    base_model_dir : str
        Base directory containing model folders
    data_file : str
        Path to data CSV file
    n_features_per_category : int
        Number of top features to show per category (default: 5)
    
    Returns:
    --------
    str or None : Path to output file if successful, None if failed
    """
    
    print(f"\n{'='*80}")
    print(f"Creating two-section figure: {response_variable} ({model_name})")
    print(f"{'='*80}")
    
    # Construct paths
    model_dir = os.path.join(base_model_dir, f'kfold_models_{response_variable}_custom')
    decline_dir = os.path.join(model_dir, f'decline_analysis_{response_variable}_{model_name}')
    
    if not os.path.exists(decline_dir):
        model_dir = os.path.join(base_model_dir, f'kfold_models_{response_variable}_custom')
        decline_dir = os.path.join(model_dir, f'decline_analysis_{response_variable}_{model_name}')
    
    if not os.path.exists(decline_dir):
        print(f"Error: Directory not found: {decline_dir}")
        return None
    
    # Load data
    try:
        shap_df = pd.read_csv(os.path.join(decline_dir, f'shap_analysis_{response_variable}.csv'))
        trends_df = pd.read_csv(os.path.join(decline_dir, f'temporal_trends_{response_variable}.csv'))
        print(f"✓ Data loaded successfully")
    except Exception as e:
        print(f"Error loading data: {e}")
        return None
    
    # Get response variable statistics
    response_trend_data = trends_df[trends_df['feature'] == response_variable]
    
    if len(response_trend_data) == 0:
        try:
            df_full = pd.read_csv(data_file)
            year_col = 'campaign_year' if 'campaign_year' in df_full.columns else 'Harvest_Year'
            
            if response_variable in df_full.columns:
                yearly_means = df_full.groupby(year_col)[response_variable].mean()
                first_year_val = yearly_means.iloc[0]
                last_year_val = yearly_means.iloc[-1]
                percent_change = ((last_year_val - first_year_val) / first_year_val * 100) if first_year_val != 0 else 0
                response_p_value = 1.0
            else:
                print(f"Error: '{response_variable}' not in dataset!")
                return None
        except Exception as e:
            print(f"Error calculating fallback statistics: {e}")
            return None
    else:
        response_trend = response_trend_data.iloc[0]
        first_year_val = response_trend['first_year_mean']
        last_year_val = response_trend['last_year_mean']
        percent_change = response_trend['percent_change']
        response_p_value = response_trend.get('p_value', response_trend.get('trend_p_value', 1.0))
    
    # Calculate SHAP importance
    exclude_cols = ['year', 'actual', 'predicted', 'Unnamed: 0', 'index']
    feature_cols = [col for col in shap_df.columns 
                   if col not in exclude_cols and not col.startswith('Unnamed')]
    
    if response_variable in feature_cols:
        feature_cols.remove(response_variable)
    
    print(f"Found {len(feature_cols)} feature columns in SHAP data")
    
    if len(feature_cols) == 0:
        print("Error: No feature columns found in SHAP data!")
        return None
    
    # Calculate SHAP importance and categorize
    shap_importance = []
    for feature in feature_cols:
        if feature in shap_df.columns:
            mean_abs_shap = shap_df[feature].abs().mean()
            mean_shap = shap_df[feature].mean()
            category = categorize_feature(feature)
            
            shap_importance.append({
                'feature': feature,
                'mean_abs_shap': mean_abs_shap,
                'mean_shap': mean_shap,
                'category': category,
                'direction': 'positive' if mean_shap > 0 else 'negative'
            })
    
    shap_df_summary = pd.DataFrame(shap_importance)
    
    if len(shap_df_summary) == 0:
        print("Error: No SHAP importance data calculated!")
        return None
    
    print(f"✓ Calculated SHAP importance for {len(shap_df_summary)} features")
    
    # Get top N features per category
    climate_features = shap_df_summary[shap_df_summary['category'] == 'climate']
    soil_features = shap_df_summary[shap_df_summary['category'] == 'soil']
    
    print(f"   - Climate features: {len(climate_features)}")
    print(f"   - Soil features: {len(soil_features)}")
    
    top_climate = climate_features.nlargest(n_features_per_category, 'mean_abs_shap')
    top_soil = soil_features.nlargest(n_features_per_category, 'mean_abs_shap')
    
    # Merge with temporal trends
    available_trend_cols = ['feature', 'percent_change']
    if 'p_value' in trends_df.columns:
        available_trend_cols.append('p_value')
    if 'significant' in trends_df.columns:
        available_trend_cols.append('significant')
    if 'trend_p_value' in trends_df.columns:
        trends_df['p_value'] = trends_df['trend_p_value']
        available_trend_cols.append('p_value')
    
    climate_data = top_climate.merge(trends_df[available_trend_cols], on='feature', how='left')
    soil_data = top_soil.merge(trends_df[available_trend_cols], on='feature', how='left')
    
    # Fill missing values
    for df in [climate_data, soil_data]:
        if 'p_value' not in df.columns:
            df['p_value'] = 1.0
        if 'significant' not in df.columns:
            df['significant'] = False
        df['p_value'] = df['p_value'].fillna(1.0)
        df['percent_change'] = df['percent_change'].fillna(0.0)
    
    # Sort by SHAP importance (ascending for display)
    climate_data = climate_data.sort_values('mean_abs_shap', ascending=True)
    soil_data = soil_data.sort_values('mean_abs_shap', ascending=True)
    
    if len(climate_data) == 0 or len(soil_data) == 0:
        print("Error: No data to plot!")
        return None
    
    print(f"✓ Ready to plot {len(climate_data)} climate + {len(soil_data)} soil features")
    
    # ========================================================================
    # CREATE FIGURE WITH TWO SECTIONS
    # ========================================================================
    
    # Calculate heights based on number of features
    climate_height = len(climate_data) * 0.35
    soil_height = len(soil_data) * 0.35
    gap_height = 0.5  # Space between sections
    total_height = climate_height + soil_height + gap_height + 1.5  # Extra for title/margins
    
    fig = plt.figure(figsize=(8, total_height))
    
    # Calculate height ratios for gridspec
    height_ratio_climate = climate_height
    height_ratio_gap = gap_height
    height_ratio_soil = soil_height
    
    # Create grid with three rows: climate, gap, soil
    gs = fig.add_gridspec(3, 1, 
                         height_ratios=[height_ratio_climate, height_ratio_gap, height_ratio_soil],
                         hspace=0.05,
                         left=0.30, right=0.85, top=0.95, bottom=0.08)
    
    # Create axes
    ax_climate = fig.add_subplot(gs[0])
    ax_soil = fig.add_subplot(gs[2])
    
    # Get max SHAP value for consistent x-axis
    max_shap = max(climate_data['mean_abs_shap'].max(), soil_data['mean_abs_shap'].max())
    
    # ========================================================================
    # PLOT CLIMATE SECTION (TOP)
    # ========================================================================
    
    y_pos_climate = np.arange(len(climate_data))
    
    # Plot bars
    bars_climate = ax_climate.barh(y_pos_climate, climate_data['mean_abs_shap'], 
                                   height=0.5,
                                   color='#6C757D',
                                   alpha=0.75,
                                   edgecolor='white',
                                   linewidth=0.5)
    
    # Add temporal change annotations (ONLY SIGNIFICANT ONES)
    annotation_x = max_shap * 1.08
    
    for i, (idx, row) in enumerate(climate_data.iterrows()):
        pct_change = row['percent_change']
        p_val = row['p_value']
        
        # Significance stars
        if p_val < 0.001:
            stars = '***'
            is_significant = True
        elif p_val < 0.01:
            stars = '**'
            is_significant = True
        elif p_val < 0.05:
            stars = '*'
            is_significant = True
        else:
            stars = ''
            is_significant = False
        
        # ONLY SHOW IF SIGNIFICANT
        if not is_significant or abs(pct_change) < 0.5:
            continue
        
        change_text = f"{pct_change:+.1f}%{stars}"
        
        # Color based on direction
        if pct_change > 0:
            text_color = TEMPORAL_COLORS['increase']
        else:
            text_color = TEMPORAL_COLORS['decrease']
        
        ax_climate.text(annotation_x, i, change_text,
                       va='center', ha='left',
                       fontsize=8,
                       color=text_color,
                       weight='bold')
    
    # Format y-axis
    feature_labels_climate = [format_feature_label(f) for f in climate_data['feature']]
    ax_climate.set_yticks(y_pos_climate)
    ax_climate.set_yticklabels(feature_labels_climate, fontsize=9, weight='bold')
    
    # X-axis for climate
    ax_climate.set_xlim(0, max_shap * 1.15)
    ax_climate.set_xlabel('Mean |SHAP value| (feature importance)', 
                         fontsize=10, weight='bold')
    ax_climate.tick_params(axis='x', which='major', labelsize=9, width=1, labelcolor='black')
    
    # Make x-axis tick labels bold
    for label in ax_climate.get_xticklabels():
        label.set_weight('bold')
    
    # Grid
    ax_climate.xaxis.grid(True, alpha=0.3, linewidth=0.5, linestyle='--', color='gray')
    ax_climate.set_axisbelow(True)
    
    # Spines
    ax_climate.spines['left'].set_linewidth(1)
    ax_climate.spines['bottom'].set_linewidth(1)
    ax_climate.spines['top'].set_visible(False)
    ax_climate.spines['right'].set_visible(False)
    
    # ========================================================================
    # PLOT SOIL SECTION (BOTTOM)
    # ========================================================================
    
    y_pos_soil = np.arange(len(soil_data))
    
    # Plot bars
    bars_soil = ax_soil.barh(y_pos_soil, soil_data['mean_abs_shap'], 
                            height=0.5,
                            color='#6C757D',
                            alpha=0.75,
                            edgecolor='white',
                            linewidth=0.5)
    
    # Add temporal change annotations (ONLY SIGNIFICANT ONES)
    for i, (idx, row) in enumerate(soil_data.iterrows()):
        pct_change = row['percent_change']
        p_val = row['p_value']
        
        # Significance stars
        if p_val < 0.001:
            stars = '***'
            is_significant = True
        elif p_val < 0.01:
            stars = '**'
            is_significant = True
        elif p_val < 0.05:
            stars = '*'
            is_significant = True
        else:
            stars = ''
            is_significant = False
        
        # ONLY SHOW IF SIGNIFICANT
        if not is_significant or abs(pct_change) < 0.5:
            continue
        
        change_text = f"{pct_change:+.1f}%{stars}"
        
        # Color based on direction
        if pct_change > 0:
            text_color = TEMPORAL_COLORS['increase']
        else:
            text_color = TEMPORAL_COLORS['decrease']
        
        ax_soil.text(annotation_x, i, change_text,
                    va='center', ha='left',
                    fontsize=8,
                    color=text_color,
                    weight='bold')
    
    # Format y-axis
    feature_labels_soil = [format_feature_label(f) for f in soil_data['feature']]
    ax_soil.set_yticks(y_pos_soil)
    ax_soil.set_yticklabels(feature_labels_soil, fontsize=9, weight='bold')
    
    # X-axis for soil
    ax_soil.set_xlim(0, max_shap * 1.15)
    ax_soil.set_xlabel('Mean |SHAP value| (feature importance)', 
                      fontsize=10, weight='bold')
    ax_soil.tick_params(axis='x', which='major', labelsize=9, width=1, labelcolor='black')
    
    # Make x-axis tick labels bold
    for label in ax_soil.get_xticklabels():
        label.set_weight('bold')
    
    # Grid
    ax_soil.xaxis.grid(True, alpha=0.3, linewidth=0.5, linestyle='--', color='gray')
    ax_soil.set_axisbelow(True)
    
    # Spines
    ax_soil.spines['left'].set_linewidth(1)
    ax_soil.spines['bottom'].set_linewidth(1)
    ax_soil.spines['top'].set_visible(False)
    ax_soil.spines['right'].set_visible(False)
    
    # ========================================================================
    # INTELLIGENT POSITIONING OF CATEGORY LABELS
    # ========================================================================
    
    # Force rendering to calculate actual label sizes
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    
    # Calculate category label positions to avoid overlap with y-axis labels
    category_x_positions = {}
    
    for ax_name, ax in [('climate', ax_climate), ('soil', ax_soil)]:
        max_label_width = 0
        for label in ax.get_yticklabels():
            bbox = label.get_window_extent(renderer=renderer)
            bbox_axes = bbox.transformed(ax.transAxes.inverted())
            label_left_extent = abs(bbox_axes.x0)
            max_label_width = max(max_label_width, label_left_extent)
        
        # Add safety margin to ensure no overlap
        safety_margin = 0.02  # 2% extra space
        category_x_positions[ax_name] = -(max_label_width + safety_margin)
        
        print(f"   {ax_name.capitalize()} category label position: {category_x_positions[ax_name]:.4f}")
    
    # ALIGN BOTH LABELS: Use the farthest (most negative) position for both
    aligned_position = min(category_x_positions.values())  # Most negative = farthest left
    print(f"   Aligned position (both labels): {aligned_position:.4f}")
    
    # Add category labels with aligned positioning
    ax_climate.text(aligned_position, 0.5, 'Climate factors',
                   transform=ax_climate.transAxes,
                   ha='right', va='center',
                   fontsize=9, weight='bold', 
                   color='black', style='italic',
                   rotation=90)
    
    ax_soil.text(aligned_position, 0.5, 'Soil factors',
                transform=ax_soil.transAxes,
                ha='right', va='center',
                fontsize=9, weight='bold', 
                color='black', style='italic',
                rotation=90)
    
    # ========================================================================
    # ADD LEGENDS AND TEMPORAL CHANGE LABELS TO BOTH PLOTS
    # ========================================================================
    
    # Create SIMPLIFIED legend
    from matplotlib.lines import Line2D
    
    legend_elements = [
        Line2D([0], [0], color='black', marker='', linewidth=0,
               label='Signif.: *** p<0.001, ** p<0.01, * p<0.05')
    ]
    
    # Add legend to climate plot
    ax_climate.legend(handles=legend_elements, 
                     loc='lower right',
                     bbox_to_anchor=(0.85, 0),
                     fontsize=7.5,
                     frameon=False)
    
    # Add temporal change label on right - positioned more inside to avoid overlap
    ax_climate.text(0.88, 0.5, 'Temporal change',
                   transform=ax_climate.transAxes,
                   rotation=270,
                   va='center',
                   ha='center',
                   fontsize=9,
                   weight='bold',
                   color='gray')
    
    # Add legend to soil plot
    ax_soil.legend(handles=legend_elements, 
                  loc='lower right',
                  bbox_to_anchor=(0.85, 0),
                  fontsize=7.5,
                  frameon=False)
    
    # Add temporal change label on right - positioned more inside to avoid overlap
    ax_soil.text(0.88, 0.5, 'Temporal change',
                transform=ax_soil.transAxes,
                rotation=270,
                va='center',
                ha='center',
                fontsize=9,
                weight='bold',
                color='gray')
    
    # ========================================================================
    # OVERALL FIGURE ELEMENTS
    # ========================================================================
    
    # Title with response variable change
    if response_p_value < 0.001:
        response_sig = ' ***'
    elif response_p_value < 0.01:
        response_sig = ' **'
    elif response_p_value < 0.05:
        response_sig = ' *'
    else:
        response_sig = ''
    
    direction_symbol = '↓' if percent_change < 0 else '↑'
    title = (f"{response_variable.replace('_', ' ').title()}: "
            f"{direction_symbol} {abs(percent_change):.1f}%{response_sig} "
            f"({first_year_val:.1f} → {last_year_val:.1f})")
    
    if 'tc_per_ha' in response_variable:
        title += " t/ha"
    elif 'percent' in response_variable or 'brix' in response_variable or 'pol' in response_variable:
        title += " %"
    
    fig.suptitle(title, fontsize=11, weight='bold', y=0.98)
    
    # ========================================================================
    # SAVE FIGURE
    # ========================================================================
    
    output_file = os.path.join(decline_dir, 
                              f'two_section_panel_{response_variable}_{model_name}.png')
    
    try:
        plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"✓ PNG saved: {output_file}")
        
        output_pdf = output_file.replace('.png', '.pdf')
        plt.savefig(output_pdf, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"✓ PDF saved: {output_pdf}")
        
        output_hires = output_file.replace('.png', '_hires.png')
        plt.savefig(output_hires, dpi=600, bbox_inches='tight', facecolor='white')
        print(f"✓ High-res PNG saved: {output_hires}")
        
        plt.close()
        return output_file
        
    except Exception as e:
        print(f"Error saving figure: {e}")
        plt.close()
        return None

# ============================================================================
# BATCH GENERATION
# ============================================================================

def create_all_two_section_figures(response_variables, model_name, base_model_dir, 
                                   data_file, n_features_per_category=5):
    """Create two-section figures for all response variables"""
    
    print("\n" + "="*80)
    print("CREATING TWO-SECTION FIGURES FOR ALL RESPONSE VARIABLES")
    print("="*80)
    
    results = {}
    
    for response_var in response_variables:
        try:
            output_file = create_enhanced_two_section_figure(
                response_variable=response_var,
                model_name=model_name,
                base_model_dir=base_model_dir,
                data_file=data_file,
                n_features_per_category=n_features_per_category
            )
            
            if output_file is not None:
                results[response_var] = {
                    'output': output_file,
                    'status': 'success'
                }
                print(f"✓ Completed: {response_var}\n")
            else:
                results[response_var] = {
                    'status': 'failed',
                    'error': 'Function returned None'
                }
                print(f"✗ Failed: {response_var}\n")
            
        except Exception as e:
            print(f"✗ Error with {response_var}: {e}\n")
            results[response_var] = {
                'status': 'failed',
                'error': str(e)
            }
    
    print("\n" + "="*80)
    print("BATCH GENERATION COMPLETE")
    print("="*80)
    
    successful = sum(1 for r in results.values() if r['status'] == 'success')
    print(f"\nSuccessful: {successful}/{len(response_variables)}")
    
    return results

# ============================================================================
# EXAMPLE USAGE
# ============================================================================

if __name__ == "__main__":
    
    DATA_FILE = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis/final_dataset_regrouped_categorical.csv'
    BASE_MODEL_DIR = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis'
    
    RESPONSE_VARIABLES = ['brix_percent',
                          'pol_juice_percent',
                          'pol_cane_percent',
                          'purity_percent',
                          'glucose_cane_percent',
                          'hugot_sugar',
                          'tc_per_ha_actual',
                          'tc_per_ha_per_week',
                          'tc_per_ha_52_weeks',
                          'fiber_percent',
                          'total_recoverable_sugar_t_per_ha']
    
    MODEL_NAME = 'XGBoost'
    N_FEATURES_PER_CATEGORY = 6
    
    results = create_all_two_section_figures(
        response_variables=RESPONSE_VARIABLES,
        model_name=MODEL_NAME,
        base_model_dir=BASE_MODEL_DIR,
        data_file=DATA_FILE,
        n_features_per_category=N_FEATURES_PER_CATEGORY
    )
    
    print("\n" + "="*80)
    print("FIGURES GENERATED:")
    print("="*80)
    
    for response_var, result in results.items():
        if result['status'] == 'success':
            print(f"\n✓ {response_var}:")
            print(f"   {result['output']}")
        else:
            print(f"\n✗ {response_var}: {result.get('error', 'Unknown error')}")
    
    print("\n" + "="*80)
    print("="*80)

SHAP + Temporal Change (TWO SECTIONS - ALL TOP FEATURES)
============================================================================================

In [ ]:
"""
Plot: SHAP + Temporal Change (TWO SECTIONS - ALL TOP FEATURES)
============================================================================================

Two separate sections showing climate and soil features with distinct spacing.
Shows ALL top features (regardless of significance), but only annotates significant changes.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
#
# ============================================================================

plt.rcParams.update({
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 9,
    'axes.labelsize': 10,
    'axes.titlesize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'axes.linewidth': 1,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'xtick.major.width': 1,
    'ytick.major.width': 1,
})

# Color scheme
SHAP_COLOR = '#E64B35'

TEMPORAL_COLORS = {
    'increase': '#DC3545',
    'decrease': '#28A745',
    'stable': '#6C757D',
}

# ============================================================================
# FEATURE CATEGORIZATION
# ============================================================================

CLIMATE_FEATURES = [
    'growing_season_rainfall_mm',
    'CDD', 'CWD',
    'SDII',
    'Rx1', 'R20',
    'TNn', 'TNx',
    'temp_mini_mean', 'temp_maxi_mean',
    'h_mini_mean', 'h_maxi_mean',
    'soil_temp_6h_mean',
    'radiation_ERA_total'
]


SOIL_FEATURES = [
    'pH_complete',
    'Exchangeable_Calcium_meq_100g_complete',
    'Exchangeable_Potassium_meq_100g_complete',
    'Exchangeable_Sodium_meq_100g_complete',
    'Available_P2O5_mg_kg_complete',
    'Total_Nitrogen_per_mille_complete',
    'Electrical_Conductivity_mmho_cm_complete',
    'Mg_K_Ratio_complete',
    'Coarse_Silt_percent_complete',
    'Coarse_Sand_percent_complete'
]

def categorize_feature(feature_name):
    """Determine if feature is climate or soil"""
    # Check exact match first
    if feature_name in CLIMATE_FEATURES:
        return 'climate'
    elif feature_name in SOIL_FEATURES:
        return 'soil'
    
    # Check partial matches (for variations in naming)
    feature_lower = feature_name.lower()
    
    # Climate indicators
    climate_keywords = ['temp', 'rain', 'radiation', 'cdd', 'cwd', 'tnn', 'tnx',
                       'ampl', 'rx1', 'r20', 'r99p', 'sdii', 'hd', 'h_maxi', 'h_mini']
    if any(kw in feature_lower for kw in climate_keywords):
        return 'climate'
    
    # Soil indicators
    soil_keywords = ['silt', 'nitrogen', 'ph', 'calcium', 'p2o5', 'conductivity',
                    'sodium', 'mg_k', 'soil']
    if any(kw in feature_lower for kw in soil_keywords):
        return 'soil'
    
    return 'other'

def format_feature_label(feature):
    """Format feature name with proper scientific notation and line breaks"""
    
    # Comprehensive mapping dictionary for all features
    feature_labels = {
        # Climate features
        'ampl_mean': 'Temperature\namplitude',
        'TNn': 'Min. night\ntemperature',
        'TNx': 'Max. night\ntemperature',
        'temp_moy_mean': 'Mean\ntemperature',
        'h_maxi_mean': 'Max.\nhumidity',
        'CDD': 'Consecutive\ndry days',
        'radiation_ERA_total': 'Solar\nradiation',
        'Rx1': 'Max. 1-day\nrainfall',
        'R20': 'Heavy rain\ndays',
        'SDII': 'Rainfall\nintensity',
        'h_mini_mean': 'Min.\nhumidity',
        'HD': 'Heat\ndays',
        'R99p': 'Extreme\nrainfall',
        'growing_season_rainfall_mm': 'Growing season\nrainfall',
        'soil_temp_6h_mean': 'Soil temperature\n(6h)',
        'CWD': 'Consecutive\nwet days',
        
        # Soil features
        'Coarse_Silt_percent_complete': 'Coarse silt\n(%)',
        'Total_Nitrogen_per_mille_complete': 'Total nitrogen\n(‰)',
        'pH_complete': 'pH',
        'Exchangeable_Calcium_meq_100g_complete': 'Exchangeable Ca\n(meq/100g)',
        'Available_P2O5_mg_kg_complete': 'Available P₂O₅\n(mg/kg)',
        'Electrical_Conductivity_mmho_cm_complete': 'Electrical conductivity\n(mmho/cm)',
        'Exchangeable_Sodium_meq_100g_complete': 'Exchangeable Na\n(meq/100g)',
        'Mg_K_Ratio_complete': 'Mg/K\nratio'
    }
    
    # Return mapped label if exists, otherwise format generically
    if feature in feature_labels:
        return feature_labels[feature]
    
    # Fallback: generic formatting
    label = feature.replace('_', ' ').replace('complete', '').strip()
    
    # Dictionary of scientific terms that should maintain specific formatting
    scientific_terms = {
        'ph': 'pH',
        'p2o5': 'P₂O₅',
        'mg k ratio': 'Mg/K ratio',
        'meq 100g': 'meq/100g',
        'mg kg': 'mg/kg',
        'mmho cm': 'mmho/cm',
        'per mille': '‰',
        'tnn': 'TNn',
        'tnx': 'TNx',
        'cdd': 'CDD',
        'cwd': 'CWD',
        'sdii': 'SDII',
        'hd': 'HD',
        'rx1': 'Rx1',
        'r20': 'R20',
        'r99p': 'R99p',
        'era': 'ERA'
    }
    
    # Split and capitalize
    words = label.split()
    formatted_words = []
    
    for word in words:
        word_lower = word.lower()
        if word_lower in scientific_terms:
            formatted_words.append(scientific_terms[word_lower])
        else:
            formatted_words.append(word.capitalize())
    
    label = ' '.join(formatted_words)
    
    # Add line breaks for long labels (>25 characters)
    if len(label) > 25:
        words = label.split()
        mid_point = len(words) // 2
        label = ' '.join(words[:mid_point]) + '\n' + ' '.join(words[mid_point:])
    
    return label

# ============================================================================
# MAIN PLOTTING FUNCTION
# ============================================================================

def create_enhanced_two_section_figure(response_variable, model_name, base_model_dir, 
                                       data_file, n_features_per_category=5):
    """
    Create enhanced two-section figure with separate climate/soil sections
    Shows ALL top features (regardless of significance)
    
    Parameters:
    -----------
    response_variable : str
        Response variable name
    model_name : str
        Model name
    base_model_dir : str
        Base directory containing model folders
    data_file : str
        Path to data CSV file
    n_features_per_category : int
        Number of top features to show per category (default: 5)
    
    Returns:
    --------
    str or None : Path to output file if successful, None if failed
    """
    
    print(f"\n{'='*80}")
    print(f"Creating two-section figure: {response_variable} ({model_name})")
    print(f"{'='*80}")
    
    # Construct paths
    model_dir = os.path.join(base_model_dir, f'kfold_models_{response_variable}_custom')
    decline_dir = os.path.join(model_dir, f'decline_analysis_{response_variable}_{model_name}')
    
    if not os.path.exists(decline_dir):
        model_dir = os.path.join(base_model_dir, f'kfold_models_{response_variable}_custom')
        decline_dir = os.path.join(model_dir, f'decline_analysis_{response_variable}_{model_name}')
    
    if not os.path.exists(decline_dir):
        print(f"Error: Directory not found: {decline_dir}")
        return None
    
    # Load data
    try:
        shap_df = pd.read_csv(os.path.join(decline_dir, f'shap_analysis_{response_variable}.csv'))
        trends_df = pd.read_csv(os.path.join(decline_dir, f'temporal_trends_{response_variable}.csv'))
        print(f"✓ Data loaded successfully")
    except Exception as e:
        print(f"Error loading data: {e}")
        return None
    
    # Get response variable statistics
    response_trend_data = trends_df[trends_df['feature'] == response_variable]
    
    if len(response_trend_data) == 0:
        try:
            df_full = pd.read_csv(data_file)
            year_col = 'campaign_year' if 'campaign_year' in df_full.columns else 'Harvest_Year'
            
            if response_variable in df_full.columns:
                yearly_means = df_full.groupby(year_col)[response_variable].mean()
                first_year_val = yearly_means.iloc[0]
                last_year_val = yearly_means.iloc[-1]
                percent_change = ((last_year_val - first_year_val) / first_year_val * 100) if first_year_val != 0 else 0
                response_p_value = 1.0
            else:
                print(f"Error: '{response_variable}' not in dataset!")
                return None
        except Exception as e:
            print(f"Error calculating fallback statistics: {e}")
            return None
    else:
        response_trend = response_trend_data.iloc[0]
        first_year_val = response_trend['first_year_mean']
        last_year_val = response_trend['last_year_mean']
        percent_change = response_trend['percent_change']
        response_p_value = response_trend.get('p_value', response_trend.get('trend_p_value', 1.0))
    
    # Calculate SHAP importance
    exclude_cols = ['year', 'actual', 'predicted', 'Unnamed: 0', 'index']
    feature_cols = [col for col in shap_df.columns 
                   if col not in exclude_cols and not col.startswith('Unnamed')]
    
    if response_variable in feature_cols:
        feature_cols.remove(response_variable)
    
    print(f"Found {len(feature_cols)} feature columns in SHAP data")
    
    if len(feature_cols) == 0:
        print("Error: No feature columns found in SHAP data!")
        return None
    
    # Calculate SHAP importance and categorize
    shap_importance = []
    for feature in feature_cols:
        if feature in shap_df.columns:
            mean_abs_shap = shap_df[feature].abs().mean()
            mean_shap = shap_df[feature].mean()
            category = categorize_feature(feature)
            
            shap_importance.append({
                'feature': feature,
                'mean_abs_shap': mean_abs_shap,
                'mean_shap': mean_shap,
                'category': category,
                'direction': 'positive' if mean_shap > 0 else 'negative'
            })
    
    shap_df_summary = pd.DataFrame(shap_importance)
    
    if len(shap_df_summary) == 0:
        print("Error: No SHAP importance data calculated!")
        return None
    
    print(f"✓ Calculated SHAP importance for {len(shap_df_summary)} features")
    
    # Get top N features per category
    climate_features = shap_df_summary[shap_df_summary['category'] == 'climate']
    soil_features = shap_df_summary[shap_df_summary['category'] == 'soil']
    
    print(f"   - Climate features: {len(climate_features)}")
    print(f"   - Soil features: {len(soil_features)}")
    
    top_climate = climate_features.nlargest(n_features_per_category, 'mean_abs_shap')
    top_soil = soil_features.nlargest(n_features_per_category, 'mean_abs_shap')
    
    # Merge with temporal trends
    available_trend_cols = ['feature', 'percent_change']
    if 'p_value' in trends_df.columns:
        available_trend_cols.append('p_value')
    if 'significant' in trends_df.columns:
        available_trend_cols.append('significant')
    if 'trend_p_value' in trends_df.columns:
        trends_df['p_value'] = trends_df['trend_p_value']
        available_trend_cols.append('p_value')
    
    climate_data = top_climate.merge(trends_df[available_trend_cols], on='feature', how='left')
    soil_data = top_soil.merge(trends_df[available_trend_cols], on='feature', how='left')
    
    # Fill missing values
    for df in [climate_data, soil_data]:
        if 'p_value' not in df.columns:
            df['p_value'] = 1.0
        if 'significant' not in df.columns:
            df['significant'] = False
        df['p_value'] = df['p_value'].fillna(1.0)
        df['percent_change'] = df['percent_change'].fillna(0.0)
    
    # Sort by SHAP importance (ascending for display)
    climate_data = climate_data.sort_values('mean_abs_shap', ascending=True)
    soil_data = soil_data.sort_values('mean_abs_shap', ascending=True)
    
    if len(climate_data) == 0 or len(soil_data) == 0:
        print("Error: No data to plot!")
        return None
    
    print(f"✓ Ready to plot {len(climate_data)} climate + {len(soil_data)} soil features")
    
    # ========================================================================
    # CREATE FIGURE WITH TWO SECTIONS
    # ========================================================================
    
    # Calculate heights based on number of features
    climate_height = len(climate_data) * 0.35
    soil_height = len(soil_data) * 0.35
    gap_height = 0.5  # Space between sections
    total_height = climate_height + soil_height + gap_height + 1.5  # Extra for title/margins
    
    fig = plt.figure(figsize=(8, total_height))
    
    # Calculate height ratios for gridspec
    height_ratio_climate = climate_height
    height_ratio_gap = gap_height
    height_ratio_soil = soil_height
    
    # Create grid with three rows: climate, gap, soil
    gs = fig.add_gridspec(3, 1, 
                         height_ratios=[height_ratio_climate, height_ratio_gap, height_ratio_soil],
                         hspace=0.05,
                         left=0.30, right=0.85, top=0.95, bottom=0.08)
    
    # Create axes
    ax_climate = fig.add_subplot(gs[0])
    ax_soil = fig.add_subplot(gs[2])
    
    # Get max SHAP value for consistent x-axis
    max_shap = max(climate_data['mean_abs_shap'].max(), soil_data['mean_abs_shap'].max())
    
    # ========================================================================
    # PLOT CLIMATE SECTION (TOP)
    # ========================================================================
    
    y_pos_climate = np.arange(len(climate_data))
    
    # Plot bars
    bars_climate = ax_climate.barh(y_pos_climate, climate_data['mean_abs_shap'], 
                                   height=0.5,
                                   color='#6C757D',
                                   alpha=0.75,
                                   edgecolor='white',
                                   linewidth=0.5)
    
    # Add temporal change annotations (ONLY SIGNIFICANT ONES)
    annotation_x = max_shap * 1.08
    
    for i, (idx, row) in enumerate(climate_data.iterrows()):
        pct_change = row['percent_change']
        p_val = row['p_value']
        
        # Significance stars
        if p_val < 0.001:
            stars = '***'
            is_significant = True
        elif p_val < 0.01:
            stars = '**'
            is_significant = True
        elif p_val < 0.05:
            stars = '*'
            is_significant = True
        else:
            stars = ''
            is_significant = False
        
        # ONLY SHOW IF SIGNIFICANT
        if not is_significant or abs(pct_change) < 0.5:
            continue
        
        change_text = f"{pct_change:+.1f}%{stars}"
        
        # Color based on direction
        if pct_change > 0:
            text_color = TEMPORAL_COLORS['increase']
        else:
            text_color = TEMPORAL_COLORS['decrease']
        
        ax_climate.text(annotation_x, i, change_text,
                       va='center', ha='left',
                       fontsize=8,
                       color=text_color,
                       weight='bold')
    
    # Format y-axis
    feature_labels_climate = [format_feature_label(f) for f in climate_data['feature']]
    ax_climate.set_yticks(y_pos_climate)
    ax_climate.set_yticklabels(feature_labels_climate, fontsize=9, weight='bold')
    
    # X-axis for climate
    ax_climate.set_xlim(0, max_shap * 1.15)
    ax_climate.set_xlabel('Mean |SHAP value| (feature importance)', 
                         fontsize=10, weight='bold')
    ax_climate.tick_params(axis='x', which='major', labelsize=9, width=1, labelcolor='black')
    
    # Make x-axis tick labels bold
    for label in ax_climate.get_xticklabels():
        label.set_weight('bold')
    
    # Grid
    ax_climate.xaxis.grid(True, alpha=0.3, linewidth=0.5, linestyle='--', color='gray')
    ax_climate.set_axisbelow(True)
    
    # Spines
    ax_climate.spines['left'].set_linewidth(1)
    ax_climate.spines['bottom'].set_linewidth(1)
    ax_climate.spines['top'].set_visible(False)
    ax_climate.spines['right'].set_visible(False)
    
    # ========================================================================
    # PLOT SOIL SECTION (BOTTOM)
    # ========================================================================
    
    y_pos_soil = np.arange(len(soil_data))
    
    # Plot bars
    bars_soil = ax_soil.barh(y_pos_soil, soil_data['mean_abs_shap'], 
                            height=0.5,
                            color='#6C757D',
                            alpha=0.75,
                            edgecolor='white',
                            linewidth=0.5)
    
    # Add temporal change annotations (ONLY SIGNIFICANT ONES)
    for i, (idx, row) in enumerate(soil_data.iterrows()):
        pct_change = row['percent_change']
        p_val = row['p_value']
        
        # Significance stars
        if p_val < 0.001:
            stars = '***'
            is_significant = True
        elif p_val < 0.01:
            stars = '**'
            is_significant = True
        elif p_val < 0.05:
            stars = '*'
            is_significant = True
        else:
            stars = ''
            is_significant = False
        
        # ONLY SHOW IF SIGNIFICANT
        if not is_significant or abs(pct_change) < 0.5:
            continue
        
        change_text = f"{pct_change:+.1f}%{stars}"
        
        # Color based on direction
        if pct_change > 0:
            text_color = TEMPORAL_COLORS['increase']
        else:
            text_color = TEMPORAL_COLORS['decrease']
        
        ax_soil.text(annotation_x, i, change_text,
                    va='center', ha='left',
                    fontsize=8,
                    color=text_color,
                    weight='bold')
    
    # Format y-axis
    feature_labels_soil = [format_feature_label(f) for f in soil_data['feature']]
    ax_soil.set_yticks(y_pos_soil)
    ax_soil.set_yticklabels(feature_labels_soil, fontsize=9, weight='bold')
    
    # X-axis for soil
    ax_soil.set_xlim(0, max_shap * 1.15)
    ax_soil.set_xlabel('Mean |SHAP value| (feature importance)', 
                      fontsize=10, weight='bold')
    ax_soil.tick_params(axis='x', which='major', labelsize=9, width=1, labelcolor='black')
    
    # Make x-axis tick labels bold
    for label in ax_soil.get_xticklabels():
        label.set_weight('bold')
    
    # Grid
    ax_soil.xaxis.grid(True, alpha=0.3, linewidth=0.5, linestyle='--', color='gray')
    ax_soil.set_axisbelow(True)
    
    # Spines
    ax_soil.spines['left'].set_linewidth(1)
    ax_soil.spines['bottom'].set_linewidth(1)
    ax_soil.spines['top'].set_visible(False)
    ax_soil.spines['right'].set_visible(False)
    
    # ========================================================================
    # INTELLIGENT POSITIONING OF CATEGORY LABELS
    # ========================================================================
    
    # Force rendering to calculate actual label sizes
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    
    # Calculate category label positions to avoid overlap with y-axis labels
    category_x_positions = {}
    
    for ax_name, ax in [('climate', ax_climate), ('soil', ax_soil)]:
        max_label_width = 0
        for label in ax.get_yticklabels():
            bbox = label.get_window_extent(renderer=renderer)
            bbox_axes = bbox.transformed(ax.transAxes.inverted())
            label_left_extent = abs(bbox_axes.x0)
            max_label_width = max(max_label_width, label_left_extent)
        
        # Add safety margin to ensure no overlap
        safety_margin = 0.02  # 2% extra space
        category_x_positions[ax_name] = -(max_label_width + safety_margin)
        
        print(f"   {ax_name.capitalize()} category label position: {category_x_positions[ax_name]:.4f}")
    
    # ALIGN BOTH LABELS: Use the farthest (most negative) position for both
    aligned_position = min(category_x_positions.values())  # Most negative = farthest left
    print(f"   Aligned position (both labels): {aligned_position:.4f}")
    
    # Add category labels with aligned positioning
    ax_climate.text(aligned_position, 0.5, 'Climate factors',
                   transform=ax_climate.transAxes,
                   ha='right', va='center',
                   fontsize=9, weight='bold', 
                   color='black', style='italic',
                   rotation=90)
    
    ax_soil.text(aligned_position, 0.5, 'Soil factors',
                transform=ax_soil.transAxes,
                ha='right', va='center',
                fontsize=9, weight='bold', 
                color='black', style='italic',
                rotation=90)
    
    # ========================================================================
    # ADD LEGENDS AND TEMPORAL CHANGE LABELS TO BOTH PLOTS
    # ========================================================================
    
    # Create SIMPLIFIED legend
    from matplotlib.lines import Line2D
    
    legend_elements = [
        Line2D([0], [0], color='black', marker='', linewidth=0,
               label='Signif.: *** p<0.001, ** p<0.01, * p<0.05')
    ]
    
    # Add legend to climate plot
    ax_climate.legend(handles=legend_elements, 
                     loc='lower right',
                     bbox_to_anchor=(0.85, 0),
                     fontsize=7.5,
                     frameon=False)
    
    # Add temporal change label on right - positioned more inside to avoid overlap
    ax_climate.text(0.88, 0.5, 'Temporal change',
                   transform=ax_climate.transAxes,
                   rotation=270,
                   va='center',
                   ha='center',
                   fontsize=9,
                   weight='bold',
                   color='gray')
    
    # Add legend to soil plot
    ax_soil.legend(handles=legend_elements, 
                  loc='lower right',
                  bbox_to_anchor=(0.85, 0),
                  fontsize=7.5,
                  frameon=False)
    
    # Add temporal change label on right - positioned more inside to avoid overlap
    ax_soil.text(0.88, 0.5, 'Temporal change',
                transform=ax_soil.transAxes,
                rotation=270,
                va='center',
                ha='center',
                fontsize=9,
                weight='bold',
                color='gray')
    
    # ========================================================================
    # OVERALL FIGURE ELEMENTS
    # ========================================================================
    
    # Title with response variable change
    if response_p_value < 0.001:
        response_sig = ' ***'
    elif response_p_value < 0.01:
        response_sig = ' **'
    elif response_p_value < 0.05:
        response_sig = ' *'
    else:
        response_sig = ''
    
    direction_symbol = '↓' if percent_change < 0 else '↑'
    title = (f"{response_variable.replace('_', ' ').title()}: "
            f"{direction_symbol} {abs(percent_change):.1f}%{response_sig} "
            f"({first_year_val:.1f} → {last_year_val:.1f})")
    
    if 'tc_per_ha' in response_variable:
        title += " t/ha"
    elif 'percent' in response_variable or 'brix' in response_variable or 'pol' in response_variable:
        title += " %"
    
    fig.suptitle(title, fontsize=11, weight='bold', y=0.98)
    
    # ========================================================================
    # SAVE FIGURE
    # ========================================================================
    
    output_file = os.path.join(decline_dir, 
                              f'two_section_panel_{response_variable}_{model_name}.png')
    
    try:
        plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"✓ PNG saved: {output_file}")
        
        output_pdf = output_file.replace('.png', '.pdf')
        plt.savefig(output_pdf, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"✓ PDF saved: {output_pdf}")
        
        output_hires = output_file.replace('.png', '_hires.png')
        plt.savefig(output_hires, dpi=600, bbox_inches='tight', facecolor='white')
        print(f"✓ High-res PNG saved: {output_hires}")
        
        plt.close()
        return output_file
        
    except Exception as e:
        print(f"Error saving figure: {e}")
        plt.close()
        return None

# ============================================================================
# BATCH GENERATION
# ============================================================================

def create_all_two_section_figures(response_variables, model_name, base_model_dir, 
                                   data_file, n_features_per_category=5):
    """Create two-section figures for all response variables"""
    
    print("\n" + "="*80)
    print("CREATING TWO-SECTION FIGURES FOR ALL RESPONSE VARIABLES")
    print("="*80)
    
    results = {}
    
    for response_var in response_variables:
        try:
            output_file = create_enhanced_two_section_figure(
                response_variable=response_var,
                model_name=model_name,
                base_model_dir=base_model_dir,
                data_file=data_file,
                n_features_per_category=n_features_per_category
            )
            
            if output_file is not None:
                results[response_var] = {
                    'output': output_file,
                    'status': 'success'
                }
                print(f"✓ Completed: {response_var}\n")
            else:
                results[response_var] = {
                    'status': 'failed',
                    'error': 'Function returned None'
                }
                print(f"✗ Failed: {response_var}\n")
            
        except Exception as e:
            print(f"✗ Error with {response_var}: {e}\n")
            results[response_var] = {
                'status': 'failed',
                'error': str(e)
            }
    
    print("\n" + "="*80)
    print("BATCH GENERATION COMPLETE")
    print("="*80)
    
    successful = sum(1 for r in results.values() if r['status'] == 'success')
    print(f"\nSuccessful: {successful}/{len(response_variables)}")
    
    return results

# ============================================================================
# EXAMPLE USAGE
# ============================================================================

if __name__ == "__main__":
    
    DATA_FILE = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis/final_dataset_regrouped_categorical.csv'
    BASE_MODEL_DIR = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis'
    
    RESPONSE_VARIABLES = ['brix_percent',
                          'pol_juice_percent',
                          'pol_cane_percent',
                          'purity_percent',
                          'glucose_cane_percent',
                          'hugot_sugar',
                          'tc_per_ha_actual',
                          'tc_per_ha_per_week',
                          'tc_per_ha_52_weeks',
                          'fiber_percent',
                          'total_recoverable_sugar_t_per_ha']
    
    MODEL_NAME = 'XGBoost'
    N_FEATURES_PER_CATEGORY = 6
    
    results = create_all_two_section_figures(
        response_variables=RESPONSE_VARIABLES,
        model_name=MODEL_NAME,
        base_model_dir=BASE_MODEL_DIR,
        data_file=DATA_FILE,
        n_features_per_category=N_FEATURES_PER_CATEGORY
    )
    
    print("\n" + "="*80)
    print("FIGURES GENERATED:")
    print("="*80)
    
    for response_var, result in results.items():
        if result['status'] == 'success':
            print(f"\n✓ {response_var}:")
            print(f"   {result['output']}")
        else:
            print(f"\n✗ {response_var}: {result.get('error', 'Unknown error')}")
    
    print("\n" + "="*80)


    ╔════════════════════════════════════════════════════════════════╗
    ║  TWO-SECTION PUBLICATION FIGURE GENERATOR (ALL TOP FEATURES)   ║
    ║                                                                ║
    ║  Features:                                                     ║
    ║  • Two distinct sections with spacing                          ║
    ║  • Separate x-axis for climate factors (top)                   ║
    ║  • Separate x-axis for soil factors (bottom)                   ║
    ║  • All bars colored in gray (#6C757D)                          ║
    ║  • Shows ALL top features (not filtered by significance)       ║
    ║  • Only SIGNIFICANT temporal changes annotated                 ║
    ║  • Aligned "Climate/Soil factors" labels                       ║
    ║  • Legend without frame, shortened to "Signif."                ║
    ║  • Temporal change label positioned inside plot                ║
    ╚════════════════════════════════════════════════════════════════╝
    


SHAP + Temporal Change (TWO SECTIONS - SIGNIFICANT ONLY - WITH REPORTING)
==========================================================================

In [ ]:
"""
Plot: SHAP + Temporal Change (TWO SECTIONS - SIGNIFICANT ONLY - WITH REPORTING)
============================================================================================================

Two separate sections showing climate and soil features with distinct spacing.
ONLY shows features that have SIGNIFICANT temporal changes (p < 0.05).
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# ============================================================================

plt.rcParams.update({
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 9,
    'axes.labelsize': 10,
    'axes.titlesize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'axes.linewidth': 1,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'xtick.major.width': 1,
    'ytick.major.width': 1,
})

# Color scheme
SHAP_COLOR = '#E64B35'

TEMPORAL_COLORS = {
    'increase': '#DC3545',
    'decrease': '#28A745',
    'stable': '#6C757D',
}

# ============================================================================
# FEATURE CATEGORIZATION
# ============================================================================

CLIMATE_FEATURES = [
    'growing_season_rainfall_mm',
    'CDD', 'CWD',
    'SDII',
    'Rx1', 'R20',
    'TNn', 'TNx',
    'temp_mini_mean', 'temp_maxi_mean',
    'h_mini_mean', 'h_maxi_mean',
    'soil_temp_6h_mean',
    'radiation_ERA_total'
]


SOIL_FEATURES = [
    'pH_complete',
    'Exchangeable_Calcium_meq_100g_complete',
    'Exchangeable_Potassium_meq_100g_complete',
    'Exchangeable_Sodium_meq_100g_complete',
    'Available_P2O5_mg_kg_complete',
    'Total_Nitrogen_per_mille_complete',
    'Electrical_Conductivity_mmho_cm_complete',
    'Mg_K_Ratio_complete',
    'Coarse_Silt_percent_complete',
    'Coarse_Sand_percent_complete'
]

def categorize_feature(feature_name):
    """Determine if feature is climate or soil"""
    # Check exact match first
    if feature_name in CLIMATE_FEATURES:
        return 'climate'
    elif feature_name in SOIL_FEATURES:
        return 'soil'
    
    # Check partial matches (for variations in naming)
    feature_lower = feature_name.lower()
    
    # Climate indicators
    climate_keywords = ['temp', 'rain', 'radiation', 'cdd', 'cwd', 'tnn', 'tnx',
                       'ampl', 'rx1', 'r20', 'r99p', 'sdii', 'hd', 'h_maxi', 'h_mini']
    if any(kw in feature_lower for kw in climate_keywords):
        return 'climate'
    
    # Soil indicators
    soil_keywords = ['silt', 'nitrogen', 'ph', 'calcium', 'p2o5', 'conductivity',
                    'sodium', 'mg_k', 'soil']
    if any(kw in feature_lower for kw in soil_keywords):
        return 'soil'
    
    return 'other'

def format_feature_label(feature):
    """Format feature name with proper scientific notation and line breaks"""
    
    # English feature mapping (using short_en labels)
    feature_labels_en = {
        # Climate features
        'ampl_mean': 'Temp Ampl\n(°C)',
        'TNn': 'TNn\n(°C)',
        'TNx': 'TNx\n(°C)',
        'temp_moy_mean': 'Mean Temp\n(°C)',
        'temp_maxi_mean': 'T° max\n(°C)',
        'temp_mini_mean': 'T° min\n(°C)',
        'h_maxi_mean': 'RH max\n(%)',
        'h_mini_mean': 'RH min\n(%)',
        'h_moy_mean': 'Mean Humid\n(%)',
        'CDD': 'CDD\n(days)',
        'CWD': 'CWD\n(days)',
        'radiation_ERA_total': 'Solar rad.\n(cal/cm²)',
        'Rx1': 'Rx1\n(mm)',
        'R20': 'R20\n(days)',
        'R10': 'R10\n(days)',
        'SDII': 'SDII\n(mm/day)',
        'HD': 'Hot Days\n(days)',
        'R99p': 'R99p\n(mm)',
        'R95p': 'R95p\n(mm)',
        'growing_season_rainfall_mm': 'Growing season\nrain (mm)',
        'soil_temp_6h_mean': 'Soil T° 6h\n(°C)',
        'soil_temp_16h_mean': 'Soil T 16h\n(°C)',
        'precipitation_total': 'Precip\n(mm)',
        'evaporation_total': 'Evaporation\n(mm)',
        'TXn': 'TXn\n(°C)',
        'TXx': 'TXx\n(°C)',
        'Rx5': 'Max 5D\n(mm)',
        
        # Soil features
        'Coarse_Silt_percent_complete': 'Coarse silt\n(%)',
        'Fine_Silt_percent_complete': 'Fine Silt\n(%)',
        'Total_Nitrogen_per_mille_complete': 'N total\n(‰)',
        'pH_complete': 'pH',
        'Exchangeable_Calcium_meq_100g_complete': 'Ca exch.\n(meq/100g)',
        'Exchangeable_Magnesium_meq_100g_complete': 'Mg\n(meq/100g)',
        'Exchangeable_Potassium_meq_100g_complete': 'K exch.\n(meq/100g)',
        'Exchangeable_Sodium_meq_100g_complete': 'Na exch.\n(meq/100g)',
        'Available_P2O5_mg_kg_complete': 'P₂O₅ avail.\n(mg/kg)',
        'Electrical_Conductivity_mmho_cm_complete': 'EC\n(mmho/cm)',
        'Mg_K_Ratio_complete': 'Mg/K ratio',
        'Mg_Ca_Ratio_complete': 'Mg/Ca',
        'Organic_Carbon_per_mille_complete': 'Org C\n(‰)',
        'Clay_percent_complete': 'Clay\n(%)',
        'Fine_Sand_percent_complete': 'Fine Sand\n(%)',
        'Coarse_Sand_percent_complete': 'Coarse sand\n(%)',
    }
    
    # Return mapped label if exists
    if feature in feature_labels_en:
        return feature_labels_en[feature]
    
    # Fallback: generic formatting
    label = feature.replace('_', ' ').replace('complete', '').strip()
    
    # Dictionary of scientific terms that should maintain specific formatting
    scientific_terms = {
        'ph': 'pH',
        'p2o5': 'P₂O₅',
        'mg k ratio': 'Mg/K ratio',
        'meq 100g': 'meq/100g',
        'mg kg': 'mg/kg',
        'mmho cm': 'mmho/cm',
        'per mille': '‰',
        'tnn': 'TNn',
        'tnx': 'TNx',
        'cdd': 'CDD',
        'cwd': 'CWD',
        'sdii': 'SDII',
        'hd': 'HD',
        'rx1': 'Rx1',
        'r20': 'R20',
        'r99p': 'R99p',
        'era': 'ERA'
    }
    
    # Split and capitalize
    words = label.split()
    formatted_words = []
    
    for word in words:
        word_lower = word.lower()
        if word_lower in scientific_terms:
            formatted_words.append(scientific_terms[word_lower])
        else:
            formatted_words.append(word.capitalize())
    
    label = ' '.join(formatted_words)
    
    # Add line breaks for long labels (>25 characters)
    if len(label) > 25:
        words = label.split()
        mid_point = len(words) // 2
        label = ' '.join(words[:mid_point]) + '\n' + ' '.join(words[mid_point:])
    
    return label


"""
# FRENCH VERSION - Uncomment to use French labels
def format_feature_label(feature):
    '''Format feature name with proper scientific notation and line breaks (French)'''
    
    # French feature mapping (using short_fr labels)
    feature_labels_fr = {
        # Climate features
        'ampl_mean': 'Ampl temp\n(°C)',
        'TNn': 'TNn\n(°C)',
        'TNx': 'TNx\n(°C)',
        'temp_moy_mean': 'Temp moyenne\n(°C)',
        'temp_maxi_mean': 'T° max\n(°C)',
        'temp_mini_mean': 'T° min\n(°C)',
        'h_maxi_mean': 'HR max\n(%)',
        'h_mini_mean': 'HR min\n(%)',
        'h_moy_mean': 'Humid moy\n(%)',
        'CDD': 'CDD\n(jours)',
        'CWD': 'CWD\n(jours)',
        'radiation_ERA_total': 'Rayonnement\n(cal/cm²)',
        'Rx1': 'Rx1\n(mm)',
        'R20': 'R20\n(jours)',
        'R10': 'R10\n(jours)',
        'SDII': 'SDII\n(mm/jour)',
        'HD': 'Jours chauds\n(jours)',
        'R99p': 'R99p\n(mm)',
        'R95p': 'R95p\n(mm)',
        'growing_season_rainfall_mm': 'Pluie saison\n(mm)',
        'soil_temp_6h_mean': 'T° sol 6h\n(°C)',
        'soil_temp_16h_mean': 'Temp sol 16h\n(°C)',
        'precipitation_total': 'Précip\n(mm)',
        'evaporation_total': 'Évaporation\n(mm)',
        'TXn': 'TXn\n(°C)',
        'TXx': 'TXx\n(°C)',
        'Rx5': 'Max 5J\n(mm)',
        
        # Soil features
        'Coarse_Silt_percent_complete': 'Limon grossier\n(%)',
        'Fine_Silt_percent_complete': 'Limon fin\n(%)',
        'Total_Nitrogen_per_mille_complete': 'N total\n(‰)',
        'pH_complete': 'pH',
        'Exchangeable_Calcium_meq_100g_complete': 'Ca éch.\n(meq/100g)',
        'Exchangeable_Magnesium_meq_100g_complete': 'Mg\n(meq/100g)',
        'Exchangeable_Potassium_meq_100g_complete': 'K éch.\n(meq/100g)',
        'Exchangeable_Sodium_meq_100g_complete': 'Na éch.\n(meq/100g)',
        'Available_P2O5_mg_kg_complete': 'P₂O₅ disp.\n(mg/kg)',
        'Electrical_Conductivity_mmho_cm_complete': 'CE\n(mmho/cm)',
        'Mg_K_Ratio_complete': 'Rapport Mg/K',
        'Mg_Ca_Ratio_complete': 'Mg/Ca',
        'Organic_Carbon_per_mille_complete': 'C organique\n(‰)',
        'Clay_percent_complete': 'Argile\n(%)',
        'Fine_Sand_percent_complete': 'Sable fin\n(%)',
        'Coarse_Sand_percent_complete': 'Sable grossier\n(%)',
    }
    
    # Return mapped label if exists
    if feature in feature_labels_fr:
        return feature_labels_fr[feature]
    
    # Fallback: generic formatting
    label = feature.replace('_', ' ').replace('complete', '').strip()
    
    # Dictionary of scientific terms that should maintain specific formatting
    scientific_terms = {
        'ph': 'pH',
        'p2o5': 'P₂O₅',
        'mg k ratio': 'Rapport Mg/K',
        'meq 100g': 'meq/100g',
        'mg kg': 'mg/kg',
        'mmho cm': 'mmho/cm',
        'per mille': '‰',
        'tnn': 'TNn',
        'tnx': 'TNx',
        'cdd': 'CDD',
        'cwd': 'CWD',
        'sdii': 'SDII',
        'hd': 'HD',
        'rx1': 'Rx1',
        'r20': 'R20',
        'r99p': 'R99p',
        'era': 'ERA'
    }
    
    # Split and capitalize
    words = label.split()
    formatted_words = []
    
    for word in words:
        word_lower = word.lower()
        if word_lower in scientific_terms:
            formatted_words.append(scientific_terms[word_lower])
        else:
            formatted_words.append(word.capitalize())
    
    label = ' '.join(formatted_words)
    
    # Add line breaks for long labels (>25 characters)
    if len(label) > 25:
        words = label.split()
        mid_point = len(words) // 2
        label = ' '.join(words[:mid_point]) + '\n' + ' '.join(words[mid_point:])
    
    return label
"""


def get_top_n_significant_features(df_category, n_features, p_threshold=0.05, category_name=''):
    """
    Get exactly N features with significant temporal changes from a category.
    Searches through the entire ranked list until N significant features are found.
    
    Parameters:
    -----------
    df_category : DataFrame
        Features from one category (climate or soil), sorted by SHAP importance
    n_features : int
        Number of significant features to find
    p_threshold : float
        P-value threshold for significance
    category_name : str
        Name of category for reporting
    
    Returns:
    --------
    DataFrame : Top N features with significant temporal changes
    """
    if len(df_category) == 0:
        return pd.DataFrame()
    
    # Sort by SHAP importance (descending)
    df_sorted = df_category.sort_values('mean_abs_shap', ascending=False).copy()
    
    # Filter for significant changes
    significant_features = df_sorted[df_sorted['p_value'] < p_threshold].copy()
    
    print(f"\n{category_name.upper()} CATEGORY ANALYSIS:")
    print(f"{'─'*70}")
    print(f"Total features in category: {len(df_sorted)}")
    print(f"Features with significant temporal change (p<{p_threshold}): {len(significant_features)}")
    print(f"Requested: {n_features} features")
    
    # Take top N (or all if fewer than N available)
    n_available = min(n_features, len(significant_features))
    selected = significant_features.head(n_available)
    
    if n_available < n_features:
        print(f"⚠ WARNING: Only {n_available} significant features available (requested {n_features})")
    else:
        print(f"✓ Found {n_available} significant features")
    
    return selected


# ============================================================================
# MAIN PLOTTING FUNCTION
# ============================================================================

def create_two_section_significant_figure(response_variable, model_name, base_model_dir, 
                                          data_file, n_features_per_category=5,
                                          p_threshold=0.05):
    """
    Create two-section figure with climate/soil split
    ONLY SHOWS FEATURES WITH SIGNIFICANT TEMPORAL CHANGES
    ALWAYS finds exactly N features per category (searches deep if needed)
    
    Parameters:
    -----------
    response_variable : str
        Response variable name
    model_name : str
        Model name
    base_model_dir : str
        Base directory containing model folders
    data_file : str
        Path to data CSV file
    n_features_per_category : int
        Number of features with significant changes to show per category (default: 5)
        Will search through entire ranked list to find this many significant features
    p_threshold : float
        P-value threshold for significance (default: 0.05)
    
    Returns:
    --------
    dict : Summary information including selected features
    """
    
    print(f"\n{'='*80}")
    print(f"ANALYZING: {response_variable} ({model_name})")
    print(f"Target: {n_features_per_category} significant features per category (p < {p_threshold})")
    print(f"{'='*80}")
    
    # Construct paths
    model_dir = os.path.join(base_model_dir, f'kfold_models_{response_variable}_custom')
    decline_dir = os.path.join(model_dir, f'decline_analysis_{response_variable}_{model_name}')
    
    if not os.path.exists(decline_dir):
        model_dir = os.path.join(base_model_dir, f'kfold_models_{response_variable}_custom')
        decline_dir = os.path.join(model_dir, f'decline_analysis_{response_variable}_{model_name}')
    
    if not os.path.exists(decline_dir):
        print(f"Error: Directory not found: {decline_dir}")
        return None
    
    # Load data
    try:
        shap_df = pd.read_csv(os.path.join(decline_dir, f'shap_analysis_{response_variable}.csv'))
        trends_df = pd.read_csv(os.path.join(decline_dir, f'temporal_trends_{response_variable}.csv'))
        print(f"✓ Data loaded successfully")
    except Exception as e:
        print(f"Error loading data: {e}")
        return None
    
    # Get response variable statistics
    response_trend_data = trends_df[trends_df['feature'] == response_variable]
    
    if len(response_trend_data) == 0:
        try:
            df_full = pd.read_csv(data_file)
            year_col = 'campaign_year' if 'campaign_year' in df_full.columns else 'Harvest_Year'
            
            if response_variable in df_full.columns:
                yearly_means = df_full.groupby(year_col)[response_variable].mean()
                first_year_val = yearly_means.iloc[0]
                last_year_val = yearly_means.iloc[-1]
                percent_change = ((last_year_val - first_year_val) / first_year_val * 100) if first_year_val != 0 else 0
                response_p_value = 1.0
            else:
                print(f"Error: '{response_variable}' not in dataset!")
                return None
        except Exception as e:
            print(f"Error calculating fallback statistics: {e}")
            return None
    else:
        response_trend = response_trend_data.iloc[0]
        first_year_val = response_trend['first_year_mean']
        last_year_val = response_trend['last_year_mean']
        percent_change = response_trend['percent_change']
        response_p_value = response_trend.get('p_value', response_trend.get('trend_p_value', 1.0))
    
    # Calculate SHAP importance
    exclude_cols = ['year', 'actual', 'predicted', 'Unnamed: 0', 'index']
    feature_cols = [col for col in shap_df.columns 
                   if col not in exclude_cols and not col.startswith('Unnamed')]
    
    if response_variable in feature_cols:
        feature_cols.remove(response_variable)
    
    print(f"\nFound {len(feature_cols)} feature columns in SHAP data")
    
    if len(feature_cols) == 0:
        print("Error: No feature columns found in SHAP data!")
        return None
    
    # Calculate SHAP importance and categorize
    shap_importance = []
    for feature in feature_cols:
        if feature in shap_df.columns:
            mean_abs_shap = shap_df[feature].abs().mean()
            mean_shap = shap_df[feature].mean()
            category = categorize_feature(feature)
            
            shap_importance.append({
                'feature': feature,
                'mean_abs_shap': mean_abs_shap,
                'mean_shap': mean_shap,
                'category': category,
                'direction': 'positive' if mean_shap > 0 else 'negative'
            })
    
    shap_df_summary = pd.DataFrame(shap_importance)
    
    if len(shap_df_summary) == 0:
        print("Error: No SHAP importance data calculated!")
        return None
    
    # Merge with temporal trends
    available_trend_cols = ['feature', 'percent_change']
    if 'p_value' in trends_df.columns:
        available_trend_cols.append('p_value')
    if 'significant' in trends_df.columns:
        available_trend_cols.append('significant')
    if 'trend_p_value' in trends_df.columns:
        trends_df['p_value'] = trends_df['trend_p_value']
        if 'p_value' not in available_trend_cols:
            available_trend_cols.append('p_value')
    
    shap_with_trends = shap_df_summary.merge(trends_df[available_trend_cols], 
                                              on='feature', how='left')
    
    # Fill missing values
    if 'p_value' not in shap_with_trends.columns:
        shap_with_trends['p_value'] = 1.0
    if 'significant' not in shap_with_trends.columns:
        shap_with_trends['significant'] = False
    shap_with_trends['p_value'] = shap_with_trends['p_value'].fillna(1.0)
    shap_with_trends['percent_change'] = shap_with_trends['percent_change'].fillna(0.0)
    
    # ========================================================================
    # KEY FEATURE SELECTION: Get exactly N significant features per category
    # ========================================================================
    
    print(f"\n{'='*80}")
    print(f"FEATURE SELECTION PROCESS")
    print(f"{'='*80}")
    
    # Separate by category
    climate_features = shap_with_trends[shap_with_trends['category'] == 'climate'].copy()
    soil_features = shap_with_trends[shap_with_trends['category'] == 'soil'].copy()
    
    # Get top N significant features per category
    top_climate = get_top_n_significant_features(climate_features, n_features_per_category, 
                                                  p_threshold, 'Climate')
    top_soil = get_top_n_significant_features(soil_features, n_features_per_category, 
                                               p_threshold, 'Soil')
    
    # ========================================================================
    # DETAILED SUMMARY TABLE
    # ========================================================================
    
    print(f"\n{'='*80}")
    print(f"SELECTED FEATURES SUMMARY")
    print(f"{'='*80}")
    
    summary_info = {
        'response_variable': response_variable,
        'climate_selected': len(top_climate),
        'soil_selected': len(top_soil),
        'total_selected': len(top_climate) + len(top_soil),
        'climate_features': [],
        'soil_features': []
    }
    
    if len(top_climate) > 0:
        print(f"\nCLIMATE FEATURES ({len(top_climate)} selected):")
        print(f"{'─'*80}")
        print(f"{'Rank':<6}{'Feature':<45}{'SHAP':<12}{'Change':<12}{'P-value':<10}{'Sig':<5}")
        print(f"{'─'*80}")
        
        for rank, (idx, row) in enumerate(top_climate.sort_values('mean_abs_shap', ascending=False).iterrows(), 1):
            stars = ''
            if row['p_value'] < 0.001:
                stars = '***'
            elif row['p_value'] < 0.01:
                stars = '**'
            elif row['p_value'] < 0.05:
                stars = '*'
            
            print(f"{rank:<6}{row['feature']:<45}{row['mean_abs_shap']:<12.4f}"
                  f"{row['percent_change']:>+10.1f}% {row['p_value']:<10.4f}{stars:<5}")
            
            summary_info['climate_features'].append({
                'rank': rank,
                'feature': row['feature'],
                'shap': row['mean_abs_shap'],
                'change': row['percent_change'],
                'p_value': row['p_value']
            })
    else:
        print(f"\nCLIMATE FEATURES: None with significant temporal change")
    
    if len(top_soil) > 0:
        print(f"\nSOIL FEATURES ({len(top_soil)} selected):")
        print(f"{'─'*80}")
        print(f"{'Rank':<6}{'Feature':<45}{'SHAP':<12}{'Change':<12}{'P-value':<10}{'Sig':<5}")
        print(f"{'─'*80}")
        
        for rank, (idx, row) in enumerate(top_soil.sort_values('mean_abs_shap', ascending=False).iterrows(), 1):
            stars = ''
            if row['p_value'] < 0.001:
                stars = '***'
            elif row['p_value'] < 0.01:
                stars = '**'
            elif row['p_value'] < 0.05:
                stars = '*'
            
            print(f"{rank:<6}{row['feature']:<45}{row['mean_abs_shap']:<12.4f}"
                  f"{row['percent_change']:>+10.1f}% {row['p_value']:<10.4f}{stars:<5}")
            
            summary_info['soil_features'].append({
                'rank': rank,
                'feature': row['feature'],
                'shap': row['mean_abs_shap'],
                'change': row['percent_change'],
                'p_value': row['p_value']
            })
    
    if len(top_climate) == 0 and len(top_soil) == 0:
        print(f"\n⚠ WARNING: No features with significant temporal changes found!")
        print(f"Cannot create plot for {response_variable}")
        summary_info['status'] = 'no_significant_features'
        return summary_info
    
    # Sort by SHAP importance for display (ascending for horizontal bar plot)
    climate_data = top_climate.sort_values('mean_abs_shap', ascending=True) if len(top_climate) > 0 else pd.DataFrame()
    soil_data = top_soil.sort_values('mean_abs_shap', ascending=True) if len(top_soil) > 0 else pd.DataFrame()
    
    print(f"\n{'='*80}")
    print(f"FINAL FEATURES TO PLOT")
    print(f"{'='*80}")
    print(f"Climate features: {len(climate_data)}")
    print(f"Soil features: {len(soil_data)}")
    
    # ========================================================================
    # CREATE FIGURE WITH TWO SECTIONS
    # ========================================================================
    
    # Calculate heights based on number of features
    climate_height = len(climate_data) * 0.35 if len(climate_data) > 0 else 0.5
    soil_height = len(soil_data) * 0.35 if len(soil_data) > 0 else 0.5
    gap_height = 0.5  # Space between sections
    total_height = climate_height + soil_height + gap_height + 1.5  # Extra for title/margins
    
    fig = plt.figure(figsize=(8, total_height))
    
    # Calculate height ratios for gridspec
    height_ratio_climate = climate_height
    height_ratio_gap = gap_height
    height_ratio_soil = soil_height
    
    # Create grid with three rows: climate, gap, soil
    gs = fig.add_gridspec(3, 1, 
                         height_ratios=[height_ratio_climate, height_ratio_gap, height_ratio_soil],
                         hspace=0.05,
                         left=0.30, right=0.85, top=0.95, bottom=0.08)
    
    # Create axes
    ax_climate = fig.add_subplot(gs[0])
    ax_soil = fig.add_subplot(gs[2])
    
    # Get max SHAP value for consistent x-axis
    max_shap = max(
        climate_data['mean_abs_shap'].max() if len(climate_data) > 0 else 0,
        soil_data['mean_abs_shap'].max() if len(soil_data) > 0 else 0
    )
    
    # ========================================================================
    # PLOT CLIMATE SECTION (TOP)
    # ========================================================================
    
    if len(climate_data) > 0:
        y_pos_climate = np.arange(len(climate_data))
        
        # Plot bars
        bars_climate = ax_climate.barh(y_pos_climate, climate_data['mean_abs_shap'], 
                                       height=0.5,
                                       color='#6C757D',
                                       alpha=0.75,
                                       edgecolor='white',
                                       linewidth=0.5)
        
        # Add temporal change annotations (ALL are significant by construction)
        annotation_x = max_shap * 1.08
        
        for i, (idx, row) in enumerate(climate_data.iterrows()):
            pct_change = row['percent_change']
            p_val = row['p_value']
            
            # Significance stars
            if p_val < 0.001:
                stars = '***'
            elif p_val < 0.01:
                stars = '**'
            elif p_val < 0.05:
                stars = '*'
            else:
                stars = ''
            
            change_text = f"{pct_change:+.1f}%{stars}"
            
            # Color based on direction
            if pct_change > 0:
                text_color = TEMPORAL_COLORS['increase']
            else:
                text_color = TEMPORAL_COLORS['decrease']
            
            ax_climate.text(annotation_x, i, change_text,
                           va='center', ha='left',
                           fontsize=8,
                           color=text_color,
                           weight='bold')
        
        # Format y-axis
        feature_labels_climate = [format_feature_label(f) for f in climate_data['feature']]
        ax_climate.set_yticks(y_pos_climate)
        ax_climate.set_yticklabels(feature_labels_climate, fontsize=9, weight='bold')
        
        # X-axis for climate
        ax_climate.set_xlim(0, max_shap * 1.15)
        ax_climate.set_xlabel('Mean |SHAP value| (feature importance)', 
                             fontsize=10, weight='bold')
        ax_climate.tick_params(axis='x', which='major', labelsize=9, width=1, labelcolor='black')
        
        # Make x-axis tick labels bold
        for label in ax_climate.get_xticklabels():
            label.set_weight('bold')
        
        # Grid
        ax_climate.xaxis.grid(True, alpha=0.3, linewidth=0.5, linestyle='--', color='gray')
        ax_climate.set_axisbelow(True)
        
        # Spines
        ax_climate.spines['left'].set_linewidth(1)
        ax_climate.spines['bottom'].set_linewidth(1)
        ax_climate.spines['top'].set_visible(False)
        ax_climate.spines['right'].set_visible(False)
    else:
        # Empty climate section
        ax_climate.text(0.5, 0.5, 'No significant climate features',
                       transform=ax_climate.transAxes,
                       ha='center', va='center',
                       fontsize=10, style='italic', color='gray')
        ax_climate.set_xlim(0, 1)
        ax_climate.set_ylim(0, 1)
        ax_climate.axis('off')
    
    # ========================================================================
    # PLOT SOIL SECTION (BOTTOM)
    # ========================================================================
    
    if len(soil_data) > 0:
        y_pos_soil = np.arange(len(soil_data))
        
        # Plot bars
        bars_soil = ax_soil.barh(y_pos_soil, soil_data['mean_abs_shap'], 
                                height=0.5,
                                color='#6C757D',
                                alpha=0.75,
                                edgecolor='white',
                                linewidth=0.5)
        
        # Add temporal change annotations (ALL are significant by construction)
        annotation_x = max_shap * 1.08
        
        for i, (idx, row) in enumerate(soil_data.iterrows()):
            pct_change = row['percent_change']
            p_val = row['p_value']
            
            # Significance stars
            if p_val < 0.001:
                stars = '***'
            elif p_val < 0.01:
                stars = '**'
            elif p_val < 0.05:
                stars = '*'
            else:
                stars = ''
            
            change_text = f"{pct_change:+.1f}%{stars}"
            
            # Color based on direction
            if pct_change > 0:
                text_color = TEMPORAL_COLORS['increase']
            else:
                text_color = TEMPORAL_COLORS['decrease']
            
            ax_soil.text(annotation_x, i, change_text,
                        va='center', ha='left',
                        fontsize=8,
                        color=text_color,
                        weight='bold')
        
        # Format y-axis
        feature_labels_soil = [format_feature_label(f) for f in soil_data['feature']]
        ax_soil.set_yticks(y_pos_soil)
        ax_soil.set_yticklabels(feature_labels_soil, fontsize=9, weight='bold')
        
        # X-axis for soil
        ax_soil.set_xlim(0, max_shap * 1.15)
        ax_soil.set_xlabel('Mean |SHAP value| (feature importance)', 
                          fontsize=10, weight='bold')
        ax_soil.tick_params(axis='x', which='major', labelsize=9, width=1, labelcolor='black')
        
        # Make x-axis tick labels bold
        for label in ax_soil.get_xticklabels():
            label.set_weight('bold')
        
        # Grid
        ax_soil.xaxis.grid(True, alpha=0.3, linewidth=0.5, linestyle='--', color='gray')
        ax_soil.set_axisbelow(True)
        
        # Spines
        ax_soil.spines['left'].set_linewidth(1)
        ax_soil.spines['bottom'].set_linewidth(1)
        ax_soil.spines['top'].set_visible(False)
        ax_soil.spines['right'].set_visible(False)
    else:
        # Empty soil section
        ax_soil.text(0.5, 0.5, 'No significant soil features',
                    transform=ax_soil.transAxes,
                    ha='center', va='center',
                    fontsize=10, style='italic', color='gray')
        ax_soil.set_xlim(0, 1)
        ax_soil.set_ylim(0, 1)
        ax_soil.axis('off')
    
    # ========================================================================
    # INTELLIGENT POSITIONING OF CATEGORY LABELS AND LEGENDS
    # ========================================================================
    
    # Force rendering to calculate actual label sizes
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    
    # Calculate category label positions to avoid overlap with y-axis labels
    category_x_positions = {}
    
    for ax_name, ax in [('climate', ax_climate), ('soil', ax_soil)]:
        if ax_name == 'climate' and len(climate_data) == 0:
            continue
        if ax_name == 'soil' and len(soil_data) == 0:
            continue
            
        max_label_width = 0
        for label in ax.get_yticklabels():
            bbox = label.get_window_extent(renderer=renderer)
            bbox_axes = bbox.transformed(ax.transAxes.inverted())
            label_left_extent = abs(bbox_axes.x0)
            max_label_width = max(max_label_width, label_left_extent)
        
        # Add safety margin to ensure no overlap
        safety_margin = 0.02  # 2% extra space
        category_x_positions[ax_name] = -(max_label_width + safety_margin)
        
        print(f"   {ax_name.capitalize()} category label position: {category_x_positions[ax_name]:.4f}")
    
    # ALIGN BOTH LABELS: Use the farthest (most negative) position for both
    if len(category_x_positions) > 0:
        aligned_position = min(category_x_positions.values())  # Most negative = farthest left
        print(f"   Aligned position (both labels): {aligned_position:.4f}")
    else:
        aligned_position = -0.12  # Fallback
    
    # Add category labels with aligned positioning
    if len(climate_data) > 0:
        ax_climate.text(aligned_position, 0.5, 'Climate factors',
                       transform=ax_climate.transAxes,
                       ha='right', va='center',
                       fontsize=9, weight='bold', 
                       color='black', style='italic',
                       rotation=90)
    
    if len(soil_data) > 0:
        ax_soil.text(aligned_position, 0.5, 'Soil factors',
                    transform=ax_soil.transAxes,
                    ha='right', va='center',
                    fontsize=9, weight='bold', 
                    color='black', style='italic',
                    rotation=90)
    
    # ========================================================================
    # ADD LEGENDS AND TEMPORAL CHANGE LABELS TO BOTH PLOTS
    # ========================================================================
    
    # Create SIMPLIFIED legend
    from matplotlib.lines import Line2D
    
    legend_elements = [
        Line2D([0], [0], color='black', marker='', linewidth=0,
               label='Signif.: *** p<0.001, ** p<0.01, * p<0.05')
    ]
    
    # Add legend to climate plot if it has data
    if len(climate_data) > 0:
        ax_climate.legend(handles=legend_elements, 
                         loc='lower right',
                         bbox_to_anchor=(0.85, 0),
                         fontsize=7.5,
                         frameon=False)
        
        # Add temporal change label on right - positioned more inside to avoid overlap
        ax_climate.text(0.88, 0.5, 'Temporal change',
                       transform=ax_climate.transAxes,
                       rotation=270,
                       va='center',
                       ha='center',
                       fontsize=9,
                       weight='bold',
                       color='gray')
    
    # Add legend to soil plot if it has data
    if len(soil_data) > 0:
        ax_soil.legend(handles=legend_elements, 
                      loc='lower right',
                      bbox_to_anchor=(0.85, 0),
                      fontsize=7.5,
                      frameon=False)
        
        # Add temporal change label on right - positioned more inside to avoid overlap
        ax_soil.text(0.88, 0.5, 'Temporal change',
                    transform=ax_soil.transAxes,
                    rotation=270,
                    va='center',
                    ha='center',
                    fontsize=9,
                    weight='bold',
                    color='gray')
    
    # ========================================================================
    # OVERALL FIGURE ELEMENTS
    # ========================================================================
    
    # Title with response variable change
    if response_p_value < 0.001:
        response_sig = ' ***'
    elif response_p_value < 0.01:
        response_sig = ' **'
    elif response_p_value < 0.05:
        response_sig = ' *'
    else:
        response_sig = ''
    
    direction_symbol = '↓' if percent_change < 0 else '↑'
    title = (f"{response_variable.replace('_', ' ').title()}: "
            f"{direction_symbol} {abs(percent_change):.1f}%{response_sig} "
            f"({first_year_val:.1f} → {last_year_val:.1f})")
    
    if 'tc_per_ha' in response_variable:
        title += " t/ha"
    elif 'percent' in response_variable or 'brix' in response_variable or 'pol' in response_variable:
        title += " %"
    
    fig.suptitle(title, fontsize=11, weight='bold', y=0.98)
    
    # ========================================================================
    # SAVE FIGURE
    # ========================================================================
    
    output_file = os.path.join(decline_dir, 
                              f'two_section_sigchange_{response_variable}_{model_name}.png')
    
    try:
        plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"\n✓ PNG saved: {output_file}")
        
        output_pdf = output_file.replace('.png', '.pdf')
        plt.savefig(output_pdf, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"✓ PDF saved: {output_pdf}")
        
        output_hires = output_file.replace('.png', '_hires.png')
        plt.savefig(output_hires, dpi=600, bbox_inches='tight', facecolor='white')
        print(f"✓ High-res PNG saved: {output_hires}")
        
        plt.close()
        
        summary_info['output_file'] = output_file
        summary_info['status'] = 'success'
        
        return summary_info
        
    except Exception as e:
        print(f"Error saving figure: {e}")
        plt.close()
        summary_info['status'] = 'failed'
        summary_info['error'] = str(e)
        return summary_info

# ============================================================================
# BATCH GENERATION
# ============================================================================

def create_all_two_section_figures(response_variables, model_name, base_model_dir, 
                                   data_file, n_features_per_category=5,
                                   p_threshold=0.05):
    """Create two-section figures for all response variables with detailed summary"""
    
    print("\n" + "="*80)
    print("CREATING TWO-SECTION FIGURES - SIGNIFICANT TEMPORAL CHANGES ONLY")
    print(f"Target: {n_features_per_category} significant features per category")
    print(f"P-value threshold: {p_threshold}")
    print("="*80)
    
    results = {}
    
    for response_var in response_variables:
        try:
            summary = create_two_section_significant_figure(
                response_variable=response_var,
                model_name=model_name,
                base_model_dir=base_model_dir,
                data_file=data_file,
                n_features_per_category=n_features_per_category,
                p_threshold=p_threshold
            )
            
            if summary is not None:
                results[response_var] = summary
                if summary.get('status') == 'success':
                    print(f"\n✓ Completed: {response_var}")
                else:
                    print(f"\n⚠ Completed with warnings: {response_var}")
            else:
                results[response_var] = {
                    'status': 'failed',
                    'error': 'Function returned None'
                }
                print(f"\n✗ Failed: {response_var}")
            
        except Exception as e:
            print(f"\n✗ Error with {response_var}: {e}")
            import traceback
            traceback.print_exc()
            results[response_var] = {
                'status': 'failed',
                'error': str(e)
            }
    
    # ========================================================================
    # FINAL COMPREHENSIVE SUMMARY
    # ========================================================================
    
    print("\n" + "="*80)
    print("COMPREHENSIVE SUMMARY - ALL RESPONSE VARIABLES")
    print("="*80)
    
    for response_var, result in results.items():
        print(f"\n{response_var.upper()}")
        print(f"{'─'*80}")
        
        if result.get('status') == 'success':
            print(f"Status: ✓ SUCCESS")
            print(f"Climate features: {result['climate_selected']}")
            print(f"Soil features: {result['soil_selected']}")
            print(f"Total features: {result['total_selected']}")
            
            if result['climate_features']:
                print(f"\nClimate features:")
                for feat in result['climate_features']:
                    stars = ''
                    if feat['p_value'] < 0.001:
                        stars = '***'
                    elif feat['p_value'] < 0.01:
                        stars = '**'
                    elif feat['p_value'] < 0.05:
                        stars = '*'
                    print(f"  {feat['rank']}. {feat['feature']} (Δ{feat['change']:+.1f}%{stars})")
            
            if result['soil_features']:
                print(f"\nSoil features:")
                for feat in result['soil_features']:
                    stars = ''
                    if feat['p_value'] < 0.001:
                        stars = '***'
                    elif feat['p_value'] < 0.01:
                        stars = '**'
                    elif feat['p_value'] < 0.05:
                        stars = '*'
                    print(f"  {feat['rank']}. {feat['feature']} (Δ{feat['change']:+.1f}%{stars})")
        else:
            print(f"Status: ✗ FAILED")
            if 'error' in result:
                print(f"Error: {result['error']}")
    
    print("\n" + "="*80)
    print("BATCH GENERATION COMPLETE")
    print("="*80)
    
    successful = sum(1 for r in results.values() if r.get('status') == 'success')
    print(f"\nSuccessful: {successful}/{len(response_variables)}")
    
    # Save summary to CSV
    summary_rows = []
    for response_var, result in results.items():
        if result.get('climate_features'):
            for feat in result['climate_features']:
                summary_rows.append({
                    'response_variable': response_var,
                    'category': 'climate',
                    'rank': feat['rank'],
                    'feature': feat['feature'],
                    'shap_importance': feat['shap'],
                    'percent_change': feat['change'],
                    'p_value': feat['p_value']
                })
        if result.get('soil_features'):
            for feat in result['soil_features']:
                summary_rows.append({
                    'response_variable': response_var,
                    'category': 'soil',
                    'rank': feat['rank'],
                    'feature': feat['feature'],
                    'shap_importance': feat['shap'],
                    'percent_change': feat['change'],
                    'p_value': feat['p_value']
                })
    
    if summary_rows:
        summary_df = pd.DataFrame(summary_rows)
        summary_file = os.path.join(base_model_dir, 'two_section_significant_features_summary.csv')
        summary_df.to_csv(summary_file, index=False)
        print(f"\n✓ Summary saved to: {summary_file}")
    
    return results

# ============================================================================
# EXAMPLE USAGE
# ============================================================================

if __name__ == "__main__":
    
    DATA_FILE = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis/final_dataset_regrouped_categorical.csv'
    BASE_MODEL_DIR = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis'
    
    RESPONSE_VARIABLES = ['brix_percent',
                          'pol_juice_percent',
                          'pol_cane_percent',
                          'purity_percent',
                          'glucose_cane_percent',
                          'hugot_sugar',
                          'tc_per_ha_actual',
                          'tc_per_ha_per_week',
                          'tc_per_ha_52_weeks',
                          'fiber_percent',
                          'total_recoverable_sugar_t_per_ha']
    
    MODEL_NAME = 'RandomForest'
    N_FEATURES_PER_CATEGORY = 6
    P_THRESHOLD = 0.05
    
    
    results = create_all_two_section_figures(
        response_variables=RESPONSE_VARIABLES,
        model_name=MODEL_NAME,
        base_model_dir=BASE_MODEL_DIR,
        data_file=DATA_FILE,
        n_features_per_category=N_FEATURES_PER_CATEGORY,
        p_threshold=P_THRESHOLD
    )
    
    print("\n" + "="*80)



    ╔════════════════════════════════════════════════════════════════╗
    ║  TWO-SECTION FIGURE GENERATOR - SIGNIFICANT CHANGES ONLY       ║
    ║                                                                ║
    ║  Features:                                                     ║
    ║  • Two distinct sections with spacing                          ║
    ║  • Separate x-axis for climate factors (top)                   ║
    ║  • Separate x-axis for soil factors (bottom)                   ║
    ║  • Shows exactly N significant features per category           ║
    ║  • All bars colored in gray (#6C757D)                          ║
    ║  • Detailed summary tables for each response variable          ║
    ║  • Comprehensive final summary across all variables            ║
    ║  • Exports summary to CSV for further analysis                 ║
    ╚════════════════════════════════════════════════════════════════╝
    

CREATING TWO-SECTION FIGURES - SIGNIFICANT TEMPORAL CHANGES ONLY
Targe

## 6. Model Interpretability: SHAP Analysis

This section performs SHAP (SHapley Additive exPlanations) analysis to interpret the trained machine learning models and quantify the contribution of each predictor variable to model predictions.

**Analysis Framework:**

**I. SHAP Value Calculation**
- Computes feature contributions for individual predictions
- Uses TreeExplainer for ensemble models (Random Forest, XGBoost)
- Uses LinearExplainer for linear models
- Applies KernelExplainer for complex architectures

**II. Feature Importance Visualization**
- **Bar plots**: Mean absolute SHAP values (global feature importance)
- **Beeswarm plots**: Distribution of SHAP values showing feature effects across all predictions
- Color-coded by feature value (red = high, blue = low)

**III. Cross-Model Comparison**
- Heatmap comparing feature importance across all trained models
- Identifies consistently important predictors
- Ranks top 10-15 features by average SHAP value

**Interpretation:** SHAP values reveal which environmental, soil, and management factors drive sugar content and yield predictions, enabling evidence-based agronomic recommendations.

**Output:** Individual SHAP plots per model, comparative heatmaps, and summary CSV files with feature importance rankings.SHAP ANALYSIS MODULE

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib
import os
import glob
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# BILINGUAL FEATURE NAME MAPPINGS
# ============================================================================

# Define feature groups with shortened names in both languages
FEATURE_MAPPINGS = {
    # Soil Properties - pH and Basic Chemistry
    'pH_complete': {
        'short_en': 'pH',
        'short_fr': 'pH',
        'full_en': 'pH',
        'full_fr': 'pH',
        'group': 'soil_chemistry'
    },
    
    # Exchangeable Elements
    'Exchangeable_Calcium_meq_100g_complete': {
        'short_en': 'Ca exch. (meq/100g)',
        'short_fr': 'Ca éch. (meq/100g)',
        'full_en': 'Exchangeable Calcium (meq/100g)',
        'full_fr': 'Calcium échangeable (meq/100g)',
        'group': 'exchangeable'
    },
    'Exchangeable_Potassium_meq_100g_complete': {
        'short_en': 'K exch. (meq/100g)',
        'short_fr': 'K éch. (meq/100g)',
        'full_en': 'Exchangeable Potassium (meq/100g)',
        'full_fr': 'Potassium échangeable (meq/100g)',
        'group': 'exchangeable'
    },
    'Exchangeable_Sodium_meq_100g_complete': {
        'short_en': 'Na exch. (meq/100g)',
        'short_fr': 'Na éch. (meq/100g)',
        'full_en': 'Exchangeable Sodium (meq/100g)',
        'full_fr': 'Sodium échangeable (meq/100g)',
        'group': 'exchangeable'
    },
    
    # Nutrients
    'Available_P2O5_mg_kg_complete': {
        'short_en': 'P₂O₅ avail. (mg/kg)',
        'short_fr': 'P₂O₅ disp. (mg/kg)',
        'full_en': 'Available P₂O₅ (mg/kg)',
        'full_fr': 'P₂O₅ disponible (mg/kg)',
        'group': 'nutrients'
    },
    'Total_Nitrogen_per_mille_complete': {
        'short_en': 'N total (‰)',
        'short_fr': 'N total (‰)',
        'full_en': 'Total Nitrogen (‰)',
        'full_fr': 'Azote total (‰)',
        'group': 'nutrients'
    },
    
    # Electrical Properties
    'Electrical_Conductivity_mmho_cm_complete': {
        'short_en': 'EC (mmho/cm)',
        'short_fr': 'CE (mmho/cm)',
        'full_en': 'Electrical Conductivity (mmho/cm)',
        'full_fr': 'Conductivité électrique (mmho/cm)',
        'group': 'soil_chemistry'
    },
    
    # Ratios
    'Mg_K_Ratio_complete': {
        'short_en': 'Mg/K ratio',
        'short_fr': 'Rapport Mg/K',
        'full_en': 'Magnesium/Potassium Ratio',
        'full_fr': 'Rapport Magnésium/Potassium',
        'group': 'ratios'
    },
    
    # Soil Texture - Granulometry
    'Coarse_Silt_percent_complete': {
        'short_en': 'Coarse silt (%)',
        'short_fr': 'Limon grossier (%)',
        'full_en': 'Coarse Silt (%)',
        'full_fr': 'Limon grossier (%)',
        'group': 'granulometry'
    },
    'Coarse_Sand_percent_complete': {
        'short_en': 'Coarse sand (%)',
        'short_fr': 'Sable grossier (%)',
        'full_en': 'Coarse Sand (%)',
        'full_fr': 'Sable grossier (%)',
        'group': 'granulometry'
    },
    
    # Climate Variables - Rainfall
    'growing_season_rainfall_mm': {
        'short_en': 'Growing season rain (mm)',
        'short_fr': 'Pluie saison (mm)',
        'full_en': 'Growing Season Rainfall (mm)',
        'full_fr': 'Pluviométrie saison culturale (mm)',
        'group': 'rainfall'
    },
    'CDD': {
        'short_en': 'CDD',
        'short_fr': 'CDD',
        'full_en': 'Consecutive Dry Days',
        'full_fr': 'Jours secs consécutifs',
        'group': 'rainfall_indices'
    },
    'SDII': {
        'short_en': 'SDII',
        'short_fr': 'SDII',
        'full_en': 'Simple Daily Intensity Index',
        'full_fr': 'Indice intensité quotidienne',
        'group': 'rainfall_indices'
    },
    'CWD': {
        'short_en': 'CWD',
        'short_fr': 'CWD',
        'full_en': 'Consecutive Wet Days',
        'full_fr': 'Jours humides consécutifs',
        'group': 'rainfall_indices'
    },
    'Rx1': {
        'short_en': 'Rx1',
        'short_fr': 'Rx1',
        'full_en': 'Max 1-day Precipitation',
        'full_fr': 'Précipitation max 1 jour',
        'group': 'rainfall_indices'
    },
    'R20': {
        'short_en': 'R20',
        'short_fr': 'R20',
        'full_en': 'Heavy Precipitation Days (>20mm)',
        'full_fr': 'Jours précipitations fortes (>20mm)',
        'group': 'rainfall_indices'
    },
    
    # Temperature Variables
    'temp_mini_mean': {
        'short_en': 'T° min (°C)',
        'short_fr': 'T° min (°C)',
        'full_en': 'Mean Minimum Temperature (°C)',
        'full_fr': 'Température minimale moyenne (°C)',
        'group': 'temperature'
    },
    'temp_maxi_mean': {
        'short_en': 'T° max (°C)',
        'short_fr': 'T° max (°C)',
        'full_en': 'Mean Maximum Temperature (°C)',
        'full_fr': 'Température maximale moyenne (°C)',
        'group': 'temperature'
    },
    'TNn': {
        'short_en': 'TNn',
        'short_fr': 'TNn',
        'full_en': 'Min Temperature of Coldest Night',
        'full_fr': 'Température min nuit la plus froide',
        'group': 'temperature_indices'
    },
    'TNx': {
        'short_en': 'TNx',
        'short_fr': 'TNx',
        'full_en': 'Max Temperature of Warmest Night',
        'full_fr': 'Température max nuit la plus chaude',
        'group': 'temperature_indices'
    },
    
    # Humidity
    'h_mini_mean': {
        'short_en': 'RH min (%)',
        'short_fr': 'HR min (%)',
        'full_en': 'Mean Minimum Relative Humidity (%)',
        'full_fr': 'Humidité relative minimale moyenne (%)',
        'group': 'humidity'
    },
    'h_maxi_mean': {
        'short_en': 'RH max (%)',
        'short_fr': 'HR max (%)',
        'full_en': 'Mean Maximum Relative Humidity (%)',
        'full_fr': 'Humidité relative maximale moyenne (%)',
        'group': 'humidity'
    },
    
    # Soil Temperature and Radiation
    'soil_temp_6h_mean': {
        'short_en': 'Soil T° 6h (°C)',
        'short_fr': 'T° sol 6h (°C)',
        'full_en': 'Mean Soil Temperature at 6h (°C)',
        'full_fr': 'Température du sol moyenne à 6h (°C)',
        'group': 'soil_climate'
    },
    'radiation_ERA_total': {
        'short_en': 'Solar rad. (cal/cm²)',
        'short_fr': 'Rayonnement (cal/cm²)',
        'full_en': 'Total Solar Radiation (cal/cm²)',
        'full_fr': 'Rayonnement solaire total (cal/cm²)',
        'group': 'radiation'
    },
    
    # Categorical variables (encoded versions)
    'variety_encoded': {
        'short_en': 'Variety',
        'short_fr': 'Variété',
        'full_en': 'Sugarcane Variety',
        'full_fr': 'Variété de canne',
        'group': 'categorical'
    },
    'crop_cycle_encoded': {
        'short_en': 'Crop cycle',
        'short_fr': 'Cycle cultural',
        'full_en': 'Crop Cycle',
        'full_fr': 'Cycle cultural',
        'group': 'categorical'
    },
    'irrigation_type_encoded': {
        'short_en': 'Irrigation',
        'short_fr': 'Irrigation',
        'full_en': 'Irrigation Type',
        'full_fr': 'Type d\'irrigation',
        'group': 'categorical'
    },
    'harvest_mode_encoded': {
        'short_en': 'Harvest mode',
        'short_fr': 'Mode de récolte',
        'full_en': 'Harvest Mode',
        'full_fr': 'Mode de récolte',
        'group': 'categorical'
    }
}

# Response variable translations
RESPONSE_VARIABLE_LABELS = {
    'brix_percent': {
        'en': 'Brix (%)',
        'fr': 'Brix (%)'
    },
    'pol_juice_percent': {
        'en': 'Pol juice (%)',
        'fr': 'Pol jus (%)'
    },
    'pol_cane_percent': {
        'en': 'Pol cane (%)',
        'fr': 'Pol canne (%)'
    },
    'purity_percent': {
        'en': 'Purity (%)',
        'fr': 'Pureté (%)'
    },
    'glucose_cane_percent': {
        'en': 'Glucose (%)',
        'fr': 'Glucose (%)'
    },
    'hugot_sugar': {
        'en': 'Hugot sugar (%)',
        'fr': 'Sucre Hugot (%)'
    },
    'tc_per_ha_actual': {
        'en': 'Cane yield (t/ha)',
        'fr': 'Rendement en canne (t/ha)'
    },
    'tc_per_ha_per_week': {
        'en': 'Cane yield per week (t/ha/week)',
        'fr': 'Rendement canne par semaine (t/ha/sem)'
    },
    'tc_per_ha_52_weeks': {
        'en': 'Cane yield 52 weeks (t/ha)',
        'fr': 'Rendement canne 52 sem (t/ha)'
    },
    'fiber_percent': {
        'en': 'Fiber in cane (%)',
        'fr': 'Fibre dans la canne (%)'
    },
    'total_recoverable_sugar_t_per_ha': {
        'en': 'Total Recov. Sugar (t ha⁻¹)',
        'fr': 'Sucre récup. tot (t ha⁻¹)'
    }
}

# Translation dictionary for plot elements
PLOT_TRANSLATIONS = {
    'en': {
        'feature_importance_title': 'Feature Importance',
        'shap_distribution_title': 'SHAP Value Distribution',
        'mean_shap_xlabel': 'mean(|SHAP value|)\n(average impact on model output magnitude)',
        'shap_value_xlabel': 'SHAP value\n(impact on model output)',
        'feature_value_label': 'Feature value',
        'comparison_title': 'Feature Importance Comparison Across All Models',
        'models_label': 'Models',
        'features_label': 'Features (Top 20)',
        'top_features_title': 'Top 10 Most Important Features (Average Across All Models)',
        'avg_shap_xlabel': 'Average Mean |SHAP value| Across All Models',
        'mean_abs_shap': 'Mean |SHAP value|',
        'model': 'Model',
        'feature': 'Feature',
        'rank': 'Rank',
        'importance': 'Importance'
    },
    'fr': {
        'feature_importance_title': 'Importance des Variables',
        'shap_distribution_title': 'Distribution des Valeurs SHAP',
        'mean_shap_xlabel': 'moyenne(|valeur SHAP|)\n(impact moyen sur la magnitude de la sortie)',
        'shap_value_xlabel': 'Valeur SHAP\n(impact sur la sortie du modèle)',
        'feature_value_label': 'Valeur de la variable',
        'comparison_title': 'Comparaison de l\'Importance des Variables pour Tous les Modèles',
        'models_label': 'Modèles',
        'features_label': 'Variables (Top 20)',
        'top_features_title': 'Top 10 des Variables les Plus Importantes (Moyenne sur Tous les Modèles)',
        'avg_shap_xlabel': 'Moyenne des |valeurs SHAP| moyennes sur tous les modèles',
        'mean_abs_shap': 'Moyenne |valeur SHAP|',
        'model': 'Modèle',
        'feature': 'Variable',
        'rank': 'Rang',
        'importance': 'Importance'
    }
}

def get_feature_name(original_name, language='en'):
    """Get translated and shortened feature name"""
    if original_name in FEATURE_MAPPINGS:
        return FEATURE_MAPPINGS[original_name][f'short_{language}']
    return original_name

def translate_feature_list(feature_names, language='en'):
    """Translate a list of feature names"""
    return [get_feature_name(name, language) for name in feature_names]

# ============================================================================
# DEFINE AVAILABLE RESPONSE VARIABLES
# ============================================================================

SUGAR_CONTENT_METRICS = {
    "brix_percent": "Brix (%)",
    "pol_juice_percent": "Pol juice (%)",
    "pol_cane_percent": "Pol cane (%)",
    "purity_percent": "Purity (%)",
    "glucose_cane_percent": "Glucose (%)",
    "hugot_sugar": "Hugot sugar (%)",
    "total_recoverable_sugar_t_per_ha": "TRS (t ha⁻¹)"
}

YIELD_PARAMETERS = {
    "tc_per_ha_actual": "Cane yield (t ha⁻¹)",
    "tc_per_ha_per_week": "Cane yield per week (t ha⁻¹/week)",
    "tc_per_ha_52_weeks": "Cane yield 52 weeks (t ha⁻¹)",
    "fiber_percent": "Fiber in Cane (%)",
}

ALL_RESPONSE_VARIABLES = {**SUGAR_CONTENT_METRICS, **YIELD_PARAMETERS}

# ============================================================================
# PLOT CONFIGURATION
# ============================================================================

def set_plot_style():
    """Set plot style"""
    plt.rcParams.update({
        'font.size': 12,
        'font.family': 'DejaVu Sans',
        'axes.labelsize': 12,
        'axes.titlesize': 14,
        'xtick.labelsize': 11,
        'ytick.labelsize': 11,
        'legend.fontsize': 11,
        'figure.titlesize': 16,
        'axes.spines.top': False,
        'axes.spines.right': False,
        'axes.grid': False,
        'axes.axisbelow': True,
        'grid.alpha': 0.3,
        'axes.edgecolor': 'gray',
        'text.color': 'gray',
        'axes.labelcolor': 'gray',
        'xtick.color': 'gray',
        'ytick.color': 'gray'
    })

set_plot_style()

# ============================================================================
# DATA LOADING (WITH RESPONSE VARIABLE PARAMETER)
# ============================================================================

def load_and_prepare_data(file_path, response_variable, custom_predictors=None, categorical_vars=None):
    """Load and prepare the original data for SHAP analysis with flexible response variable"""
    print(f"Loading and preparing data for SHAP analysis...")
    print(f"Response variable: {response_variable} - {ALL_RESPONSE_VARIABLES.get(response_variable, response_variable)}")

    df = pd.read_csv(file_path)
    
    # Check if response variable exists
    if response_variable not in df.columns:
        raise ValueError(f"Response variable '{response_variable}' not found in dataset!")
    
    # Drop ALL other response variables (keep only target)
    all_response_cols = list(ALL_RESPONSE_VARIABLES.keys())
    cols_to_drop = [col for col in all_response_cols 
                    if col != response_variable and col in df.columns]
    
    if cols_to_drop:
        print(f"Dropping other response variables: {cols_to_drop}")
        df = df.drop(columns=cols_to_drop)

    if custom_predictors is not None:
        print(f"Using custom predictors: {custom_predictors}")

        year_column = None
        for col in ['campaign_year', 'Harvest_Year', 'year']:
            if col in df.columns:
                year_column = col
                break

        columns_to_keep = [response_variable] + custom_predictors
        if year_column:
            columns_to_keep.append(year_column)
        if categorical_vars:
            columns_to_keep.extend(categorical_vars)

        missing_predictors = [col for col in custom_predictors if col not in df.columns]
        if missing_predictors:
            raise ValueError(f"Custom predictors not found in dataset: {missing_predictors}")
        
        if categorical_vars:
            missing_categorical = [col for col in categorical_vars if col not in df.columns]
            if missing_categorical:
                raise ValueError(f"Categorical variables not found: {missing_categorical}")

        df = df[columns_to_keep]
        print(f"Using {len(custom_predictors)} custom predictors")

    df = df.dropna(subset=[response_variable])

    year_column = None
    for col in ['campaign_year', 'Harvest_Year', 'year']:
        if col in df.columns:
            year_column = col
            break

    encoders = {}
    
    if categorical_vars:
        print(f"\nEncoding categorical variables for SHAP: {categorical_vars}")
        for cat_var in categorical_vars:
            if cat_var in df.columns:
                cat_encoder = LabelEncoder()
                df[f'{cat_var}_encoded'] = cat_encoder.fit_transform(df[cat_var].astype(str))
                encoders[cat_var] = cat_encoder
                print(f"   {cat_var}: {len(cat_encoder.classes_)} unique values")

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    for col in numeric_cols:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].median())

    if custom_predictors is not None:
        available_predictors = [col for col in custom_predictors if col in df.columns]
        features = available_predictors.copy()
        
        if categorical_vars:
            for cat_var in categorical_vars:
                encoded_col = f'{cat_var}_encoded'
                if encoded_col in df.columns:
                    features.append(encoded_col)
            print(f"Added encoded categorical variables: {[f'{cv}_encoded' for cv in categorical_vars]}")
    else:
        exclude_cols = [response_variable]
        if year_column:
            exclude_cols.extend([year_column, f'{year_column}_encoded'])
        if categorical_vars:
            exclude_cols.extend(categorical_vars)
        exclude_cols.extend([col for col in all_response_cols if col != response_variable])

        features = [col for col in numeric_cols if col not in exclude_cols]

    target = response_variable

    print(f"Features for SHAP analysis: {len(features)} features")
    print(f"Features: {features[:10]}{'...' if len(features) > 10 else ''}")

    X = df[features].copy()
    y = df[target].astype(float)

    if year_column and year_column in df.columns:
        try:
            year_strata = df[year_column].astype(str)
            X_temp, X_test, y_temp, y_test = train_test_split(
                X, y, test_size=0.2, random_state=42, stratify=year_strata
            )
            X_train, X_val, y_train, y_val = train_test_split(
                X_temp, y_temp, test_size=0.25, random_state=42
            )
        except:
            X_temp, X_test, y_temp, y_test = train_test_split(
                X, y, test_size=0.2, random_state=42
            )
            X_train, X_val, y_train, y_val = train_test_split(
                X_temp, y_temp, test_size=0.25, random_state=42
            )
    else:
        X_temp, X_test, y_temp, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )
        X_train, X_val, y_train, y_val = train_test_split(
            X_temp, y_temp, test_size=0.25, random_state=42
        )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print(f"Data split: Train={len(X_train_scaled)}, Val={len(X_val)}, Test={len(X_test_scaled)}")

    return X_train_scaled, X_test_scaled, features, scaler, X_train, X_test, encoders

def load_saved_models(models_dir, response_variable):
    """Load all saved models from the directory for specific response variable"""
    print(f"Loading models from {models_dir}...")
    
    if not os.path.exists(models_dir):
        print(f"ERROR: Models directory does not exist: {models_dir}")
        return {}
    
    all_files = os.listdir(models_dir)
    print(f"Found {len(all_files)} total files in directory")
    
    models = {}
    
    patterns = [
        f"*{response_variable}.pkl",
        f"*_{response_variable}.pkl",
        f"*{response_variable}_*.pkl",
    ]
    
    model_files = []
    for pattern in patterns:
        found = glob.glob(os.path.join(models_dir, pattern))
        model_files.extend(found)
    
    model_files = list(set(model_files))
    
    if not model_files:
        print(f"No files matched patterns for '{response_variable}'")
        print(f"Trying to load all .pkl files in directory...")
        model_files = glob.glob(os.path.join(models_dir, "*.pkl"))
    
    if not model_files:
        print(f"No .pkl files found in {models_dir}")
        print(f"Files in directory: {all_files[:10]}")
        return {}
    
    print(f"Found {len(model_files)} potential model files")

    for model_file in model_files:
        try:
            base_name = os.path.basename(model_file)
            if any(skip in base_name.lower() for skip in ['best_model_info', 'encoders_info', 'scaler', 'feature']):
                continue

            model_name = base_name.replace('.pkl', '')
            for resp_var in ALL_RESPONSE_VARIABLES.keys():
                model_name = model_name.replace(f'_{resp_var}', '').replace(resp_var, '')
            model_name = model_name.replace('__', '_').strip('_').replace('_', ' ')
            
            model_data = joblib.load(model_file)
            
            if isinstance(model_data, dict):
                if 'model' in model_data:
                    models[model_name] = model_data['model']
                elif 'best_model' in model_data:
                    models[model_name] = model_data['best_model']
                elif 'estimator' in model_data:
                    models[model_name] = model_data['estimator']
                else:
                    model_key = [k for k in model_data.keys() if 'model' in k.lower()]
                    if model_key:
                        models[model_name] = model_data[model_key[0]]
                    else:
                        print(f"  Skipping {base_name}: dict structure not recognized (keys: {list(model_data.keys())})")
                        continue
            else:
                models[model_name] = model_data
                
            print(f"  ✓ Loaded: {model_name}")
            
        except Exception as e:
            print(f"  ✗ Error loading {os.path.basename(model_file)}: {str(e)}")
            continue

    if not models:
        print(f"\nWARNING: No models successfully loaded!")
        print(f"Directory: {models_dir}")
        print(f"Files checked: {len(model_files)}")
        print(f"\nFirst few files in directory:")
        for f in all_files[:5]:
            print(f"  - {f}")
    
    return models

def get_shap_explainer(model, X_background, model_name):
    """Get appropriate SHAP explainer for the model"""

    tree_models = ['RandomForest', 'ExtraTrees', 'GradientBoosting', 'XGBoost', 'LightGBM', 'CatBoost']
    if any(tree_model.lower() in model_name.lower() for tree_model in tree_models):
        try:
            explainer = shap.TreeExplainer(model)
            return explainer, 'tree'
        except Exception as e:
            print(f"    TreeExplainer failed for {model_name}: {str(e)}")

    linear_models = ['LinearRegression', 'Ridge', 'Lasso', 'ElasticNet']
    if any(linear_model.lower() in model_name.lower() for linear_model in linear_models):
        try:
            explainer = shap.LinearExplainer(model, X_background)
            return explainer, 'linear'
        except Exception as e:
            print(f"    LinearExplainer failed for {model_name}: {str(e)}")

    ensemble_models = ['Voting', 'Weighted']
    if any(ensemble_model.lower() in model_name.lower() for ensemble_model in ensemble_models):
        print(f"    Using KernelExplainer for ensemble model: {model_name}")
        background_sample = shap.sample(X_background, min(50, len(X_background)))
        explainer = shap.KernelExplainer(model.predict, background_sample)
        return explainer, 'kernel'

    if 'mlp' in model_name.lower():
        print(f"    Using KernelExplainer for neural network: {model_name}")
        background_sample = shap.sample(X_background, min(50, len(X_background)))
        explainer = shap.KernelExplainer(model.predict, background_sample)
        return explainer, 'kernel'

    print(f"    Using KernelExplainer (default) for: {model_name}")
    background_sample = shap.sample(X_background, min(100, len(X_background)))
    explainer = shap.KernelExplainer(model.predict, background_sample)
    return explainer, 'kernel'

def calculate_shap_values(model, explainer, X_test, explainer_type, max_samples=500):
    """Calculate SHAP values with appropriate method"""

    if len(X_test) > max_samples:
        sample_indices = np.random.choice(len(X_test), max_samples, replace=False)
        X_test_sample = X_test[sample_indices]
    else:
        X_test_sample = X_test

    print(f"    Calculating SHAP values for {len(X_test_sample)} samples...")

    try:
        if explainer_type == 'tree':
            shap_values = explainer.shap_values(X_test_sample)
        elif explainer_type == 'linear':
            shap_values = explainer.shap_values(X_test_sample)
        else:
            shap_values = explainer.shap_values(X_test_sample, nsamples=100)

        print(f"    Successfully calculated SHAP values: shape {np.array(shap_values).shape}")
        return shap_values, X_test_sample

    except Exception as e:
        print(f"    Error calculating SHAP values: {str(e)}")
        raise

def create_shap_plots(shap_values, X_test_sample, feature_names, model_name, output_dir, 
                      response_variable, language='en'):
    """Create publication-ready SHAP plots in specified language"""

    print(f"    Creating {language.upper()} SHAP plots for {model_name}...")
    
    # Get translations
    trans = PLOT_TRANSLATIONS[language]
    
    # Translate feature names
    feature_names_translated = translate_feature_list(feature_names, language)

    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    feature_importance = pd.DataFrame({
        'feature': feature_names_translated,
        'feature_original': feature_names,
        'importance': mean_abs_shap
    }).sort_values('importance', ascending=False)

    max_features = min(15, len(feature_names))
    top_features = feature_importance.head(max_features)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

    top_features_sorted = top_features.sort_values('importance', ascending=True)
    bar_color = '#1f77b4'

    y_positions = np.arange(len(top_features_sorted))
    bars = ax1.barh(y_positions, top_features_sorted['importance'],
                    color=bar_color, alpha=0.8, height=0.7, edgecolor='white', linewidth=0.5)

    ax1.set_yticks(y_positions)
    ax1.set_yticklabels(top_features_sorted['feature'], fontsize=11, color='gray')
    ax1.set_xlabel(trans['mean_shap_xlabel'], fontsize=11, color='gray')
    ax1.set_title(trans['feature_importance_title'], fontsize=13, fontweight='bold', pad=15)

    ax1.grid(False)
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    ax1.spines['left'].set_color('gray')
    ax1.spines['bottom'].set_color('gray')

    ax1.margins(y=0.01, x=0.05)
    ax1.tick_params(axis='both', colors='gray', labelsize=11)
    ax1.tick_params(axis='x', rotation=0)

    for i, (idx, row) in enumerate(top_features_sorted.iterrows()):
        ax1.text(row['importance'] + max(top_features_sorted['importance']) * 0.01,
                i, f'{row["importance"]:.3f}',
                va='center', ha='left', fontsize=9, color='gray')

    # Create mapping for SHAP beeswarm plot
    feature_name_mapping = dict(zip(feature_names, feature_names_translated))
    
    explanation = shap.Explanation(values=shap_values,
                                   data=X_test_sample,
                                   feature_names=feature_names_translated)

    plt.sca(ax2)

    shap.plots.beeswarm(explanation, max_display=max_features, show=False,
                       color_bar_label=trans['feature_value_label'])

    ax2.set_title(trans['shap_distribution_title'], fontsize=13, fontweight='bold', pad=15)
    ax2.tick_params(axis='both', colors='gray', labelsize=11)
    ax2.tick_params(axis='y', labelleft=False)
    ax2.tick_params(axis='x', rotation=0)
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
    ax2.spines['left'].set_visible(False)
    ax2.spines['bottom'].set_color('gray')
    ax2.set_xlabel(trans['shap_value_xlabel'], fontsize=11, color='gray')
    ax2.grid(False)

    ax2.axvline(x=0, color='gray', linestyle='--', alpha=0.5, linewidth=1)

    ax1.set_ylim(-0.5, len(top_features_sorted) - 0.5)
    ax2.set_ylim(-0.5, len(top_features_sorted) - 0.5)

    plt.subplots_adjust(left=0.12, right=0.95, top=0.88, bottom=0.15, wspace=0.25)

    safe_model_name = model_name.replace(' ', '_').replace('/', '_')
    output_file = os.path.join(output_dir, f'shap_analysis_{safe_model_name}_{language}.png')
    plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white',
                edgecolor='none', pad_inches=0.2)

    pdf_output_file = os.path.join(output_dir, f'shap_analysis_{safe_model_name}_{language}.pdf')
    plt.savefig(pdf_output_file, bbox_inches='tight', facecolor='white',
                edgecolor='none', pad_inches=0.2)

    plt.close()

    print(f"    Saved {language.upper()} SHAP plots: {output_file} and {pdf_output_file}")

    return mean_abs_shap

def create_shap_summary_comparison(all_shap_results, output_dir, language='en'):
    """Create a publication-ready summary comparison of feature importance across all models"""

    if not all_shap_results:
        return

    print(f"Creating {language.upper()} SHAP summary comparison...")
    
    # Get translations
    trans = PLOT_TRANSLATIONS[language]

    feature_importance_df = pd.DataFrame()

    for model_name, results in all_shap_results.items():
        # Translate feature names
        feature_names_translated = translate_feature_list(results['feature_names'], language)
        
        importance_data = pd.DataFrame({
            'feature': feature_names_translated,
            'importance': results['mean_abs_shap'],
            'model': model_name
        })
        feature_importance_df = pd.concat([feature_importance_df, importance_data], ignore_index=True)

    pivot_df = feature_importance_df.pivot(index='feature', columns='model', values='importance')

    avg_importance = pivot_df.mean(axis=1).sort_values(ascending=False)
    pivot_df = pivot_df.loc[avg_importance.index]

    pivot_df = pivot_df.head(20)

    fig, ax = plt.subplots(figsize=(14, 10))

    cmap = plt.cm.YlOrRd

    sns.heatmap(pivot_df,
                annot=True,
                fmt='.3f',
                cmap=cmap,
                cbar_kws={'label': trans['mean_abs_shap'], 'shrink': 0.8},
                linewidths=0.5,
                linecolor='white',
                square=False,
                ax=ax)

    ax.set_title(trans['comparison_title'],
                fontsize=16, fontweight='bold', pad=20, color='#333333')
    ax.set_xlabel(trans['models_label'], fontsize=13, color='#333333')
    ax.set_ylabel(trans['features_label'], fontsize=13, color='#333333')

    plt.xticks(rotation=45, ha='right', fontsize=11)
    plt.yticks(rotation=0, fontsize=11)

    plt.tight_layout()

    comparison_file = os.path.join(output_dir, f'shap_feature_importance_comparison_{language}.png')
    plt.savefig(comparison_file, dpi=300, bbox_inches='tight', facecolor='white',
                edgecolor='none', pad_inches=0.2)

    pdf_comparison_file = os.path.join(output_dir, f'shap_feature_importance_comparison_{language}.pdf')
    plt.savefig(pdf_comparison_file, bbox_inches='tight', facecolor='white',
                edgecolor='none', pad_inches=0.2)

    plt.close()

    print(f"    Saved {language.upper()} comparison plots: {comparison_file} and {pdf_comparison_file}")

    create_top_features_summary(pivot_df, output_dir, language)

def create_top_features_summary(pivot_df, output_dir, language='en'):
    """Create a summary bar plot of top features across all models"""
    
    # Get translations
    trans = PLOT_TRANSLATIONS[language]

    avg_importance = pivot_df.mean(axis=1).sort_values(ascending=False).head(10)

    fig, ax = plt.subplots(figsize=(12, 8))

    avg_importance_sorted = avg_importance.sort_values(ascending=True)

    bars = ax.barh(range(len(avg_importance_sorted)), avg_importance_sorted.values,
                   color='#2E8B57', alpha=0.8, height=0.7,
                   edgecolor='white', linewidth=0.5)

    ax.set_yticks(range(len(avg_importance_sorted)))
    ax.set_yticklabels(avg_importance_sorted.index, fontsize=12)
    ax.set_xlabel(trans['avg_shap_xlabel'], fontsize=13, color='#333333')
    ax.set_title(trans['top_features_title'],
                fontsize=14, fontweight='bold', pad=15, color='#333333')

    ax.grid(axis='x', alpha=0.3, linestyle='-', linewidth=0.8)
    ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    for i, value in enumerate(avg_importance_sorted.values):
        ax.text(value + max(avg_importance_sorted.values) * 0.01, i,
                f'{value:.3f}', va='center', ha='left', fontsize=10)

    plt.tight_layout()

    summary_file = os.path.join(output_dir, f'shap_top_features_summary_{language}.png')
    plt.savefig(summary_file, dpi=300, bbox_inches='tight', facecolor='white',
                edgecolor='none', pad_inches=0.2)

    pdf_summary_file = os.path.join(output_dir, f'shap_top_features_summary_{language}.pdf')
    plt.savefig(pdf_summary_file, bbox_inches='tight', facecolor='white',
                edgecolor='none', pad_inches=0.2)

    plt.close()

    print(f"    Saved {language.upper()} top features summary: {summary_file} and {pdf_summary_file}")

def save_shap_results(all_shap_results, output_dir, response_variable):
    """Save detailed SHAP results to files with both English and French feature names"""

    print("Saving detailed SHAP results...")

    # English version
    summary_data_en = []
    for model_name, results in all_shap_results.items():
        feature_names_en = translate_feature_list(results['feature_names'], 'en')
        for i, feature in enumerate(feature_names_en):
            summary_data_en.append({
                'model': model_name,
                'feature': feature,
                'feature_original': results['feature_names'][i],
                'mean_abs_shap': results['mean_abs_shap'][i],
                'explainer_type': results['explainer_type']
            })

    summary_df_en = pd.DataFrame(summary_data_en)
    summary_file_en = os.path.join(output_dir, f'shap_feature_importance_summary_en.csv')
    summary_df_en.to_csv(summary_file_en, index=False)
    print(f"    Saved English summary: {summary_file_en}")

    # French version
    summary_data_fr = []
    for model_name, results in all_shap_results.items():
        feature_names_fr = translate_feature_list(results['feature_names'], 'fr')
        for i, feature in enumerate(feature_names_fr):
            summary_data_fr.append({
                'modèle': model_name,
                'variable': feature,
                'variable_originale': results['feature_names'][i],
                'moyenne_abs_shap': results['mean_abs_shap'][i],
                'type_explicateur': results['explainer_type']
            })

    summary_df_fr = pd.DataFrame(summary_data_fr)
    summary_file_fr = os.path.join(output_dir, f'shap_feature_importance_summary_fr.csv')
    summary_df_fr.to_csv(summary_file_fr, index=False)
    print(f"    Saved French summary: {summary_file_fr}")

    # Top features - English
    top_features_data_en = []
    for model_name, results in all_shap_results.items():
        feature_names_en = translate_feature_list(results['feature_names'], 'en')
        feature_importance = pd.DataFrame({
            'feature': feature_names_en,
            'feature_original': results['feature_names'],
            'importance': results['mean_abs_shap']
        }).sort_values('importance', ascending=False).head(10)

        for idx, (_, row) in enumerate(feature_importance.iterrows()):
            top_features_data_en.append({
                'model': model_name,
                'rank': idx + 1,
                'feature': row['feature'],
                'feature_original': row['feature_original'],
                'importance': row['importance']
            })

    top_features_df_en = pd.DataFrame(top_features_data_en)
    top_features_file_en = os.path.join(output_dir, f'shap_top_features_by_model_en.csv')
    top_features_df_en.to_csv(top_features_file_en, index=False)
    print(f"    Saved English top features: {top_features_file_en}")

    # Top features - French
    top_features_data_fr = []
    for model_name, results in all_shap_results.items():
        feature_names_fr = translate_feature_list(results['feature_names'], 'fr')
        feature_importance = pd.DataFrame({
            'variable': feature_names_fr,
            'variable_originale': results['feature_names'],
            'importance': results['mean_abs_shap']
        }).sort_values('importance', ascending=False).head(10)

        for idx, (_, row) in enumerate(feature_importance.iterrows()):
            top_features_data_fr.append({
                'modèle': model_name,
                'rang': idx + 1,
                'variable': row['variable'],
                'variable_originale': row['variable_originale'],
                'importance': row['importance']
            })

    top_features_df_fr = pd.DataFrame(top_features_data_fr)
    top_features_file_fr = os.path.join(output_dir, f'shap_top_features_by_model_fr.csv')
    top_features_df_fr.to_csv(top_features_file_fr, index=False)
    print(f"    Saved French top features: {top_features_file_fr}")

def main(data_file, response_variable, custom_predictors=None, categorical_vars=None, 
         base_output_dir=None, languages=['en', 'fr']):
    """Main function to run SHAP analysis for specified response variable in multiple languages"""
    
    # Validate response variable
    if response_variable not in ALL_RESPONSE_VARIABLES:
        raise ValueError(f"Response variable '{response_variable}' not recognized!\n"
                        f"Available options: {list(ALL_RESPONSE_VARIABLES.keys())}")
    
    # Set up directories based on response variable
    if base_output_dir is None:
        base_output_dir = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis'
    
    # Models directory (where models were saved)
    models_dir = os.path.join(base_output_dir, f'kfold_models_{response_variable}')
    if categorical_vars:
        models_dir += '_with_categorical'
    elif custom_predictors:
        models_dir += '_custom'
    models_dir = os.path.join(models_dir, 'models')
    
    # Output directory for SHAP results
    output_dir = os.path.join(base_output_dir, f'shap_analysis_{response_variable}')
    if categorical_vars:
        output_dir += '_with_categorical'
    elif custom_predictors:
        output_dir += '_custom'
    
    print("\n" + "="*80)
    print(f"BILINGUAL SHAP ANALYSIS FOR: {response_variable}")
    print(f"Display name: {ALL_RESPONSE_VARIABLES[response_variable]}")
    print(f"Languages: {', '.join([lang.upper() for lang in languages])}")
    print("="*80)
    print(f"Data file: {data_file}")
    print(f"Models directory: {models_dir}")
    print(f"Output directory: {output_dir}")
    if custom_predictors:
        print(f"Custom predictors: {len(custom_predictors)} features")
    if categorical_vars:
        print(f"Categorical variables: {categorical_vars}")
    print()

    all_shap_results = {}

    try:
        os.makedirs(output_dir, exist_ok=True)

        X_train_scaled, X_test_scaled, feature_names, scaler, X_train_raw, X_test_raw, encoders = load_and_prepare_data(
            data_file, response_variable, custom_predictors, categorical_vars
        )
        print(f"Data loaded: {len(X_train_scaled)} training samples, {len(X_test_scaled)} test samples")
        print(f"Features: {len(feature_names)}")

        models = load_saved_models(models_dir, response_variable)

        if not models:
            print(f"\n{'='*80}")
            print(f"WARNING: No models found for {response_variable}!")
            print(f"{'='*80}")
            print(f"Models directory: {models_dir}")
            print(f"\nPlease check:")
            print(f"1. Directory exists: {os.path.exists(models_dir)}")
            print(f"2. Model files are in the correct location")
            print(f"3. Response variable name matches exactly")
            
            return {
                'output_dir': output_dir,
                'n_models_analyzed': 0,
                'response_variable': response_variable,
                'status': 'no_models_found'
            }

        print(f"\nFound {len(models)} models for SHAP analysis\n")

        for model_name, model in models.items():
            print(f"Analyzing {model_name}...")

            try:
                explainer, explainer_type = get_shap_explainer(model, X_train_scaled, model_name)
                print(f"    Using {explainer_type} explainer")

                shap_values, X_test_sample = calculate_shap_values(
                    model, explainer, X_test_scaled, explainer_type
                )

                # Create plots in all specified languages
                mean_abs_shap = None
                for language in languages:
                    mean_abs_shap = create_shap_plots(
                        shap_values, X_test_sample, feature_names, model_name, output_dir,
                        response_variable, language
                    )

                all_shap_results[model_name] = {
                    'shap_values': shap_values,
                    'mean_abs_shap': mean_abs_shap,
                    'feature_names': feature_names,
                    'explainer_type': explainer_type
                }

                print(f"    ✓ Completed SHAP analysis for {model_name} in {len(languages)} language(s)\n")

            except Exception as e:
                print(f"    ✗ Error analyzing {model_name}: {str(e)}\n")
                continue

        if all_shap_results:
            # Create summary comparisons in all languages
            for language in languages:
                create_shap_summary_comparison(all_shap_results, output_dir, language)
            
            save_shap_results(all_shap_results, output_dir, response_variable)

            print(f"\n=== BILINGUAL SHAP ANALYSIS COMPLETE ===")
            print(f"Response variable: {response_variable} - {ALL_RESPONSE_VARIABLES[response_variable]}")
            print(f"Results saved to: {output_dir}")
            print(f"Individual plots: {len(all_shap_results)} models analyzed")
            print(f"Languages: {', '.join([lang.upper() for lang in languages])}")
            print("Summary comparison plots created in all languages")
            print("Both PNG and PDF versions saved for publication")

        else:
            print("No models were successfully analyzed!")

    except Exception as e:
        print(f"Error during SHAP analysis: {str(e)}")
        import traceback
        traceback.print_exc()
        return {
            'output_dir': output_dir,
            'n_models_analyzed': 0,
            'response_variable': response_variable,
            'status': 'error',
            'error': str(e)
        }
    
    return {
        'output_dir': output_dir,
        'n_models_analyzed': len(all_shap_results),
        'response_variable': response_variable,
        'languages': languages,
        'status': 'success'
    }

# ============================================================================
# USAGE EXAMPLES
# ============================================================================

if __name__ == "__main__":
    
    # Same predictors as in your original code
    custom_features = [
        'pH_complete', 'Exchangeable_Calcium_meq_100g_complete', 
        'Exchangeable_Potassium_meq_100g_complete', 'Exchangeable_Sodium_meq_100g_complete',
        'Available_P2O5_mg_kg_complete', 'Total_Nitrogen_per_mille_complete', 
        'Electrical_Conductivity_mmho_cm_complete', 'Mg_K_Ratio_complete',
        'Coarse_Silt_percent_complete', 'Coarse_Sand_percent_complete', 
        'growing_season_rainfall_mm', 'CDD', 'SDII',
        'temp_mini_mean', 'temp_maxi_mean', 'h_mini_mean', 'h_maxi_mean',
        'soil_temp_6h_mean', 'radiation_ERA_total', 'CWD', 'Rx1', 'R20', 'TNn', 'TNx'
    ]
    
    categorical_features = ['variety', 'crop_cycle', 'irrigation_type', 'harvest_mode']
    
    # File path
    data_file = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis/final_dataset_regrouped_categorical.csv'
    # Base output directory
    base_output = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis'
    
    print("="*80)
    print("BILINGUAL SHAP ANALYSIS PIPELINE - MULTIPLE RESPONSE VARIABLES")
    print("="*80)
    print("\nAvailable response variables:")
    print("\nSUGAR CONTENT METRICS:")
    for key, label in SUGAR_CONTENT_METRICS.items():
        print(f"  - {key}: {label}")
    print("\nYIELD PARAMETERS:")
    for key, label in YIELD_PARAMETERS.items():
        print(f"  - {key}: {label}")
    print("="*80 + "\n")
    
    # Track results for all analyses
    all_results = {}
    
    try:
        # Example 1: SHAP for brix_percent (both English and French)
        print("\n### RUNNING BILINGUAL SHAP ANALYSIS FOR: brix_percent ###\n")
        results_brix = main(
            data_file=data_file,
            response_variable='brix_percent',
            custom_predictors=custom_features,
            categorical_vars=categorical_features,
            base_output_dir=base_output,
            languages=['en', 'fr']  # Generate both languages
        )
        all_results['brix_percent'] = results_brix
        
        # Example 2: SHAP for hugot_sugar
        print("\n\n### RUNNING BILINGUAL SHAP ANALYSIS FOR: hugot_sugar ###\n")
        results_hugot = main(
            data_file=data_file,
            response_variable='hugot_sugar',
            custom_predictors=custom_features,
            categorical_vars=categorical_features,
            base_output_dir=base_output,
            languages=['en', 'fr']
        )
        all_results['hugot_sugar'] = results_hugot
        
        # Example 3: SHAP for tc_per_ha_actual
        print("\n\n### RUNNING BILINGUAL SHAP ANALYSIS FOR: tc_per_ha_actual ###\n")
        results_yield = main(
            data_file=data_file,
            response_variable='tc_per_ha_actual',
            custom_predictors=custom_features,
            categorical_vars=categorical_features,
            base_output_dir=base_output,
            languages=['en', 'fr']
        )
        all_results['tc_per_ha_actual'] = results_yield

        # Example 4: SHAP for purity_percent
        print("\n### RUNNING BILINGUAL SHAP ANALYSIS FOR: brix_percent ###\n")
        results_purity = main(
            data_file=data_file,
            response_variable='purity_percent',
            custom_predictors=custom_features,
            categorical_vars=categorical_features,
            base_output_dir=base_output,
            languages=['en', 'fr']  # Generate both languages
        )
        all_results['purity_percent'] = results_purity

        # Example 5: SHAP for TRS
        print("\n### RUNNING BILINGUAL SHAP ANALYSIS FOR: total_recoverable_sugar_t_per_ha ###\n")
        results_trs = main(
            data_file=data_file,
            response_variable='total_recoverable_sugar_t_per_ha',
            custom_predictors=custom_features,
            categorical_vars=categorical_features,
            base_output_dir=base_output,
            languages=['en', 'fr']  # Generate both languages
        )
        all_results['brix_percent'] = results_trs           
        print("\n" + "="*80)
        print("ALL BILINGUAL SHAP ANALYSES COMPLETED!")
        print("="*80)
        print("\nSummary of Results:")
        
        for var_name, results in all_results.items():
            display_name = ALL_RESPONSE_VARIABLES[var_name]
            print(f"\n{var_name} ({display_name}):")
            print(f"   Status: {results['status']}")
            print(f"   Models analyzed: {results['n_models_analyzed']}")
            if 'languages' in results:
                print(f"   Languages: {', '.join([lang.upper() for lang in results['languages']])}")
            print(f"   Output: {results['output_dir']}")
            if results['status'] == 'error':
                print(f"   Error: {results['error']}")
        
        print("\n" + "="*80)
        print("FEATURE NAME TRANSLATIONS:")
        print("="*80)
        print("\nSample translations used:")
        for original, mapping in list(FEATURE_MAPPINGS.items())[:5]:
            print(f"  {original}:")
            print(f"    EN: {mapping['short_en']}")
            print(f"    FR: {mapping['short_fr']}")
        print("  ...")
        
    except Exception as e:
        print(f"\nUnexpected Error: {e}")
        import traceback
        traceback.print_exc()

BILINGUAL SHAP ANALYSIS PIPELINE - MULTIPLE RESPONSE VARIABLES

Available response variables:

SUGAR CONTENT METRICS:
  - brix_percent: Brix (%)
  - pol_juice_percent: Pol juice (%)
  - pol_cane_percent: Pol cane (%)
  - purity_percent: Purity (%)
  - glucose_cane_percent: Glucose (%)
  - hugot_sugar: Hugot sugar (%)
  - total_recoverable_sugar_t_per_ha: TRS (t ha⁻¹)

YIELD PARAMETERS:
  - tc_per_ha_actual: Cane yield (t ha⁻¹)
  - tc_per_ha_per_week: Cane yield per week (t ha⁻¹/week)
  - tc_per_ha_52_weeks: Cane yield 52 weeks (t ha⁻¹)
  - fiber_percent: Fiber in Cane (%)


### RUNNING BILINGUAL SHAP ANALYSIS FOR: brix_percent ###


BILINGUAL SHAP ANALYSIS FOR: brix_percent
Display name: Brix (%)
Languages: EN, FR
Data file: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis/final_dataset_regrouped_categorical.csv
Models directory: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis\kfold_models_brix_percent_with_categorical\models
Out

## 6. HEATMAP SHAP VISUALIZATION

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# DEFINE AVAILABLE RESPONSE VARIABLES
# ============================================================================

SUGAR_CONTENT_METRICS = {
    "brix_percent": "Brix (%)",
    "pol_juice_percent": "Pol juice (%)",
    "pol_cane_percent": "Pol cane (%)",
    "purity_percent": "Purity (%)",
    "glucose_cane_percent": "Glucose (%)",
    "hugot_sugar": "Hugot sugar (%)"
}

YIELD_PARAMETERS = {
    "tc_per_ha_actual": "Cane yield (t ha⁻¹)",
    "tc_per_ha_per_week": "Cane yield per week (t ha⁻¹/week)",
    "tc_per_ha_52_weeks": "Cane yield 52 weeks (t ha⁻¹)",
    "fiber_percent": "Fiber in Cane (%)",
}

ALL_RESPONSE_VARIABLES = {**SUGAR_CONTENT_METRICS, **YIELD_PARAMETERS}

# ============================================================================
# FEATURE NAME MAPPING AND CATEGORIZATION
# ============================================================================

# Scientific names for features (shortened and formatted)
FEATURE_NAME_MAPPING = {
    # Climate features
    'temp_moy_mean': 'Mean temp.',
    'ampl_mean': 'Temp.\namplitude',
    'h_maxi_mean': 'Max\nhumidity',
    'h_mini_mean': 'Min\nhumidity',
    'radiation_ERA_total': 'Total\nradiation',
    'soil_temp_6h_mean': 'Soil temp.\n(6h)',
    'growing_season_rainfall_mm': 'Growing\nseason rain',
    'TNn': 'TNn',
    'TNx': 'TNx',
    'CDD': 'CDD',
    'CWD': 'CWD',
    'HD': 'HD',
    'Rx1': 'Rx1day',
    'R20': 'R20mm',
    'R99p': 'R99p',
    'SDII': 'SDII',
    
    # Soil features
    'pH_complete': 'pH',
    'Coarse_Silt_percent_complete': 'Coarse\nsilt (%)',
    'Total_Nitrogen_per_mille_complete': 'Total N\n(‰)',
    'Exchangeable_Calcium_meq_100g_complete': 'Exch. Ca\n(meq/100g)',
    'Exchangeable_Sodium_meq_100g_complete': 'Exch. Na\n(meq/100g)',
    'Available_P2O5_mg_kg_complete': 'Avail. P₂O₅\n(mg/kg)',
    'Electrical_Conductivity_mmho_cm_complete': 'Elec. cond.\n(mmho/cm)',
    'Mg_K_Ratio_complete': 'Mg/K',
    
    # Management features
    'variety_encoded': 'Variety',
    'irrigation_encoded': 'Irrigation',
    'harvest_encoded': 'Harvest\ntype',
    'crop_cycle_encoded': 'Crop\ncycle',
}

# Feature categories for color coding
FEATURE_CATEGORIES = {
    'climate': [
        'temp_moy_mean', 'ampl_mean', 'h_maxi_mean', 'h_mini_mean',
        'radiation_ERA_total', 'soil_temp_6h_mean', 'growing_season_rainfall_mm',
        'TNn', 'TNx', 'CDD', 'CWD', 'HD', 'Rx1', 'R20', 'R99p', 'SDII'
    ],
    'soil': [
        'pH_complete', 'Coarse_Silt_percent_complete', 'Total_Nitrogen_per_mille_complete',
        'Exchangeable_Calcium_meq_100g_complete', 'Exchangeable_Sodium_meq_100g_complete',
        'Available_P2O5_mg_kg_complete', 'Electrical_Conductivity_mmho_cm_complete',
        'Mg_K_Ratio_complete'
    ],
    'management': [
        'variety_encoded', 'irrigation_encoded', 'harvest_encoded', 'crop_cycle_encoded',
        'variety', 'irrigation', 'harvest', 'crop_cycle'  # Both encoded and non-encoded
    ]
}

# Color codes for each category
CATEGORY_COLORS = {
    'climate': '#2E86AB',      # Blue
    'soil': '#A23B72',         # Purple-pink
    'management': '#06A77D',   # Green
    'other': '#7F8C8D'         # Gray
}

def get_feature_category(feature_name):
    """Get the category of a feature"""
    # Remove _encoded suffix for checking
    clean_name = feature_name.replace('_encoded', '')
    
    for category, features in FEATURE_CATEGORIES.items():
        if feature_name in features or clean_name in features:
            return category
    return 'other'

def format_feature_name(feature_name):
    """Format feature name for display"""
    # Use mapping if available
    if feature_name in FEATURE_NAME_MAPPING:
        return FEATURE_NAME_MAPPING[feature_name]
    
    # Check without _encoded suffix
    clean_name = feature_name.replace('_encoded', '')
    if clean_name in FEATURE_NAME_MAPPING:
        return FEATURE_NAME_MAPPING[clean_name]
    
    # Otherwise return cleaned version
    clean = feature_name.replace('_encoded', '').replace('_complete', '')
    return clean.replace('_', ' ').title()

# ============================================================================
# LOAD SHAP RESULTS FROM CSV FILES - CUSTOM FOLDERS ONLY
# ============================================================================

def load_shap_results(base_dir, response_variables=None):
    """
    Load SHAP results ONLY from folders ending with _custom
    
    This function will:
    - Only look for folders named: shap_analysis_{response_var}_custom
    - Skip any folders with 'with_categorical' in the name
    - Report which folders are found/skipped
    """
    
    if response_variables is None:
        response_variables = list(ALL_RESPONSE_VARIABLES.keys())
    
    all_results = {}
    
    print("\n" + "="*80)
    print("LOADING SHAP RESULTS FROM _CUSTOM FOLDERS ONLY")
    print("="*80)
    
    for resp_var in response_variables:
        # ONLY look for folders ending with _custom
        shap_dir = os.path.join(base_dir, f'shap_analysis_{resp_var}_custom')
        
        # Skip if directory doesn't exist
        if not os.path.exists(shap_dir):
            print(f"⚠ Skipping {resp_var}: _custom folder not found at {shap_dir}")
            continue
        
        # Double check it's not a _with_categorical folder
        if 'with_categorical' in shap_dir:
            print(f"⚠ Skipping {resp_var}: contains 'with_categorical' in path")
            continue
        
        summary_file = os.path.join(shap_dir, 'shap_feature_importance_summary.csv')
        
        if not os.path.exists(summary_file):
            print(f"⚠ Skipping {resp_var}: summary CSV not found in {shap_dir}")
            continue
        
        try:
            df = pd.read_csv(summary_file)
            all_results[resp_var] = df
            print(f"✓ Loaded SHAP results for {resp_var} from _custom folder: {len(df)} entries")
        except Exception as e:
            print(f"✗ Error loading {resp_var}: {str(e)}")
    
    print("="*80)
    
    if not all_results:
        print("\n⚠ WARNING: No SHAP results loaded from _custom folders!")
        print("   Please ensure folders are named: shap_analysis_{response_var}_custom")
    
    return all_results

def consolidate_shap_data(all_results):
    """Consolidate SHAP results from all response variables"""
    
    consolidated = []
    
    for resp_var, df in all_results.items():
        df_copy = df.copy()
        df_copy['response_variable'] = resp_var
        df_copy['response_label'] = ALL_RESPONSE_VARIABLES.get(resp_var, resp_var)
        consolidated.append(df_copy)
    
    if not consolidated:
        raise ValueError("No SHAP results were successfully loaded!")
    
    final_df = pd.concat(consolidated, ignore_index=True)
    
    print(f"\n✓ Consolidated {len(final_df)} SHAP entries from _custom folders")
    print(f"  Response variables: {final_df['response_variable'].nunique()}")
    print(f"  Models: {final_df['model'].nunique()}")
    print(f"  Features: {final_df['feature'].nunique()}")
    
    return final_df

def create_comprehensive_heatmap(shap_df, output_dir, top_n_features=25, 
                                 response_variable_subset=None):
    """Create comprehensive heatmap with improved formatting"""
    
    os.makedirs(output_dir, exist_ok=True)
    
    if response_variable_subset:
        plot_df = shap_df[shap_df['response_variable'] == response_variable_subset].copy()
        title_suffix = f" - {ALL_RESPONSE_VARIABLES.get(response_variable_subset, response_variable_subset)}"
        filename_suffix = f"_{response_variable_subset}"
    else:
        plot_df = shap_df.copy()
        title_suffix = ""
        filename_suffix = "_all_variables"
    
    # Create pivot table
    pivot_df = plot_df.pivot_table(
        values='mean_abs_shap',
        index='feature',
        columns='model',
        aggfunc='mean'
    )
    
    # Sort by average importance
    avg_importance = pivot_df.mean(axis=1).sort_values(ascending=False)
    pivot_df = pivot_df.loc[avg_importance.index]
    pivot_df_top = pivot_df.head(top_n_features)
    
    print(f"\n{'='*80}")
    print(f"Creating heatmap{title_suffix}")
    print(f"Features: {len(pivot_df_top)}, Models: {len(pivot_df_top.columns)}")
    
    # MEMORY OPTIMIZATION: Limit figure size
    n_features = len(pivot_df_top)
    n_models = len(pivot_df_top.columns)
    
    # More conservative figure sizing to avoid memory error
    fig_height = min(16, max(10, n_features * 0.35))
    fig_width = min(20, max(12, n_models * 0.5))
    
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    
    # Custom colormap
    colors = ['#FFFFFF', '#FEF5E7', '#FAD7A0', '#F8C471', '#F39C12', 
              '#E67E22', '#DC7633', '#CA6F1E', '#A04000']
    cmap = LinearSegmentedColormap.from_list('shap_importance', colors, N=256)
    
    # Create heatmap with BOLD annotation
    sns.heatmap(pivot_df_top,
                annot=True,
                fmt='.3f',
                cmap=cmap,
                cbar_kws={
                    'label': 'Mean |SHAP value|',
                    'shrink': 0.7,
                    'aspect': 25
                },
                linewidths=1.5,
                linecolor='#34495E',
                square=False,
                ax=ax,
                annot_kws={'fontsize': 8, 'fontweight': 'bold', 'color': '#2C3E50'})
    
    # Format feature names and get colors
    formatted_features = []
    feature_colors = []
    for feat in pivot_df_top.index:
        formatted_features.append(format_feature_name(feat))
        category = get_feature_category(feat)
        feature_colors.append(CATEGORY_COLORS[category])
    
    # Set y-axis labels with colors and BOLD
    ax.set_yticks(np.arange(len(formatted_features)) + 0.5)
    ax.set_yticklabels(formatted_features, fontsize=10, fontweight='bold')
    
    # Color individual y-axis labels
    for ticklabel, color in zip(ax.get_yticklabels(), feature_colors):
        ticklabel.set_color(color)
    
    # Format x-axis labels (models) - BOLD
    model_labels = [m.replace('_', ' ').title() for m in pivot_df_top.columns]
    ax.set_xticklabels(model_labels, rotation=45, ha='right', 
                       fontsize=9, fontweight='bold')
    
    # Title and labels - BOLD
    ax.set_title(f'Feature Importance Across Models{title_suffix}',
                fontsize=16, fontweight='bold', pad=20, color='#2C3E50')
    ax.set_xlabel('Models', fontsize=13, fontweight='bold', color='#2C3E50')
    ax.set_ylabel('Features (ranked by importance)', 
                 fontsize=13, fontweight='bold', color='#2C3E50')
    
    # Add external grid (thick border)
    for spine in ax.spines.values():
        spine.set_edgecolor('#2C3E50')
        spine.set_linewidth(2.5)
    
    # Add legend for feature categories
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=CATEGORY_COLORS['climate'], label='Climate', edgecolor='black', linewidth=1),
        Patch(facecolor=CATEGORY_COLORS['soil'], label='Soil', edgecolor='black', linewidth=1),
        Patch(facecolor=CATEGORY_COLORS['management'], label='Management', edgecolor='black', linewidth=1)
    ]
    ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.15, 1),
             fontsize=10, frameon=True, title='Feature Type', title_fontsize=11,
             fancybox=True, shadow=True)
    
    plt.tight_layout()
    
    # Save with reduced DPI initially to avoid memory issues
    base_filename = f'shap_comprehensive_heatmap_custom{filename_suffix}'
    
    # PNG at 600 DPI
    png_file = os.path.join(output_dir, f'{base_filename}.png')
    plt.savefig(png_file, dpi=600, bbox_inches='tight', 
                facecolor='white', edgecolor='none', pad_inches=0.3)
    print(f"✓ Saved PNG (600 DPI): {png_file}")
    
    # PDF
    pdf_file = os.path.join(output_dir, f'{base_filename}.pdf')
    plt.savefig(pdf_file, bbox_inches='tight', 
                facecolor='white', edgecolor='none', pad_inches=0.3)
    print(f"✓ Saved PDF: {pdf_file}")
    
    plt.close()
    
    # Save data
    csv_file = os.path.join(output_dir, f'{base_filename}_data.csv')
    pivot_df_top.to_csv(csv_file)
    print(f"✓ Saved data: {csv_file}")
    
    return pivot_df_top

def create_response_variable_comparison(shap_df, output_dir, top_n_features=15):
    """Create faceted heatmap with memory optimization"""
    
    response_vars = shap_df['response_variable'].unique()
    n_vars = len(response_vars)
    
    if n_vars < 2:
        print("Only one response variable, skipping comparison")
        return
    
    print(f"\n{'='*80}")
    print(f"Creating comparison with {n_vars} variables")
    print(f"{'='*80}")
    
    # MEMORY OPTIMIZATION: Use smaller figure and fewer features
    n_cols = min(2, n_vars)  # Reduced from 3 to 2
    n_rows = int(np.ceil(n_vars / n_cols))
    
    # More conservative sizing
    fig, axes = plt.subplots(n_rows, n_cols, 
                            figsize=(n_cols * 7, n_rows * 5),
                            squeeze=False)
    axes_flat = axes.flatten()
    
    # Get global min/max
    all_pivot_dfs = []
    for resp_var in response_vars:
        subset = shap_df[shap_df['response_variable'] == resp_var]
        pivot = subset.pivot_table(
            values='mean_abs_shap',
            index='feature',
            columns='model',
            aggfunc='mean'
        )
        avg_imp = pivot.mean(axis=1).sort_values(ascending=False)
        pivot = pivot.loc[avg_imp.index].head(top_n_features)
        all_pivot_dfs.append(pivot)
    
    vmin = min([df.min().min() for df in all_pivot_dfs])
    vmax = max([df.max().max() for df in all_pivot_dfs])
    
    # Custom colormap
    colors = ['#FFFFFF', '#FEF5E7', '#FAD7A0', '#F8C471', '#F39C12', 
              '#E67E22', '#DC7633', '#CA6F1E', '#A04000']
    cmap = LinearSegmentedColormap.from_list('shap_importance', colors, N=256)
    
    # Create each subplot
    for idx, (resp_var, pivot_df) in enumerate(zip(response_vars, all_pivot_dfs)):
        ax = axes_flat[idx]
        
        sns.heatmap(pivot_df,
                    annot=True,
                    fmt='.2f',
                    cmap=cmap,
                    vmin=vmin,
                    vmax=vmax,
                    cbar_kws={'label': 'Mean |SHAP|', 'shrink': 0.7},
                    linewidths=1,
                    linecolor='#34495E',
                    ax=ax,
                    annot_kws={'fontsize': 7, 'fontweight': 'bold'})
        
        # Format feature names
        formatted_features = [format_feature_name(f) for f in pivot_df.index]
        ax.set_yticklabels(formatted_features, fontsize=8, fontweight='bold')
        
        # Color y-axis labels
        for ticklabel, feat in zip(ax.get_yticklabels(), pivot_df.index):
            category = get_feature_category(feat)
            ticklabel.set_color(CATEGORY_COLORS[category])
        
        title = ALL_RESPONSE_VARIABLES.get(resp_var, resp_var)
        ax.set_title(title, fontsize=11, fontweight='bold', pad=10)
        ax.set_xlabel('Models', fontsize=9, fontweight='bold')
        ax.set_ylabel('Features', fontsize=9, fontweight='bold')
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right', 
                fontsize=7, fontweight='bold')
    
    # Hide extra subplots
    for idx in range(n_vars, len(axes_flat)):
        axes_flat[idx].set_visible(False)
    
    plt.suptitle('Feature Importance Across Response Variables (Custom Models)',
                fontsize=14, fontweight='bold', y=0.995)
    
    plt.tight_layout()
    
    # Save
    base_filename = 'shap_response_variable_comparison_custom'
    
    png_file = os.path.join(output_dir, f'{base_filename}.png')
    plt.savefig(png_file, dpi=400, bbox_inches='tight',  # Reduced DPI for memory
                facecolor='white', edgecolor='none', pad_inches=0.3)
    print(f"\n✓ Saved comparison PNG: {png_file}")
    
    pdf_file = os.path.join(output_dir, f'{base_filename}.pdf')
    plt.savefig(pdf_file, bbox_inches='tight', 
                facecolor='white', edgecolor='none', pad_inches=0.3)
    print(f"✓ Saved comparison PDF: {pdf_file}")
    
    plt.close()

def create_top_features_summary(shap_df, output_dir, top_n=15):
    """Create summary with color-coded features"""
    
    # Calculate average importance
    feature_avg = shap_df.groupby('feature')['mean_abs_shap'].mean().sort_values(ascending=False)
    top_features = feature_avg.head(top_n)
    
    # Create bar plot
    fig, ax = plt.subplots(figsize=(12, 8))
    
    y_pos = np.arange(len(top_features))
    
    # Get colors based on feature category
    bar_colors = [CATEGORY_COLORS[get_feature_category(feat)] 
                  for feat in top_features.index[::-1]]
    
    bars = ax.barh(y_pos, top_features.values[::-1], 
                   color=bar_colors, alpha=0.8, height=0.7,
                   edgecolor='#2C3E50', linewidth=1.5)
    
    # Format feature names
    formatted_features = [format_feature_name(f) for f in top_features.index[::-1]]
    ax.set_yticks(y_pos)
    ax.set_yticklabels(formatted_features, fontsize=11, fontweight='bold')
    
    # Color y-axis labels
    for ticklabel, feat in zip(ax.get_yticklabels(), top_features.index[::-1]):
        category = get_feature_category(feat)
        ticklabel.set_color(CATEGORY_COLORS[category])
    
    ax.set_xlabel('Average Mean |SHAP value|', 
                 fontsize=13, fontweight='bold', color='#2C3E50')
    ax.set_title(f'Top {top_n} Most Important Features (Custom Models Overall)', 
                fontsize=15, fontweight='bold', pad=15, color='#2C3E50')
    
    # Add value labels
    for i, value in enumerate(top_features.values[::-1]):
        ax.text(value + max(top_features.values) * 0.01, i,
                f'{value:.4f}', va='center', ha='left', 
                fontsize=9, fontweight='bold')
    
    # Styling
    ax.grid(axis='x', alpha=0.3, linestyle='-', linewidth=0.8)
    ax.set_axisbelow(True)
    
    # Thick external border
    for spine in ax.spines.values():
        spine.set_edgecolor('#2C3E50')
        spine.set_linewidth(2.5)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=CATEGORY_COLORS['climate'], label='Climate', 
              edgecolor='black', linewidth=1),
        Patch(facecolor=CATEGORY_COLORS['soil'], label='Soil', 
              edgecolor='black', linewidth=1),
        Patch(facecolor=CATEGORY_COLORS['management'], label='Management', 
              edgecolor='black', linewidth=1)
    ]
    ax.legend(handles=legend_elements, loc='lower right',
             fontsize=10, frameon=True, title='Feature Type', 
             title_fontsize=11, fancybox=True, shadow=True)
    
    plt.tight_layout()
    
    # Save
    base_filename = 'shap_overall_top_features_custom'
    
    png_file = os.path.join(output_dir, f'{base_filename}.png')
    plt.savefig(png_file, dpi=600, bbox_inches='tight', 
                facecolor='white', edgecolor='none', pad_inches=0.3)
    print(f"\n✓ Saved top features PNG: {png_file}")
    
    pdf_file = os.path.join(output_dir, f'{base_filename}.pdf')
    plt.savefig(pdf_file, bbox_inches='tight', 
                facecolor='white', edgecolor='none', pad_inches=0.3)
    print(f"✓ Saved top features PDF: {pdf_file}")
    
    plt.close()
    
    # Save data
    csv_file = os.path.join(output_dir, f'{base_filename}_data.csv')
    top_features.to_csv(csv_file, header=['mean_abs_shap'])
    print(f"✓ Saved data: {csv_file}")

def generate_summary_report(shap_df, output_dir):
    """Generate comprehensive summary report"""
    
    report = []
    report.append("="*80)
    report.append("COMPREHENSIVE SHAP ANALYSIS SUMMARY (CUSTOM MODELS ONLY)")
    report.append("="*80)
    report.append("")
    
    report.append(f"Total entries: {len(shap_df):,}")
    report.append(f"Response variables: {shap_df['response_variable'].nunique()}")
    report.append(f"Unique models: {shap_df['model'].nunique()}")
    report.append(f"Unique features: {shap_df['feature'].nunique()}")
    report.append("")
    
    # Response variables
    report.append("Response Variables:")
    for resp_var in shap_df['response_variable'].unique():
        count = len(shap_df[shap_df['response_variable'] == resp_var])
        label = ALL_RESPONSE_VARIABLES.get(resp_var, resp_var)
        # Replace special unicode characters with ASCII equivalents
        label = label.replace('⁻', '-').replace('₂', '2').replace('₅', '5')
        report.append(f"  - {resp_var} ({label}): {count} entries")
    report.append("")
    
    # Top features by category
    report.append("Top Features by Category:")
    for category in ['climate', 'soil', 'management']:
        report.append(f"\n{category.upper()}:")
        category_features = [f for f in shap_df['feature'].unique() 
                           if get_feature_category(f) == category]
        if category_features:
            cat_importance = shap_df[shap_df['feature'].isin(category_features)]\
                .groupby('feature')['mean_abs_shap'].mean().sort_values(ascending=False)
            for i, (feat, value) in enumerate(list(cat_importance.items())[:5], 1):
                display_name = format_feature_name(feat)
                # Replace special unicode characters
                display_name = display_name.replace('₂', '2').replace('₅', '5')
                report.append(f"  {i}. {display_name}: {value:.4f}")
    report.append("")
    
    # Save report with UTF-8 encoding to handle special characters
    report_file = os.path.join(output_dir, 'comprehensive_shap_summary_custom.txt')
    with open(report_file, 'w', encoding='utf-8') as f:
        f.write('\n'.join(report))
    
    print(f"\n✓ Saved summary: {report_file}")
    print('\n'.join(report))

# ============================================================================
# MAIN FUNCTION
# ============================================================================

def main(base_dir, output_dir=None, response_variables=None, 
         top_n_features=25, create_individual_heatmaps=False):
    """Main function with memory optimization - CUSTOM MODELS ONLY"""
    
    if output_dir is None:
        output_dir = os.path.join(base_dir, 'comprehensive_shap_heatmaps_custom')
    
    print("\n" + "="*80)
    print("COMPREHENSIVE SHAP HEATMAP GENERATOR (CUSTOM MODELS ONLY)")
    print("="*80)
    print(f"Base directory: {base_dir}")
    print(f"Output directory: {output_dir}")
    print(f"Top N features: {top_n_features}")
    print(f"Target folders: shap_analysis_{{response_var}}_custom")
    print("="*80 + "\n")
    
    # Load results (only from _custom folders)
    all_results = load_shap_results(base_dir, response_variables)
    
    if not all_results:
        print("\n✗ No SHAP results found in _custom folders!")
        print("\nPlease ensure your folders are named:")
        print("  shap_analysis_{response_var}_custom")
        return
    
    # Consolidate
    shap_df = consolidate_shap_data(all_results)
    
    # Create main heatmap
    print("\n" + "="*80)
    print("CREATING COMPREHENSIVE HEATMAP")
    print("="*80)
    pivot_all = create_comprehensive_heatmap(
        shap_df, output_dir, 
        top_n_features=top_n_features
    )
    
    # Individual heatmaps (optional, memory intensive)
    if create_individual_heatmaps:
        print("\n" + "="*80)
        print("CREATING INDIVIDUAL HEATMAPS")
        print("="*80)
        for resp_var in shap_df['response_variable'].unique():
            try:
                create_comprehensive_heatmap(
                    shap_df, output_dir,
                    top_n_features=top_n_features,
                    response_variable_subset=resp_var
                )
            except MemoryError:
                print(f"⚠ Memory error for {resp_var}, skipping...")
                continue
    
    # Comparison (with reduced features)
    try:
        create_response_variable_comparison(shap_df, output_dir, top_n_features=15)
    except MemoryError:
        print("⚠ Memory error in comparison, skipping...")
    
    # Top features summary
    create_top_features_summary(shap_df, output_dir, top_n=15)
    
    # Generate report
    generate_summary_report(shap_df, output_dir)

# ============================================================================
# USAGE
# ============================================================================

if __name__ == "__main__":
    
    base_directory = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis'
    
    main(
        base_dir=base_directory,
        output_dir=None,  # Will create 'comprehensive_shap_heatmaps_custom' folder
        response_variables=None,  # Will process all available response variables
        top_n_features=15,
        create_individual_heatmaps=True  # Set to False to avoid memory issues
    )


COMPREHENSIVE SHAP HEATMAP GENERATOR (CUSTOM MODELS ONLY)
Base directory: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis
Output directory: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis\comprehensive_shap_heatmaps_custom
Top N features: 15
Target folders: shap_analysis_{response_var}_custom


LOADING SHAP RESULTS FROM _CUSTOM FOLDERS ONLY
✓ Loaded SHAP results for brix_percent from _custom folder: 240 entries
✓ Loaded SHAP results for pol_juice_percent from _custom folder: 240 entries
✓ Loaded SHAP results for pol_cane_percent from _custom folder: 240 entries
✓ Loaded SHAP results for purity_percent from _custom folder: 240 entries
✓ Loaded SHAP results for glucose_cane_percent from _custom folder: 240 entries
✓ Loaded SHAP results for hugot_sugar from _custom folder: 240 entries
✓ Loaded SHAP results for tc_per_ha_actual from _custom folder: 240 entries
✓ Loaded SHAP results for tc_per_ha_per_week from _custom folder: 240 en

# HEATMAP SHAP WITH CATEGORICAL

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# DEFINE AVAILABLE RESPONSE VARIABLES
# ============================================================================

SUGAR_CONTENT_METRICS = {
    "brix_percent": "Brix (%)",
    "pol_juice_percent": "Pol juice (%)",
    "pol_cane_percent": "Pol cane (%)",
    "purity_percent": "Purity (%)",
    "glucose_cane_percent": "Glucose (%)",
    "hugot_sugar": "Hugot sugar (%)"
}

YIELD_PARAMETERS = {
    "tc_per_ha_actual": "Cane yield (t ha⁻¹)",
    "tc_per_ha_per_week": "Cane yield per week (t ha⁻¹/week)",
    "tc_per_ha_52_weeks": "Cane yield 52 weeks (t ha⁻¹)",
    "fiber_percent": "Fiber in Cane (%)",
}

ALL_RESPONSE_VARIABLES = {**SUGAR_CONTENT_METRICS, **YIELD_PARAMETERS}

# ============================================================================
# FEATURE NAME MAPPING AND CATEGORIZATION
# ============================================================================

# Scientific names for features (shortened and formatted)
FEATURE_NAME_MAPPING = {
    # Climate features
    'temp_moy_mean': 'Mean temp.',
    'ampl_mean': 'Temp.\namplitude',
    'h_maxi_mean': 'Max\nhumidity',
    'h_mini_mean': 'Min\nhumidity',
    'radiation_ERA_total': 'Total\nradiation',
    'soil_temp_6h_mean': 'Soil temp.\n(6h)',
    'growing_season_rainfall_mm': 'Growing\nseason rain',
    'TNn': 'TNn',
    'TNx': 'TNx',
    'CDD': 'CDD',
    'CWD': 'CWD',
    'HD': 'HD',
    'Rx1': 'Rx1day',
    'R20': 'R20mm',
    'R99p': 'R99p',
    'SDII': 'SDII',
    
    # Soil features
    'pH_complete': 'pH',
    'Coarse_Silt_percent_complete': 'Coarse\nsilt (%)',
    'Total_Nitrogen_per_mille_complete': 'Total N\n(‰)',
    'Exchangeable_Calcium_meq_100g_complete': 'Exch. Ca\n(meq/100g)',
    'Exchangeable_Sodium_meq_100g_complete': 'Exch. Na\n(meq/100g)',
    'Available_P2O5_mg_kg_complete': 'Avail. P₂O₅\n(mg/kg)',
    'Electrical_Conductivity_mmho_cm_complete': 'Elec. cond.\n(mmho/cm)',
    'Mg_K_Ratio_complete': 'Mg/K',
    
    # Management features
    'variety_encoded': 'Variety',
    'irrigation_encoded': 'Irrigation',
    'harvest_encoded': 'Harvest\ntype',
    'crop_cycle_encoded': 'Crop\ncycle',
}

# Feature categories for color coding
FEATURE_CATEGORIES = {
    'climate': [
        'temp_moy_mean', 'ampl_mean', 'h_maxi_mean', 'h_mini_mean',
        'radiation_ERA_total', 'soil_temp_6h_mean', 'growing_season_rainfall_mm',
        'TNn', 'TNx', 'CDD', 'CWD', 'HD', 'Rx1', 'R20', 'R99p', 'SDII'
    ],
    'soil': [
        'pH_complete', 'Coarse_Silt_percent_complete', 'Total_Nitrogen_per_mille_complete',
        'Exchangeable_Calcium_meq_100g_complete', 'Exchangeable_Sodium_meq_100g_complete',
        'Available_P2O5_mg_kg_complete', 'Electrical_Conductivity_mmho_cm_complete',
        'Mg_K_Ratio_complete'
    ],
    'management': [
        'variety_encoded', 'irrigation_encoded', 'harvest_encoded', 'crop_cycle_encoded',
        'variety', 'irrigation', 'harvest', 'crop_cycle'  # Both encoded and non-encoded
    ]
}

# Color codes for each category
CATEGORY_COLORS = {
    'climate': '#2E86AB',      # Blue
    'soil': '#A23B72',         # Purple-pink
    'management': '#06A77D',   # Green
    'other': '#7F8C8D'         # Gray
}

def get_feature_category(feature_name):
    """Get the category of a feature"""
    # Remove _encoded suffix for checking
    clean_name = feature_name.replace('_encoded', '')
    
    for category, features in FEATURE_CATEGORIES.items():
        if feature_name in features or clean_name in features:
            return category
    return 'other'

def format_feature_name(feature_name):
    """Format feature name for display"""
    # Use mapping if available
    if feature_name in FEATURE_NAME_MAPPING:
        return FEATURE_NAME_MAPPING[feature_name]
    
    # Check without _encoded suffix
    clean_name = feature_name.replace('_encoded', '')
    if clean_name in FEATURE_NAME_MAPPING:
        return FEATURE_NAME_MAPPING[clean_name]
    
    # Otherwise return cleaned version
    clean = feature_name.replace('_encoded', '').replace('_complete', '')
    return clean.replace('_', ' ').title()

# ============================================================================
# LOAD SHAP RESULTS FROM CSV FILES
# ============================================================================

def load_shap_results(base_dir, response_variables=None):
    """Load SHAP results from all available CSV files"""
    
    if response_variables is None:
        response_variables = list(ALL_RESPONSE_VARIABLES.keys())
    
    all_results = {}
    
    for resp_var in response_variables:
        # Try different directory naming patterns
        possible_dirs = [
            os.path.join(base_dir, f'shap_analysis_{resp_var}'),
            os.path.join(base_dir, f'shap_analysis_{resp_var}_with_categorical'),
            os.path.join(base_dir, f'shap_analysis_{resp_var}_custom'),
        ]
        
        shap_dir = None
        for dir_path in possible_dirs:
            if os.path.exists(dir_path):
                shap_dir = dir_path
                break
        
        if shap_dir is None:
            continue
        
        summary_file = os.path.join(shap_dir, 'shap_feature_importance_summary.csv')
        
        if not os.path.exists(summary_file):
            continue
        
        try:
            df = pd.read_csv(summary_file)
            all_results[resp_var] = df
            print(f"✓ Loaded SHAP results for {resp_var}: {len(df)} entries")
        except Exception as e:
            print(f"✗ Error loading {resp_var}: {str(e)}")
    
    return all_results

def consolidate_shap_data(all_results):
    """Consolidate SHAP results from all response variables"""
    
    consolidated = []
    
    for resp_var, df in all_results.items():
        df_copy = df.copy()
        df_copy['response_variable'] = resp_var
        df_copy['response_label'] = ALL_RESPONSE_VARIABLES.get(resp_var, resp_var)
        consolidated.append(df_copy)
    
    if not consolidated:
        raise ValueError("No SHAP results were successfully loaded!")
    
    final_df = pd.concat(consolidated, ignore_index=True)
    
    print(f"\n✓ Consolidated {len(final_df)} SHAP entries")
    print(f"  Models: {final_df['model'].nunique()}")
    print(f"  Features: {final_df['feature'].nunique()}")
    
    return final_df

def create_comprehensive_heatmap(shap_df, output_dir, top_n_features=25, 
                                 response_variable_subset=None):
    """Create comprehensive heatmap with improved formatting"""
    
    os.makedirs(output_dir, exist_ok=True)
    
    if response_variable_subset:
        plot_df = shap_df[shap_df['response_variable'] == response_variable_subset].copy()
        title_suffix = f" - {ALL_RESPONSE_VARIABLES.get(response_variable_subset, response_variable_subset)}"
        filename_suffix = f"_{response_variable_subset}"
    else:
        plot_df = shap_df.copy()
        title_suffix = ""
        filename_suffix = "_all_variables"
    
    # Create pivot table
    pivot_df = plot_df.pivot_table(
        values='mean_abs_shap',
        index='feature',
        columns='model',
        aggfunc='mean'
    )
    
    # Sort by average importance
    avg_importance = pivot_df.mean(axis=1).sort_values(ascending=False)
    pivot_df = pivot_df.loc[avg_importance.index]
    pivot_df_top = pivot_df.head(top_n_features)
    
    print(f"\n{'='*80}")
    print(f"Creating heatmap{title_suffix}")
    print(f"Features: {len(pivot_df_top)}, Models: {len(pivot_df_top.columns)}")
    
    # MEMORY OPTIMIZATION: Limit figure size
    n_features = len(pivot_df_top)
    n_models = len(pivot_df_top.columns)
    
    # More conservative figure sizing to avoid memory error
    fig_height = min(16, max(10, n_features * 0.35))
    fig_width = min(20, max(12, n_models * 0.5))
    
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    
    # Custom colormap
    colors = ['#FFFFFF', '#FEF5E7', '#FAD7A0', '#F8C471', '#F39C12', 
              '#E67E22', '#DC7633', '#CA6F1E', '#A04000']
    cmap = LinearSegmentedColormap.from_list('shap_importance', colors, N=256)
    
    # Create heatmap with BOLD annotation
    sns.heatmap(pivot_df_top,
                annot=True,
                fmt='.3f',
                cmap=cmap,
                cbar_kws={
                    'label': 'Mean |SHAP value|',
                    'shrink': 0.7,
                    'aspect': 25
                },
                linewidths=1.5,
                linecolor='#34495E',
                square=False,
                ax=ax,
                annot_kws={'fontsize': 8, 'fontweight': 'bold', 'color': '#2C3E50'})
    
    # Format feature names and get colors
    formatted_features = []
    feature_colors = []
    for feat in pivot_df_top.index:
        formatted_features.append(format_feature_name(feat))
        category = get_feature_category(feat)
        feature_colors.append(CATEGORY_COLORS[category])
    
    # Set y-axis labels with colors and BOLD
    ax.set_yticks(np.arange(len(formatted_features)) + 0.5)
    ax.set_yticklabels(formatted_features, fontsize=10, fontweight='bold')
    
    # Color individual y-axis labels
    for ticklabel, color in zip(ax.get_yticklabels(), feature_colors):
        ticklabel.set_color(color)
    
    # Format x-axis labels (models) - BOLD
    model_labels = [m.replace('_', ' ').title() for m in pivot_df_top.columns]
    ax.set_xticklabels(model_labels, rotation=45, ha='right', 
                       fontsize=9, fontweight='bold')
    
    # Title and labels - BOLD
    ax.set_title(f'Feature Importance Across Models{title_suffix}',
                fontsize=16, fontweight='bold', pad=20, color='#2C3E50')
    ax.set_xlabel('Models', fontsize=13, fontweight='bold', color='#2C3E50')
    ax.set_ylabel('Features (ranked by importance)', 
                 fontsize=13, fontweight='bold', color='#2C3E50')
    
    # Add external grid (thick border)
    for spine in ax.spines.values():
        spine.set_edgecolor('#2C3E50')
        spine.set_linewidth(2.5)
    
    # Add legend for feature categories
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=CATEGORY_COLORS['climate'], label='Climate', edgecolor='black', linewidth=1),
        Patch(facecolor=CATEGORY_COLORS['soil'], label='Soil', edgecolor='black', linewidth=1),
        Patch(facecolor=CATEGORY_COLORS['management'], label='Management', edgecolor='black', linewidth=1)
    ]
    ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.15, 1),
             fontsize=10, frameon=True, title='Feature Type', title_fontsize=11,
             fancybox=True, shadow=True)
    
    plt.tight_layout()
    
    # Save with reduced DPI initially to avoid memory issues
    base_filename = f'shap_comprehensive_heatmap{filename_suffix}'
    
    # PNG at 600 DPI
    png_file = os.path.join(output_dir, f'{base_filename}.png')
    plt.savefig(png_file, dpi=600, bbox_inches='tight', 
                facecolor='white', edgecolor='none', pad_inches=0.3)
    print(f"✓ Saved PNG (600 DPI): {png_file}")
    
    # PDF
    pdf_file = os.path.join(output_dir, f'{base_filename}.pdf')
    plt.savefig(pdf_file, bbox_inches='tight', 
                facecolor='white', edgecolor='none', pad_inches=0.3)
    print(f"✓ Saved PDF: {pdf_file}")
    
    plt.close()
    
    # Save data
    csv_file = os.path.join(output_dir, f'{base_filename}_data.csv')
    pivot_df_top.to_csv(csv_file)
    print(f"✓ Saved data: {csv_file}")
    
    return pivot_df_top

def create_response_variable_comparison(shap_df, output_dir, top_n_features=15):
    """Create faceted heatmap with memory optimization"""
    
    response_vars = shap_df['response_variable'].unique()
    n_vars = len(response_vars)
    
    if n_vars < 2:
        print("Only one response variable, skipping comparison")
        return
    
    print(f"\n{'='*80}")
    print(f"Creating comparison with {n_vars} variables")
    print(f"{'='*80}")
    
    # MEMORY OPTIMIZATION: Use smaller figure and fewer features
    n_cols = min(2, n_vars)  # Reduced from 3 to 2
    n_rows = int(np.ceil(n_vars / n_cols))
    
    # More conservative sizing
    fig, axes = plt.subplots(n_rows, n_cols, 
                            figsize=(n_cols * 7, n_rows * 5),
                            squeeze=False)
    axes_flat = axes.flatten()
    
    # Get global min/max
    all_pivot_dfs = []
    for resp_var in response_vars:
        subset = shap_df[shap_df['response_variable'] == resp_var]
        pivot = subset.pivot_table(
            values='mean_abs_shap',
            index='feature',
            columns='model',
            aggfunc='mean'
        )
        avg_imp = pivot.mean(axis=1).sort_values(ascending=False)
        pivot = pivot.loc[avg_imp.index].head(top_n_features)
        all_pivot_dfs.append(pivot)
    
    vmin = min([df.min().min() for df in all_pivot_dfs])
    vmax = max([df.max().max() for df in all_pivot_dfs])
    
    # Custom colormap
    colors = ['#FFFFFF', '#FEF5E7', '#FAD7A0', '#F8C471', '#F39C12', 
              '#E67E22', '#DC7633', '#CA6F1E', '#A04000']
    cmap = LinearSegmentedColormap.from_list('shap_importance', colors, N=256)
    
    # Create each subplot
    for idx, (resp_var, pivot_df) in enumerate(zip(response_vars, all_pivot_dfs)):
        ax = axes_flat[idx]
        
        sns.heatmap(pivot_df,
                    annot=True,
                    fmt='.2f',
                    cmap=cmap,
                    vmin=vmin,
                    vmax=vmax,
                    cbar_kws={'label': 'Mean |SHAP|', 'shrink': 0.7},
                    linewidths=1,
                    linecolor='#34495E',
                    ax=ax,
                    annot_kws={'fontsize': 7, 'fontweight': 'bold'})
        
        # Format feature names
        formatted_features = [format_feature_name(f) for f in pivot_df.index]
        ax.set_yticklabels(formatted_features, fontsize=8, fontweight='bold')
        
        # Color y-axis labels
        for ticklabel, feat in zip(ax.get_yticklabels(), pivot_df.index):
            category = get_feature_category(feat)
            ticklabel.set_color(CATEGORY_COLORS[category])
        
        title = ALL_RESPONSE_VARIABLES.get(resp_var, resp_var)
        ax.set_title(title, fontsize=11, fontweight='bold', pad=10)
        ax.set_xlabel('Models', fontsize=9, fontweight='bold')
        ax.set_ylabel('Features', fontsize=9, fontweight='bold')
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right', 
                fontsize=7, fontweight='bold')
    
    # Hide extra subplots
    for idx in range(n_vars, len(axes_flat)):
        axes_flat[idx].set_visible(False)
    
    plt.suptitle('Feature Importance Across Response Variables',
                fontsize=14, fontweight='bold', y=0.995)
    
    plt.tight_layout()
    
    # Save
    base_filename = 'shap_response_variable_comparison'
    
    png_file = os.path.join(output_dir, f'{base_filename}.png')
    plt.savefig(png_file, dpi=400, bbox_inches='tight',  # Reduced DPI for memory
                facecolor='white', edgecolor='none', pad_inches=0.3)
    print(f"\n✓ Saved comparison PNG: {png_file}")
    
    pdf_file = os.path.join(output_dir, f'{base_filename}.pdf')
    plt.savefig(pdf_file, bbox_inches='tight', 
                facecolor='white', edgecolor='none', pad_inches=0.3)
    print(f"✓ Saved comparison PDF: {pdf_file}")
    
    plt.close()

def create_top_features_summary(shap_df, output_dir, top_n=15):
    """Create summary with color-coded features"""
    
    # Calculate average importance
    feature_avg = shap_df.groupby('feature')['mean_abs_shap'].mean().sort_values(ascending=False)
    top_features = feature_avg.head(top_n)
    
    # Create bar plot
    fig, ax = plt.subplots(figsize=(12, 8))
    
    y_pos = np.arange(len(top_features))
    
    # Get colors based on feature category
    bar_colors = [CATEGORY_COLORS[get_feature_category(feat)] 
                  for feat in top_features.index[::-1]]
    
    bars = ax.barh(y_pos, top_features.values[::-1], 
                   color=bar_colors, alpha=0.8, height=0.7,
                   edgecolor='#2C3E50', linewidth=1.5)
    
    # Format feature names
    formatted_features = [format_feature_name(f) for f in top_features.index[::-1]]
    ax.set_yticks(y_pos)
    ax.set_yticklabels(formatted_features, fontsize=11, fontweight='bold')
    
    # Color y-axis labels
    for ticklabel, feat in zip(ax.get_yticklabels(), top_features.index[::-1]):
        category = get_feature_category(feat)
        ticklabel.set_color(CATEGORY_COLORS[category])
    
    ax.set_xlabel('Average Mean |SHAP value|', 
                 fontsize=13, fontweight='bold', color='#2C3E50')
    ax.set_title(f'Top {top_n} Most Important Features (Overall)', 
                fontsize=15, fontweight='bold', pad=15, color='#2C3E50')
    
    # Add value labels
    for i, value in enumerate(top_features.values[::-1]):
        ax.text(value + max(top_features.values) * 0.01, i,
                f'{value:.4f}', va='center', ha='left', 
                fontsize=9, fontweight='bold')
    
    # Styling
    ax.grid(axis='x', alpha=0.3, linestyle='-', linewidth=0.8)
    ax.set_axisbelow(True)
    
    # Thick external border
    for spine in ax.spines.values():
        spine.set_edgecolor('#2C3E50')
        spine.set_linewidth(2.5)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=CATEGORY_COLORS['climate'], label='Climate', 
              edgecolor='black', linewidth=1),
        Patch(facecolor=CATEGORY_COLORS['soil'], label='Soil', 
              edgecolor='black', linewidth=1),
        Patch(facecolor=CATEGORY_COLORS['management'], label='Management', 
              edgecolor='black', linewidth=1)
    ]
    ax.legend(handles=legend_elements, loc='lower right',
             fontsize=10, frameon=True, title='Feature Type', 
             title_fontsize=11, fancybox=True, shadow=True)
    
    plt.tight_layout()
    
    # Save
    base_filename = 'shap_overall_top_features'
    
    png_file = os.path.join(output_dir, f'{base_filename}.png')
    plt.savefig(png_file, dpi=600, bbox_inches='tight', 
                facecolor='white', edgecolor='none', pad_inches=0.3)
    print(f"\n✓ Saved top features PNG: {png_file}")
    
    pdf_file = os.path.join(output_dir, f'{base_filename}.pdf')
    plt.savefig(pdf_file, bbox_inches='tight', 
                facecolor='white', edgecolor='none', pad_inches=0.3)
    print(f"✓ Saved top features PDF: {pdf_file}")
    
    plt.close()
    
    # Save data
    csv_file = os.path.join(output_dir, f'{base_filename}_data.csv')
    top_features.to_csv(csv_file, header=['mean_abs_shap'])
    print(f"✓ Saved data: {csv_file}")

def generate_summary_report(shap_df, output_dir):
    """Generate comprehensive summary report"""
    
    report = []
    report.append("="*80)
    report.append("COMPREHENSIVE SHAP ANALYSIS SUMMARY")
    report.append("="*80)
    report.append("")
    
    report.append(f"Total entries: {len(shap_df):,}")
    report.append(f"Response variables: {shap_df['response_variable'].nunique()}")
    report.append(f"Unique models: {shap_df['model'].nunique()}")
    report.append(f"Unique features: {shap_df['feature'].nunique()}")
    report.append("")
    
    # Response variables
    report.append("Response Variables:")
    for resp_var in shap_df['response_variable'].unique():
        count = len(shap_df[shap_df['response_variable'] == resp_var])
        label = ALL_RESPONSE_VARIABLES.get(resp_var, resp_var)
        # Replace special unicode characters with ASCII equivalents
        label = label.replace('⁻', '-').replace('₂', '2').replace('₅', '5')
        report.append(f"  - {resp_var} ({label}): {count} entries")
    report.append("")
    
    # Top features by category
    report.append("Top Features by Category:")
    for category in ['climate', 'soil', 'management']:
        report.append(f"\n{category.upper()}:")
        category_features = [f for f in shap_df['feature'].unique() 
                           if get_feature_category(f) == category]
        if category_features:
            cat_importance = shap_df[shap_df['feature'].isin(category_features)]\
                .groupby('feature')['mean_abs_shap'].mean().sort_values(ascending=False)
            for i, (feat, value) in enumerate(list(cat_importance.items())[:5], 1):
                display_name = format_feature_name(feat)
                # Replace special unicode characters
                display_name = display_name.replace('₂', '2').replace('₅', '5')
                report.append(f"  {i}. {display_name}: {value:.4f}")
    report.append("")
    
    # Save report with UTF-8 encoding to handle special characters
    report_file = os.path.join(output_dir, 'comprehensive_shap_summary.txt')
    with open(report_file, 'w', encoding='utf-8') as f:
        f.write('\n'.join(report))
    
    print(f"\n✓ Saved summary: {report_file}")
    print('\n'.join(report))

# ============================================================================
# MAIN FUNCTION
# ============================================================================

def main(base_dir, output_dir=None, response_variables=None, 
         top_n_features=25, create_individual_heatmaps=False):
    """Main function with memory optimization"""
    
    if output_dir is None:
        output_dir = os.path.join(base_dir, 'comprehensive_shap_heatmaps')
    
    print("\n" + "="*80)
    print("COMPREHENSIVE SHAP HEATMAP GENERATOR")
    print("="*80)
    print(f"Base directory: {base_dir}")
    print(f"Output directory: {output_dir}")
    print(f"Top N features: {top_n_features}")
    print("="*80 + "\n")
    
    # Load results
    all_results = load_shap_results(base_dir, response_variables)
    
    if not all_results:
        print("\n✗ No SHAP results found!")
        return
    
    # Consolidate
    shap_df = consolidate_shap_data(all_results)
    
    # Create main heatmap
    print("\n" + "="*80)
    print("CREATING COMPREHENSIVE HEATMAP")
    print("="*80)
    pivot_all = create_comprehensive_heatmap(
        shap_df, output_dir, 
        top_n_features=top_n_features
    )
    
    # Individual heatmaps (optional, memory intensive)
    if create_individual_heatmaps:
        print("\n" + "="*80)
        print("CREATING INDIVIDUAL HEATMAPS")
        print("="*80)
        for resp_var in shap_df['response_variable'].unique():
            try:
                create_comprehensive_heatmap(
                    shap_df, output_dir,
                    top_n_features=top_n_features,
                    response_variable_subset=resp_var
                )
            except MemoryError:
                print(f"⚠ Memory error for {resp_var}, skipping...")
                continue
    
    # Comparison (with reduced features)
    try:
        create_response_variable_comparison(shap_df, output_dir, top_n_features=15)
    except MemoryError:
        print("⚠ Memory error in comparison, skipping...")
    
    # Top features summary
    create_top_features_summary(shap_df, output_dir, top_n=15)
    
    # Generate report
    generate_summary_report(shap_df, output_dir)

# ============================================================================
# USAGE
# ============================================================================

if __name__ == "__main__":
    
    base_directory = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis'
    
    # Run with memory-safe settings
    main(
        base_dir=base_directory,
        output_dir=None,
        response_variables=None,
        top_n_features=20,
        create_individual_heatmaps=True  # Set to False to avoid memory issues
    )


COMPREHENSIVE SHAP HEATMAP GENERATOR
Base directory: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis
Output directory: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis\comprehensive_shap_heatmaps
Top N features: 20

✓ Loaded SHAP results for brix_percent: 280 entries
✓ Loaded SHAP results for pol_juice_percent: 280 entries
✓ Loaded SHAP results for pol_cane_percent: 280 entries
✓ Loaded SHAP results for purity_percent: 280 entries
✓ Loaded SHAP results for glucose_cane_percent: 280 entries
✓ Loaded SHAP results for hugot_sugar: 280 entries
✓ Loaded SHAP results for tc_per_ha_actual: 280 entries
✓ Loaded SHAP results for tc_per_ha_per_week: 280 entries
✓ Loaded SHAP results for tc_per_ha_52_weeks: 280 entries
✓ Loaded SHAP results for fiber_percent: 280 entries

✓ Consolidated 2800 SHAP entries
  Models: 10
  Features: 28

CREATING COMPREHENSIVE HEATMAP

Creating heatmap
Features: 20, Models: 10
✓ Saved PNG (600 DPI): C:/Users/U0

## 7. Partial Dependence Plots: Feature-Response Relationships

This section visualizes the marginal effect of individual features on model predictions using Partial Dependence Plots (PDPs). PDPs reveal how each predictor variable influences the predicted response while averaging over all other features.

**Analysis Components:**

**I. Dual-Response Visualization**
- Simultaneous plotting of two response variables (e.g., cane yield + Brix content)
- Dual y-axes with color-coded lines (blue/orange)
- Identifies shared vs. divergent predictor relationships

**II. Smoothing & Confidence Intervals**
- LOWESS smoothing with Gaussian filtering for publication-quality curves
- Bootstrap-based 95% confidence intervals (25 iterations)
- Prevents over-interpretation of noisy patterns

**III. Data Distribution Context**
- **Rug plots**: Show observed data distribution along x-axis
- Black marks filtered to PDP x-range (avoid plotting extreme outliers)
- X-axis limits focus on PDP curve range for clarity

**IV. SHAP-Informed Feature Selection**
- Features ranked by SHAP importance displayed as subplot annotations (R1#3, R2#7)
- Prioritizes most influential predictors

**Output:** Multi-page grid plots (3×4 layout, 350 DPI) and individual high-resolution plots (600 DPI) for each feature, saved as PNG and PDF.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.inspection import partial_dependence
from scipy.interpolate import make_interp_spline
from scipy.ndimage import gaussian_filter1d
from statsmodels.nonparametric.smoothers_lowess import lowess
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.size': 14,
    'font.family': 'DejaVu Sans',
    'axes.labelsize': 16,
    'axes.titlesize': 18,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 12,
    'figure.titlesize': 20,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': False,
    'axes.axisbelow': True,
    'axes.edgecolor': '#333333',
    'text.color': '#333333',
    'axes.labelcolor': '#333333',
    'xtick.color': '#333333',
    'ytick.color': '#333333',
    'axes.linewidth': 1.2,
    'xtick.major.width': 1.0,
    'ytick.major.width': 1.0
})

# ============================================================================
# LANGUAGE SUPPORT
# ============================================================================

LANGUAGES = ['en', 'fr']

TRANSLATIONS = {
    'en': {
        'page_title': 'Partial Dependence Plots (Page {page}/{total})',
        'not_available': 'Not available',
        'categorical': 'Categorical (excluded)',
        'error': 'Error',
        'rank_suffix': {1: 'st', 2: 'nd', 3: 'rd'},  # 1st, 2nd, 3rd
        'rank_default': 'th',  # 4th, 5th, etc.
    },
    'fr': {
        'page_title': 'Graphiques de dépendance partielle (Page {page}/{total})',
        'not_available': 'Non disponible',
        'categorical': 'Catégorique (exclu)',
        'error': 'Erreur',
        'rank_suffix': {1: 'er', 2: 'e', 3: 'e'},  # 1er, 2e, 3e
        'rank_default': 'e',  # 4e, 5e, etc.
    }
}

def format_rank(rank, lang='en'):
    """Format rank with appropriate suffix (1st/1er, 2nd/2e, etc.)"""
    if rank is None:
        return ''
    
    translations = TRANSLATIONS[lang]
    
    if rank in translations['rank_suffix']:
        suffix = translations['rank_suffix'][rank]
    else:
        suffix = translations['rank_default']
    
    return f"{rank}{suffix}"

# ============================================================================
# RESPONSE VARIABLE DEFINITIONS AND COLOR PALETTES
# ============================================================================

SUGAR_CONTENT_METRICS = {
    "brix_percent": {"en": "Brix (%)", "fr": "Brix (%)"},
    "pol_juice_percent": {"en": "Pol juice (%)", "fr": "Pol jus (%)"},
    "pol_cane_percent": {"en": "Pol cane (%)", "fr": "Pol canne (%)"},
    "purity_percent": {"en": "Purity (%)", "fr": "Pureté (%)"},
    "glucose_cane_percent": {"en": "Glucose (%)", "fr": "Glucose (%)"},
    "hugot_sugar": {"en": "Sucrose (%)", "fr": "Saccharose (%)"}
}

YIELD_PARAMETERS = {
    "tc_per_ha_actual": {"en": "Cane yield (t ha⁻¹)", "fr": "Rendement canne (t ha⁻¹)"},
    "tc_per_ha_per_week": {"en": "Cane yield per week (t ha⁻¹/week)", "fr": "Rendement canne/semaine (t ha⁻¹/semaine)"},
    "tc_per_ha_52_weeks": {"en": "Cane yield 52 weeks (t ha⁻¹)", "fr": "Rendement canne 52 semaines (t ha⁻¹)"},
    "fiber_percent": {"en": "Fiber in Cane (%)", "fr": "Fibre dans la canne (%)"},
    "total_recoverable_sugar_t_per_ha": {"en": "Total Recoverable Sugar (t ha⁻¹)", "fr": "Sucre récupérable total (t ha⁻¹)"}
}

ALL_RESPONSE_VARIABLES = {**SUGAR_CONTENT_METRICS, **YIELD_PARAMETERS}

# COLOR PALETTE FOR UP TO 5 RESPONSE VARIABLES
RESPONSE_COLORS = {
    'tc_per_ha_actual': '#1f77b4',           # Blue
    'brix_percent': '#2ca02c',               #
    'hugot_sugar': '#ff7f0e',                # 
    'purity_percent': '#d62728',             # Red
    'total_recoverable_sugar_t_per_ha': '#2ca02c'  # Purple
}

FALLBACK_COLORS = ['#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']

def get_response_color(response_var, index=0):
    """Get color for a response variable"""
    if response_var in RESPONSE_COLORS:
        return RESPONSE_COLORS[response_var]
    else:
        return FALLBACK_COLORS[index % len(FALLBACK_COLORS)]

def get_response_label(response_var, lang='en'):
    """Get response variable label in specified language"""
    if response_var in ALL_RESPONSE_VARIABLES:
        return ALL_RESPONSE_VARIABLES[response_var][lang]
    return response_var

# ============================================================================
# SMOOTHING FUNCTIONS
# ============================================================================

def ultra_smooth_curve_perfect_monotonic(x, y, method='lowess', frac=0.4, n_bootstrap=25):
    """PERFECT SMOOTHING for PDP curves"""
    try:
        sorted_indices = np.argsort(x)
        x_sorted = x[sorted_indices]
        y_sorted = y[sorted_indices]

        if len(x_sorted) <= 3:
            return x, y, None, None

        unique_x, inverse_indices = np.unique(x_sorted, return_inverse=True)
        unique_y = np.array([np.mean(y_sorted[inverse_indices == i]) for i in range(len(unique_x))])

        if len(unique_x) < 5:
            return x, y, None, None

        if method == 'lowess':
            smoothed = lowess(unique_y, unique_x, frac=frac, it=10, return_sorted=True)
            smooth_x = smoothed[:, 0]
            smooth_y = smoothed[:, 1]

            sigma_base = len(smooth_y) * 0.15
            for i in range(6):
                sigma = sigma_base * (1.0 - i * 0.1)
                smooth_y = gaussian_filter1d(smooth_y, sigma=sigma)

            smooth_y = gaussian_filter1d(smooth_y, sigma=len(smooth_y) * 0.2)

            try:
                n_spline = min(50, len(smooth_x))
                x_spline = np.linspace(smooth_x.min(), smooth_x.max(), n_spline)
                spline = make_interp_spline(smooth_x, smooth_y, k=2, bc_type='natural')
                y_spline = spline(x_spline)
                y_spline = gaussian_filter1d(y_spline, sigma=len(y_spline) * 0.25)
                smooth_x = x_spline
                smooth_y = y_spline
            except:
                pass

            if n_bootstrap > 0:
                bootstrap_curves = []
                n_points = len(unique_x)

                for _ in range(n_bootstrap):
                    bootstrap_indices = np.random.choice(n_points, n_points, replace=True)
                    boot_x = unique_x[bootstrap_indices]
                    boot_y = unique_y[bootstrap_indices]
                    boot_sort_idx = np.argsort(boot_x)
                    boot_x_sorted = boot_x[boot_sort_idx]
                    boot_y_sorted = boot_y[boot_sort_idx]

                    try:
                        boot_smoothed = lowess(boot_y_sorted, boot_x_sorted, frac=frac, it=8, return_sorted=True)
                        boot_y_smooth = boot_smoothed[:, 1]
                        for i in range(4):
                            sigma = len(boot_y_smooth) * 0.12 * (1.0 - i * 0.1)
                            boot_y_smooth = gaussian_filter1d(boot_y_smooth, sigma=sigma)
                        boot_y_smooth = gaussian_filter1d(boot_y_smooth, sigma=len(boot_y_smooth) * 0.2)
                        boot_interp_y = np.interp(smooth_x, boot_smoothed[:, 0], boot_y_smooth)
                        bootstrap_curves.append(boot_interp_y)
                    except:
                        continue

                if bootstrap_curves:
                    bootstrap_curves = np.array(bootstrap_curves)
                    ci_lower = np.percentile(bootstrap_curves, 2.5, axis=0)
                    ci_upper = np.percentile(bootstrap_curves, 97.5, axis=0)
                    ci_lower = gaussian_filter1d(ci_lower, sigma=len(ci_lower) * 0.15)
                    ci_upper = gaussian_filter1d(ci_upper, sigma=len(ci_upper) * 0.15)
                    return smooth_x, smooth_y, ci_lower, ci_upper

            return smooth_x, smooth_y, None, None

    except Exception as e:
        print(f"Perfect smoothing failed: {e}")
        return x, y, None, None

# ============================================================================
# MODEL AND DATA LOADING
# ============================================================================

def load_single_model(models_dir, model_name, response_var):
    """Load a single model for a response variable"""
    print(f"  Loading {model_name} model for {response_var}...")
    
    model_file = os.path.join(models_dir, f'{model_name.replace(" ", "_")}_{response_var}.pkl')
    if not os.path.exists(model_file):
        raise FileNotFoundError(f"Model not found: {model_file}")
    
    model_data = joblib.load(model_file)
    if isinstance(model_data, dict):
        model = model_data.get('model', model_data)
        encoders = model_data.get('encoders', {})
        feature_names = model_data.get('feature_names', None)
    else:
        model = model_data
        encoders = {}
        feature_names = None
    
    n_features = model.n_features_in_ if hasattr(model, 'n_features_in_') else None
    
    print(f"    ✓ Loaded (features: {n_features}, encoders: {len(encoders)})")
    
    return model, encoders, feature_names

def load_multiple_models(base_dir, model_name, response_variables, folder_suffix):
    """Load models for multiple response variables"""
    print(f"\n{'='*80}")
    print(f"LOADING MODELS FOR {len(response_variables)} RESPONSE VARIABLE(S)")
    print(f"{'='*80}")
    
    models = {}
    encoders_dict = {}
    feature_names_dict = {}
    
    for response_var in response_variables:
        models_dir = os.path.join(base_dir, f'kfold_models_{response_var}_{folder_suffix}', 'models')
        
        model, encoders, feature_names = load_single_model(models_dir, model_name, response_var)
        
        models[response_var] = model
        encoders_dict[response_var] = encoders
        feature_names_dict[response_var] = feature_names
    
    print(f"{'='*80}\n")
    return models, encoders_dict, feature_names_dict

def load_and_prepare_data_for_model(file_path, response_variable, model, custom_predictors=None, 
                                     categorical_vars=None, encoders=None):
    """Load and prepare data for a specific model using exact model features"""
    print(f"\n  Loading and preparing data for {response_variable} model...")
    
    df = pd.read_csv(file_path)
    
    target_col = response_variable
    
    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found!")
    
    df = df.dropna(subset=[target_col])
    
    year_column = None
    for col in ['campaign_year', 'Harvest_Year', 'year']:
        if col in df.columns:
            year_column = col
            break
    
    expected_n_features = model.n_features_in_ if hasattr(model, 'n_features_in_') else None
    print(f"    Model expects {expected_n_features} features")
    
    if custom_predictors is not None:
        columns_to_keep = [target_col] + custom_predictors
        if year_column:
            columns_to_keep.append(year_column)
        if categorical_vars:
            columns_to_keep.extend(categorical_vars)
        df = df[[col for col in columns_to_keep if col in df.columns]]
    
    if categorical_vars and encoders:
        print(f"    Encoding categorical variables: {categorical_vars}")
        for cat_var in categorical_vars:
            if cat_var in df.columns and cat_var in encoders:
                cat_encoder = encoders[cat_var]
                df[f'{cat_var}_encoded'] = df[cat_var].apply(
                    lambda x: cat_encoder.transform([str(x).upper()])[0] 
                    if str(x).upper() in cat_encoder.classes_ else -1
                )
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    for col in numeric_cols:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].median())
    
    if custom_predictors is not None:
        features = [col for col in custom_predictors if col in df.columns]
        if categorical_vars:
            for cat_var in categorical_vars:
                encoded_col = f'{cat_var}_encoded'
                if encoded_col in df.columns:
                    features.append(encoded_col)
    else:
        exclude_cols = [target_col]
        if year_column:
            exclude_cols.extend([year_column, f'{year_column}_encoded'])
        if categorical_vars:
            exclude_cols.extend(categorical_vars)
        features = [col for col in numeric_cols if col not in exclude_cols]
    
    print(f"    Initial feature count: {len(features)}")
    
    if expected_n_features and len(features) > expected_n_features:
        excess = len(features) - expected_n_features
        print(f"    ⚠ Trimming {excess} features to match model")
        features = features[:expected_n_features]
    
    missing_features = [f for f in features if f not in df.columns]
    if missing_features:
        raise ValueError(f"Cannot find required features: {missing_features}")
    
    X = df[features].copy()
    y = df[target_col].astype(float)
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    numeric_only_features = [f for f in features if not f.endswith('_encoded')]
    X_train_unscaled_numeric = X_train[numeric_only_features].copy()
    
    print(f"    ✓ Data prepared: {len(X_train_scaled)} samples, {len(features)} features")
    
    return X_train_scaled, X_test_scaled, y_train, y_test, features, scaler, df, X_train_unscaled_numeric

# ============================================================================
# SHAP IMPORTANCE LOADING
# ============================================================================

def load_shap_importance(shap_results_file, model_name):
    """Load SHAP feature importance from CSV file"""
    if not os.path.exists(shap_results_file):
        print(f"    ⚠ SHAP file not found: {shap_results_file}")
        return None
    
    try:
        shap_df = pd.read_csv(shap_results_file)
        model_shap = shap_df[shap_df['model'] == model_name].sort_values('mean_abs_shap', ascending=False)
        
        if model_shap.empty:
            print(f"    ⚠ No SHAP data found for model: {model_name}")
            return None
        
        features = model_shap['feature_original'].tolist()
        importance = model_shap['mean_abs_shap'].tolist()
        
        print(f"    ✓ Loaded SHAP importance for {len(features)} features")
        return dict(zip(features, importance))
        
    except Exception as e:
        print(f"    ✗ Error loading SHAP file: {str(e)}")
        return None

def get_feature_ranks_from_shap(base_dir, response_variables, model_name, folder_suffix):
    """Get feature ranks from SHAP analysis for multiple response variables"""
    print(f"\n{'='*80}")
    print("EXTRACTING FEATURE RANKS FROM SHAP ANALYSIS")
    print(f"{'='*80}")
    
    all_ranks = {}
    
    for response_var in response_variables:
        shap_dir = os.path.join(base_dir, f'shap_analysis_{response_var}_{folder_suffix.replace("models", "results")}')
        shap_file = os.path.join(shap_dir, 'shap_feature_importance_summary_en.csv')
        
        print(f"\n  {response_var}:")
        importance = load_shap_importance(shap_file, model_name)
        
        if importance:
            sorted_features = sorted(importance.items(), key=lambda x: x[1], reverse=True)
            ranks = {feat: rank+1 for rank, (feat, _) in enumerate(sorted_features)}
            all_ranks[response_var] = ranks
            print(f"    ✓ Ranked {len(ranks)} features")
        else:
            all_ranks[response_var] = {}
    
    print(f"{'='*80}\n")
    return all_ranks

# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def get_short_feature_name(feature_name):
    """Get a short, filesystem-safe name for a feature"""
    
    # Mapping of long feature names to short versions for file saving
    short_names = {
        # Climate variables
        'temp_moy_mean': 'temp_mean',
        'temp_maxi_mean': 'temp_max',
        'temp_mini_mean': 'temp_min',
        'precipitation_total': 'precip',
        'growing_season_rainfall_mm': 'gs_rainfall',
        'radiation_ERA_total': 'radiation',
        'evaporation_total': 'evap',
        'ampl_mean': 'temp_ampl',
        'h_mini_mean': 'rh_min',
        'h_maxi_mean': 'rh_max',
        'h_moy_mean': 'rh_mean',
        
        # Extreme climate indices
        'CWD': 'cwd',
        'CDD': 'cdd',
        'SDII': 'sdii',
        'R10': 'r10',
        'R20': 'r20',
        'Rx1': 'rx1',
        'Rx5': 'rx5',
        'R95p': 'r95p',
        'R99p': 'r99p',
        'HD': 'hot_days',
        'TNn': 'tnn',
        'TNx': 'tnx',
        'TXn': 'txn',
        'TXx': 'txx',
        
        # Soil temperature
        'soil_temp_6h_mean': 'soil_t_6h',
        'soil_temp_16h_mean': 'soil_t_16h',
        
        # Soil properties
        'pH_complete': 'ph',
        'Clay_percent_complete': 'clay',
        'Fine_Silt_percent_complete': 'fine_silt',
        'Coarse_Silt_percent_complete': 'coarse_silt',
        'Fine_Sand_percent_complete': 'fine_sand',
        'Coarse_Sand_percent_complete': 'coarse_sand',
        'Total_Nitrogen_per_mille_complete': 'n_total',
        'Organic_Carbon_per_mille_complete': 'org_c',
        'Exchangeable_Calcium_meq_100g_complete': 'ca_exch',
        'Exchangeable_Magnesium_meq_100g_complete': 'mg_exch',
        'Exchangeable_Potassium_meq_100g_complete': 'k_exch',
        'Exchangeable_Sodium_meq_100g_complete': 'na_exch',
        'Available_P2O5_mg_kg_complete': 'p2o5_avail',
        'Electrical_Conductivity_mmho_cm_complete': 'ec',
        'Mg_K_Ratio_complete': 'mg_k_ratio',
        'Mg_Ca_Ratio_complete': 'mg_ca_ratio',
    }
    
    # Return short name if exists, otherwise create a safe version
    if feature_name in short_names:
        return short_names[feature_name]
    else:
        # Fallback: just clean up the name
        safe_name = feature_name.replace('_complete', '').replace('_percent', '_pct')
        safe_name = safe_name.replace('_mean', '').replace('_total', '')
        safe_name = safe_name[:40]  # Max 40 chars
        return safe_name.lower()

def transform_to_original_scale(scaled_values, feature_idx, scaler, feature_names):
    """Transform scaled values back to original scale"""
    n_features = len(feature_names)
    dummy_scaled = np.zeros((len(scaled_values), n_features))
    dummy_scaled[:, feature_idx] = scaled_values
    dummy_original = scaler.inverse_transform(dummy_scaled)
    return dummy_original[:, feature_idx]

def get_feature_display_info(feature_name, lang='en'):
    """Get display label and units for features in specified language"""
    
    # English feature mapping
    feature_mapping_en = {
        'temp_moy_mean': {'label': 'Mean Temp', 'unit': '°C'},
        'temp_maxi_mean': {'label': 'T° max', 'unit': '°C'},
        'temp_mini_mean': {'label': 'T° min', 'unit': '°C'},
        'precipitation_total': {'label': 'Precip', 'unit': 'mm'},
        'growing_season_rainfall_mm': {'label': 'Growing season rain', 'unit': 'mm'},
        'radiation_ERA_total': {'label': 'Solar rad.', 'unit': 'cal/cm²'},
        'evaporation_total': {'label': 'Evaporation', 'unit': 'mm'},
        'CWD': {'label': 'CWD', 'unit': 'days'},
        'CDD': {'label': 'CDD', 'unit': 'days'},
        'SDII': {'label': 'SDII', 'unit': 'mm/day'},
        'R10': {'label': 'R10', 'unit': 'days'},
        'R20': {'label': 'R20', 'unit': 'days'},
        'Rx1': {'label': 'Rx1', 'unit': 'mm'},
        'Rx5': {'label': 'Max 5D', 'unit': 'mm'},
        'R95p': {'label': 'R95p', 'unit': 'mm'},
        'R99p': {'label': 'R99p', 'unit': 'mm'},
        'HD': {'label': 'Hot Days', 'unit': 'days'},
        'TNn': {'label': 'TNn', 'unit': '°C'},
        'TNx': {'label': 'TNx', 'unit': '°C'},
        'TXn': {'label': 'TXn', 'unit': '°C'},
        'TXx': {'label': 'TXx', 'unit': '°C'},
        'soil_temp_6h_mean': {'label': 'Soil T° 6h', 'unit': '°C'},
        'soil_temp_16h_mean': {'label': 'Soil T 16h', 'unit': '°C'},
        'pH_complete': {'label': 'pH', 'unit': ''},
        'Clay_percent_complete': {'label': 'Clay', 'unit': '%'},
        'Fine_Silt_percent_complete': {'label': 'Fine Silt', 'unit': '%'},
        'Coarse_Silt_percent_complete': {'label': 'Coarse silt', 'unit': '%'},
        'Fine_Sand_percent_complete': {'label': 'Fine Sand', 'unit': '%'},
        'Coarse_Sand_percent_complete': {'label': 'Coarse sand', 'unit': '%'},
        'Total_Nitrogen_per_mille_complete': {'label': 'N total', 'unit': '‰'},
        'Organic_Carbon_per_mille_complete': {'label': 'Org C', 'unit': '‰'},
        'Exchangeable_Calcium_meq_100g_complete': {'label': 'Ca exch.', 'unit': 'meq/100g'},
        'Exchangeable_Magnesium_meq_100g_complete': {'label': 'Mg', 'unit': 'meq/100g'},
        'Exchangeable_Potassium_meq_100g_complete': {'label': 'K exch.', 'unit': 'meq/100g'},
        'Mg_K_Ratio_complete': {'label': 'Mg/K ratio', 'unit': ''},
        'Mg_Ca_Ratio_complete': {'label': 'Mg/Ca', 'unit': ''},
        'ampl_mean': {'label': 'Temp Ampl', 'unit': '°C'},
        'h_mini_mean': {'label': 'RH min', 'unit': '%'},
        'h_maxi_mean': {'label': 'RH max', 'unit': '%'},
        'h_moy_mean': {'label': 'Mean Humid', 'unit': '%'},
        'Available_P2O5_mg_kg_complete': {'label': 'P₂O₅ avail.', 'unit': 'mg/kg'},
        'Electrical_Conductivity_mmho_cm_complete': {'label': 'EC', 'unit': 'mmho/cm'},
        'Exchangeable_Sodium_meq_100g_complete': {'label': 'Na exch.', 'unit': 'meq/100g'},
    }

    # French feature mapping
    feature_mapping_fr = {
        'temp_moy_mean': {'label': 'Temp moyenne', 'unit': '°C'},
        'temp_maxi_mean': {'label': 'T° max', 'unit': '°C'},
        'temp_mini_mean': {'label': 'T° min', 'unit': '°C'},
        'precipitation_total': {'label': 'Précip', 'unit': 'mm'},
        'growing_season_rainfall_mm': {'label': 'Pluie saison', 'unit': 'mm'},
        'radiation_ERA_total': {'label': 'Rayonnement', 'unit': 'cal/cm²'},
        'evaporation_total': {'label': 'Évaporation', 'unit': 'mm'},
        'CWD': {'label': 'CWD', 'unit': 'jours'},
        'CDD': {'label': 'CDD', 'unit': 'jours'},
        'SDII': {'label': 'SDII', 'unit': 'mm/jour'},
        'R10': {'label': 'R10', 'unit': 'jours'},
        'R20': {'label': 'R20', 'unit': 'jours'},
        'Rx1': {'label': 'Rx1', 'unit': 'mm'},
        'Rx5': {'label': 'Max 5J', 'unit': 'mm'},
        'R95p': {'label': 'R95p', 'unit': 'mm'},
        'R99p': {'label': 'R99p', 'unit': 'mm'},
        'HD': {'label': 'Jours chauds', 'unit': 'jours'},
        'TNn': {'label': 'TNn', 'unit': '°C'},
        'TNx': {'label': 'TNx', 'unit': '°C'},
        'TXn': {'label': 'TXn', 'unit': '°C'},
        'TXx': {'label': 'TXx', 'unit': '°C'},
        'soil_temp_6h_mean': {'label': 'T° sol 6h', 'unit': '°C'},
        'soil_temp_16h_mean': {'label': 'Temp sol 16h', 'unit': '°C'},
        'pH_complete': {'label': 'pH', 'unit': ''},
        'Clay_percent_complete': {'label': 'Argile', 'unit': '%'},
        'Fine_Silt_percent_complete': {'label': 'Limon fin', 'unit': '%'},
        'Coarse_Silt_percent_complete': {'label': 'Limon grossier', 'unit': '%'},
        'Fine_Sand_percent_complete': {'label': 'Sable fin', 'unit': '%'},
        'Coarse_Sand_percent_complete': {'label': 'Sable grossier', 'unit': '%'},
        'Total_Nitrogen_per_mille_complete': {'label': 'N total', 'unit': '‰'},
        'Organic_Carbon_per_mille_complete': {'label': 'C organique', 'unit': '‰'},
        'Exchangeable_Calcium_meq_100g_complete': {'label': 'Ca éch.', 'unit': 'meq/100g'},
        'Exchangeable_Magnesium_meq_100g_complete': {'label': 'Mg', 'unit': 'meq/100g'},
        'Exchangeable_Potassium_meq_100g_complete': {'label': 'K éch.', 'unit': 'meq/100g'},
        'Mg_K_Ratio_complete': {'label': 'Rapport Mg/K', 'unit': ''},
        'Mg_Ca_Ratio_complete': {'label': 'Mg/Ca', 'unit': ''},
        'ampl_mean': {'label': 'Ampl temp', 'unit': '°C'},
        'h_mini_mean': {'label': 'HR min', 'unit': '%'},
        'h_maxi_mean': {'label': 'HR max', 'unit': '%'},
        'h_moy_mean': {'label': 'Humid moy', 'unit': '%'},
        'Available_P2O5_mg_kg_complete': {'label': 'P₂O₅ disp.', 'unit': 'mg/kg'},
        'Electrical_Conductivity_mmho_cm_complete': {'label': 'CE', 'unit': 'mmho/cm'},
        'Exchangeable_Sodium_meq_100g_complete': {'label': 'Na éch.', 'unit': 'meq/100g'},
    }
    
    # Select appropriate mapping
    feature_mapping = feature_mapping_fr if lang == 'fr' else feature_mapping_en
    
    # Get info or create default
    if feature_name in feature_mapping:
        info = feature_mapping[feature_name]
    else:
        # Default: clean up feature name
        default_label = feature_name.replace('_', ' ').title()
        info = {'label': default_label, 'unit': ''}
    
    return info['label'], info['unit']

def add_rug_plot(ax, data_values, x_min, x_max, color='#333333', alpha=0.3, height=0.03):
    """Add rug plot to show distribution of observed data points within PDP range"""
    try:
        data_values_filtered = data_values[(data_values >= x_min) & (data_values <= x_max)]
        
        if len(data_values_filtered) == 0:
            return
        
        y_min, y_max = ax.get_ylim()
        y_range = y_max - y_min
        
        rug_y_position = y_min + (y_range * 0.02)
        rug_height = y_range * height
        
        if len(data_values_filtered) > 1000:
            sample_indices = np.random.choice(len(data_values_filtered), 1000, replace=False)
            data_values_sample = data_values_filtered[sample_indices]
        else:
            data_values_sample = data_values_filtered
        
        for val in data_values_sample:
            ax.plot([val, val], [rug_y_position, rug_y_position + rug_height],
                   color=color, alpha=alpha, linewidth=0.8, zorder=1)
                   
    except Exception as e:
        print(f"      ⚠ Could not add rug plot: {str(e)}")

# ============================================================================
# PDP COMPUTATION
# ============================================================================

def compute_all_pdp_ranges(models_dict, X_train_dict, feature_names_dict, scaler_dict, response_variables):
    """Pre-compute PDP ranges for all features to establish global y-axis limits"""
    print("\n" + "="*80)
    print("PRE-COMPUTING PDP RANGES FOR CONSISTENT Y-AXIS SCALING")
    print("="*80)
    
    all_features_sets = [set([f for f in feature_names_dict[rv] if not f.endswith('_encoded')]) 
                        for rv in response_variables]
    all_features = sorted(list(set.union(*all_features_sets)))
    
    print(f"Total numeric features: {len(all_features)}")
    print(f"(Excluded categorical encoded features)\n")
    
    all_values_by_response = {rv: [] for rv in response_variables}
    
    for i, feature in enumerate(all_features):
        print(f"  [{i+1}/{len(all_features)}] Computing range for {feature}...")
        
        for response_var in response_variables:
            if feature not in feature_names_dict[response_var]:
                continue
            
            try:
                feature_idx = feature_names_dict[response_var].index(feature)
                model = models_dict[response_var]
                X_train = X_train_dict[response_var]
                scaler = scaler_dict[response_var]
                
                pd_result = partial_dependence(
                    model, X_train, features=[feature_idx], grid_resolution=200
                )
                
                if hasattr(pd_result, 'average'):
                    values = pd_result.average[0]
                    scaled_grid = pd_result.grid_values[0]
                else:
                    values = pd_result[0][0]
                    scaled_grid = pd_result[1][0]
                
                original_grid = transform_to_original_scale(
                    scaled_grid, feature_idx, scaler, feature_names_dict[response_var]
                )
                
                smooth_x, smooth_y, ci_lower, ci_upper = ultra_smooth_curve_perfect_monotonic(
                    original_grid, values, method='lowess', frac=0.4, n_bootstrap=15
                )
                
                all_values_by_response[response_var].extend(smooth_y)
                if ci_lower is not None:
                    all_values_by_response[response_var].extend(ci_lower)
                    all_values_by_response[response_var].extend(ci_upper)
                    
            except Exception as e:
                print(f"      ⚠ Error for {response_var}: {str(e)}")
                continue
    
    global_limits = {}
    print(f"\n{'='*80}")
    print("COMPUTED GLOBAL Y-AXIS RANGES:")
    print(f"{'='*80}")
    
    for response_var in response_variables:
        if all_values_by_response[response_var]:
            y_min = np.min(all_values_by_response[response_var])
            y_max = np.max(all_values_by_response[response_var])
            y_range = y_max - y_min
            padding = y_range * 0.10
            global_limits[response_var] = (y_min - padding, y_max + padding)
            print(f"  {response_var}: [{y_min - padding:.6f}, {y_max + padding:.6f}]")
        else:
            global_limits[response_var] = None
            print(f"  {response_var}: No data")
    
    print(f"{'='*80}\n")
    return global_limits

# ============================================================================
# PLOTTING FUNCTIONS
# ============================================================================

def create_single_pdp_on_axis(ax, models_dict, X_train_dict, X_train_unscaled_dict,
                              feature_name, scaler_dict, feature_names_dict,
                              response_variables, all_ranks, global_limits, lang='en'):
    """Create a single PDP plot on given axis for multiple response variables"""
    
    translations = TRANSLATIONS[lang]
    
    # Skip categorical encoded features
    if feature_name.endswith('_encoded'):
        ax.text(0.5, 0.5, translations['categorical'], ha='center', va='center', 
                transform=ax.transAxes, fontsize=14, color='gray')
        ax.set_title(feature_name, fontsize=18, fontweight='600')
        return False
    
    # Check which response variables have this feature
    available_responses = [rv for rv in response_variables if feature_name in feature_names_dict[rv]]
    
    if not available_responses:
        ax.text(0.5, 0.5, translations['not_available'], ha='center', va='center', 
                transform=ax.transAxes, fontsize=14)
        ax.set_title(feature_name, fontsize=18, fontweight='600')
        return False
    
    try:
        feature_label, unit = get_feature_display_info(feature_name, lang)
        
        lines = []
        labels = []
        axes = []
        
        x_min_pdp = None
        x_max_pdp = None
        
        # Plot each response variable
        for i, response_var in enumerate(available_responses):
            model = models_dict[response_var]
            X_train = X_train_dict[response_var]
            scaler = scaler_dict[response_var]
            feature_names = feature_names_dict[response_var]
            feature_idx = feature_names.index(feature_name)
            
            # Compute PDP
            pd_result = partial_dependence(
                model, X_train, features=[feature_idx], grid_resolution=200
            )
            
            if hasattr(pd_result, 'average'):
                values = pd_result.average[0]
                scaled_grid = pd_result.grid_values[0]
            else:
                values = pd_result[0][0]
                scaled_grid = pd_result[1][0]
            
            original_grid = transform_to_original_scale(
                scaled_grid, feature_idx, scaler, feature_names
            )
            
            smooth_x, smooth_y, ci_lower, ci_upper = ultra_smooth_curve_perfect_monotonic(
                original_grid, values, method='lowess', frac=0.4, n_bootstrap=15
            )
            
            # Update x-range
            if x_min_pdp is None:
                x_min_pdp = smooth_x.min()
                x_max_pdp = smooth_x.max()
            else:
                x_min_pdp = min(x_min_pdp, smooth_x.min())
                x_max_pdp = max(x_max_pdp, smooth_x.max())
            
            # Get color for this response variable
            color = get_response_color(response_var, i)
            
            # Create axis (first uses main ax, others use twinx)
            if i == 0:
                current_ax = ax
            else:
                current_ax = ax.twinx()
                # Position additional y-axes
                if i > 1:
                    current_ax.spines['right'].set_position(('outward', 60 * (i - 1)))
            
            axes.append(current_ax)
            
            # Plot confidence interval
            if ci_lower is not None:
                current_ax.fill_between(smooth_x, ci_lower, ci_upper,
                                       alpha=0.15, color=color, zorder=2)
            
            # Get rank for label - SIMPLIFIED: just show rank like "1st" or "2nd"
            rank = all_ranks[response_var].get(feature_name)
            label = format_rank(rank, lang)
            
            # Plot line
            line, = current_ax.plot(smooth_x, smooth_y, linewidth=3.5, color=color,
                                   label=label, solid_capstyle='round', zorder=5)
            
            # Style y-axis
            current_ax.tick_params(axis='y', labelcolor=color, labelsize=13)
            current_ax.set_ylabel(get_response_label(response_var, lang), color=color, 
                                 fontsize=16, fontweight='600')
            
            # Apply global limits
            if global_limits[response_var] is not None:
                current_ax.set_ylim(global_limits[response_var])
            
            lines.append(line)
            labels.append(label)
        
        # Set x-axis limits
        if x_min_pdp is not None and x_max_pdp is not None:
            x_range = x_max_pdp - x_min_pdp
            x_padding = x_range * 0.02
            ax.set_xlim(x_min_pdp - x_padding, x_max_pdp + x_padding)
        
        # Add rug plot
        if x_min_pdp is not None and x_max_pdp is not None:
            first_response = available_responses[0]
            X_unscaled = X_train_unscaled_dict[first_response]
            if X_unscaled is not None and feature_name in X_unscaled.columns:
                feature_data = X_unscaled[feature_name].values
                add_rug_plot(ax, feature_data, x_min_pdp, x_max_pdp, 
                           color='#333333', alpha=0.3, height=0.03)
        
        # Title
        title_text = feature_label
        if unit:
            title_text += f' ({unit})'
        ax.set_title(title_text, fontsize=18, fontweight='600', pad=12)
        
        # Legend
        ax.legend(lines, labels, loc='upper right', fontsize=12, framealpha=0.95, 
                 edgecolor='#cccccc', handlelength=1.5, borderpad=0.5)
        
        # Grid
        ax.grid(True, alpha=0.25, linestyle=':', linewidth=0.7, zorder=0)
        ax.set_axisbelow(True)
        
        # X-axis formatting
        ax.set_xlabel('', fontsize=13)
        ax.tick_params(axis='x', labelsize=13)
        
        return True
        
    except Exception as e:
        print(f"    ✗ Error plotting {feature_name}: {str(e)}")
        ax.text(0.5, 0.5, translations['error'], ha='center', va='center', 
                transform=ax.transAxes, fontsize=13, color='red')
        ax.set_title(feature_name, fontsize=18, fontweight='600')
        return False

def save_individual_plot(models_dict, X_train_dict, X_train_unscaled_dict,
                        feature_name, scaler_dict, feature_names_dict,
                        response_variables, all_ranks, global_limits, 
                        output_dir, model_name, lang='en'):
    """Save individual PDP plot at 600 DPI in specified language"""
    
    fig = None
    try:
        fig, ax = plt.subplots(figsize=(4.5, 3.5))
        
        success = create_single_pdp_on_axis(
            ax, models_dict, X_train_dict, X_train_unscaled_dict,
            feature_name, scaler_dict, feature_names_dict,
            response_variables, all_ranks, global_limits, lang
        )
        
        if success:
            # Create the directory
            individual_dir = os.path.join(output_dir, f'individual_plots_{lang}')
            individual_dir = os.path.normpath(individual_dir)
            os.makedirs(individual_dir, exist_ok=True)
            
            # Use short name for file system
            short_name = get_short_feature_name(feature_name)
            filename = f'pdp_{short_name}_{model_name.replace(" ", "_")}_{lang}.png'
            output_file = os.path.join(individual_dir, filename)
            output_file = os.path.normpath(output_file)
            
            plt.savefig(output_file, dpi=600, bbox_inches='tight', facecolor='white')
            plt.close(fig)
            
            return output_file
        else:
            plt.close(fig)
            return None
            
    except Exception as e:
        print(f"      ✗ Error saving individual plot for {feature_name} ({lang}): {str(e)}")
        if fig is not None:
            plt.close(fig)
        return None

def create_grid_pdp_plots(models_dict, X_train_dict, X_train_unscaled_dict,
                         feature_names_dict, response_variables,
                         output_dir, model_name, scaler_dict, all_ranks,
                         plots_per_page=12):
    """Create bilingual grid PDP plots for multiple response variables"""
    
    print(f"\n{'='*80}")
    print(f"CREATING BILINGUAL GRID PDP PLOTS FOR {len(response_variables)} RESPONSE VARIABLE(S)")
    print(f"Model: {model_name}")
    print(f"Plots per page: {plots_per_page} (3×4 grid)")
    print(f"Languages: English & French")
    print(f"{'='*80}\n")
    
    # Pre-compute global y-axis ranges
    global_limits = compute_all_pdp_ranges(
        models_dict, X_train_dict, feature_names_dict, scaler_dict, response_variables
    )
    
    # Get union of all features
    all_features_sets = [set(feature_names_dict[rv]) for rv in response_variables]
    all_features = sorted(list(set.union(*all_features_sets)))
    print(f"Total unique features: {len(all_features)}\n")
    
    # Create output directory with proper path handling
    model_output_dir = os.path.join(output_dir, model_name.replace(" ", "_"))
    model_output_dir = os.path.normpath(model_output_dir)
    os.makedirs(model_output_dir, exist_ok=True)
    print(f"Output directory: {model_output_dir}\n")
    
    # Calculate pages
    n_features = len(all_features)
    n_pages = int(np.ceil(n_features / plots_per_page))
    
    print(f"Will create {n_pages} page(s) for {n_features} features (in 2 languages)\n")
    
    created_files = {'en': [], 'fr': []}
    individual_files = {'en': [], 'fr': []}
    filename_mapping = {'en': {}, 'fr': {}}  # Track feature -> filename mapping
    
    # Generate plots for both languages
    for lang in LANGUAGES:
        print(f"\n{'='*80}")
        print(f"GENERATING {lang.upper()} PLOTS")
        print(f"{'='*80}\n")
        
        for page_num in range(n_pages):
            start_idx = page_num * plots_per_page
            end_idx = min(start_idx + plots_per_page, n_features)
            page_features = all_features[start_idx:end_idx]
            
            print(f"\n{'='*60}")
            print(f"Page {page_num + 1}/{n_pages} ({lang.upper()}): Features {start_idx + 1}-{end_idx}")
            print(f"{'='*60}")
            
            # Create figure
            fig = plt.figure(figsize=(16, 16))
            gs = fig.add_gridspec(4, 3, hspace=0.50, wspace=0.50,
                                 left=0.08, right=0.92, top=0.94, bottom=0.06)
            
            # Page title
            response_names = ', '.join([rv[:10] for rv in response_variables])
            title_template = TRANSLATIONS[lang]['page_title']
            fig.suptitle(f'{model_name} - {title_template.format(page=page_num + 1, total=n_pages)}\n{response_names}',
                        fontsize=16, fontweight='bold', y=0.97)
            
            # Plot each feature
            for i, feature in enumerate(page_features):
                row = i // 3
                col = i % 3
                
                ax = fig.add_subplot(gs[row, col])
                
                # Check availability and ranks
                available_in = []
                for rv in response_variables:
                    if feature in feature_names_dict[rv]:
                        rank = all_ranks[rv].get(feature)
                        rank_formatted = format_rank(rank, lang) if rank else ''
                        available_in.append(f"{rv[:5]}#{rank_formatted}")
                
                rank_str = f" [{', '.join(available_in)}]" if available_in else ""
                
                print(f"  [{i+1}/{len(page_features)}] Plotting {feature}{rank_str}...")
                
                success = create_single_pdp_on_axis(
                    ax, models_dict, X_train_dict, X_train_unscaled_dict,
                    feature, scaler_dict, feature_names_dict,
                    response_variables, all_ranks, global_limits, lang
                )
                
                if success:
                    print(f"      ✓ Success")
                    
                    # Save individual plot
                    print(f"      → Saving individual 600 DPI export ({lang})...")
                    individual_file = save_individual_plot(
                        models_dict, X_train_dict, X_train_unscaled_dict,
                        feature, scaler_dict, feature_names_dict,
                        response_variables, all_ranks, global_limits, 
                        model_output_dir, model_name, lang
                    )
                    
                    if individual_file:
                        individual_files[lang].append(individual_file)
                        filename_mapping[lang][feature] = os.path.basename(individual_file)
                        print(f"      ✓ Saved: {os.path.basename(individual_file)}")
                else:
                    print(f"      ⚠ Skipped or error")
            
            # Save the page with proper path handling
            output_file = os.path.join(model_output_dir, 
                                       f'grid_pdp_page_{page_num + 1:02d}_{model_name.replace(" ", "_")}_{lang}.png')
            output_file = os.path.normpath(output_file)
            
            plt.savefig(output_file, dpi=350, bbox_inches='tight', facecolor='white')
            
            # Also save as PDF
            pdf_file = output_file.replace('.png', '.pdf')
            plt.savefig(pdf_file, bbox_inches='tight', facecolor='white')
            
            plt.close()
            
            created_files[lang].append(output_file)
            print(f"\n  ✓ Saved page {page_num + 1} ({lang.upper()}): {output_file}")
            print(f"  ✓ Saved PDF ({lang.upper()}): {pdf_file}")
    
    # Save filename mappings
    for lang in LANGUAGES:
        if filename_mapping[lang]:
            mapping_file = os.path.join(model_output_dir, f'filename_mapping_{lang}.csv')
            mapping_file = os.path.normpath(mapping_file)
            mapping_df = pd.DataFrame([
                {'feature_name': feat, 'filename': fname} 
                for feat, fname in filename_mapping[lang].items()
            ])
            mapping_df.to_csv(mapping_file, index=False)
            print(f"\n✓ Saved filename mapping ({lang.upper()}): {mapping_file}")
    
    print(f"\n{'='*80}")
    print(f"BILINGUAL GRID PDP PLOTS COMPLETE")
    print(f"English: Created {len(created_files['en'])} pages + {len(individual_files['en'])} individual plots")
    print(f"French: Created {len(created_files['fr'])} pages + {len(individual_files['fr'])} individual plots")
    print(f"Output directory: {model_output_dir}")
    print(f"{'='*80}\n")
    
    return created_files, model_output_dir, individual_files

# ============================================================================
# MAIN FUNCTION
# ============================================================================

def main_grid_pdp_analysis(response_variables, 
                          data_file=None, 
                          base_dir=None,
                          model_name='RandomForest', 
                          custom_predictors=None, 
                          categorical_vars=None,
                          plots_per_page=12):
    """
    Main function for bilingual grid PDP analysis with flexible number of response variables (1-5)
    
    Parameters:
    -----------
    response_variables : list of str
        List of response variables (1 to 5)
    """
    
    # Validate inputs
    if not isinstance(response_variables, list):
        response_variables = [response_variables]
    
    if len(response_variables) < 1 or len(response_variables) > 5:
        raise ValueError(f"Must specify 1-5 response variables. Got {len(response_variables)}")
    
    for rv in response_variables:
        if rv not in ALL_RESPONSE_VARIABLES:
            raise ValueError(f"Invalid response variable: {rv}. Must be one of {list(ALL_RESPONSE_VARIABLES.keys())}")
    
    print("\n" + "="*90)
    print(f"BILINGUAL GRID PDP ANALYSIS - {len(response_variables)} RESPONSE VARIABLE(S)")
    print("="*90)

    # Set default paths
    if data_file is None:
        data_file = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis/final_dataset_regrouped_categorical.csv'
    
    if base_dir is None:
        base_dir = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis'
    
    # Determine folder suffix
    if categorical_vars:
        folder_suffix = 'with_categorical'
    elif custom_predictors:
        folder_suffix = 'custom'
    else:
        folder_suffix = 'models'
    
    # Construct output directory with proper path handling
    response_names_joined = '_'.join(response_variables)
    output_dir = os.path.join(base_dir, f'combined_pdp_grid_bilingual_{response_names_joined}_{folder_suffix}')
    output_dir = os.path.normpath(output_dir)
    
    print(f"\nConfiguration:")
    print(f"  • Response Variables ({len(response_variables)}):")
    for rv in response_variables:
        print(f"      - {rv}: EN={get_response_label(rv, 'en')}, FR={get_response_label(rv, 'fr')} (color: {get_response_color(rv)})")
    print(f"  • Model: {model_name}")
    print(f"  • Plots per page: {plots_per_page}")
    print(f"  • Languages: English & French")
    print(f"  • Output: {output_dir}\n")
    
    try:
        # Load all models
        models, encoders_dict, feature_names_dict = load_multiple_models(
            base_dir, model_name, response_variables, folder_suffix
        )
        
        # Prepare data for all response variables
        X_train_dict = {}
        X_test_dict = {}
        y_train_dict = {}
        y_test_dict = {}
        feature_names_dict_final = {}
        scaler_dict = {}
        X_train_unscaled_dict = {}
        
        print(f"\n{'='*80}")
        print("PREPARING DATA FOR ALL RESPONSE VARIABLES")
        print(f"{'='*80}")
        
        for rv in response_variables:
            (X_train, X_test, y_train, y_test,
             features, scaler, df, X_train_unscaled) = load_and_prepare_data_for_model(
                data_file, rv, models[rv], custom_predictors, categorical_vars, 
                encoders_dict[rv]
            )
            
            X_train_dict[rv] = X_train
            X_test_dict[rv] = X_test
            y_train_dict[rv] = y_train
            y_test_dict[rv] = y_test
            feature_names_dict_final[rv] = features
            scaler_dict[rv] = scaler
            X_train_unscaled_dict[rv] = X_train_unscaled
        
        print(f"{'='*80}\n")
        
        # Get feature ranks from SHAP
        all_ranks = get_feature_ranks_from_shap(
            base_dir, response_variables, model_name, folder_suffix
        )
        
        # Create bilingual grid PDP plots
        created_files, model_output_dir, individual_files = create_grid_pdp_plots(
            models, X_train_dict, X_train_unscaled_dict,
            feature_names_dict_final, response_variables,
            output_dir, model_name, scaler_dict, all_ranks,
            plots_per_page=plots_per_page
        )
        
        # Create summary
        all_features_sets = [set(feature_names_dict_final[rv]) for rv in response_variables]
        all_features = sorted(list(set.union(*all_features_sets)))
        
        summary_data = []
        for feature in all_features:
            feature_label_en, unit_en = get_feature_display_info(feature, 'en')
            feature_label_fr, unit_fr = get_feature_display_info(feature, 'fr')
            
            row = {
                'feature': feature,
                'feature_label_en': feature_label_en,
                'feature_label_fr': feature_label_fr,
                'unit': unit_en
            }
            
            for rv in response_variables:
                in_model = feature in feature_names_dict_final[rv]
                row[f'in_{rv}_model'] = in_model
                row[f'{rv}_rank'] = all_ranks[rv].get(feature, 'N/A') if in_model else 'N/A'
            
            summary_data.append(row)
        
        summary_df = pd.DataFrame(summary_data)
        summary_file = os.path.join(model_output_dir, f'feature_summary_{model_name.replace(" ", "_")}.csv')
        summary_file = os.path.normpath(summary_file)
        summary_df.to_csv(summary_file, index=False)
        print(f"✓ Saved feature summary: {summary_file}\n")
        
        # Final summary
        print("\n" + "="*90)
        print("BILINGUAL ANALYSIS COMPLETE")
        print("="*90)
        print(f"✓ Analyzed {len(response_variables)} response variable(s)")
        print(f"✓ English: {len(created_files['en'])} grid pages + {len(individual_files['en'])} individual plots")
        print(f"✓ French: {len(created_files['fr'])} grid pages + {len(individual_files['fr'])} individual plots")
        print(f"✓ Grid plots (350 DPI): {model_output_dir}")
        print(f"✓ Individual plots (600 DPI):")
        print(f"    - English: {os.path.join(model_output_dir, 'individual_plots_en')}")
        print(f"    - French: {os.path.join(model_output_dir, 'individual_plots_fr')}")
        print("="*90 + "\n")
        
        return created_files, summary_df, model_output_dir, individual_files
        
    except Exception as e:
        print(f"\n✗ ERROR: {str(e)}")
        import traceback
        traceback.print_exc()
        return None, None, None, None

# ============================================================================
# EXAMPLE USAGE
# ============================================================================

if __name__ == "__main__":
    custom_features = [
        'pH_complete', 'Exchangeable_Calcium_meq_100g_complete', 
        'Exchangeable_Potassium_meq_100g_complete', 'Exchangeable_Sodium_meq_100g_complete',
        'Available_P2O5_mg_kg_complete', 'Total_Nitrogen_per_mille_complete', 
        'Electrical_Conductivity_mmho_cm_complete', 'Mg_K_Ratio_complete',
        'Coarse_Silt_percent_complete', 'Coarse_Sand_percent_complete', 
        'growing_season_rainfall_mm', 'CDD', 'SDII',
        'temp_mini_mean', 'temp_maxi_mean', 'h_mini_mean', 'h_maxi_mean',
        'soil_temp_6h_mean', 'radiation_ERA_total', 'CWD', 'Rx1', 'R20', 'TNn', 'TNx'
    ]
    
    categorical_features = ['variety', 'crop_cycle', 'irrigation_type', 'harvest_mode']
    
    print("\n" + "="*90)
    print("BILINGUAL PDP ANALYSIS - SUPPORTS 1-5 RESPONSE VARIABLES")
    print("="*90)
    print("\nSupported response variables:")
    print("  • tc_per_ha_actual (Blue)")
    print("  • brix_percent (Orange)")
    print("  • hugot_sugar (Green)")
    print("  • purity_percent (Red)")
    print("  • total_recoverable_sugar_t_per_ha (Purple)")
    print("="*90 + "\n")
    
    # ========================================================================
    # EXAMPLE 1: Single response variable
    # ========================================================================
    print("\n" + "="*90)
    print("EXAMPLE 1: SINGLE RESPONSE VARIABLE (BILINGUAL)")
    print("="*90)
    
    results_single = main_grid_pdp_analysis(
        response_variables=['total_recoverable_sugar_t_per_ha'],
        model_name='RandomForest',
        custom_predictors=custom_features,
        categorical_vars=categorical_features,
        plots_per_page=12
    )
    
    # ========================================================================
    # EXAMPLE 2: Two response variables
    # ========================================================================
    print("\n" + "="*90)
    print("EXAMPLE 2: TWO RESPONSE VARIABLES (BILINGUAL)")
    print("="*90)
    
    results_two = main_grid_pdp_analysis(
        response_variables=['tc_per_ha_actual', 'hugot_sugar'],
        model_name='RandomForest',
        custom_predictors=custom_features,
        categorical_vars=categorical_features,
        plots_per_page=12
    )
    
    print("\n" + "="*90)
    print("ALL EXAMPLES COMPLETED")
    print("="*90)
    print("Each plot is generated in both English and French")
    print("Legends show only rank (e.g., '1st', '2nd', '1er', '2e')")
    print("\nExample usage:")
    print("  results = main_grid_pdp_analysis(")
    print("      response_variables=['brix_percent', 'hugot_sugar'],")
    print("      model_name='RandomForest',")
    print("      ...)")
    print("="*90 + "\n")


BILINGUAL PDP ANALYSIS - SUPPORTS 1-5 RESPONSE VARIABLES

Supported response variables:
  • tc_per_ha_actual (Blue)
  • brix_percent (Orange)
  • hugot_sugar (Green)
  • purity_percent (Red)
  • total_recoverable_sugar_t_per_ha (Purple)


EXAMPLE 1: SINGLE RESPONSE VARIABLE (BILINGUAL)

BILINGUAL GRID PDP ANALYSIS - 1 RESPONSE VARIABLE(S)

Configuration:
  • Response Variables (1):
      - total_recoverable_sugar_t_per_ha: EN=Total Recoverable Sugar (t ha⁻¹), FR=Sucre récupérable total (t ha⁻¹) (color: #2ca02c)
  • Model: RandomForest
  • Plots per page: 12
  • Languages: English & French
  • Output: C:\Users\U078-N800\Desktop\Ml_Internship\Copy\Ml_Training\Regrouped_Analysis\combined_pdp_grid_bilingual_total_recoverable_sugar_t_per_ha_with_categorical


LOADING MODELS FOR 1 RESPONSE VARIABLE(S)
  Loading RandomForest model for total_recoverable_sugar_t_per_ha...
    ✓ Loaded (features: 28, encoders: 5)


PREPARING DATA FOR ALL RESPONSE VARIABLES

  Loading and preparing data for tota

KeyboardInterrupt: 

## 8. Stratified Partial Dependence Analysis

This section extends PDP analysis by stratifying predictions across categorical variables (variety, crop cycle, irrigation type, harvest mode) to reveal how feature-response relationships vary between management practices or cultivars.

**Analytical Approach:**

**I. Category-Specific PDPs**
- Generate separate PDP curves for top N categories (e.g., top 6 varieties by sample size)
- Each category rendered with distinct color from RColorBrewer palette
- Consistent color mapping across all plots (alphabetically sorted categories)

**II. Statistical Rigor**
- Minimum sample threshold (default: 30 observations per category)
- Bootstrap confidence intervals (100 iterations, 95% CI) for each stratified curve
- LOWESS smoothing with spline interpolation for publication-ready curves

**III. Category Filtering**
- **Crop cycle**: Automatically excludes late ratoons (9th+, "Beyond 12th") with insufficient data
- Ensures balanced representation and prevents spurious patterns

**IV. Visualization Features**
- **Transparent legends**: Smart positioning to avoid overlapping curves/rug marks
- **Dual output formats**: Standalone plots (600 DPI) + combined multi-panel figures
- **Rug plots**: Data distribution within PDP range, filtered by category

**Output:** Individual stratified PDPs per feature + multi-panel summary figure showing all top features side-by-side for direct comparison across categories.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import glob
import re
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.inspection import partial_dependence, PartialDependenceDisplay
from scipy.interpolate import make_interp_spline
from scipy.ndimage import gaussian_filter1d
from statsmodels.nonparametric.smoothers_lowess import lowess
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# EXCLUDED CATEGORIES (for crop_cycle stratification)
# ============================================================================

EXCLUDED_CATEGORIES = [
    '9th Ratoon',
    '10th Ratoon', 
    '11th Ratoon',
    'Beyond 12th Ratoon'
]

# ============================================================================
# DEFINE AVAILABLE RESPONSE VARIABLES
# ============================================================================

SUGAR_CONTENT_METRICS = {
    "brix_percent": "Brix (%)",
    "pol_juice_percent": "Pol juice (%)",
    "pol_cane_percent": "Pol cane (%)",
    "purity_percent": "Purity (%)",
    "glucose_cane_percent": "Glucose (%)",
    "hugot_sugar": "Hugot sugar (%)"
}

YIELD_PARAMETERS = {
    "tc_per_ha_actual": "Cane yield (t ha⁻¹)",
    "tc_per_ha_per_week": "Cane yield per week (t ha⁻¹/week)",
    "tc_per_ha_52_weeks": "Cane yield 52 weeks (t ha⁻¹)",
    "fiber_percent": "Fiber in Cane (%)",
}

ALL_RESPONSE_VARIABLES = {**SUGAR_CONTENT_METRICS, **YIELD_PARAMETERS}

# ============================================================================

# ============================================================================

plt.rcParams.update({
    'font.size': 8,
    'font.family': 'Arial',
    'axes.labelsize': 8,
    'axes.titlesize': 9,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 6.5,
    'figure.titlesize': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': False,
    'axes.axisbelow': True,
    'axes.edgecolor': '#2d2d2d',
    'text.color': '#2d2d2d',
    'axes.labelcolor': '#2d2d2d',
    'xtick.color': '#2d2d2d',
    'ytick.color': '#2d2d2d',
    'axes.linewidth': 0.75,
    'xtick.major.width': 0.75,
    'ytick.major.width': 0.75,
    'xtick.major.size': 3,
    'ytick.major.size': 3,
    'figure.dpi': 300,
    'savefig.dpi': 600,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.05
})

# ============================================================================
# COLOR PALETTE - RColorBrewer-inspired (consistent with categorical analysis)
# ============================================================================

#CATEGORY_COLORS = [
#    '#E41A1C', '#377EB8', '#4DAF4A', '#984EA3', '#FF7F00', '#FFFF33', '#A65628', '#F781BF', '#999999',
#    '#1B9E77', '#D95F02', '#7570B3', '#E7298A', '#66A61E', '#E6AB02', '#A6761D', '#666666',
#    '#A6CEE3', '#1F78B4', '#B2DF8A', '#33A02C', '#FB9A99', '#E31A1C', '#FDBF6F', '#FF7F00',
#    '#8DD3C7', '#FFFFB3', '#BEBADA', '#FB8072', '#80B1D3',
#]

#For harvest_mode
CATEGORY_COLORS = ['#D4A373', '#8B4513']
#For irrigation_type
#CATEGORY_COLORS = ['#4A90E2', '#E63946']

def get_consistent_category_colors(categories):
    """Assign consistent colors to categories based on ALPHABETICAL sorting"""
    sorted_categories = sorted(categories)
    color_map = {}
    for i, cat in enumerate(sorted_categories):
        color_map[str(cat)] = CATEGORY_COLORS[i % len(CATEGORY_COLORS)]
    return color_map

# ============================================================================
# RUG PLOT FUNCTION
# ============================================================================

def add_rug_plot(ax, data_values, x_min, x_max, color='#333333', alpha=0.3, height=0.03):
    """Add rug plot filtered to PDP x-axis range"""
    try:
        data_values_filtered = data_values[(data_values >= x_min) & (data_values <= x_max)]
        if len(data_values_filtered) == 0:
            return
        
        y_min, y_max = ax.get_ylim()
        y_range = y_max - y_min
        rug_y_position = y_min + (y_range * 0.02)
        rug_height = y_range * height
        
        if len(data_values_filtered) > 1000:
            sample_indices = np.random.choice(len(data_values_filtered), 1000, replace=False)
            data_values_sample = data_values_filtered[sample_indices]
        else:
            data_values_sample = data_values_filtered
        
        for val in data_values_sample:
            ax.plot([val, val], [rug_y_position, rug_y_position + rug_height],
                   color=color, alpha=alpha, linewidth=0.6, zorder=1)
    except Exception:
        pass

# ============================================================================
# SMOOTHING FUNCTIONS
# ============================================================================

def ultra_smooth_curve_perfect_monotonic(x, y, method='lowess', frac=0.4, n_bootstrap=100, confidence_level=0.95):
    """PERFECT SMOOTHING for PDP curves"""
    try:
        sorted_indices = np.argsort(x)
        x_sorted = x[sorted_indices]
        y_sorted = y[sorted_indices]
        
        if len(x_sorted) <= 3:
            return x, y, None, None
        
        unique_x, inverse_indices = np.unique(x_sorted, return_inverse=True)
        unique_y = np.array([np.mean(y_sorted[inverse_indices == i]) for i in range(len(unique_x))])
        
        if len(unique_x) < 5:
            return x, y, None, None
        
        if method == 'lowess':
            smoothed = lowess(unique_y, unique_x, frac=frac, it=10, return_sorted=True)
            smooth_x = smoothed[:, 0]
            smooth_y = smoothed[:, 1]
            
            sigma_base = len(smooth_y) * 0.15
            for i in range(6):
                sigma = sigma_base * (1.0 - i * 0.1)
                smooth_y = gaussian_filter1d(smooth_y, sigma=sigma)
            
            smooth_y = gaussian_filter1d(smooth_y, sigma=len(smooth_y) * 0.2)
            
            try:
                n_spline = min(50, len(smooth_x))
                x_spline = np.linspace(smooth_x.min(), smooth_x.max(), n_spline)
                spline = make_interp_spline(smooth_x, smooth_y, k=2, bc_type='natural')
                y_spline = spline(x_spline)
                y_spline = gaussian_filter1d(y_spline, sigma=len(y_spline) * 0.25)
                smooth_x = x_spline
                smooth_y = y_spline
            except:
                pass
            
            if n_bootstrap > 0:
                bootstrap_curves = []
                n_points = len(unique_x)
                
                for _ in range(n_bootstrap):
                    bootstrap_indices = np.random.choice(n_points, n_points, replace=True)
                    boot_x = unique_x[bootstrap_indices]
                    boot_y = unique_y[bootstrap_indices]
                    boot_sort_idx = np.argsort(boot_x)
                    boot_x_sorted = boot_x[boot_sort_idx]
                    boot_y_sorted = boot_y[boot_sort_idx]
                    
                    try:
                        boot_smoothed = lowess(boot_y_sorted, boot_x_sorted, frac=frac, it=8, return_sorted=True)
                        boot_y_smooth = boot_smoothed[:, 1]
                        for i in range(4):
                            sigma = len(boot_y_smooth) * 0.12 * (1.0 - i * 0.1)
                            boot_y_smooth = gaussian_filter1d(boot_y_smooth, sigma=sigma)
                        boot_y_smooth = gaussian_filter1d(boot_y_smooth, sigma=len(boot_y_smooth) * 0.2)
                        boot_interp_y = np.interp(smooth_x, boot_smoothed[:, 0], boot_y_smooth)
                        bootstrap_curves.append(boot_interp_y)
                    except:
                        continue
                
                if bootstrap_curves:
                    bootstrap_curves = np.array(bootstrap_curves)
                    alpha = (1 - confidence_level) / 2
                    ci_lower = np.percentile(bootstrap_curves, alpha * 100, axis=0)
                    ci_upper = np.percentile(bootstrap_curves, (1 - alpha) * 100, axis=0)
                    ci_lower = gaussian_filter1d(ci_lower, sigma=len(ci_lower) * 0.15)
                    ci_upper = gaussian_filter1d(ci_upper, sigma=len(ci_upper) * 0.15)
                    return smooth_x, smooth_y, ci_lower, ci_upper
            
            return smooth_x, smooth_y, None, None
    except Exception as e:
        return x, y, None, None

# ============================================================================
# CATEGORY FILTERING FUNCTIONS
# ============================================================================

def filter_excluded_categories(df, cat_var):
    """Filter out excluded categories from dataframe (for crop_cycle)"""
    initial_len = len(df)
    
    if cat_var.lower() in ['crop_cycle', 'cycle']:
        df = df.copy()
        df[cat_var] = df[cat_var].astype(str)
        excluded_str = [str(x) for x in EXCLUDED_CATEGORIES]
        df_filtered = df[~df[cat_var].isin(excluded_str)].copy()
        n_excluded = initial_len - len(df_filtered)
        
        if n_excluded > 0:
            print(f"\n⚠ Excluded {n_excluded} rows with crop cycles: {', '.join(EXCLUDED_CATEGORIES)}")
            print(f"  Remaining rows: {len(df_filtered):,}")
        
        return df_filtered, n_excluded
    else:
        return df, 0

# ============================================================================
# STRATIFICATION HELPER FUNCTIONS
# ============================================================================

def get_top_n_varieties(X_train_raw, stratify_feature, n_top=6, min_samples=30):
    """Get top N varieties by sample count"""
    X_train_filtered, n_excluded = filter_excluded_categories(X_train_raw, stratify_feature)
    unique_categories = X_train_filtered[stratify_feature].dropna().unique()
    
    category_sample_counts = {}
    for category in unique_categories:
        category_mask = X_train_filtered[stratify_feature] == category
        n_samples = category_mask.sum()
        if n_samples >= min_samples:
            category_sample_counts[category] = n_samples
    
    sorted_categories = sorted(category_sample_counts.items(), 
                              key=lambda x: x[1], reverse=True)
    top_varieties = [cat for cat, count in sorted_categories[:n_top]]
    
    print(f"\n{'='*60}")
    print(f"TOP {n_top} {stratify_feature.upper()} BY SAMPLE COUNT:")
    print(f"{'='*60}")
    for i, (variety, count) in enumerate(sorted_categories[:n_top], 1):
        print(f"  {i}. {variety}: {count} samples")
    print(f"{'='*60}\n")
    
    return top_varieties

def transform_to_original_scale(scaled_values, feature_idx, scaler, feature_names):
    """Transform scaled values back to original scale"""
    n_features = len(feature_names)
    dummy_scaled = np.zeros((len(scaled_values), n_features))
    dummy_scaled[:, feature_idx] = scaled_values
    dummy_original = scaler.inverse_transform(dummy_scaled)
    return dummy_original[:, feature_idx]

# ============================================================================
# FEATURE DISPLAY INFO
# ============================================================================

def get_feature_display_info(feature_name):
    """Get display label and units for features"""
    feature_mapping = {
        'temp_moy_mean': {'label': 'Mean temperature', 'unit': '°C'},
        'temp_maxi_mean': {'label': 'Maximum temperature', 'unit': '°C'},
        'temp_mini_mean': {'label': 'Minimum temperature', 'unit': '°C'},
        'precipitation_total': {'label': 'Total precipitation', 'unit': 'mm'},
        'growing_season_rainfall_mm': {'label': 'Growing season rainfall', 'unit': 'mm'},
        'radiation_ERA_total': {'label': 'Solar radiation', 'unit': 'MJ m⁻²'},
        'evaporation_total': {'label': 'Total evaporation', 'unit': 'mm'},
        'CWD': {'label': 'Consecutive wet days', 'unit': 'days'},
        'CDD': {'label': 'Consecutive dry days', 'unit': 'days'},
        'SDII': {'label': 'Daily intensity index', 'unit': 'mm day⁻¹'},
        'R10': {'label': 'Heavy precipitation days', 'unit': 'days'},
        'R20': {'label': 'Very heavy precip. days', 'unit': 'days'},
        'Rx1': {'label': 'Max 1-day precipitation', 'unit': 'mm'},
        'Rx5': {'label': 'Max 5-day precipitation', 'unit': 'mm'},
        'R95p': {'label': 'Very wet days precip.', 'unit': 'mm'},
        'R99p': {'label': 'Extremely wet days precip.', 'unit': 'mm'},
        'HD': {'label': 'Hot days', 'unit': 'days'},
        'TNn': {'label': 'Min daily min temp.', 'unit': '°C'},
        'TNx': {'label': 'Max daily min temp.', 'unit': '°C'},
        'TXn': {'label': 'Min daily max temp.', 'unit': '°C'},
        'TXx': {'label': 'Max daily max temp.', 'unit': '°C'},
        'soil_temp_6h_mean': {'label': 'Soil temperature (6h)', 'unit': '°C'},
        'soil_temp_16h_mean': {'label': 'Soil temperature (16h)', 'unit': '°C'},
        'pH_complete': {'label': 'Soil pH', 'unit': ''},
        'Clay_percent_complete': {'label': 'Clay content', 'unit': '%'},
        'Fine_Silt_percent_complete': {'label': 'Fine silt', 'unit': '%'},
        'Coarse_Silt_percent_complete': {'label': 'Coarse silt', 'unit': '%'},
        'Fine_Sand_percent_complete': {'label': 'Fine sand', 'unit': '%'},
        'Coarse_Sand_percent_complete': {'label': 'Coarse sand', 'unit': '%'},
        'Total_Nitrogen_per_mille_complete': {'label': 'Total nitrogen', 'unit': '‰'},
        'Organic_Carbon_per_mille_complete': {'label': 'Organic carbon', 'unit': '‰'},
        'Exchangeable_Calcium_meq_100g_complete': {'label': 'Exchangeable Ca', 'unit': 'meq 100g⁻¹'},
        'Exchangeable_Magnesium_meq_100g_complete': {'label': 'Exchangeable Mg', 'unit': 'meq 100g⁻¹'},
        'Exchangeable_Potassium_meq_100g_complete': {'label': 'Exchangeable K', 'unit': 'meq 100g⁻¹'},
        'Exchangeable_Sodium_meq_100g_complete': {'label': 'Exchangeable Na', 'unit': 'meq 100g⁻¹'},
        'Available_P2O5_mg_kg_complete': {'label': 'Available P₂O₅', 'unit': 'mg kg⁻¹'},
        'Electrical_Conductivity_mmho_cm_complete': {'label': 'Electrical conductivity', 'unit': 'mmho cm⁻¹'},
        'Mg_K_Ratio_complete': {'label': 'Mg:K ratio', 'unit': ''},
        'Mg_Ca_Ratio_complete': {'label': 'Mg:Ca ratio', 'unit': ''},
        'ampl_mean': {'label': 'Temperature amplitude', 'unit': '°C'},
        'h_mini_mean': {'label': 'Min humidity', 'unit': '%'},
        'h_maxi_mean': {'label': 'Max humidity', 'unit': '%'},
        'h_moy_mean': {'label': 'Mean humidity', 'unit': '%'},
    }
    
    info = feature_mapping.get(feature_name, {'label': feature_name.replace('_', ' '), 'unit': ''})
    return info['label'], info['unit']

# ============================================================================
# LEGEND POSITION OPTIMIZATION
# ============================================================================

def find_optimal_legend_position(ax, category_data, valid_categories):
    """Find optimal legend position avoiding rug marks and curves"""
    all_curves = []
    for cat in valid_categories:
        data = category_data[cat]
        all_curves.append((data['x'], data['y']))
    
    x_min, x_max = ax.get_xlim()
    y_min, y_max = ax.get_ylim()
    x_range = x_max - x_min
    y_range = y_max - y_min
    
    rug_zone_top = y_min + 0.15 * y_range
    
    regions = {
        'upper right': {'x': (x_min + 0.6*x_range, x_max), 'y': (y_min + 0.7*y_range, y_max)},
        'upper left': {'x': (x_min, x_min + 0.4*x_range), 'y': (y_min + 0.7*y_range, y_max)},
        'center right': {'x': (x_min + 0.65*x_range, x_max), 'y': (rug_zone_top + 0.2*y_range, y_min + 0.65*y_range)},
        'center left': {'x': (x_min, x_min + 0.35*x_range), 'y': (rug_zone_top + 0.2*y_range, y_min + 0.65*y_range)},
    }
    
    region_scores = {}
    for region_name, bounds in regions.items():
        score = 0
        for x_vals, y_vals in all_curves:
            in_region = ((x_vals >= bounds['x'][0]) & (x_vals <= bounds['x'][1]) &
                        (y_vals >= bounds['y'][0]) & (y_vals <= bounds['y'][1]))
            score += np.sum(in_region)
        region_scores[region_name] = int(score)
    
    best_region = min(region_scores, key=region_scores.get)
    print(f"      Legend scores: {region_scores}, Selected: {best_region}")
    
    return best_region

# ============================================================================
# ENCODING HELPER FUNCTIONS
# ============================================================================

def normalize_ordinal(text):
    """Normalize ordinal numbers (1ST -> 1st, 2ND -> 2nd)"""
    text = str(text).strip()
    def ordinal_replacement(match):
        num = match.group(1)
        suffix = match.group(2).lower()
        return f"{num}{suffix}"
    text = re.sub(r'(\d+)(ST|ND|RD|TH)\b', ordinal_replacement, text, flags=re.IGNORECASE)
    return text

def encode_categorical_value(value, encoder, cat_var):
    """Encode categorical value with robust string matching"""
    if pd.isna(value):
        return np.nan
    
    original = str(value).strip()
    normalized = normalize_ordinal(original)
    
    attempts = [
        normalized.title(), original, original.title(), original.upper(), 
        original.lower(), normalized, normalized.upper(), normalized.lower(),
    ]
    
    for attempt in attempts:
        try:
            if attempt in encoder.classes_:
                return encoder.transform([attempt])[0]
        except:
            continue
    
    return -1

# ============================================================================
# STRATIFIED PDP PLOTTING FUNCTIONS
# ============================================================================

def create_stratified_pdp_for_feature_enhanced(model, X_train_scaled, feature_idx, feature_name,
                                     stratify_feature, stratify_encoder, X_train_raw,
                                     scaler, feature_names, output_dir, model_name, 
                                     response_label="Effect",
                                     min_samples=30, top_n_varieties=6, figsize=(3.5, 2.8)):
    """Create publication-quality stratified PDP with transparent legend"""
    print(f"    Creating stratified PDP for {feature_name} by {stratify_feature}...")
    
    try:
        top_varieties = get_top_n_varieties(X_train_raw, stratify_feature, 
                                           n_top=top_n_varieties, min_samples=min_samples)
        
        if not top_varieties:
            print(f"    ✗ No varieties with sufficient samples")
            return None, None
        
        stratify_col = f'{stratify_feature}_encoded'
        if stratify_col not in feature_names:
            print(f"    ✗ Stratify feature {stratify_col} not found")
            return None, None
        
        stratify_idx = feature_names.index(stratify_col)
        
        category_data = {}
        valid_categories = []
        all_pdp_x = []
        
        for category in top_varieties:
            try:
                category_mask = X_train_raw[stratify_feature] == category
                n_samples = category_mask.sum()
                
                encoded_val = encode_categorical_value(category, stratify_encoder, stratify_feature)
                if encoded_val == -1:
                    print(f"    ⚠ Could not encode: {category}")
                    continue
                
                dummy_unscaled = np.zeros((1, len(feature_names)))
                dummy_unscaled[0, stratify_idx] = encoded_val
                dummy_scaled = scaler.transform(dummy_unscaled)
                
                X_category = X_train_scaled[category_mask.values]
                if len(X_category) < 10:
                    continue
                
                pd_results = partial_dependence(
                    model, X_category, features=[feature_idx], grid_resolution=200
                )
                
                if hasattr(pd_results, 'average'):
                    pd_values = pd_results.average[0]
                    scaled_grid_values = pd_results.grid_values[0]
                else:
                    pd_values = pd_results[0][0]
                    scaled_grid_values = pd_results[1][0]
                
                original_grid_values = transform_to_original_scale(
                    scaled_grid_values, feature_idx, scaler, feature_names
                )
                
                smooth_x, smooth_y, ci_lower, ci_upper = ultra_smooth_curve_perfect_monotonic(
                    original_grid_values, pd_values, method='lowess', 
                    frac=0.40, n_bootstrap=100, confidence_level=0.95
                )
                
                category_data[category] = {
                    'x': smooth_x, 'y': smooth_y,
                    'ci_lower': ci_lower, 'ci_upper': ci_upper,
                    'n_samples': n_samples
                }
                valid_categories.append(category)
                all_pdp_x.extend(smooth_x)
                
            except Exception:
                continue
        
        if not valid_categories:
            print(f"    ✗ No valid categories")
            return None, None
        
        color_map = get_consistent_category_colors(valid_categories)
        for cat in valid_categories:
            category_data[cat]['color'] = color_map[str(cat)]
        
        n_varieties = len(valid_categories)
        
        all_x, all_y = [], []
        for cat in valid_categories:
            data = category_data[cat]
            all_x.extend(data['x'])
            all_y.extend(data['y'])
            if data['ci_lower'] is not None:
                all_y.extend(data['ci_lower'])
                all_y.extend(data['ci_upper'])
        
        x_min, x_max = np.min(all_x), np.max(all_x)
        y_min, y_max = np.min(all_y), np.max(all_y)
        x_range = x_max - x_min
        y_range = y_max - y_min
        x_buffer = x_range * 0.03
        y_buffer = y_range * 0.08
        
        fig, ax = plt.subplots(figsize=figsize)
        
        for category in valid_categories:
            data = category_data[category]
            
            if data['ci_lower'] is not None:
                ax.fill_between(data['x'], data['ci_lower'], data['ci_upper'],
                              alpha=0.15, color=data['color'], linewidth=0, zorder=2)
            
            ax.plot(data['x'], data['y'], linewidth=1.8, color=data['color'], alpha=0.9,
                   label=f"{str(category)[:15]}", solid_capstyle='round', zorder=5)
        
        ax.set_xlim(x_min - x_buffer, x_max + x_buffer)
        ax.set_ylim(y_min - y_buffer, y_max + y_buffer)
        
        # Add rug marks
        if feature_name in X_train_raw.columns:
            feature_data = X_train_raw[feature_name].dropna().values
            add_rug_plot(ax, feature_data, x_min, x_max, color='#333333', alpha=0.3, height=0.03)
        
        # Zero line
        y_lim = ax.get_ylim()
        if y_lim[0] < 0 < y_lim[1]:
            ax.axhline(0, color='#cccccc', linestyle='-', linewidth=0.75, alpha=0.8, zorder=1)
        
        feature_label, unit = get_feature_display_info(feature_name)
        xlabel = f"{feature_label}" + (f" ({unit})" if unit else "")
        
        ax.set_xlabel(xlabel, fontsize=8, labelpad=3)
        ax.set_ylabel(f'{response_label}', fontsize=8, labelpad=3)
        ax.set_title(f'{feature_label}', fontsize=9, fontweight='bold', pad=6)
        
        # Transparent legend
        legend_loc = find_optimal_legend_position(ax, category_data, valid_categories)
        stratify_display = stratify_feature.replace('_', ' ').capitalize()
        
        if n_varieties <= 3:
            legend = ax.legend(loc=legend_loc, frameon=False, fontsize=5, ncol=1, 
                             title=stratify_display, title_fontsize=5.5,
                             handlelength=1.0, labelspacing=0.2, borderpad=0.2)
        else:
            legend = ax.legend(loc=legend_loc, frameon=False, fontsize=4.5, ncol=2,
                             title=stratify_display, title_fontsize=5,
                             handlelength=0.9, labelspacing=0.15, columnspacing=0.6, borderpad=0.2)
        
        legend.get_title().set_fontweight('bold')
        for text in legend.get_texts():
            text.set_alpha(0.9)
        legend.get_title().set_alpha(0.9)
        
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.tick_params(axis='both', labelsize=7, length=3, width=0.75, pad=2)
        
        plt.tight_layout()
        
        safe_feature_name = feature_name.replace(' ', '_')
        safe_stratify_name = stratify_feature.replace(' ', '_')
        
        output_file = os.path.join(output_dir, 
                                   f'pdp_stratified_{safe_feature_name}_by_{safe_stratify_name}.png')
        plt.savefig(output_file, dpi=600, bbox_inches='tight', facecolor='white')
        
        pdf_file = output_file.replace('.png', '.pdf')
        plt.savefig(pdf_file, bbox_inches='tight', facecolor='white')
        
        plt.close()
        
        print(f"    ✓ Created stratified PDP ({n_varieties} varieties) - legend at {legend_loc}")
        return output_file, category_data
        
    except Exception as e:
        print(f"    ✗ Error: {str(e)}")
        plt.close()
        return None, None

def create_combined_stratified_pdp_panel(model, X_train_scaled, feature_names, top_features,
                                        stratify_feature, stratify_encoder, X_train_raw,
                                        scaler, output_dir, model_name, response_label="Effect",
                                        categorical_features=None,
                                        min_samples=30, top_n_varieties=6):
    """Create combined panel of stratified PDPs"""
    print(f"\n=== CREATING COMBINED PANEL (TOP {top_n_varieties} VARIETIES) ===\n")
    
    top_varieties = get_top_n_varieties(X_train_raw, stratify_feature, 
                                       n_top=top_n_varieties, min_samples=min_samples)
    
    if not top_varieties:
        print("No varieties with sufficient samples")
        return None
    
    # Filter to numeric features only
    numeric_features = []
    for feature in top_features:
        is_categorical = False
        if categorical_features:
            for cat_var in categorical_features:
                if f'{cat_var}_encoded' == feature:
                    is_categorical = True
                    break
        if not is_categorical:
            numeric_features.append(feature)
    
    n_features = len(numeric_features)
    if n_features == 0:
        return None
    
    if n_features <= 4:
        n_cols = 2
    elif n_features <= 9:
        n_cols = 3
    else:
        n_cols = 4
    
    n_rows = int(np.ceil(n_features / n_cols))
    fig_width = 7.0
    fig_height = 2.2 * n_rows
    fig = plt.figure(figsize=(fig_width, fig_height))
    
    stratify_col = f'{stratify_feature}_encoded'
    stratify_idx = feature_names.index(stratify_col)
    
    color_map = get_consistent_category_colors(top_varieties)
    
    for plot_idx, feature in enumerate(numeric_features):
        try:
            feature_idx = feature_names.index(feature)
            ax = plt.subplot(n_rows, n_cols, plot_idx + 1)
            
            category_data = {}
            all_x, all_y = [], []
            
            for category in top_varieties:
                try:
                    category_mask = X_train_raw[stratify_feature] == category
                    encoded_val = encode_categorical_value(category, stratify_encoder, stratify_feature)
                    if encoded_val == -1:
                        continue
                    
                    dummy_unscaled = np.zeros((1, len(feature_names)))
                    dummy_unscaled[0, stratify_idx] = encoded_val
                    dummy_scaled = scaler.transform(dummy_unscaled)
                    
                    X_category = X_train_scaled[category_mask.values]
                    if len(X_category) < 10:
                        continue
                    
                    pd_results = partial_dependence(model, X_category, features=[feature_idx], grid_resolution=150)
                    
                    if hasattr(pd_results, 'average'):
                        pd_values = pd_results.average[0]
                        scaled_grid_values = pd_results.grid_values[0]
                    else:
                        pd_values = pd_results[0][0]
                        scaled_grid_values = pd_results[1][0]
                    
                    original_grid_values = transform_to_original_scale(
                        scaled_grid_values, feature_idx, scaler, feature_names
                    )
                    
                    smooth_x, smooth_y, ci_lower, ci_upper = ultra_smooth_curve_perfect_monotonic(
                        original_grid_values, pd_values, method='lowess', 
                        frac=0.40, n_bootstrap=50, confidence_level=0.95
                    )
                    
                    category_data[category] = {
                        'x': smooth_x, 'y': smooth_y,
                        'ci_lower': ci_lower, 'ci_upper': ci_upper
                    }
                    
                    all_x.extend(smooth_x)
                    all_y.extend(smooth_y)
                    if ci_lower is not None:
                        all_y.extend(ci_lower)
                        all_y.extend(ci_upper)
                except Exception:
                    continue
            
            if not category_data:
                continue
            
            x_min, x_max = np.min(all_x), np.max(all_x)
            y_min, y_max = np.min(all_y), np.max(all_y)
            x_range = x_max - x_min
            y_range = y_max - y_min
            
            ax.set_xlim(x_min - x_range*0.02, x_max + x_range*0.02)
            ax.set_ylim(y_min - y_range*0.06, y_max + y_range*0.06)
            
            for category in top_varieties:
                if category not in category_data:
                    continue
                data = category_data[category]
                color = color_map[category]
                
                if data['ci_lower'] is not None:
                    ax.fill_between(data['x'], data['ci_lower'], data['ci_upper'],
                                  alpha=0.12, color=color, linewidth=0, zorder=2)
                
                ax.plot(data['x'], data['y'], linewidth=1.5, color=color, alpha=0.9,
                        solid_capstyle='round', zorder=5)
            
            # Add rug marks
            if feature in X_train_raw.columns:
                feature_data = X_train_raw[feature].dropna().values
                add_rug_plot(ax, feature_data, x_min, x_max, color='#333333', alpha=0.25, height=0.025)
            
            # Zero line
            y_lim = ax.get_ylim()
            if y_lim[0] < 0 < y_lim[1]:
                ax.axhline(0, color='#cccccc', linestyle='-', linewidth=0.6, alpha=0.7, zorder=1)
            
            feature_label, unit = get_feature_display_info(feature)
            xlabel = feature_label + (f"\n({unit})" if unit else "")
            ax.set_xlabel(xlabel, fontsize=7, labelpad=2)
            
            if plot_idx % n_cols == 0:
                ax.set_ylabel(f'{response_label}', fontsize=7, labelpad=2)
            
            ax.set_title(feature_label, fontsize=8, fontweight='bold', pad=4)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.tick_params(axis='both', labelsize=6.5, length=2.5, width=0.6, pad=1.5)
            
            print(f"    ✓ Panel {plot_idx+1}/{n_features}: {feature}")
            
        except Exception as e:
            print(f"    ✗ Error for {feature}: {str(e)}")
            continue
    
    handles = [plt.Line2D([0], [0], color=color_map[c], linewidth=1.8) for c in top_varieties]
    labels = [str(c)[:15] for c in top_varieties]
    stratify_display = stratify_feature.replace('_', ' ').capitalize()
    
    ncol_legend = min(len(top_varieties), 6)
    fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.02),
               frameon=False, fontsize=6, ncol=ncol_legend,
               title=stratify_display, title_fontsize=6.5,
               handlelength=1.5, labelspacing=0.4, columnspacing=1.0, borderpad=0.4)
    
    plt.suptitle(f'Stratified Partial Dependence: {model_name} (Top {len(top_varieties)} Varieties)',
                 fontsize=10, fontweight='bold', y=0.995)
    
    plt.tight_layout(rect=[0, 0.05, 1, 0.99], h_pad=2.0, w_pad=1.5)
    
    safe_stratify = stratify_feature.replace(' ', '_')
    output_file = os.path.join(output_dir, f'pdp_stratified_panel_by_{safe_stratify}.png')
    pdf_file = os.path.join(output_dir, f'pdp_stratified_panel_by_{safe_stratify}.pdf')
    
    plt.savefig(output_file, dpi=600, bbox_inches='tight', facecolor='white')
    plt.savefig(pdf_file, bbox_inches='tight', facecolor='white')
    plt.close()
    
    print(f"\n✓ Created combined panel: {output_file}")
    return output_file

def create_all_stratified_pdps_enhanced(model, X_train_scaled, feature_names, top_features,
                                       stratify_feature, stratify_encoder, X_train_raw,
                                       scaler, output_dir, model_name, response_label="Effect",
                                       categorical_features=None,
                                       min_samples=30, top_n_varieties=6):
    """Create both standalone and combined stratified PDPs"""
    print(f"\n=== CREATING STRATIFIED PDPs BY {stratify_feature.upper()} ===")
    
    numeric_features = []
    for feature in top_features:
        is_categorical = False
        if categorical_features:
            for cat_var in categorical_features:
                if f'{cat_var}_encoded' == feature:
                    is_categorical = True
                    break
        if not is_categorical:
            numeric_features.append(feature)
    
    print(f"Creating stratified PDPs for {len(numeric_features)} numeric features\n")
    
    stratified_plots = []
    all_category_data = {}
    
    for feature in numeric_features:
        try:
            feature_idx = feature_names.index(feature)
            
            plot_file, cat_data = create_stratified_pdp_for_feature_enhanced(
                model, X_train_scaled, feature_idx, feature,
                stratify_feature, stratify_encoder, X_train_raw,
                scaler, feature_names, output_dir, model_name,
                response_label=response_label,
                min_samples=min_samples, top_n_varieties=top_n_varieties
            )
            
            if plot_file:
                stratified_plots.append(plot_file)
                all_category_data[feature] = cat_data
                
        except Exception as e:
            print(f"    ✗ Error for {feature}: {str(e)}")
            continue
    
    print(f"\n✓ Created {len(stratified_plots)} standalone stratified plots")
    
    panel_file = create_combined_stratified_pdp_panel(
        model, X_train_scaled, feature_names, top_features,
        stratify_feature, stratify_encoder, X_train_raw,
        scaler, output_dir, model_name, response_label,
        categorical_features, min_samples=min_samples, top_n_varieties=top_n_varieties
    )
    
    return stratified_plots, panel_file

# ============================================================================
# DATA LOADING FUNCTIONS
# ============================================================================

def load_and_prepare_data(file_path, response_variable, custom_predictors=None, 
                         categorical_vars=None, model_encoders=None):
    """Load and prepare data for PDP analysis - USE MODEL ENCODERS"""
    print(f"Loading data for {response_variable}...")
    
    df = pd.read_csv(file_path)
    
    if response_variable not in df.columns:
        raise ValueError(f"Response variable '{response_variable}' not found!")
    
    all_response_cols = list(ALL_RESPONSE_VARIABLES.keys())
    cols_to_drop = [col for col in all_response_cols 
                    if col != response_variable and col in df.columns]
    
    if cols_to_drop:
        df = df.drop(columns=cols_to_drop)
    
    if custom_predictors is not None:
        year_column = None
        for col in ['campaign_year', 'Harvest_Year', 'year']:
            if col in df.columns:
                year_column = col
                break
        
        columns_to_keep = [response_variable] + custom_predictors
        if year_column:
            columns_to_keep.append(year_column)
        if categorical_vars:
            columns_to_keep.extend(categorical_vars)
        
        df = df[columns_to_keep]
    
    df = df.dropna(subset=[response_variable])
    
    encoders = model_encoders if model_encoders is not None else {}
    
    if categorical_vars:
        print(f"\nEncoding categorical variables: {categorical_vars}")
        for cat_var in categorical_vars:
            if cat_var in df.columns:
                if cat_var in encoders:
                    print(f"   Using model encoder for {cat_var}")
                    cat_encoder = encoders[cat_var]
                    
                    df[f'{cat_var}_encoded'] = df[cat_var].apply(
                        lambda x: encode_categorical_value(x, cat_encoder, cat_var)
                    )
                    
                    valid_count = (df[f'{cat_var}_encoded'] != -1).sum()
                    invalid_count = (df[f'{cat_var}_encoded'] == -1).sum()
                    print(f"     Valid: {valid_count}, Invalid: {invalid_count}")
                    
                    if valid_count > 0:
                        df = df[df[f'{cat_var}_encoded'] != -1]
                    else:
                        raise ValueError(f"No valid encodings for {cat_var}!")
                else:
                    print(f"   Creating new encoder for {cat_var}")
                    cat_encoder = LabelEncoder()
                    df[f'{cat_var}_encoded'] = cat_encoder.fit_transform(df[cat_var].astype(str))
                    encoders[cat_var] = cat_encoder
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    for col in numeric_cols:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].median())
    
    if custom_predictors is not None:
        available_predictors = [col for col in custom_predictors if col in df.columns]
        features = available_predictors.copy()
        
        if categorical_vars:
            for cat_var in categorical_vars:
                encoded_col = f'{cat_var}_encoded'
                if encoded_col in df.columns:
                    features.append(encoded_col)
    else:
        year_column = None
        for col in ['campaign_year', 'Harvest_Year', 'year']:
            if col in df.columns:
                year_column = col
                break
        
        exclude_cols = [response_variable]
        if year_column:
            exclude_cols.extend([year_column, f'{year_column}_encoded'])
        if categorical_vars:
            exclude_cols.extend(categorical_vars)
        exclude_cols.extend([col for col in all_response_cols if col != response_variable])
        features = [col for col in numeric_cols if col not in exclude_cols]
    
    X = df[features].copy()
    y = df[response_variable].astype(float)
    
    X_with_categorical = X.copy()
    if categorical_vars:
        for cat_var in categorical_vars:
            if cat_var in df.columns:
                X_with_categorical[cat_var] = df[cat_var]
    
    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)
    
    X_with_cat_temp, _, _, _ = train_test_split(X_with_categorical, y, test_size=0.2, random_state=42)
    X_train_with_cat, _, _, _ = train_test_split(X_with_cat_temp, y_temp, test_size=0.25, random_state=42)
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    X_val_scaled = scaler.transform(X_val)
    
    return X_train_scaled, X_val_scaled, X_test_scaled, y_train, y_val, y_test, features, scaler, X_train_with_cat, encoders

def load_model_and_get_top_features(models_dir, response_variable, model_name='RandomForest', shap_results_file=None):
    """Load model and get top features - INCLUDING ENCODERS"""
    print(f"Loading {model_name} model for {response_variable}...")
    
    model_file = os.path.join(models_dir, f'{model_name.replace(" ", "_")}_{response_variable}.pkl')
    if not os.path.exists(model_file):
        raise FileNotFoundError(f"Model file not found: {model_file}")
    
    model_data = joblib.load(model_file)
    
    if isinstance(model_data, dict):
        model = model_data.get('model', model_data)
        model_encoders = model_data.get('encoders', {})
        print(f"  Loaded model with {len(model_encoders)} encoders")
    else:
        model = model_data
        model_encoders = {}
    
    top_features = None
    if shap_results_file and os.path.exists(shap_results_file):
        try:
            shap_df = pd.read_csv(shap_results_file)
            model_shap = shap_df[shap_df['model'] == model_name].sort_values('mean_abs_shap', ascending=False)
            if not model_shap.empty:
                top_features = model_shap['feature'].head(12).tolist()
                print(f"  Loaded top {len(top_features)} features from SHAP")
        except Exception as e:
            print(f"  Could not load SHAP: {str(e)}")
    
    return model, top_features, model_encoders

def get_top_features_from_model(model, feature_names, n_features=12):
    """Get top features from model importance"""
    if hasattr(model, 'feature_importances_'):
        importance_df = pd.DataFrame({
            'feature': feature_names,
            'importance': model.feature_importances_
        }).sort_values('importance', ascending=False)
        return importance_df.head(n_features)['feature'].tolist()
    else:
        return feature_names[:n_features]

# ============================================================================
# MAIN STRATIFIED PDP ANALYSIS FUNCTION
# ============================================================================

def perform_stratified_pdp_analysis(data_file, models_dir, output_dir, response_variable,
                                   model_name='RandomForest',
                                   custom_predictors=None,
                                   categorical_vars=None,
                                   shap_results_dir=None,
                                   stratify_by=None,
                                   min_samples=30,
                                   top_n_varieties=6,
                                   n_top_features=12):
    """Perform stratified PDP analysis with publication-quality plots"""
    
    print(f"\n{'='*80}")
    print(f"STRATIFIED PDP ANALYSIS FOR: {response_variable}")
    print(f"Display name: {ALL_RESPONSE_VARIABLES[response_variable]}")
    print(f"{'='*80}")
    print(f"Model: {model_name}")
    print(f"Top {n_top_features} numeric features")
    if stratify_by:
        print(f"Stratification: By '{stratify_by}' (TOP {top_n_varieties} categories)")
        if stratify_by.lower() in ['crop_cycle', 'cycle']:
            print(f"⚠ Crop cycle filtering: Excluding {EXCLUDED_CATEGORIES}")
    
    os.makedirs(output_dir, exist_ok=True)
    
    response_label = ALL_RESPONSE_VARIABLES.get(response_variable, response_variable)
    
    shap_results_file = None
    if shap_results_dir:
        shap_results_file = os.path.join(shap_results_dir, f'shap_feature_importance_summary.csv')
    
    model, top_features_shap, model_encoders = load_model_and_get_top_features(
        models_dir, response_variable, model_name, shap_results_file
    )
    
    X_train_scaled, X_val_scaled, X_test_scaled, y_train, y_val, y_test, feature_names, scaler, X_train_raw, encoders = load_and_prepare_data(
        data_file, response_variable, custom_predictors, categorical_vars, model_encoders
    )
    print(f"Data loaded: {len(X_train_scaled)} training samples")
    
    if top_features_shap:
        top_features = top_features_shap
    else:
        top_features = get_top_features_from_model(model, feature_names, n_features=n_top_features)
    
    # Filter to numeric features only
    numeric_features = []
    for feature in top_features:
        is_categorical = False
        if categorical_vars:
            for cat_var in categorical_vars:
                if f'{cat_var}_encoded' == feature:
                    is_categorical = True
                    break
        if not is_categorical:
            numeric_features.append(feature)
    
    # Ensure we have exactly n_top_features
    if len(numeric_features) < n_top_features:
        all_numeric_features = []
        for feature in feature_names:
            is_categorical = False
            if categorical_vars:
                for cat_var in categorical_vars:
                    if f'{cat_var}_encoded' == feature:
                        is_categorical = True
                        break
            if not is_categorical:
                all_numeric_features.append(feature)
        
        if hasattr(model, 'feature_importances_'):
            importance_df = pd.DataFrame({
                'feature': feature_names,
                'importance': model.feature_importances_
            })
            importance_df = importance_df[importance_df['feature'].isin(all_numeric_features)]
            importance_df = importance_df.sort_values('importance', ascending=False)
            numeric_features = importance_df.head(n_top_features)['feature'].tolist()
        else:
            numeric_features = all_numeric_features[:n_top_features]
    
    numeric_features = numeric_features[:n_top_features]
    top_features = numeric_features
    
    print(f"\n{'='*60}")
    print(f"TOP {len(numeric_features)} NUMERIC FEATURES:")
    print(f"{'='*60}")
    for i, feat in enumerate(numeric_features, 1):
        feat_label, unit = get_feature_display_info(feat)
        print(f"  {i:2d}. {feat_label}")
    print(f"{'='*60}\n")
    
    stratified_plots = []
    panel_file = None
    
    if stratify_by and categorical_vars and stratify_by in categorical_vars:
        stratify_encoder = encoders.get(stratify_by)
        if stratify_encoder:
            stratified_plots, panel_file = create_all_stratified_pdps_enhanced(
                model, X_train_scaled, feature_names, top_features,
                stratify_by, stratify_encoder, X_train_raw,
                scaler, output_dir, model_name, response_label,
                categorical_vars,
                min_samples=min_samples, top_n_varieties=top_n_varieties
            )
        else:
            print(f"\n⚠ Warning: No encoder for '{stratify_by}'")
    
    print(f"\n{'='*80}")
    print("STRATIFIED PDP ANALYSIS COMPLETE")
    print(f"{'='*80}")
    print(f"Response variable: {response_variable} - {response_label}")
    print(f"Results saved to: {output_dir}")
    if stratified_plots:
        print(f"✓ Standalone plots: {len(stratified_plots)}")
    if panel_file:
        print(f"✓ Combined panel: {panel_file}")
    
    return {
        'output_dir': output_dir,
        'stratified_plots': stratified_plots,
        'panel_file': panel_file,
        'response_variable': response_variable
    }
# ============================================================================
# MAIN FUNCTION WITH FLEXIBLE RESPONSE VARIABLES
# ============================================================================

def main(data_file, response_variable, custom_predictors=None, categorical_vars=None,
         base_output_dir=None, model_name='RandomForest',
         stratify_by=None, min_samples=30, top_n_varieties=6, n_top_features=12):
    """Main function with flexible response variables"""
    
    if response_variable not in ALL_RESPONSE_VARIABLES:
        raise ValueError(f"Response variable '{response_variable}' not recognized!\n"
                        f"Available: {list(ALL_RESPONSE_VARIABLES.keys())}")
    
    if base_output_dir is None:
        base_output_dir = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Irrigation_Split_Analysis/GAG'
    
    # Determine folder suffix
    if categorical_vars:
        folder_suffix = 'with_categorical'
    elif custom_predictors:
        folder_suffix = 'custom'
    else:
        folder_suffix = 'models'
    
    # Construct paths following the pattern
    models_dir = os.path.join(base_output_dir, f'kfold_models_{response_variable}_{folder_suffix}', 'models')
    shap_results_dir = os.path.join(base_output_dir, f'shap_analysis_{response_variable}_{folder_suffix}')
    output_dir = os.path.join(base_output_dir, f'pdp_stratified_{response_variable}_{folder_suffix}')
    
    # Create model-specific subfolder
    output_dir = os.path.join(output_dir, model_name.replace(" ", "_"))
    
    print(f"\n{'='*80}")
    print(f"STRATIFIED PDP ANALYSIS CONFIGURATION")
    print(f"{'='*80}")
    print(f"Data file: {data_file}")
    print(f"Response variable: {response_variable} - {ALL_RESPONSE_VARIABLES[response_variable]}")
    print(f"Models directory: {models_dir}")
    print(f"SHAP directory: {shap_results_dir}")
    print(f"Output directory: {output_dir}")
    print(f"Model: {model_name}")
    if custom_predictors:
        print(f"Custom predictors: {len(custom_predictors)} features")
    if categorical_vars:
        print(f"Categorical variables: {categorical_vars}")
    if stratify_by:
        print(f"Stratify by: {stratify_by} (Top {top_n_varieties} categories)")
    
    try:
        results = perform_stratified_pdp_analysis(
            data_file=data_file,
            models_dir=models_dir,
            output_dir=output_dir,
            response_variable=response_variable,
            model_name=model_name,
            custom_predictors=custom_predictors,
            categorical_vars=categorical_vars,
            shap_results_dir=shap_results_dir,
            stratify_by=stratify_by,
            min_samples=min_samples,
            top_n_varieties=top_n_varieties,
            n_top_features=n_top_features
        )
        
        print(f"\n{'='*80}")
        print("STRATIFIED PDP ANALYSIS COMPLETED SUCCESSFULLY")
        print(f"{'='*80}")
        
        return results
        
    except Exception as e:
        print(f"\nError: {str(e)}")
        import traceback
        traceback.print_exc()
        raise

# ============================================================================
# USAGE EXAMPLE
# ============================================================================

if __name__ == "__main__":
    
    custom_features = [
        'ampl_mean', 'TNn', 'TNx', 'Clay_percent_complete',
        'Total_Nitrogen_per_mille_complete', 'pH_complete', 'temp_moy_mean', 
        'Exchangeable_Calcium_meq_100g_complete', 'h_maxi_mean', 'CDD', 
        'radiation_ERA_total', 'Rx1', 'R20', 'SDII',
        'h_mini_mean', 'HD', 'Available_P2O5_mg_kg_complete', 
        'Electrical_Conductivity_mmho_cm_complete', 
        'Exchangeable_Sodium_meq_100g_complete',
        'R99p', 'growing_season_rainfall_mm', 'soil_temp_6h_mean', 
        'CWD', 'Mg_K_Ratio_complete'
    ]
    
    categorical_features = ['variety', 'irrigation_type', 'harvest_mode', 'crop_cycle']
    
    data_file = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis/final_dataset_regrouped_categorical.csv'
    base_output = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis'
    
    print("="*80)
    print("STRATIFIED PDP ANALYSIS PIPELINE - FLEXIBLE RESPONSE VARIABLES")
    
    try:
        # Example: Stratified PDP for tc_per_ha_actual by variety
        print("\n### RUNNING STRATIFIED PDP FOR: tc_per_ha_actual (by variety) ###\n")
        results_yield = main(
            data_file=data_file,
            response_variable='brix_percent',
            custom_predictors=custom_features,
            categorical_vars=categorical_features,
            base_output_dir=base_output,
            model_name='RandomForest',
            stratify_by='harvest_mode',
            min_samples=30,
            top_n_varieties=16,
            n_top_features=12
        )
    
      
        print("\n" + "="*80)
        print("ANALYSIS COMPLETED SUCCESSFULLY!")
        print("="*80)
        
    except Exception as e:
        print(f"\n❌ ERROR: {str(e)}")
        import traceback
        traceback.print_exc()
        

STRATIFIED PDP ANALYSIS PIPELINE - FLEXIBLE RESPONSE VARIABLES

Key features:
  • Flexible response variables from predefined list
  • Automatic path construction based on response variable
  • Transparent legends with smart positioning
  • Consistent colors across all analyses
  • Publication-quality plots at 600 DPI


### RUNNING STRATIFIED PDP FOR: tc_per_ha_actual (by variety) ###


STRATIFIED PDP ANALYSIS CONFIGURATION
Data file: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis/final_dataset_regrouped_categorical.csv
Response variable: brix_percent - Brix (%)
Models directory: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis\kfold_models_brix_percent_with_categorical\models
SHAP directory: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis\shap_analysis_brix_percent_with_categorical
Output directory: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis\pdp_stratified_brix_perc

## 7bis. Stratified PDP : Fixed axis values

This section creates side-by-side comparative visualizations of categorical partial dependence plots for simultaneous evaluation of yield and sugar content responses across management practices.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import glob
import re
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.inspection import partial_dependence, PartialDependenceDisplay
from scipy.interpolate import make_interp_spline
from scipy.ndimage import gaussian_filter1d
from statsmodels.nonparametric.smoothers_lowess import lowess
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# EXCLUDED CATEGORIES (for crop_cycle stratification)
# ============================================================================

EXCLUDED_CATEGORIES = [
    '9th Ratoon',
    '10th Ratoon', 
    '11th Ratoon',
    'Beyond 12th Ratoon'
]

# ============================================================================
# DEFINE AVAILABLE RESPONSE VARIABLES
# ============================================================================

SUGAR_CONTENT_METRICS = {
    "brix_percent": "Brix (%)",
    "pol_juice_percent": "Pol juice (%)",
    "pol_cane_percent": "Pol cane (%)",
    "purity_percent": "Purity (%)",
    "glucose_cane_percent": "Glucose (%)",
    "hugot_sugar": "Hugot sugar (%)"
}

YIELD_PARAMETERS = {
    "tc_per_ha_actual": "Cane yield (t ha⁻¹)",
    "tc_per_ha_per_week": "Cane yield per week (t ha⁻¹/week)",
    "tc_per_ha_52_weeks": "Cane yield 52 weeks (t ha⁻¹)",
    "fiber_percent": "Fiber in Cane (%)",
}

ALL_RESPONSE_VARIABLES = {**SUGAR_CONTENT_METRICS, **YIELD_PARAMETERS}

# ============================================================================
# PUBLICATION-QUALITY PLOT CONFIGURATION
# ============================================================================

plt.rcParams.update({
    'font.size': 8,
    'font.family': 'Arial',
    'axes.labelsize': 8,
    'axes.titlesize': 9,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 6.5,
    'figure.titlesize': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': False,
    'axes.axisbelow': True,
    'axes.edgecolor': '#2d2d2d',
    'text.color': '#2d2d2d',
    'axes.labelcolor': '#2d2d2d',
    'xtick.color': '#2d2d2d',
    'ytick.color': '#2d2d2d',
    'axes.linewidth': 0.75,
    'xtick.major.width': 0.75,
    'ytick.major.width': 0.75,
    'xtick.major.size': 3,
    'ytick.major.size': 3,
    'figure.dpi': 300,
    'savefig.dpi': 600,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.05
})

# ============================================================================
# COLOR PALETTE - RColorBrewer-inspired (consistent with categorical analysis)
# ============================================================================

CATEGORY_COLORS = [
    '#E41A1C', '#377EB8', '#4DAF4A', '#984EA3', '#FF7F00', '#FFFF33', '#A65628', '#F781BF', '#999999',
    '#1B9E77', '#D95F02', '#7570B3', '#E7298A', '#66A61E', '#E6AB02', '#A6761D', '#666666',
    '#A6CEE3', '#1F78B4', '#B2DF8A', '#33A02C', '#FB9A99', '#E31A1C', '#FDBF6F', '#FF7F00',
    '#8DD3C7', '#FFFFB3', '#BEBADA', '#FB8072', '#80B1D3',
]

#For harvest_mode
#CATEGORY_COLORS = ['#D4A373', '#8B4513']
#For irrigation_type
#CATEGORY_COLORS = ['#4A90E2', '#E63946']

def get_consistent_category_colors(categories):
    """Assign consistent colors to categories based on ALPHABETICAL sorting"""
    sorted_categories = sorted(categories)
    color_map = {}
    for i, cat in enumerate(sorted_categories):
        color_map[str(cat)] = CATEGORY_COLORS[i % len(CATEGORY_COLORS)]
    return color_map

# ============================================================================
# RUG PLOT FUNCTION
# ============================================================================

def add_rug_plot(ax, data_values, x_min, x_max, color='#333333', alpha=0.3, height=0.03):
    """Add rug plot filtered to PDP x-axis range"""
    try:
        data_values_filtered = data_values[(data_values >= x_min) & (data_values <= x_max)]
        if len(data_values_filtered) == 0:
            return
        
        y_min, y_max = ax.get_ylim()
        y_range = y_max - y_min
        rug_y_position = y_min + (y_range * 0.02)
        rug_height = y_range * height
        
        if len(data_values_filtered) > 1000:
            sample_indices = np.random.choice(len(data_values_filtered), 1000, replace=False)
            data_values_sample = data_values_filtered[sample_indices]
        else:
            data_values_sample = data_values_filtered
        
        for val in data_values_sample:
            ax.plot([val, val], [rug_y_position, rug_y_position + rug_height],
                   color=color, alpha=alpha, linewidth=0.6, zorder=1)
    except Exception:
        pass

# ============================================================================
# SMOOTHING FUNCTIONS
# ============================================================================

def ultra_smooth_curve_perfect_monotonic(x, y, method='lowess', frac=0.4, n_bootstrap=100, confidence_level=0.95):
    """PERFECT SMOOTHING for PDP curves"""
    try:
        sorted_indices = np.argsort(x)
        x_sorted = x[sorted_indices]
        y_sorted = y[sorted_indices]
        
        if len(x_sorted) <= 3:
            return x, y, None, None
        
        unique_x, inverse_indices = np.unique(x_sorted, return_inverse=True)
        unique_y = np.array([np.mean(y_sorted[inverse_indices == i]) for i in range(len(unique_x))])
        
        if len(unique_x) < 5:
            return x, y, None, None
        
        if method == 'lowess':
            smoothed = lowess(unique_y, unique_x, frac=frac, it=10, return_sorted=True)
            smooth_x = smoothed[:, 0]
            smooth_y = smoothed[:, 1]
            
            sigma_base = len(smooth_y) * 0.15
            for i in range(6):
                sigma = sigma_base * (1.0 - i * 0.1)
                smooth_y = gaussian_filter1d(smooth_y, sigma=sigma)
            
            smooth_y = gaussian_filter1d(smooth_y, sigma=len(smooth_y) * 0.2)
            
            try:
                n_spline = min(50, len(smooth_x))
                x_spline = np.linspace(smooth_x.min(), smooth_x.max(), n_spline)
                spline = make_interp_spline(smooth_x, smooth_y, k=2, bc_type='natural')
                y_spline = spline(x_spline)
                y_spline = gaussian_filter1d(y_spline, sigma=len(y_spline) * 0.25)
                smooth_x = x_spline
                smooth_y = y_spline
            except:
                pass
            
            if n_bootstrap > 0:
                bootstrap_curves = []
                n_points = len(unique_x)
                
                for _ in range(n_bootstrap):
                    bootstrap_indices = np.random.choice(n_points, n_points, replace=True)
                    boot_x = unique_x[bootstrap_indices]
                    boot_y = unique_y[bootstrap_indices]
                    boot_sort_idx = np.argsort(boot_x)
                    boot_x_sorted = boot_x[boot_sort_idx]
                    boot_y_sorted = boot_y[boot_sort_idx]
                    
                    try:
                        boot_smoothed = lowess(boot_y_sorted, boot_x_sorted, frac=frac, it=8, return_sorted=True)
                        boot_y_smooth = boot_smoothed[:, 1]
                        for i in range(4):
                            sigma = len(boot_y_smooth) * 0.12 * (1.0 - i * 0.1)
                            boot_y_smooth = gaussian_filter1d(boot_y_smooth, sigma=sigma)
                        boot_y_smooth = gaussian_filter1d(boot_y_smooth, sigma=len(boot_y_smooth) * 0.2)
                        boot_interp_y = np.interp(smooth_x, boot_smoothed[:, 0], boot_y_smooth)
                        bootstrap_curves.append(boot_interp_y)
                    except:
                        continue
                
                if bootstrap_curves:
                    bootstrap_curves = np.array(bootstrap_curves)
                    alpha = (1 - confidence_level) / 2
                    ci_lower = np.percentile(bootstrap_curves, alpha * 100, axis=0)
                    ci_upper = np.percentile(bootstrap_curves, (1 - alpha) * 100, axis=0)
                    ci_lower = gaussian_filter1d(ci_lower, sigma=len(ci_lower) * 0.15)
                    ci_upper = gaussian_filter1d(ci_upper, sigma=len(ci_upper) * 0.15)
                    return smooth_x, smooth_y, ci_lower, ci_upper
            
            return smooth_x, smooth_y, None, None
    except Exception as e:
        return x, y, None, None

# ============================================================================
# CATEGORY FILTERING FUNCTIONS
# ============================================================================

def filter_excluded_categories(df, cat_var):
    """Filter out excluded categories from dataframe (for crop_cycle)"""
    initial_len = len(df)
    
    if cat_var.lower() in ['crop_cycle', 'cycle']:
        df = df.copy()
        df[cat_var] = df[cat_var].astype(str)
        excluded_str = [str(x) for x in EXCLUDED_CATEGORIES]
        df_filtered = df[~df[cat_var].isin(excluded_str)].copy()
        n_excluded = initial_len - len(df_filtered)
        
        if n_excluded > 0:
            print(f"\n⚠ Excluded {n_excluded} rows with crop cycles: {', '.join(EXCLUDED_CATEGORIES)}")
            print(f"  Remaining rows: {len(df_filtered):,}")
        
        return df_filtered, n_excluded
    else:
        return df, 0

# ============================================================================
# STRATIFICATION HELPER FUNCTIONS
# ============================================================================

def get_top_n_varieties(X_train_raw, stratify_feature, n_top=6, min_samples=30):
    """Get top N varieties by sample count"""
    X_train_filtered, n_excluded = filter_excluded_categories(X_train_raw, stratify_feature)
    unique_categories = X_train_filtered[stratify_feature].dropna().unique()
    
    category_sample_counts = {}
    for category in unique_categories:
        category_mask = X_train_filtered[stratify_feature] == category
        n_samples = category_mask.sum()
        if n_samples >= min_samples:
            category_sample_counts[category] = n_samples
    
    sorted_categories = sorted(category_sample_counts.items(), 
                              key=lambda x: x[1], reverse=True)
    top_varieties = [cat for cat, count in sorted_categories[:n_top]]
    
    print(f"\n{'='*60}")
    print(f"TOP {n_top} {stratify_feature.upper()} BY SAMPLE COUNT:")
    print(f"{'='*60}")
    for i, (variety, count) in enumerate(sorted_categories[:n_top], 1):
        print(f"  {i}. {variety}: {count} samples")
    print(f"{'='*60}\n")
    
    return top_varieties

def transform_to_original_scale(scaled_values, feature_idx, scaler, feature_names):
    """Transform scaled values back to original scale"""
    n_features = len(feature_names)
    dummy_scaled = np.zeros((len(scaled_values), n_features))
    dummy_scaled[:, feature_idx] = scaled_values
    dummy_original = scaler.inverse_transform(dummy_scaled)
    return dummy_original[:, feature_idx]

# ============================================================================
# FEATURE DISPLAY INFO
# ============================================================================

def get_feature_display_info(feature_name):
    """Get display label and units for features"""
    feature_mapping = {
        'temp_moy_mean': {'label': 'Mean temperature', 'unit': '°C'},
        'temp_maxi_mean': {'label': 'Maximum temperature', 'unit': '°C'},
        'temp_mini_mean': {'label': 'Minimum temperature', 'unit': '°C'},
        'precipitation_total': {'label': 'Total precipitation', 'unit': 'mm'},
        'growing_season_rainfall_mm': {'label': 'Growing season rainfall', 'unit': 'mm'},
        'radiation_ERA_total': {'label': 'Solar radiation', 'unit': 'MJ m⁻²'},
        'evaporation_total': {'label': 'Total evaporation', 'unit': 'mm'},
        'CWD': {'label': 'Consecutive wet days', 'unit': 'days'},
        'CDD': {'label': 'Consecutive dry days', 'unit': 'days'},
        'SDII': {'label': 'Daily intensity index', 'unit': 'mm day⁻¹'},
        'R10': {'label': 'Heavy precipitation days', 'unit': 'days'},
        'R20': {'label': 'Very heavy precip. days', 'unit': 'days'},
        'Rx1': {'label': 'Max 1-day precipitation', 'unit': 'mm'},
        'Rx5': {'label': 'Max 5-day precipitation', 'unit': 'mm'},
        'R95p': {'label': 'Very wet days precip.', 'unit': 'mm'},
        'R99p': {'label': 'Extremely wet days precip.', 'unit': 'mm'},
        'HD': {'label': 'Hot days', 'unit': 'days'},
        'TNn': {'label': 'Min daily min temp.', 'unit': '°C'},
        'TNx': {'label': 'Max daily min temp.', 'unit': '°C'},
        'TXn': {'label': 'Min daily max temp.', 'unit': '°C'},
        'TXx': {'label': 'Max daily max temp.', 'unit': '°C'},
        'soil_temp_6h_mean': {'label': 'Soil temperature (6h)', 'unit': '°C'},
        'soil_temp_16h_mean': {'label': 'Soil temperature (16h)', 'unit': '°C'},
        'pH_complete': {'label': 'Soil pH', 'unit': ''},
        'Clay_percent_complete': {'label': 'Clay content', 'unit': '%'},
        'Fine_Silt_percent_complete': {'label': 'Fine silt', 'unit': '%'},
        'Coarse_Silt_percent_complete': {'label': 'Coarse silt', 'unit': '%'},
        'Fine_Sand_percent_complete': {'label': 'Fine sand', 'unit': '%'},
        'Coarse_Sand_percent_complete': {'label': 'Coarse sand', 'unit': '%'},
        'Total_Nitrogen_per_mille_complete': {'label': 'Total nitrogen', 'unit': '‰'},
        'Organic_Carbon_per_mille_complete': {'label': 'Organic carbon', 'unit': '‰'},
        'Exchangeable_Calcium_meq_100g_complete': {'label': 'Exchangeable Ca', 'unit': 'meq 100g⁻¹'},
        'Exchangeable_Magnesium_meq_100g_complete': {'label': 'Exchangeable Mg', 'unit': 'meq 100g⁻¹'},
        'Exchangeable_Potassium_meq_100g_complete': {'label': 'Exchangeable K', 'unit': 'meq 100g⁻¹'},
        'Exchangeable_Sodium_meq_100g_complete': {'label': 'Exchangeable Na', 'unit': 'meq 100g⁻¹'},
        'Available_P2O5_mg_kg_complete': {'label': 'Available P₂O₅', 'unit': 'mg kg⁻¹'},
        'Electrical_Conductivity_mmho_cm_complete': {'label': 'Electrical conductivity', 'unit': 'mmho cm⁻¹'},
        'Mg_K_Ratio_complete': {'label': 'Mg:K ratio', 'unit': ''},
        'Mg_Ca_Ratio_complete': {'label': 'Mg:Ca ratio', 'unit': ''},
        'ampl_mean': {'label': 'Temperature amplitude', 'unit': '°C'},
        'h_mini_mean': {'label': 'Min humidity', 'unit': '%'},
        'h_maxi_mean': {'label': 'Max humidity', 'unit': '%'},
        'h_moy_mean': {'label': 'Mean humidity', 'unit': '%'},
    }
    
    info = feature_mapping.get(feature_name, {'label': feature_name.replace('_', ' '), 'unit': ''})
    return info['label'], info['unit']

# ============================================================================
# LEGEND POSITION OPTIMIZATION
# ============================================================================

def find_optimal_legend_position(ax, category_data, valid_categories):
    """Find optimal legend position avoiding rug marks and curves"""
    all_curves = []
    for cat in valid_categories:
        data = category_data[cat]
        all_curves.append((data['x'], data['y']))
    
    x_min, x_max = ax.get_xlim()
    y_min, y_max = ax.get_ylim()
    x_range = x_max - x_min
    y_range = y_max - y_min
    
    rug_zone_top = y_min + 0.15 * y_range
    
    regions = {
        'upper right': {'x': (x_min + 0.6*x_range, x_max), 'y': (y_min + 0.7*y_range, y_max)},
        'upper left': {'x': (x_min, x_min + 0.4*x_range), 'y': (y_min + 0.7*y_range, y_max)},
        'center right': {'x': (x_min + 0.65*x_range, x_max), 'y': (rug_zone_top + 0.2*y_range, y_min + 0.65*y_range)},
        'center left': {'x': (x_min, x_min + 0.35*x_range), 'y': (rug_zone_top + 0.2*y_range, y_min + 0.65*y_range)},
    }
    
    region_scores = {}
    for region_name, bounds in regions.items():
        score = 0
        for x_vals, y_vals in all_curves:
            in_region = ((x_vals >= bounds['x'][0]) & (x_vals <= bounds['x'][1]) &
                        (y_vals >= bounds['y'][0]) & (y_vals <= bounds['y'][1]))
            score += np.sum(in_region)
        region_scores[region_name] = int(score)
    
    best_region = min(region_scores, key=region_scores.get)
    print(f"      Legend scores: {region_scores}, Selected: {best_region}")
    
    return best_region

# ============================================================================
# ENCODING HELPER FUNCTIONS
# ============================================================================

def normalize_ordinal(text):
    """Normalize ordinal numbers (1ST -> 1st, 2ND -> 2nd)"""
    text = str(text).strip()
    def ordinal_replacement(match):
        num = match.group(1)
        suffix = match.group(2).lower()
        return f"{num}{suffix}"
    text = re.sub(r'(\d+)(ST|ND|RD|TH)\b', ordinal_replacement, text, flags=re.IGNORECASE)
    return text

def encode_categorical_value(value, encoder, cat_var):
    """Encode categorical value with robust string matching"""
    if pd.isna(value):
        return np.nan
    
    original = str(value).strip()
    normalized = normalize_ordinal(original)
    
    attempts = [
        normalized.title(), original, original.title(), original.upper(), 
        original.lower(), normalized, normalized.upper(), normalized.lower(),
    ]
    
    for attempt in attempts:
        try:
            if attempt in encoder.classes_:
                return encoder.transform([attempt])[0]
        except:
            continue
    
    return -1

# ============================================================================
# STRATIFIED PDP PLOTTING FUNCTIONS
# ============================================================================
def compute_reference_ylim(model, X_train_scaled, feature_name, feature_names, 
                          stratify_feature, stratify_encoder, X_train_raw,
                          scaler, top_varieties, min_samples=30):
    """
    Compute y-axis limits based on a reference feature's PDP values across all categories.
    This ensures all plots use the same y-axis scale for comparability.
    
    Parameters:
    -----------
    model : trained model
    X_train_scaled : scaled training data
    feature_name : str, name of reference feature (e.g., 'HD')
    feature_names : list of all feature names
    stratify_feature : str, categorical variable for stratification
    stratify_encoder : LabelEncoder for stratify_feature
    X_train_raw : raw training data with categorical variables
    scaler : StandardScaler
    top_varieties : list of top variety categories
    min_samples : int, minimum samples per category
    
    Returns:
    --------
    tuple : (y_min, y_max) for fixed axis limits
    """
    print(f"\n{'='*60}")
    print(f"COMPUTING Y-AXIS LIMITS FROM REFERENCE FEATURE: {feature_name}")
    print(f"{'='*60}")
    
    try:
        feature_idx = feature_names.index(feature_name)
        stratify_col = f'{stratify_feature}_encoded'
        stratify_idx = feature_names.index(stratify_col)
        
        all_y_values = []
        
        for category in top_varieties:
            try:
                category_mask = X_train_raw[stratify_feature] == category
                n_samples = category_mask.sum()
                
                if n_samples < min_samples:
                    continue
                
                encoded_val = encode_categorical_value(category, stratify_encoder, stratify_feature)
                if encoded_val == -1:
                    continue
                
                X_category = X_train_scaled[category_mask.values]
                if len(X_category) < 10:
                    continue
                
                # Compute partial dependence
                pd_results = partial_dependence(
                    model, X_category, features=[feature_idx], grid_resolution=200
                )
                
                if hasattr(pd_results, 'average'):
                    pd_values = pd_results.average[0]
                    scaled_grid_values = pd_results.grid_values[0]
                else:
                    pd_values = pd_results[0][0]
                    scaled_grid_values = pd_results[1][0]
                
                original_grid_values = transform_to_original_scale(
                    scaled_grid_values, feature_idx, scaler, feature_names
                )
                
                # Smooth the curve
                smooth_x, smooth_y, ci_lower, ci_upper = ultra_smooth_curve_perfect_monotonic(
                    original_grid_values, pd_values, method='lowess', 
                    frac=0.40, n_bootstrap=100, confidence_level=0.95
                )
                
                # Collect all y values including confidence intervals
                all_y_values.extend(smooth_y)
                if ci_lower is not None:
                    all_y_values.extend(ci_lower)
                    all_y_values.extend(ci_upper)
                
                print(f"  ✓ {category}: {len(smooth_y)} points, "
                      f"range [{np.min(smooth_y):.4f}, {np.max(smooth_y):.4f}]")
                
            except Exception as e:
                print(f"  ✗ {category}: {str(e)}")
                continue
        
        if not all_y_values:
            print("  ⚠ Warning: No valid PDP values computed")
            return None, None
        
        # Compute overall limits with buffer
        y_min = np.min(all_y_values)
        y_max = np.max(all_y_values)
        y_range = y_max - y_min
        y_buffer = y_range * 0.08
        
        final_y_min = y_min - y_buffer
        final_y_max = y_max + y_buffer
        
        print(f"\n{'='*60}")
        print(f"COMPUTED Y-AXIS LIMITS:")
        print(f"  Raw range: [{y_min:.4f}, {y_max:.4f}]")
        print(f"  With buffer (8%): [{final_y_min:.4f}, {final_y_max:.4f}]")
        print(f"{'='*60}\n")
        
        return final_y_min, final_y_max
        
    except Exception as e:
        print(f"  ✗ Error computing reference limits: {str(e)}")
        return None, None

def create_stratified_pdp_for_feature_fixed_axis(model, X_train_scaled, feature_idx, feature_name,
                                     stratify_feature, stratify_encoder, X_train_raw,
                                     scaler, feature_names, output_dir, model_name, 
                                     response_label="Effect",
                                     min_samples=30, top_n_varieties=6, 
                                     fixed_ylim=None, figsize=(3.5, 2.8)):
    """
    Create publication-quality stratified PDP with FIXED y-axis limits for comparability.
    
    NEW PARAMETER:
    --------------
    fixed_ylim : tuple (y_min, y_max) or None
        If provided, forces all plots to use these y-axis limits
    """
    print(f"    Creating stratified PDP for {feature_name} by {stratify_feature}...")
    
    try:
        top_varieties = get_top_n_varieties(X_train_raw, stratify_feature, 
                                           n_top=top_n_varieties, min_samples=min_samples)
        
        if not top_varieties:
            print(f"    ✗ No varieties with sufficient samples")
            return None, None
        
        stratify_col = f'{stratify_feature}_encoded'
        if stratify_col not in feature_names:
            print(f"    ✗ Stratify feature {stratify_col} not found")
            return None, None
        
        stratify_idx = feature_names.index(stratify_col)
        
        category_data = {}
        valid_categories = []
        all_pdp_x = []
        
        for category in top_varieties:
            try:
                category_mask = X_train_raw[stratify_feature] == category
                n_samples = category_mask.sum()
                
                encoded_val = encode_categorical_value(category, stratify_encoder, stratify_feature)
                if encoded_val == -1:
                    print(f"    ⚠ Could not encode: {category}")
                    continue
                
                dummy_unscaled = np.zeros((1, len(feature_names)))
                dummy_unscaled[0, stratify_idx] = encoded_val
                dummy_scaled = scaler.transform(dummy_unscaled)
                
                X_category = X_train_scaled[category_mask.values]
                if len(X_category) < 10:
                    continue
                
                pd_results = partial_dependence(
                    model, X_category, features=[feature_idx], grid_resolution=200
                )
                
                if hasattr(pd_results, 'average'):
                    pd_values = pd_results.average[0]
                    scaled_grid_values = pd_results.grid_values[0]
                else:
                    pd_values = pd_results[0][0]
                    scaled_grid_values = pd_results[1][0]
                
                original_grid_values = transform_to_original_scale(
                    scaled_grid_values, feature_idx, scaler, feature_names
                )
                
                smooth_x, smooth_y, ci_lower, ci_upper = ultra_smooth_curve_perfect_monotonic(
                    original_grid_values, pd_values, method='lowess', 
                    frac=0.40, n_bootstrap=100, confidence_level=0.95
                )
                
                category_data[category] = {
                    'x': smooth_x, 'y': smooth_y,
                    'ci_lower': ci_lower, 'ci_upper': ci_upper,
                    'n_samples': n_samples
                }
                valid_categories.append(category)
                all_pdp_x.extend(smooth_x)
                
            except Exception:
                continue
        
        if not valid_categories:
            print(f"    ✗ No valid categories")
            return None, None
        
        color_map = get_consistent_category_colors(valid_categories)
        for cat in valid_categories:
            category_data[cat]['color'] = color_map[str(cat)]
        
        n_varieties = len(valid_categories)
        
        # ====================================================================
        # MODIFIED SECTION: USE FIXED Y-AXIS LIMITS IF PROVIDED
        # ====================================================================
        
        if fixed_ylim is not None:
            # Use provided fixed limits
            y_min, y_max = fixed_ylim
            print(f"    → Using FIXED y-axis: [{y_min:.4f}, {y_max:.4f}]")
        else:
            # Compute limits from data (original behavior)
            all_x, all_y = [], []
            for cat in valid_categories:
                data = category_data[cat]
                all_x.extend(data['x'])
                all_y.extend(data['y'])
                if data['ci_lower'] is not None:
                    all_y.extend(data['ci_lower'])
                    all_y.extend(data['ci_upper'])
            
            y_min, y_max = np.min(all_y), np.max(all_y)
            y_range = y_max - y_min
            y_buffer = y_range * 0.08
            y_min = y_min - y_buffer
            y_max = y_max + y_buffer
            print(f"    → Using AUTO y-axis: [{y_min:.4f}, {y_max:.4f}]")
        
        # Compute x-axis limits (always from data)
        all_x = []
        for cat in valid_categories:
            all_x.extend(category_data[cat]['x'])
        x_min, x_max = np.min(all_x), np.max(all_x)
        x_range = x_max - x_min
        x_buffer = x_range * 0.03
        
        # ====================================================================
        
        fig, ax = plt.subplots(figsize=figsize)
        
        for category in valid_categories:
            data = category_data[category]
            
            if data['ci_lower'] is not None:
                ax.fill_between(data['x'], data['ci_lower'], data['ci_upper'],
                              alpha=0.15, color=data['color'], linewidth=0, zorder=2)
            
            ax.plot(data['x'], data['y'], linewidth=1.8, color=data['color'], alpha=0.9,
                   label=f"{str(category)[:15]}", solid_capstyle='round', zorder=5)
        
        # Apply axis limits
        ax.set_xlim(x_min - x_buffer, x_max + x_buffer)
        ax.set_ylim(y_min, y_max)
        
        # Add rug marks
        if feature_name in X_train_raw.columns:
            feature_data = X_train_raw[feature_name].dropna().values
            add_rug_plot(ax, feature_data, x_min, x_max, color='#333333', alpha=0.3, height=0.03)
        
        # Zero line
        if y_min < 0 < y_max:
            ax.axhline(0, color='#cccccc', linestyle='-', linewidth=0.75, alpha=0.8, zorder=1)
        
        feature_label, unit = get_feature_display_info(feature_name)
        xlabel = f"{feature_label}" + (f" ({unit})" if unit else "")
        
        ax.set_xlabel(xlabel, fontsize=8, labelpad=3)
        ax.set_ylabel(f'{response_label}', fontsize=8, labelpad=3)
        ax.set_title(f'{feature_label}', fontsize=9, fontweight='bold', pad=6)
        
        # Transparent legend
        legend_loc = find_optimal_legend_position(ax, category_data, valid_categories)
        stratify_display = stratify_feature.replace('_', ' ').capitalize()
        
        if n_varieties <= 3:
            legend = ax.legend(loc=legend_loc, frameon=False, fontsize=5, ncol=1, 
                             title=stratify_display, title_fontsize=5.5,
                             handlelength=1.0, labelspacing=0.2, borderpad=0.2)
        else:
            legend = ax.legend(loc=legend_loc, frameon=False, fontsize=4.5, ncol=2,
                             title=stratify_display, title_fontsize=5,
                             handlelength=0.9, labelspacing=0.15, columnspacing=0.6, borderpad=0.2)
        
        legend.get_title().set_fontweight('bold')
        for text in legend.get_texts():
            text.set_alpha(0.9)
        legend.get_title().set_alpha(0.9)
        
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.tick_params(axis='both', labelsize=7, length=3, width=0.75, pad=2)
        
        plt.tight_layout()
        
        safe_feature_name = feature_name.replace(' ', '_')
        safe_stratify_name = stratify_feature.replace(' ', '_')
        
        output_file = os.path.join(output_dir, 
                                   f'pdp_stratified_{safe_feature_name}_by_{safe_stratify_name}.png')
        plt.savefig(output_file, dpi=600, bbox_inches='tight', facecolor='white')
        
        pdf_file = output_file.replace('.png', '.pdf')
        plt.savefig(pdf_file, bbox_inches='tight', facecolor='white')
        
        plt.close()
        
        axis_type = "FIXED" if fixed_ylim is not None else "AUTO"
        print(f"    ✓ Created stratified PDP ({n_varieties} varieties) - {axis_type} y-axis, legend at {legend_loc}")
        return output_file, category_data
        
    except Exception as e:
        print(f"    ✗ Error: {str(e)}")
        plt.close()
        return None, None

def create_combined_stratified_pdp_panel_fixed_axis(model, X_train_scaled, feature_names, top_features,
                                        stratify_feature, stratify_encoder, X_train_raw,
                                        scaler, output_dir, model_name, response_label="Effect",
                                        categorical_features=None,
                                        min_samples=30, top_n_varieties=6, fixed_ylim=None):
    """
    Create combined panel of stratified PDPs with FIXED y-axis limits.
    
    NEW PARAMETER:
    --------------
    fixed_ylim : tuple (y_min, y_max) or None
        If provided, forces all subplots to use these y-axis limits
    """
    print(f"\n=== CREATING COMBINED PANEL (TOP {top_n_varieties} VARIETIES) ===")
    if fixed_ylim:
        print(f"Using FIXED y-axis: [{fixed_ylim[0]:.4f}, {fixed_ylim[1]:.4f}]\n")
    else:
        print("Using AUTO y-axis (independent per subplot)\n")
    
    top_varieties = get_top_n_varieties(X_train_raw, stratify_feature, 
                                       n_top=top_n_varieties, min_samples=min_samples)
    
    if not top_varieties:
        print("No varieties with sufficient samples")
        return None
    
    # Filter to numeric features only
    numeric_features = []
    for feature in top_features:
        is_categorical = False
        if categorical_features:
            for cat_var in categorical_features:
                if f'{cat_var}_encoded' == feature:
                    is_categorical = True
                    break
        if not is_categorical:
            numeric_features.append(feature)
    
    n_features = len(numeric_features)
    if n_features == 0:
        return None
    
    if n_features <= 4:
        n_cols = 2
    elif n_features <= 9:
        n_cols = 3
    else:
        n_cols = 4
    
    n_rows = int(np.ceil(n_features / n_cols))
    fig_width = 7.0
    fig_height = 2.2 * n_rows
    fig = plt.figure(figsize=(fig_width, fig_height))
    
    stratify_col = f'{stratify_feature}_encoded'
    stratify_idx = feature_names.index(stratify_col)
    
    color_map = get_consistent_category_colors(top_varieties)
    
    for plot_idx, feature in enumerate(numeric_features):
        try:
            feature_idx = feature_names.index(feature)
            ax = plt.subplot(n_rows, n_cols, plot_idx + 1)
            
            category_data = {}
            all_x, all_y = [], []
            
            for category in top_varieties:
                try:
                    category_mask = X_train_raw[stratify_feature] == category
                    encoded_val = encode_categorical_value(category, stratify_encoder, stratify_feature)
                    if encoded_val == -1:
                        continue
                    
                    dummy_unscaled = np.zeros((1, len(feature_names)))
                    dummy_unscaled[0, stratify_idx] = encoded_val
                    dummy_scaled = scaler.transform(dummy_unscaled)
                    
                    X_category = X_train_scaled[category_mask.values]
                    if len(X_category) < 10:
                        continue
                    
                    pd_results = partial_dependence(model, X_category, features=[feature_idx], grid_resolution=150)
                    
                    if hasattr(pd_results, 'average'):
                        pd_values = pd_results.average[0]
                        scaled_grid_values = pd_results.grid_values[0]
                    else:
                        pd_values = pd_results[0][0]
                        scaled_grid_values = pd_results[1][0]
                    
                    original_grid_values = transform_to_original_scale(
                        scaled_grid_values, feature_idx, scaler, feature_names
                    )
                    
                    smooth_x, smooth_y, ci_lower, ci_upper = ultra_smooth_curve_perfect_monotonic(
                        original_grid_values, pd_values, method='lowess', 
                        frac=0.40, n_bootstrap=50, confidence_level=0.95
                    )
                    
                    category_data[category] = {
                        'x': smooth_x, 'y': smooth_y,
                        'ci_lower': ci_lower, 'ci_upper': ci_upper
                    }
                    
                    all_x.extend(smooth_x)
                    all_y.extend(smooth_y)
                    if ci_lower is not None:
                        all_y.extend(ci_lower)
                        all_y.extend(ci_upper)
                except Exception:
                    continue
            
            if not category_data:
                continue
            
            # ================================================================
            # MODIFIED SECTION: USE FIXED Y-AXIS LIMITS IF PROVIDED
            # ================================================================
            
            if fixed_ylim is not None:
                # Use provided fixed limits
                y_min, y_max = fixed_ylim
            else:
                # Compute limits from subplot data (original behavior)
                y_min, y_max = np.min(all_y), np.max(all_y)
                y_range = y_max - y_min
                y_min = y_min - y_range*0.06
                y_max = y_max + y_range*0.06
            
            # Compute x-axis limits (always from data)
            x_min, x_max = np.min(all_x), np.max(all_x)
            x_range = x_max - x_min
            
            ax.set_xlim(x_min - x_range*0.02, x_max + x_range*0.02)
            ax.set_ylim(y_min, y_max)
            
            # ================================================================
            
            for category in top_varieties:
                if category not in category_data:
                    continue
                data = category_data[category]
                color = color_map[category]
                
                if data['ci_lower'] is not None:
                    ax.fill_between(data['x'], data['ci_lower'], data['ci_upper'],
                                  alpha=0.12, color=color, linewidth=0, zorder=2)
                
                ax.plot(data['x'], data['y'], linewidth=1.5, color=color, alpha=0.9,
                        solid_capstyle='round', zorder=5)
            
            # Add rug marks
            if feature in X_train_raw.columns:
                feature_data = X_train_raw[feature].dropna().values
                add_rug_plot(ax, feature_data, x_min, x_max, color='#333333', alpha=0.25, height=0.025)
            
            # Zero line
            if y_min < 0 < y_max:
                ax.axhline(0, color='#cccccc', linestyle='-', linewidth=0.6, alpha=0.7, zorder=1)
            
            feature_label, unit = get_feature_display_info(feature)
            xlabel = feature_label + (f"\n({unit})" if unit else "")
            ax.set_xlabel(xlabel, fontsize=7, labelpad=2)
            
            if plot_idx % n_cols == 0:
                ax.set_ylabel(f'{response_label}', fontsize=7, labelpad=2)
            
            ax.set_title(feature_label, fontsize=8, fontweight='bold', pad=4)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.tick_params(axis='both', labelsize=6.5, length=2.5, width=0.6, pad=1.5)
            
            axis_type = "FIXED" if fixed_ylim is not None else "AUTO"
            print(f"    ✓ Panel {plot_idx+1}/{n_features}: {feature} [{axis_type} y-axis]")
            
        except Exception as e:
            print(f"    ✗ Error for {feature}: {str(e)}")
            continue
    
    handles = [plt.Line2D([0], [0], color=color_map[c], linewidth=1.8) for c in top_varieties]
    labels = [str(c)[:15] for c in top_varieties]
    stratify_display = stratify_feature.replace('_', ' ').capitalize()
    
    ncol_legend = min(len(top_varieties), 6)
    fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.02),
               frameon=False, fontsize=6, ncol=ncol_legend,
               title=stratify_display, title_fontsize=6.5,
               handlelength=1.5, labelspacing=0.4, columnspacing=1.0, borderpad=0.4)
    
    axis_info = f" (Fixed Y-axis)" if fixed_ylim else ""
    plt.suptitle(f'Stratified Partial Dependence: {model_name} (Top {len(top_varieties)} Varieties){axis_info}',
                 fontsize=10, fontweight='bold', y=0.995)
    
    plt.tight_layout(rect=[0, 0.05, 1, 0.99], h_pad=2.0, w_pad=1.5)
    
    safe_stratify = stratify_feature.replace(' ', '_')
    suffix = '_fixed_axis' if fixed_ylim else ''
    output_file = os.path.join(output_dir, f'pdp_stratified_panel_by_{safe_stratify}{suffix}.png')
    pdf_file = os.path.join(output_dir, f'pdp_stratified_panel_by_{safe_stratify}{suffix}.pdf')
    
    plt.savefig(output_file, dpi=600, bbox_inches='tight', facecolor='white')
    plt.savefig(pdf_file, bbox_inches='tight', facecolor='white')
    plt.close()
    
    print(f"\n✓ Created combined panel: {output_file}")
    return output_file

def create_all_stratified_pdps_fixed_axis(model, X_train_scaled, feature_names, top_features,
                                       stratify_feature, stratify_encoder, X_train_raw,
                                       scaler, output_dir, model_name, response_label="Effect",
                                       categorical_features=None,
                                       min_samples=30, top_n_varieties=6,
                                       reference_feature=None):
    """
    Create both standalone and combined stratified PDPs with fixed y-axis scaling.
    
    NEW PARAMETER:
    --------------
    reference_feature : str or None
        Feature name to use for computing fixed y-axis limits (e.g., 'HD').
        If None, uses auto-scaling (original behavior).
    """
    print(f"\n=== CREATING STRATIFIED PDPs BY {stratify_feature.upper()} ===")
    
    numeric_features = []
    for feature in top_features:
        is_categorical = False
        if categorical_features:
            for cat_var in categorical_features:
                if f'{cat_var}_encoded' == feature:
                    is_categorical = True
                    break
        if not is_categorical:
            numeric_features.append(feature)
    
    print(f"Creating stratified PDPs for {len(numeric_features)} numeric features")
    
    # ========================================================================
    # NEW SECTION: COMPUTE FIXED Y-AXIS LIMITS IF REFERENCE FEATURE PROVIDED
    # ========================================================================
    
    fixed_ylim = None
    if reference_feature is not None:
        if reference_feature not in numeric_features:
            print(f"\n⚠ Warning: Reference feature '{reference_feature}' not in top features!")
            print(f"Available features: {numeric_features[:5]}...")
        else:
            top_varieties = get_top_n_varieties(X_train_raw, stratify_feature, 
                                               n_top=top_n_varieties, min_samples=min_samples)
            
            y_min, y_max = compute_reference_ylim(
                model, X_train_scaled, reference_feature, feature_names,
                stratify_feature, stratify_encoder, X_train_raw,
                scaler, top_varieties, min_samples=min_samples
            )
            
            if y_min is not None and y_max is not None:
                fixed_ylim = (y_min, y_max)
                print(f"\n✓ Will use FIXED y-axis for all plots: [{y_min:.4f}, {y_max:.4f}]")
            else:
                print(f"\n⚠ Could not compute reference limits, using AUTO scaling")
    else:
        print(f"\nNo reference feature specified - using AUTO y-axis scaling")
    
    # ========================================================================
    
    print()
    
    stratified_plots = []
    all_category_data = {}
    
    for feature in numeric_features:
        try:
            feature_idx = feature_names.index(feature)
            
            plot_file, cat_data = create_stratified_pdp_for_feature_fixed_axis(
                model, X_train_scaled, feature_idx, feature,
                stratify_feature, stratify_encoder, X_train_raw,
                scaler, feature_names, output_dir, model_name,
                response_label=response_label,
                min_samples=min_samples, top_n_varieties=top_n_varieties,
                fixed_ylim=fixed_ylim  # NEW: Pass fixed limits
            )
            
            if plot_file:
                stratified_plots.append(plot_file)
                all_category_data[feature] = cat_data
                
        except Exception as e:
            print(f"    ✗ Error for {feature}: {str(e)}")
            continue
    
    axis_type = "FIXED" if fixed_ylim else "AUTO"
    print(f"\n✓ Created {len(stratified_plots)} standalone stratified plots with {axis_type} y-axis")
    
    panel_file = create_combined_stratified_pdp_panel_fixed_axis(
        model, X_train_scaled, feature_names, top_features,
        stratify_feature, stratify_encoder, X_train_raw,
        scaler, output_dir, model_name, response_label,
        categorical_features, min_samples=min_samples, 
        top_n_varieties=top_n_varieties,
        fixed_ylim=fixed_ylim  # NEW: Pass fixed limits
    )
    
    return stratified_plots, panel_file

# ============================================================================
# DATA LOADING FUNCTIONS
# ============================================================================

def load_and_prepare_data(file_path, response_variable, custom_predictors=None, 
                         categorical_vars=None, model_encoders=None):
    """Load and prepare data for PDP analysis - USE MODEL ENCODERS"""
    print(f"Loading data for {response_variable}...")
    
    df = pd.read_csv(file_path)
    
    if response_variable not in df.columns:
        raise ValueError(f"Response variable '{response_variable}' not found!")
    
    all_response_cols = list(ALL_RESPONSE_VARIABLES.keys())
    cols_to_drop = [col for col in all_response_cols 
                    if col != response_variable and col in df.columns]
    
    if cols_to_drop:
        df = df.drop(columns=cols_to_drop)
    
    if custom_predictors is not None:
        year_column = None
        for col in ['campaign_year', 'Harvest_Year', 'year']:
            if col in df.columns:
                year_column = col
                break
        
        columns_to_keep = [response_variable] + custom_predictors
        if year_column:
            columns_to_keep.append(year_column)
        if categorical_vars:
            columns_to_keep.extend(categorical_vars)
        
        df = df[columns_to_keep]
    
    df = df.dropna(subset=[response_variable])
    
    encoders = model_encoders if model_encoders is not None else {}
    
    if categorical_vars:
        print(f"\nEncoding categorical variables: {categorical_vars}")
        for cat_var in categorical_vars:
            if cat_var in df.columns:
                if cat_var in encoders:
                    print(f"   Using model encoder for {cat_var}")
                    cat_encoder = encoders[cat_var]
                    
                    df[f'{cat_var}_encoded'] = df[cat_var].apply(
                        lambda x: encode_categorical_value(x, cat_encoder, cat_var)
                    )
                    
                    valid_count = (df[f'{cat_var}_encoded'] != -1).sum()
                    invalid_count = (df[f'{cat_var}_encoded'] == -1).sum()
                    print(f"     Valid: {valid_count}, Invalid: {invalid_count}")
                    
                    if valid_count > 0:
                        df = df[df[f'{cat_var}_encoded'] != -1]
                    else:
                        raise ValueError(f"No valid encodings for {cat_var}!")
                else:
                    print(f"   Creating new encoder for {cat_var}")
                    cat_encoder = LabelEncoder()
                    df[f'{cat_var}_encoded'] = cat_encoder.fit_transform(df[cat_var].astype(str))
                    encoders[cat_var] = cat_encoder
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    for col in numeric_cols:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].median())
    
    if custom_predictors is not None:
        available_predictors = [col for col in custom_predictors if col in df.columns]
        features = available_predictors.copy()
        
        if categorical_vars:
            for cat_var in categorical_vars:
                encoded_col = f'{cat_var}_encoded'
                if encoded_col in df.columns:
                    features.append(encoded_col)
    else:
        year_column = None
        for col in ['campaign_year', 'Harvest_Year', 'year']:
            if col in df.columns:
                year_column = col
                break
        
        exclude_cols = [response_variable]
        if year_column:
            exclude_cols.extend([year_column, f'{year_column}_encoded'])
        if categorical_vars:
            exclude_cols.extend(categorical_vars)
        exclude_cols.extend([col for col in all_response_cols if col != response_variable])
        features = [col for col in numeric_cols if col not in exclude_cols]
    
    X = df[features].copy()
    y = df[response_variable].astype(float)
    
    X_with_categorical = X.copy()
    if categorical_vars:
        for cat_var in categorical_vars:
            if cat_var in df.columns:
                X_with_categorical[cat_var] = df[cat_var]
    
    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)
    
    X_with_cat_temp, _, _, _ = train_test_split(X_with_categorical, y, test_size=0.2, random_state=42)
    X_train_with_cat, _, _, _ = train_test_split(X_with_cat_temp, y_temp, test_size=0.25, random_state=42)
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    X_val_scaled = scaler.transform(X_val)
    
    return X_train_scaled, X_val_scaled, X_test_scaled, y_train, y_val, y_test, features, scaler, X_train_with_cat, encoders

def load_model_and_get_top_features(models_dir, response_variable, model_name='RandomForest', shap_results_file=None):
    """Load model and get top features - INCLUDING ENCODERS"""
    print(f"Loading {model_name} model for {response_variable}...")
    
    model_file = os.path.join(models_dir, f'{model_name.replace(" ", "_")}_{response_variable}.pkl')
    if not os.path.exists(model_file):
        raise FileNotFoundError(f"Model file not found: {model_file}")
    
    model_data = joblib.load(model_file)
    
    if isinstance(model_data, dict):
        model = model_data.get('model', model_data)
        model_encoders = model_data.get('encoders', {})
        print(f"  Loaded model with {len(model_encoders)} encoders")
    else:
        model = model_data
        model_encoders = {}
    
    top_features = None
    if shap_results_file and os.path.exists(shap_results_file):
        try:
            shap_df = pd.read_csv(shap_results_file)
            model_shap = shap_df[shap_df['model'] == model_name].sort_values('mean_abs_shap', ascending=False)
            if not model_shap.empty:
                top_features = model_shap['feature'].head(12).tolist()
                print(f"  Loaded top {len(top_features)} features from SHAP")
        except Exception as e:
            print(f"  Could not load SHAP: {str(e)}")
    
    return model, top_features, model_encoders

def get_top_features_from_model(model, feature_names, n_features=12):
    """Get top features from model importance"""
    if hasattr(model, 'feature_importances_'):
        importance_df = pd.DataFrame({
            'feature': feature_names,
            'importance': model.feature_importances_
        }).sort_values('importance', ascending=False)
        return importance_df.head(n_features)['feature'].tolist()
    else:
        return feature_names[:n_features]

# ============================================================================
# MAIN STRATIFIED PDP ANALYSIS FUNCTION
# ============================================================================

def perform_stratified_pdp_analysis_fixed_axis(data_file, models_dir, output_dir, response_variable,
                                   model_name='RandomForest',
                                   custom_predictors=None,
                                   categorical_vars=None,
                                   shap_results_dir=None,
                                   stratify_by=None,
                                   min_samples=30,
                                   top_n_varieties=6,
                                   n_top_features=12,
                                   reference_feature=None):
    """
    Perform stratified PDP analysis with FIXED y-axis scaling option.
    
    NEW PARAMETER:
    --------------
    reference_feature : str or None
        Feature name to use as reference for y-axis limits (e.g., 'HD').
        All plots will use the same y-axis scale as this reference feature.
        If None, each plot uses independent auto-scaling.
    """
    
    print(f"\n{'='*80}")
    print(f"STRATIFIED PDP ANALYSIS FOR: {response_variable}")
    print(f"Display name: {ALL_RESPONSE_VARIABLES[response_variable]}")
    print(f"{'='*80}")
    print(f"Model: {model_name}")
    print(f"Top {n_top_features} numeric features")
    if stratify_by:
        print(f"Stratification: By '{stratify_by}' (TOP {top_n_varieties} categories)")
        if stratify_by.lower() in ['crop_cycle', 'cycle']:
            print(f"⚠ Crop cycle filtering: Excluding {EXCLUDED_CATEGORIES}")
    if reference_feature:
        print(f"Y-axis scaling: FIXED (based on '{reference_feature}')")
    else:
        print(f"Y-axis scaling: AUTO (independent per plot)")
    
    os.makedirs(output_dir, exist_ok=True)
    
    response_label = ALL_RESPONSE_VARIABLES.get(response_variable, response_variable)
    
    shap_results_file = None
    if shap_results_dir:
        shap_results_file = os.path.join(shap_results_dir, f'shap_feature_importance_summary.csv')
    
    model, top_features_shap, model_encoders = load_model_and_get_top_features(
        models_dir, response_variable, model_name, shap_results_file
    )
    
    X_train_scaled, X_val_scaled, X_test_scaled, y_train, y_val, y_test, feature_names, scaler, X_train_raw, encoders = load_and_prepare_data(
        data_file, response_variable, custom_predictors, categorical_vars, model_encoders
    )
    print(f"Data loaded: {len(X_train_scaled)} training samples")
    
    if top_features_shap:
        top_features = top_features_shap
    else:
        top_features = get_top_features_from_model(model, feature_names, n_features=n_top_features)
    
    # Filter to numeric features only
    numeric_features = []
    for feature in top_features:
        is_categorical = False
        if categorical_vars:
            for cat_var in categorical_vars:
                if f'{cat_var}_encoded' == feature:
                    is_categorical = True
                    break
        if not is_categorical:
            numeric_features.append(feature)
    
    # Ensure we have exactly n_top_features
    if len(numeric_features) < n_top_features:
        all_numeric_features = []
        for feature in feature_names:
            is_categorical = False
            if categorical_vars:
                for cat_var in categorical_vars:
                    if f'{cat_var}_encoded' == feature:
                        is_categorical = True
                        break
            if not is_categorical:
                all_numeric_features.append(feature)
        
        if hasattr(model, 'feature_importances_'):
            importance_df = pd.DataFrame({
                'feature': feature_names,
                'importance': model.feature_importances_
            })
            importance_df = importance_df[importance_df['feature'].isin(all_numeric_features)]
            importance_df = importance_df.sort_values('importance', ascending=False)
            numeric_features = importance_df.head(n_top_features)['feature'].tolist()
        else:
            numeric_features = all_numeric_features[:n_top_features]
    
    numeric_features = numeric_features[:n_top_features]
    top_features = numeric_features
    
    print(f"\n{'='*60}")
    print(f"TOP {len(numeric_features)} NUMERIC FEATURES:")
    print(f"{'='*60}")
    for i, feat in enumerate(numeric_features, 1):
        feat_label, unit = get_feature_display_info(feat)
        ref_marker = " ← REFERENCE" if feat == reference_feature else ""
        print(f"  {i:2d}. {feat_label}{ref_marker}")
    print(f"{'='*60}\n")
    
    stratified_plots = []
    panel_file = None
    
    if stratify_by and categorical_vars and stratify_by in categorical_vars:
        stratify_encoder = encoders.get(stratify_by)
        if stratify_encoder:
            stratified_plots, panel_file = create_all_stratified_pdps_fixed_axis(
                model, X_train_scaled, feature_names, top_features,
                stratify_by, stratify_encoder, X_train_raw,
                scaler, output_dir, model_name, response_label,
                categorical_vars,
                min_samples=min_samples, top_n_varieties=top_n_varieties,
                reference_feature=reference_feature  # NEW: Pass reference feature
            )
        else:
            print(f"\n⚠ Warning: No encoder for '{stratify_by}'")
    
    print(f"\n{'='*80}")
    print("STRATIFIED PDP ANALYSIS COMPLETE")
    print(f"{'='*80}")
    print(f"Response variable: {response_variable} - {response_label}")
    print(f"Results saved to: {output_dir}")
    if reference_feature:
        print(f"Y-axis: FIXED (based on '{reference_feature}')")
    else:
        print(f"Y-axis: AUTO (independent scaling)")
    if stratified_plots:
        print(f"✓ Standalone plots: {len(stratified_plots)}")
    if panel_file:
        print(f"✓ Combined panel: {panel_file}")
    
    return {
        'output_dir': output_dir,
        'stratified_plots': stratified_plots,
        'panel_file': panel_file,
        'response_variable': response_variable,
        'fixed_ylim_used': reference_feature is not None
    }
# ============================================================================
# MAIN FUNCTION WITH FLEXIBLE RESPONSE VARIABLES
# ============================================================================

def main(data_file, response_variable, custom_predictors=None, categorical_vars=None,
         base_output_dir=None, model_name='RandomForest',
         stratify_by=None, min_samples=30, top_n_varieties=6, n_top_features=12,
         reference_feature=None):
    """
    Main function with FIXED y-axis scaling option.
    
    NEW PARAMETER:
    --------------
    reference_feature : str or None
        Feature name to use as reference for y-axis limits (e.g., 'HD').
        Example: reference_feature='HD' will compute y-axis limits from HD's PDP
        and apply those limits to all other features for direct comparability.
    """
    
    if response_variable not in ALL_RESPONSE_VARIABLES:
        raise ValueError(f"Response variable '{response_variable}' not recognized!\n"
                        f"Available: {list(ALL_RESPONSE_VARIABLES.keys())}")
    
    if base_output_dir is None:
        base_output_dir = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Irrigation_Split_Analysis/GAG'
    
    # Determine folder suffix
    if categorical_vars:
        folder_suffix = 'with_categorical'
    elif custom_predictors:
        folder_suffix = 'custom'
    else:
        folder_suffix = 'models'
    
    # Construct paths following the pattern
    models_dir = os.path.join(base_output_dir, f'kfold_models_{response_variable}_{folder_suffix}', 'models')
    shap_results_dir = os.path.join(base_output_dir, f'shap_analysis_{response_variable}_{folder_suffix}')
    
    # Add suffix to output directory if using fixed axis
    output_subdir = f'pdp_stratified_{response_variable}_{folder_suffix}'
    if reference_feature:
        output_subdir += '_fixed_axis'
    output_dir = os.path.join(base_output_dir, output_subdir)
    
    # Create model-specific subfolder
    output_dir = os.path.join(output_dir, model_name.replace(" ", "_"))
    
    print(f"\n{'='*80}")
    print(f"STRATIFIED PDP ANALYSIS CONFIGURATION")
    print(f"{'='*80}")
    print(f"Data file: {data_file}")
    print(f"Response variable: {response_variable} - {ALL_RESPONSE_VARIABLES[response_variable]}")
    print(f"Models directory: {models_dir}")
    print(f"SHAP directory: {shap_results_dir}")
    print(f"Output directory: {output_dir}")
    print(f"Model: {model_name}")
    if custom_predictors:
        print(f"Custom predictors: {len(custom_predictors)} features")
    if categorical_vars:
        print(f"Categorical variables: {categorical_vars}")
    if stratify_by:
        print(f"Stratify by: {stratify_by} (Top {top_n_varieties} categories)")
    if reference_feature:
        print(f"Y-axis: FIXED (reference: '{reference_feature}')")
    else:
        print(f"Y-axis: AUTO (independent)")
    
    try:
        results = perform_stratified_pdp_analysis_fixed_axis(
            data_file=data_file,
            models_dir=models_dir,
            output_dir=output_dir,
            response_variable=response_variable,
            model_name=model_name,
            custom_predictors=custom_predictors,
            categorical_vars=categorical_vars,
            shap_results_dir=shap_results_dir,
            stratify_by=stratify_by,
            min_samples=min_samples,
            top_n_varieties=top_n_varieties,
            n_top_features=n_top_features,
            reference_feature=reference_feature  # NEW: Pass reference feature
        )
        
        print(f"\n{'='*80}")
        print("STRATIFIED PDP ANALYSIS COMPLETED SUCCESSFULLY")
        print(f"{'='*80}")
        
        return results
        
    except Exception as e:
        print(f"\nError: {str(e)}")
        import traceback
        traceback.print_exc()
        raise


# ============================================================================
# USAGE EXAMPLE WITH FIXED Y-AXIS
# ============================================================================

if __name__ == "__main__":
    
    custom_features = [
        'ampl_mean', 'TNn', 'TNx', 'Coarse_Silt_percent_complete',
        'Total_Nitrogen_per_mille_complete', 'pH_complete', 'temp_moy_mean', 
        'Exchangeable_Calcium_meq_100g_complete', 'h_maxi_mean', 'CDD', 
        'radiation_ERA_total', 'Rx1', 'R20', 'SDII',
        'h_mini_mean', 'HD', 'Available_P2O5_mg_kg_complete', 
        'Electrical_Conductivity_mmho_cm_complete', 
        'Exchangeable_Sodium_meq_100g_complete',
        'R99p', 'growing_season_rainfall_mm', 'soil_temp_6h_mean', 
        'CWD', 'Mg_K_Ratio_complete'
    ]
    
    categorical_features = ['variety', 'irrigation_type', 'harvest_mode', 'crop_cycle']
    
    data_file = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis/final_dataset_regrouped_categorical.csv'
    base_output = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis'
    
    print("="*80)
    print("STRATIFIED PDP ANALYSIS WITH FIXED Y-AXIS OPTION")
    print("="*80 + "\n")
    
    try:
        # Example 1: FIXED y-axis based on 'HD' feature
        print("\n### EXAMPLE 1: FIXED Y-AXIS (Reference: HD) ###\n")
        results_fixed = main(
            data_file=data_file,
            response_variable='tc_per_ha_actual',
            custom_predictors=custom_features,
            categorical_vars=categorical_features,
            base_output_dir=base_output,
            model_name='RandomForest',
            stratify_by='variety',
            min_samples=30,
            top_n_varieties=16,
            n_top_features=14,
            reference_feature='HD'  # Use HD as reference for y-axis
        )
        
        print("\n" + "="*80)
        print("ANALYSIS COMPLETED SUCCESSFULLY!")
        print("="*80)
        
    except Exception as e:
        print(f"\n❌ ERROR: {str(e)}")
        import traceback
        traceback.print_exc()


STRATIFIED PDP ANALYSIS WITH FIXED Y-AXIS OPTION

Key features:
  • NEW: Fixed y-axis scaling based on reference feature
  • All plots use same y-axis limits for direct comparison
  • Reference feature (e.g., 'HD') determines the scale
  • Transparent legends with smart positioning
  • Publication-quality plots at 600 DPI


### EXAMPLE 1: FIXED Y-AXIS (Reference: HD) ###


STRATIFIED PDP ANALYSIS CONFIGURATION
Data file: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis/final_dataset_regrouped_categorical.csv
Response variable: tc_per_ha_actual - Cane yield (t ha⁻¹)
Models directory: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis\kfold_models_tc_per_ha_actual_with_categorical\models
SHAP directory: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis\shap_analysis_tc_per_ha_actual_with_categorical
Output directory: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis\pdp_stratified_

## 8. Categorical Partial Dependence: Side-by-Side Comparison

This section creates side-by-side comparative visualizations of categorical partial dependence plots for simultaneous evaluation of yield and sugar content responses across management practices.

**Visualization Strategy:**

**I. Dual-Response Bar Plots**
- **Layout**: 1×2 panel configuration (yield | Brix content)
- Each categorical variable (variety, crop cycle, irrigation, harvest mode) gets its own figure
- Color-coded by response: Blue (yield), Orange (Brix)
- Consistent category ordering across both panels

**II. Statistical Inference**
- **Parametric approach**: ANOVA + Tukey HSD (when normality & homoscedasticity hold)
- **Non-parametric approach**: Kruskal-Wallis + Dunn's test (otherwise)
- Compact letter display (CLD) for pairwise comparisons
- Significance letters positioned above confidence intervals

**III. Uncertainty Quantification**
- Bootstrap confidence intervals (100 iterations, 95% CI)
- Error bars on each category bar
- Shapiro-Wilk normality test per category
- Levene's test for variance homogeneity

**IV. Category Filtering & Ordering**
- **Variety**: Custom order by mean yield performance
- **Crop cycle**: Excludes late ratoons (9th+) with sparse data
- **X-axis labels**: 45° rotation for readability

**Output:** Individual high-resolution PNG files (600 DPI) for each categorical variable, enabling direct comparison of how yield and quality respond to different management practices.

In [ ]:
"""
MULTI-PANEL STATISTICAL COMPARISON
===================================
Creates 3-panel figures showing statistical comparisons across categorical variables

Author: Agricultural ML Analysis
Date: 2025
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import kruskal
from itertools import combinations
import warnings
import os
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.size': 9,
    'font.family': 'Arial',
    'axes.labelsize': 9,
    'axes.titlesize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.titlesize': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': False,
    'axes.axisbelow': True,
    'axes.edgecolor': '#000000',
    'text.color': '#000000',
    'axes.labelcolor': '#000000',
    'xtick.color': '#000000',
    'ytick.color': '#000000',
    'axes.linewidth': 1.0,
    'lines.linewidth': 1.0,
    'patch.linewidth': 0.5,
    'xtick.major.width': 1.0,
    'ytick.major.width': 1.0,
    'xtick.major.size': 3.5,
    'ytick.major.size': 3.5,
    'figure.dpi': 600,
    'savefig.dpi': 600,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.05,
})

RESPONSE_COLORS = {
    "tc_per_ha_actual": '#1f77b4',  # Blue for cane yield
    "hugot_sugar": '#ff7f0e',       # Orange for sucrose
    "total_recoverable_sugar_t_per_ha": '#2ca02c'  # Green for TPER
}

# Response variable labels
RESPONSE_LABELS = {
    'tc_per_ha_actual': r'Cane Yield (t ha$^{-1}$)',
    'hugot_sugar': 'Sucrose (%)',
    'total_recoverable_sugar_t_per_ha': r'TPER (t ha$^{-1}$)'
}

# Subplot labels
SUBPLOT_LABELS = {
    'tc_per_ha_actual': 'Cane Yield',
    'hugot_sugar': 'Sucrose',
    'total_recoverable_sugar_t_per_ha': 'TPER'
}

IRRIGATION_COLORS = {'GRV': '#E63946', 'GAG': '#4A90E2'}
HARVEST_COLORS = {'MANUELLE': '#D4A373', 'MECANIQUE': '#8B4513'}
DEFAULT_BINARY_COLORS = {0: '#1f77b4', 1: '#ff7f0e', 2: '#2ca02c', 3: '#d62728'}

# Category orders
CROP_CYCLE_ORDER = ['Plant Cane', '1st Ratoon', '2nd Ratoon', '3rd Ratoon', 
                    '4th Ratoon', '5th Ratoon', '6th Ratoon', '7th Ratoon', '8th Ratoon']

EXCLUDED_CATEGORIES = ['9th Ratoon', '10th Ratoon', '11th Ratoon', 'Beyond 12th Ratoon']

BINARY_ORDERS = {
    'irrigation': ['GRV', 'GAG', 'Sprinkler', 'Drip'],
    'irrigation_type': ['GRV', 'GAG', 'Sprinkler', 'Drip'],
    'harvest': ['MANUELLE', 'MECANIQUE', 'Manual', 'Machine'],
    'harvest_method': ['MANUELLE', 'MECANIQUE', 'Manual', 'Machine'],
    'harvest_mode': ['MANUELLE', 'MECANIQUE', 'Manual', 'Machine'],
    'recolte': ['MANUELLE', 'MECANIQUE', 'Manual', 'Machine'],
}


def is_binary_variable(cat_var, unique_values):
    """Check if variable is binary or few-level"""
    n_levels = len(unique_values)
    return n_levels == 2, 2 <= n_levels <= 4


def get_binary_variable_order(cat_var, unique_values):
    """Get fixed order for binary/few-level variables"""
    cat_var_lower = cat_var.lower()
    
    for key in BINARY_ORDERS:
        if key in cat_var_lower:
            predefined_order = BINARY_ORDERS[key]
            filtered_order = []
            for val in predefined_order:
                if val in unique_values:
                    filtered_order.append(val)
                else:
                    for actual_val in unique_values:
                        if str(actual_val).upper() == val.upper():
                            filtered_order.append(actual_val)
                            break
            
            for val in unique_values:
                if val not in filtered_order:
                    filtered_order.append(val)
            
            return filtered_order
    
    _, is_few_level = is_binary_variable(cat_var, unique_values)
    if is_few_level:
        return sorted(unique_values)
    
    return None


def get_binary_colors(cat_var, categories):
    """Get color map for binary/few-level variables"""
    cat_var_lower = cat_var.lower()
    
    irrigation_keywords = ['irrigation', 'water', 'irrig', 'arrosage']
    if any(keyword in cat_var_lower for keyword in irrigation_keywords):
        color_map = {}
        for i, cat in enumerate(categories):
            cat_upper = str(cat).upper()
            if cat_upper in IRRIGATION_COLORS:
                color_map[str(cat)] = IRRIGATION_COLORS[cat_upper]
            else:
                color_map[str(cat)] = DEFAULT_BINARY_COLORS.get(i, CATEGORY_COLORS[i % len(CATEGORY_COLORS)])
        return color_map
    
    harvest_keywords = ['harvest', 'recolte', 'harvester', 'moisson', 'mode']
    if any(keyword in cat_var_lower for keyword in harvest_keywords):
        color_map = {}
        for i, cat in enumerate(categories):
            cat_upper = str(cat).upper()
            if cat_upper in HARVEST_COLORS:
                color_map[str(cat)] = HARVEST_COLORS[cat_upper]
            else:
                color_map[str(cat)] = DEFAULT_BINARY_COLORS.get(i, CATEGORY_COLORS[i % len(CATEGORY_COLORS)])
        return color_map
    
    _, is_few_level = is_binary_variable(cat_var, categories)
    if is_few_level:
        return {str(cat): DEFAULT_BINARY_COLORS.get(i, CATEGORY_COLORS[i % len(CATEGORY_COLORS)]) 
                for i, cat in enumerate(categories)}
    
    return None


def format_p_value(p_value):
    """
    Format p-value for display
    Returns formatted string: < 0.001, < 0.01, < 0.05, or ns
    
    Parameters:
    -----------
    p_value : float
        P-value from statistical test
        
    Returns:
    --------
    str : Formatted p-value string
    """
    if p_value < 0.001:
        return "< 0.001"
    elif p_value < 0.01:
        return "< 0.01"
    elif p_value < 0.05:
        return "< 0.05"
    else:
        return "ns"  # not significant
    """Assign consistent colors to categories"""
    df = df.copy()
    df[cat_var] = df[cat_var].fillna('MISSING').astype(str)
    
    all_categories = df[df[cat_var] != 'MISSING'][cat_var].unique()
    all_categories = [str(c) for c in all_categories]
    
    binary_colors = get_binary_colors(cat_var, all_categories)
    if binary_colors is not None:
        return binary_colors
    
    all_categories_sorted = sorted(all_categories)
    color_map = {cat: CATEGORY_COLORS[i % len(CATEGORY_COLORS)] 
                 for i, cat in enumerate(all_categories_sorted)}
    
    return color_map


def filter_excluded_categories(df, cat_var):
    """Filter out excluded categories"""
    initial_len = len(df)
    df = df.copy()
    df[cat_var] = df[cat_var].fillna('MISSING').astype(str)
    df_filtered = df[df[cat_var] != 'MISSING'].copy()
    n_missing = initial_len - len(df_filtered)
    
    if cat_var.lower() in ['crop_cycle', 'cycle']:
        excluded_str = [str(x) for x in EXCLUDED_CATEGORIES]
        df_filtered = df_filtered[~df_filtered[cat_var].isin(excluded_str)].copy()
    
    return df_filtered, n_missing


def get_category_order(cat_var, categories):
    """
    Define custom ordering for categorical variables
    Matches the PDP style ordering
    """
    if cat_var == 'variety':
        variety_order = [
            'N 14', 'FR 8783', 'CP 722086', 'N 19', 'CP 742005', 'R 579',
            'SP 75184', 'R 575', 'RB 72454', 'OTHER VARIETIES', 'SP 701284',
            'SP 703370', 'B 63118', 'DB 66113', 'SP 711081', 'CO 6806'
        ]
        ordered = [cat for cat in variety_order if cat in categories]
        remaining = [cat for cat in categories if cat not in ordered]
        return ordered + sorted(remaining)
    
    elif cat_var == 'crop_cycle':
        excluded = ['9th Ratoon', '10th Ratoon', '11th Ratoon', '12th Ratoon',
                   'Beyond 12 Ratoon', 'Beyond 12th Ratoon']
        filtered = [cat for cat in categories if cat not in excluded]
        
        cycle_order = ['Plant Cane', '1st Ratoon', '2nd Ratoon', '3rd Ratoon',
                      '4th Ratoon', '5th Ratoon', '6th Ratoon', '7th Ratoon', '8th Ratoon']
        ordered = [cat for cat in cycle_order if cat in filtered]
        remaining = [cat for cat in filtered if cat not in ordered]
        return ordered + sorted(remaining)
    
    elif cat_var == 'irrigation_type':
        irrigation_order = ['GRV', 'GAG']
        ordered = [cat for cat in irrigation_order if cat in categories]
        remaining = [cat for cat in categories if cat not in ordered]
        return ordered + sorted(remaining)
    
    elif cat_var == 'harvest_mode':
        harvest_order = ['MANUELLE', 'MECANIQUE']
        ordered = [cat for cat in harvest_order if cat in categories]
        remaining = [cat for cat in categories if cat not in ordered]
        return ordered + sorted(remaining)
    
    return sorted(categories)


def posthoc_dunn_manual(groups, group_names, p_adjust='bonferroni'):
    """Manual Dunn's post-hoc test"""
    n_groups = len(groups)
    n_total = sum(len(g) for g in groups)
    all_data = np.concatenate(groups)
    ranks = stats.rankdata(all_data)
    
    start_idx = 0
    mean_ranks = []
    n_sizes = []
    
    for group in groups:
        end_idx = start_idx + len(group)
        group_ranks = ranks[start_idx:end_idx]
        mean_ranks.append(np.mean(group_ranks))
        n_sizes.append(len(group))
        start_idx = end_idx
    
    p_values = np.ones((n_groups, n_groups))
    
    for i, j in combinations(range(n_groups), 2):
        z_score = (mean_ranks[i] - mean_ranks[j]) / np.sqrt(
            (n_total * (n_total + 1) / 12) * (1/n_sizes[i] + 1/n_sizes[j])
        )
        p_val = 2 * (1 - stats.norm.cdf(abs(z_score)))
        p_values[i, j] = p_val
        p_values[j, i] = p_val
    
    p_df = pd.DataFrame(p_values, index=group_names, columns=group_names)
    
    if p_adjust == 'bonferroni':
        n_comparisons = n_groups * (n_groups - 1) / 2
        for i, j in combinations(range(n_groups), 2):
            p_df.iloc[i, j] = min(p_df.iloc[i, j] * n_comparisons, 1.0)
            p_df.iloc[j, i] = p_df.iloc[i, j]
    
    return p_df


def assign_significance_letters(p_values, alpha=0.05, means_dict=None):
    """Assign significance letters"""
    categories = p_values.index.tolist()
    
    if means_dict is not None:
        valid_means = {k: v for k, v in means_dict.items() if not pd.isna(v) and k in categories}
        if len(valid_means) > 0:
            categories_ordered = sorted(valid_means.keys(), key=lambda x: valid_means[x], reverse=True)
        else:
            categories_ordered = sorted(categories)
    else:
        categories_ordered = sorted(categories)
    
    if len(categories_ordered) <= 1:
        return {cat: 'a' for cat in categories_ordered}
    
    sig_matrix = (p_values < alpha).astype(bool)
    letters = {cat: [] for cat in categories_ordered}
    available_letters = [chr(97 + i) for i in range(26)]
    letter_idx = 0
    assigned = set()
    
    for i, cat in enumerate(categories_ordered):
        if cat in assigned:
            continue
        
        current_letter = available_letters[letter_idx]
        letter_idx += 1
        group = [cat]
        
        for j in range(i + 1, len(categories_ordered)):
            other_cat = categories_ordered[j]
            if other_cat not in assigned:
                can_join = all(not sig_matrix.loc[group_member, other_cat] for group_member in group)
                if can_join:
                    group.append(other_cat)
        
        for cat_in_group in group:
            letters[cat_in_group].append(current_letter)
            assigned.add(cat_in_group)
    
    letters = {cat: ''.join(sorted(letters[cat])) if letters[cat] else 'a' 
               for cat in categories_ordered}
    
    return letters


def compute_statistics(df, cat_var, response_var, categories_order=None):
    """Compute statistics for a categorical variable and response variable"""
    df_filtered, _ = filter_excluded_categories(df, cat_var)
    df_filtered = df_filtered.copy()
    df_filtered[cat_var] = df_filtered[cat_var].astype(str)
    
    unique_values = df_filtered[cat_var].unique()
    _, is_few_level = is_binary_variable(cat_var, unique_values)
    n_categories = len(unique_values)
    
    stats_data = []
    categories_with_data = []
    
    for cat in df_filtered[cat_var].unique():
        cat_str = str(cat)
        cat_data = df_filtered[df_filtered[cat_var] == cat_str][response_var].dropna()
        if len(cat_data) >= 3:
            stats_data.append({
                'category': cat_str,
                'mean': float(cat_data.mean()),
                'std': float(cat_data.std()),
                'n': int(len(cat_data))
            })
            categories_with_data.append(cat_str)
    
    if len(stats_data) == 0:
        return None, {}, 1.0, 'none', None
    
    stats_df = pd.DataFrame(stats_data)
    
    # Order categories
    if categories_order is not None:
        categories_order_filtered = [c for c in categories_order if c in categories_with_data]
        if len(categories_order_filtered) > 0:
            stats_df = stats_df.set_index('category').reindex(categories_order_filtered).reset_index()
            stats_df = stats_df.dropna(subset=['mean'])
    else:
        categories_order_filtered = get_category_order(cat_var, categories_with_data)
        if categories_order_filtered is None:
            stats_df = stats_df.sort_values('mean', ascending=False)
        else:
            categories_order_filtered = [c for c in categories_order_filtered if c in categories_with_data]
            if len(categories_order_filtered) > 0:
                stats_df = stats_df.set_index('category').reindex(categories_order_filtered).reset_index()
                stats_df = stats_df.dropna(subset=['mean'])
    
    categories_order = stats_df['category'].tolist()
    n_categories = len(categories_order)
    
    # Statistical tests
    if len(stats_df) < 2:
        sig_letters = {cat: 'a' for cat in categories_order}
        return stats_df, sig_letters, 1.0, 'none', is_few_level
    
    groups = [df_filtered[df_filtered[cat_var] == str(cat)][response_var].dropna().values 
              for cat in categories_order]
    
    # For binary comparisons (2 groups), use Mann-Whitney U test
    if n_categories == 2:
        from scipy.stats import mannwhitneyu
        _, p_overall = mannwhitneyu(groups[0], groups[1], alternative='two-sided')
        test_name = 'Mann-Whitney'
        
        # For 2 groups, significance is determined by the overall test
        if p_overall < 0.05:
            # Determine which group has higher mean
            if stats_df.iloc[0]['mean'] > stats_df.iloc[1]['mean']:
                sig_letters = {categories_order[0]: 'a', categories_order[1]: 'b'}
            else:
                sig_letters = {categories_order[0]: 'b', categories_order[1]: 'a'}
        else:
            sig_letters = {cat: 'a' for cat in categories_order}
        
        return stats_df, sig_letters, p_overall, test_name, is_few_level
    
    # For 3+ groups, check normality and use appropriate test
    normality_results = []
    for group in groups:
        if len(group) >= 3:
            try:
                _, p_val = stats.shapiro(group)
                normality_results.append(p_val > 0.05)
            except:
                normality_results.append(False)
        else:
            normality_results.append(False)
    
    use_parametric = all(normality_results)
    
    try:
        if use_parametric:
            f_stat, p_overall = stats.f_oneway(*groups)
            test_name = 'ANOVA'
            from statsmodels.stats.multicomp import pairwise_tukeyhsd
            all_data = []
            all_labels = []
            for cat, group in zip(categories_order, groups):
                all_data.extend([float(x) for x in group])
                all_labels.extend([str(cat)] * len(group))
            
            tukey_result = pairwise_tukeyhsd(all_data, all_labels, alpha=0.05)
            p_values_matrix = pd.DataFrame(np.ones((len(categories_order), len(categories_order))),
                                          index=categories_order, columns=categories_order)
            
            for i in range(len(tukey_result.summary().data) - 1):
                row = tukey_result.summary().data[i + 1]
                cat1, cat2, p_val = str(row[0]), str(row[1]), float(row[3])
                if cat1 in p_values_matrix.index and cat2 in p_values_matrix.columns:
                    p_values_matrix.loc[cat1, cat2] = p_val
                    p_values_matrix.loc[cat2, cat1] = p_val
        else:
            h_stat, p_overall = kruskal(*groups)
            test_name = 'Kruskal-Wallis'
            p_values_matrix = posthoc_dunn_manual(groups, categories_order, p_adjust='bonferroni')
    except:
        p_overall = 1.0
        test_name = 'none'
        sig_letters = {cat: 'a' for cat in categories_order}
        return stats_df, sig_letters, p_overall, test_name, is_few_level
    
    if p_overall < 0.05:
        means_dict = dict(zip(stats_df['category'], stats_df['mean']))
        sig_letters = assign_significance_letters(p_values_matrix, alpha=0.05, means_dict=means_dict)
    else:
        sig_letters = {cat: 'a' for cat in categories_order}
    
    return stats_df, sig_letters, p_overall, test_name, is_few_level


def create_1x3_categorical_plot(df, cat_var, response_vars, output_path=None):
    """
    Create 1x3 plot for ONE categorical variable showing 3 response variables
    Styled to match the PDP publication style
    
    Parameters:
    -----------
    df : DataFrame
        Input data
    cat_var : str
        Categorical variable to analyze
    response_vars : list
        List of 3 response variables
    output_path : str, optional
        Path to save figure
    """
    if len(response_vars) != 3:
        raise ValueError("Must provide exactly 3 response variables")
    
    print(f"\n{'='*80}")
    print(f"Creating 1x3 plot for: {cat_var}")
    print(f"{'='*80}")
    
    # Get categories for this variable
    df_clean, _ = filter_excluded_categories(df, cat_var)
    categories = df_clean[cat_var].unique()
    categories_order = get_category_order(cat_var, categories)
    
    if not categories_order:
        categories_order = sorted(categories)
    
    n_categories = len(categories_order)
    print(f"  Processing {n_categories} categories: {categories_order}")
    
    # Calculate dynamic width
    base_width_per_subplot = 2.4
    if n_categories > 6:
        extra_width = (n_categories - 6) * 0.15
        width_per_subplot = base_width_per_subplot + extra_width
    else:
        width_per_subplot = base_width_per_subplot
    
    total_width = min(width_per_subplot * 3, 11.0)
    fig_height = 2.8
    
    fig, axes = plt.subplots(1, 3, figsize=(total_width, fig_height))
    print(f"  Figure size: {total_width:.1f}\" x {fig_height}\"")
    
    # Plot each response variable
    for resp_idx, response_var in enumerate(response_vars):
        ax = axes[resp_idx]
        color = RESPONSE_COLORS[response_var]
        
        print(f"\n  Processing {response_var}...")
        
        # Compute statistics
        stats_df, sig_letters, p_value, test_name, is_few_level = compute_statistics(
            df, cat_var, response_var, categories_order
        )
        
        if stats_df is None or len(stats_df) == 0:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            ax.set_xticks([])
            ax.set_yticks([])
            continue
        
        # Format p-value for display
        p_value_text = format_p_value(p_value)
        
        # Bar width
        bar_width = 0.5 if (is_few_level and len(stats_df) <= 4) else 0.5  # Fixed at 0.5
        
        # Plot bars
        x_pos = np.arange(len(stats_df))
        bars = ax.bar(x_pos, stats_df['mean'], bar_width,
                     color=color, alpha=0.85,
                     edgecolor='white', linewidth=1.5, zorder=3)
        
        # Add error bars
        errors = np.array([[stats_df.iloc[i]['mean'] - stats_df.iloc[i]['mean'] + stats_df.iloc[i]['std'],
                           stats_df.iloc[i]['std']] for i in range(len(stats_df))]).T
        
        ax.errorbar(x_pos, stats_df['mean'], yerr=stats_df['std'],
                   fmt='none', ecolor='#000000', capsize=3.5,
                   capthick=1.0, alpha=0.8, linewidth=1.0, zorder=4)
        
        # Adjust y-limits - start at 40% of max value, not at min
        y_max_data = (stats_df['mean'] + stats_df['std']).max()
        y_min_plot = y_max_data * 0.4  # Start at 40% of max
        y_range = y_max_data - y_min_plot
        
        # Add significance letters
        letter_offset = y_range * 0.03
        
        for i, cat in enumerate(stats_df['category']):
            letter = sig_letters.get(str(cat), 'a')
            y_pos_letter = stats_df.iloc[i]['mean'] + stats_df.iloc[i]['std'] + letter_offset
            
            ax.text(i, y_pos_letter, letter,
                   ha='center', va='bottom',
                   fontsize=6.5, fontweight='bold', color=color,
                   bbox=dict(boxstyle='round,pad=0.15', facecolor='white',
                           edgecolor=color, alpha=0.9, linewidth=0.8),
                   zorder=5)
        
        # Adjust y-limits for letters
        y_lim_max = y_max_data + y_range * 0.15
        ax.set_ylim(y_min_plot, y_lim_max)
        
        # Format x-axis - horizontal for binary variables, 45° for others
        # Check if this is a binary variable (2 categories)
        if len(stats_df) == 2 and cat_var.lower() in ['irrigation_type', 'harvest_mode', 'irrigation', 'harvest']:
            # Horizontal labels for binary variables
            ax.set_xticks(x_pos)
            ax.set_xticklabels(stats_df['category'], rotation=0, ha='center',
                              fontsize=7.5, fontweight='normal')
        else:
            # 45° rotation for multi-level variables
            ax.set_xticks(x_pos)
            ax.set_xticklabels(stats_df['category'], rotation=45, ha='right',
                              fontsize=7.5, fontweight='normal')
        
        ax.tick_params(axis='x', width=1.0, length=3.5)
        
        # Y-axis labels with color matching
        ax.set_ylabel(RESPONSE_LABELS[response_var],
                     fontsize=8.5, fontweight='bold', color=color)
        ax.tick_params(axis='y', labelcolor=color, labelsize=7.5,
                      width=1.0, length=3.5, colors=color)
        
        # Gridlines
        ax.yaxis.grid(True, linestyle='--', linewidth=0.5, alpha=0.3,
                     color=color, zorder=1)
        
        # Format y-axis ticks
        from matplotlib.ticker import MaxNLocator
        ax.yaxis.set_major_locator(MaxNLocator(nbins=5, prune='both'))
        
        # Spines - matching PDP style
        ax.spines['left'].set_color(color)
        ax.spines['left'].set_linewidth(1.5)
        ax.spines['bottom'].set_linewidth(1.0)
        ax.spines['bottom'].set_color('#000000')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        
        # Subplot titles with letter labels - NO parentheses, use period
        subplot_letter = ['a.', 'b.', 'c.'][resp_idx]
        subplot_label = SUBPLOT_LABELS[response_var]
        subplot_title = f"{subplot_letter} {subplot_label}"
        ax.set_title(subplot_title, fontsize=9, fontweight='bold',
                    loc='left', pad=6)
        
        # Add p-value annotation with formatted p-value
        # Use short test names: MW for Mann-Whitney, KW for Kruskal-Wallis, ANOVA
        if test_name == 'Mann-Whitney':
            test_abbrev = 'MW'
        elif test_name == 'Kruskal-Wallis':
            test_abbrev = 'KW'
        elif test_name == 'ANOVA':
            test_abbrev = 'ANOVA'
        else:
            test_abbrev = test_name
        
        ax.text(0.02, 0.98, f'{test_abbrev}: p {p_value_text}', 
               transform=ax.transAxes, fontsize=6.5, va='top',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # Main title
    cat_display = cat_var.replace('_', ' ').title()
    fig.suptitle(f'Statistical Comparison: {cat_display}',
                fontsize=10, fontweight='bold', y=0.99)
    
    # Adjust layout
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    
    if output_path:
        plt.savefig(output_path, dpi=600, bbox_inches='tight',
                    facecolor='white', edgecolor='none', format='png')
        print(f"\n✓ Saved: {output_path}")
        plt.close()
    else:
        plt.show()
        plt.close()
    
    return output_path


def batch_1x3_analysis(data_file, categorical_vars, response_vars, output_dir):
    """
    Create separate 1x3 figures for EACH categorical variable
    
    Parameters:
    -----------
    data_file : str
        Path to data file
    categorical_vars : list
        All categorical variables to analyze
    response_vars : list
        The 3 response variables
    output_dir : str
        Output directory
    """
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"\n{'='*80}")
    print(f"BATCH 1x3 STATISTICAL ANALYSIS")
    print(f"{'='*80}")
    print(f"\nCategorical variables: {len(categorical_vars)}")
    print(f"Response variables: {response_vars}")
    print(f"Output directory: {output_dir}")
    
    # Load data
    df = pd.read_csv(data_file)
    
    output_files = []
    
    # Create one 1x3 plot per categorical variable
    for cat_var in categorical_vars:
        if cat_var not in df.columns:
            print(f"\n⚠ Skipping {cat_var} - not in dataset")
            continue
        
        output_path = os.path.join(output_dir, f'stats_1x3_{cat_var}.png')
        
        try:
            create_1x3_categorical_plot(df, cat_var, response_vars, output_path)
            output_files.append(output_path)
        except Exception as e:
            print(f"\n✗ Error creating plot for {cat_var}: {e}")
            import traceback
            traceback.print_exc()
    
    print(f"\n{'='*80}")
    print(f"BATCH ANALYSIS COMPLETE")
    print(f"{'='*80}")
    print(f"Generated {len(output_files)} figures (1 per categorical variable)")
    print(f"Saved to: {output_dir}")
    
    return output_files


# Example usage
if __name__ == "__main__":
    # Configuration
    DATA_FILE = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis/final_dataset_regrouped_categorical.csv'
    OUTPUT_DIR = 'statistical_comparison_1x3'
    
    CATEGORICAL_VARS = ['variety', 'irrigation_type', 'harvest_mode', 'crop_cycle']
    
    RESPONSE_VARIABLES = [
        'tc_per_ha_actual',
        'hugot_sugar',
        'total_recoverable_sugar_t_per_ha'
    ]
    
    
    # Run batch analysis - creates one 1x3 plot per categorical variable
    output_files = batch_1x3_analysis(DATA_FILE, CATEGORICAL_VARS, RESPONSE_VARIABLES, OUTPUT_DIR)
    
    print(f"\n{'='*80}")
    print("✓ ANALYSIS COMPLETE!")
    print(f"{'='*80}")
    print(f"\nGenerated {len(output_files)} publication-ready plots:")
    for i, cat_var in enumerate(CATEGORICAL_VARS, 1):
        print(f"  {i}. stats_1x3_{cat_var}.png")
    print(f"\nOutput directory: {OUTPUT_DIR}")
    print(f"{'='*80}")


    ╔════════════════════════════════════════════════════════════════════════════╗
    ║                                                                            ║
    ║         1x3 STATISTICAL COMPARISON - PUBLICATION STYLE                     ║
    ║                                                                            ║
    ║              One plot per categorical variable                             ║
    ║              Styled to match PDP publication format                        ║
    ║                                                                            ║
    ╚════════════════════════════════════════════════════════════════════════════╝
    

BATCH 1x3 STATISTICAL ANALYSIS

Categorical variables: 4
Response variables: ['tc_per_ha_actual', 'hugot_sugar', 'total_recoverable_sugar_t_per_ha']
Output directory: statistical_comparison_1x3

Creating 1x3 plot for: variety
  Processing 16 categories: ['N 14', 'FR 8783', 'CP 722086', 'N 19', 'CP 742005', 'R 579', 'SP 75184', 

## 9. Scatterplots

This section creates scatterplots of observed values vs fitted values 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import joblib
import os
import json
from pathlib import Path
from scipy.stats import gaussian_kde
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# FONT SETUP 
# ============================================================================

# Check available fonts and set appropriate fallback
available_fonts = [f.name for f in fm.fontManager.ttflist]
if 'DejaVu Sans' in available_fonts:
    font_family = 'DejaVu Sans'
elif 'Liberation Sans' in available_fonts:
    font_family = 'Liberation Sans'
else:
    font_family = 'sans-serif'

print(f"Using font: {font_family}\n")

plt.rcParams.update({
    'font.family': font_family,
    'font.size': 8,
    'axes.labelsize': 9,
    'axes.titlesize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 7,
    'figure.titlesize': 10,
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.major.size': 3,
    'ytick.major.size': 3,
    'lines.linewidth': 1.0,
    'pdf.fonttype': 42,  # TrueType fonts for compatibility
    'ps.fonttype': 42,
})

# ============================================================================
# RESPONSE VARIABLE CONFIGURATIONS - BILINGUAL
# ============================================================================

RESPONSE_VARIABLE_CONFIG = {
    'tc_per_ha_actual': {
        'display_name': {
            'en': 'Cane yield (t ha⁻¹)',
            'fr': 'Rendement en canne (t ha⁻¹)'
        },
        'unit': 't ha⁻¹',
        'ylabel_observed': {
            'en': 'Observed yield (t ha⁻¹)',
            'fr': 'Rendement observé (t ha⁻¹)'
        },
        'ylabel_predicted': {
            'en': 'Predicted yield (t ha⁻¹)',
            'fr': 'Rendement prédit (t ha⁻¹)'
        },
        'colormap': 'viridis'
    },
    'brix_percent': {
        'display_name': {
            'en': 'Brix (%)',
            'fr': 'Brix (%)'
        },
        'unit': '%',
        'ylabel_observed': {
            'en': 'Observed Brix (%)',
            'fr': 'Brix observé (%)'
        },
        'ylabel_predicted': {
            'en': 'Predicted Brix (%)',
            'fr': 'Brix prédit (%)'
        },
        'colormap': 'plasma'
    },
    'pol_cane_percent': {
        'display_name': {
            'en': 'Pol cane (%)',
            'fr': 'Pol canne (%)'
        },
        'unit': '%',
        'ylabel_observed': {
            'en': 'Observed Pol cane (%)',
            'fr': 'Pol canne observé (%)'
        },
        'ylabel_predicted': {
            'en': 'Predicted Pol cane (%)',
            'fr': 'Pol canne prédit (%)'
        },
        'colormap': 'viridis'
    },
    'pol_juice_percent': {
        'display_name': {
            'en': 'Pol juice (%)',
            'fr': 'Pol jus (%)'
        },
        'unit': '%',
        'ylabel_observed': {
            'en': 'Observed Pol juice (%)',
            'fr': 'Pol jus observé (%)'
        },
        'ylabel_predicted': {
            'en': 'Predicted Pol juice (%)',
            'fr': 'Pol jus prédit (%)'
        },
        'colormap': 'plasma'
    },
    'purity_percent': {
        'display_name': {
            'en': 'Purity (%)',
            'fr': 'Pureté (%)'
        },
        'unit': '%',
        'ylabel_observed': {
            'en': 'Observed Purity (%)',
            'fr': 'Pureté observée (%)'
        },
        'ylabel_predicted': {
            'en': 'Predicted Purity (%)',
            'fr': 'Pureté prédite (%)'
        },
        'colormap': 'cividis'
    },
    'glucose_cane_percent': {
        'display_name': {
            'en': 'Glucose (%)',
            'fr': 'Glucose (%)'
        },
        'unit': '%',
        'ylabel_observed': {
            'en': 'Observed Glucose (%)',
            'fr': 'Glucose observé (%)'
        },
        'ylabel_predicted': {
            'en': 'Predicted Glucose (%)',
            'fr': 'Glucose prédit (%)'
        },
        'colormap': 'magma'
    },
    'fiber_percent': {
        'display_name': {
            'en': 'Fiber (%)',
            'fr': 'Fibre (%)'
        },
        'unit': '%',
        'ylabel_observed': {
            'en': 'Observed Fiber (%)',
            'fr': 'Fibre observée (%)'
        },
        'ylabel_predicted': {
            'en': 'Predicted Fiber (%)',
            'fr': 'Fibre prédite (%)'
        },
        'colormap': 'inferno'
    }
}

# Translation dictionary for other elements
TRANSLATIONS = {
    'en': {
        'point_density': 'Point density',
        'fit_line': 'Fit',
        'one_to_one': '1:1 line'
    },
    'fr': {
        'point_density': 'Densité de points',
        'fit_line': 'Ajustement',
        'one_to_one': 'Ligne 1:1'
    }
}

# Models to plot
SELECTED_MODELS = ['RandomForest', 'XGBoost', 'LinearRegression']

# ============================================================================
# DENSITY CALCULATION (from GAM code)
# ============================================================================

def calculate_point_density(x, y, n_bins=50):
    """Calculate density for each point using KDE"""
    # Remove any NaN values
    mask = ~(np.isnan(x) | np.isnan(y))
    x_clean = x[mask]
    y_clean = y[mask]
    
    # Calculate the point density
    xy = np.vstack([x_clean, y_clean])
    
    try:
        kde = gaussian_kde(xy)
        density = kde(xy)
    except:
        # Fallback to simple binning if KDE fails
        density = np.ones(len(x_clean))
    
    # Normalize density
    density = (density - density.min()) / (density.max() - density.min())
    
    return density, mask

# ============================================================================
# DATA LOADING AND PREPARATION
# ============================================================================

def load_data_and_prepare_features(data_path, response_variable, custom_predictors=None, categorical_vars=None):
    """Load and prepare data with categorical encoding (matching training pipeline)"""
    print(f"\nLoading data for {response_variable}...")
    df = pd.read_csv(data_path)
    
    # Check if response variable exists
    if response_variable not in df.columns:
        raise ValueError(f"Response variable '{response_variable}' not found in dataset!")
    
    # Handle missing target values
    initial_count = len(df)
    df = df.dropna(subset=[response_variable])
    final_count = len(df)
    
    if initial_count - final_count > 0:
        print(f"   Dropped {initial_count - final_count} observations with missing {response_variable}")
    
    # Identify year column
    year_column = None
    for col in ['campaign_year', 'Harvest_Year', 'year']:
        if col in df.columns:
            year_column = col
            break
    
    # Encode categorical variables
    encoders = {}
    if categorical_vars:
        print(f"   Encoding categorical variables: {categorical_vars}")
        for cat_var in categorical_vars:
            if cat_var in df.columns:
                cat_encoder = LabelEncoder()
                df[f'{cat_var}_encoded'] = cat_encoder.fit_transform(df[cat_var].astype(str))
                encoders[cat_var] = cat_encoder
    
    # Prepare features
    if custom_predictors is not None:
        FEATURES = [col for col in custom_predictors if col in df.columns]
        # Add encoded categorical variables
        if categorical_vars:
            for cat_var in categorical_vars:
                encoded_col = f'{cat_var}_encoded'
                if encoded_col in df.columns:
                    FEATURES.append(encoded_col)
        print(f"   Using {len(FEATURES)} features")
    else:
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        exclude_cols = [response_variable]
        if year_column:
            exclude_cols.extend([year_column, f'{year_column}_encoded'])
        if categorical_vars:
            exclude_cols.extend(categorical_vars)
        FEATURES = [col for col in numeric_cols if col not in exclude_cols]
    
    # Fill missing values
    for col in FEATURES:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].median())
    
    X = df[FEATURES].copy()
    y = df[response_variable].copy()
    
    print(f"   Data loaded: {X.shape[0]} samples, {X.shape[1]} features")
    print(f"   {response_variable} range: {y.min():.2f} - {y.max():.2f}")
    
    return X, y, FEATURES, encoders

# ============================================================================
# CV PREDICTIONS
# ============================================================================

def get_cv_predictions(model, scaler, X, y, n_splits=5):
    """Get cross-validation predictions for plotting"""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    all_predictions = []
    all_actuals = []
    
    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        # Scale data
        scaler_fold = StandardScaler()
        X_train_scaled = scaler_fold.fit_transform(X_train)
        X_val_scaled = scaler_fold.transform(X_val)
        
        # Clone and train model
        model_fold = model.__class__(**model.get_params())
        model_fold.fit(X_train_scaled, y_train)
        y_pred = model_fold.predict(X_val_scaled)
        
        all_predictions.extend(y_pred)
        all_actuals.extend(y_val)
    
    return np.array(all_actuals), np.array(all_predictions)

# ============================================================================
# METRICS CALCULATION
# ============================================================================

def calculate_metrics(y_true, y_pred):
    """Calculate performance metrics"""
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    bias = np.mean(y_pred - y_true)
    
    # Calculate RPD and RPIQ
    sd = np.std(y_true)
    q75, q25 = np.percentile(y_true, [75, 25])
    iqr = q75 - q25
    rpd = sd / rmse if rmse > 0 else 0
    rpiq = iqr / rmse if rmse > 0 else 0
    
    return {
        'r2': r2,
        'rmse': rmse,
        'mae': mae,
        'bias': bias,
        'rpd': rpd,
        'rpiq': rpiq,
        'n': len(y_true)
    }

# ============================================================================
# PLOTTING FUNCTIONS 
# ============================================================================

def create_scatter_plot_density(ax, y_true, y_pred, model_name, metrics, response_config, language='en'):
    """
    Create a single scatter plot with Nature style and density coloring
    Compact black axes, no grid, no top/right spines
    Bilingual support (English/French)
    """
    
    # Get translations
    trans = TRANSLATIONS[language]
    
    # Calculate density
    density, mask = calculate_point_density(y_true, y_pred)
    
    # Get clean data
    y_true_clean = y_true[mask]
    y_pred_clean = y_pred[mask]
    
    # Sort by density so densest points are on top
    idx = density.argsort()
    y_true_sorted = y_true_clean[idx]
    y_pred_sorted = y_pred_clean[idx]
    density_sorted = density[idx]
    
    # Create scatter plot with density coloring
    scatter = ax.scatter(y_true_sorted, y_pred_sorted, 
                        c=density_sorted, 
                        cmap=response_config['colormap'],
                        s=12,
                        alpha=0.6,
                        edgecolors='none',
                        rasterized=True)
    
    # Add 1:1 line
    lims = [
        np.min([ax.get_xlim(), ax.get_ylim()]),
        np.max([ax.get_xlim(), ax.get_ylim()]),
    ]
    ax.plot(lims, lims, 'k--', linewidth=1.0, alpha=0.6, zorder=1, 
            label=trans['one_to_one'])
    
    # Add best fit line
    z = np.polyfit(y_true_clean, y_pred_clean, 1)
    p = np.poly1d(z)
    ax.plot(y_true_clean, p(y_true_clean), 
            color='red', linewidth=1.2, alpha=0.8, zorder=2,
            label=f'{trans["fit_line"]}: y={z[0]:.3f}x+{z[1]:.2f}')
    
    # Set labels (bilingual)
    ax.set_xlabel(response_config['ylabel_observed'][language], fontweight='normal')
    ax.set_ylabel(response_config['ylabel_predicted'][language], fontweight='normal')
    ax.set_title(model_name, fontweight='bold', pad=10)
    
    # COMPACT BLACK AXES - Remove top and right spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('black')
    ax.spines['bottom'].set_color('black')
    ax.spines['left'].set_linewidth(0.8)
    ax.spines['bottom'].set_linewidth(0.8)
    
    # Set tick colors to black
    ax.tick_params(axis='both', colors='black', which='both')
    ax.tick_params(axis='x', direction='out', length=3, width=0.8)
    ax.tick_params(axis='y', direction='out', length=3, width=0.8)
    
    # Remove grid
    ax.grid(False)
    
    # Tight axis limits for compact appearance
    ax.autoscale(enable=True, axis='both', tight=True)
    
    # Add colorbar with smaller size
    cbar = plt.colorbar(scatter, ax=ax, pad=0.02, 
                       aspect=20,
                       shrink=0.6)
    cbar.set_label(trans['point_density'], rotation=270, labelpad=10, fontsize=6)
    cbar.ax.tick_params(labelsize=5, width=0.5, length=1.5, colors='black')
    cbar.outline.set_linewidth(0.5)
    cbar.outline.set_edgecolor('black')
    
    # Add statistics box with Nature styling
    stats_text = (
        f'$R^2$ = {metrics["r2"]:.3f}\n'
        f'RMSE = {metrics["rmse"]:.3f} {response_config["unit"]}\n'
        f'MAE = {metrics["mae"]:.3f} {response_config["unit"]}\n'
        f'n = {metrics["n"]:,}'
    )
    
    # Position text box in upper left
    props = dict(boxstyle='round,pad=0.5', facecolor='white', 
                edgecolor='black', linewidth=0.5, alpha=0.9)
    ax.text(0.05, 0.95, stats_text, transform=ax.transAxes, 
            fontsize=7, verticalalignment='top', bbox=props,
            family='monospace', color='black')
    
    # Add legend
    legend = ax.legend(loc='lower right', frameon=True, fontsize=6.5,
                      handlelength=1.5, handletextpad=0.5)
    legend.get_frame().set_linewidth(0.5)
    legend.get_frame().set_edgecolor('black')
    
    # Set equal aspect ratio
    ax.set_aspect('equal', adjustable='box')

# ============================================================================
# MAIN PLOTTING FUNCTION - STANDALONE ONLY
# ============================================================================

def create_standalone_plots(base_models_dir, data_path, response_variable, 
                           custom_predictors=None, categorical_vars=None, 
                           models_to_plot=None, n_splits=5, base_output_dir=None,
                           languages=['en', 'fr']):
    """
    Create standalone scatter plots for each model in multiple languages
    Output: Individual plots in English AND French
    All saved at 600 DPI in Nature journal style with compact black axes
    """
    
    # Validate response variable
    if response_variable not in RESPONSE_VARIABLE_CONFIG:
        raise ValueError(f"Response variable '{response_variable}' not configured!\n"
                        f"Available: {list(RESPONSE_VARIABLE_CONFIG.keys())}")
    
    response_config = RESPONSE_VARIABLE_CONFIG[response_variable]
    
    if models_to_plot is None:
        models_to_plot = SELECTED_MODELS
    
    print("\n" + "="*80)
    print(f"CREATING BILINGUAL STANDALONE PLOTS FOR: {response_variable}")
    print(f"Languages: {', '.join(languages)}")
    print("="*80)
    
    # Construct models directory path
    models_dir = Path(base_models_dir) / f'kfold_models_{response_variable}_with_categorical' / 'models'
    
    if not models_dir.exists():
        # Try without categorical suffix
        models_dir = Path(base_models_dir) / f'kfold_models_{response_variable}' / 'models'
        if not models_dir.exists():
            raise FileNotFoundError(f"Models directory not found: {models_dir}")
    
    print(f"Models directory: {models_dir}")
    
    # Load data
    X, y, FEATURES, encoders = load_data_and_prepare_features(
        data_path, response_variable, custom_predictors, categorical_vars
    )
    
    # Create output directory
    if base_output_dir is None:
        base_output_dir = Path(base_models_dir) / f'standalone_plots_{response_variable}'
    else:
        base_output_dir = Path(base_output_dir) / f'{response_variable}_standalone'
    
    base_output_dir.mkdir(parents=True, exist_ok=True)
    print(f"Output directory: {base_output_dir}")
    
    # Store results
    model_results = {}
    
    # Process each model
    print("\nProcessing models...")
    for model_name in models_to_plot:
        model_file = models_dir / f'{model_name}_{response_variable}.pkl'
        
        if not model_file.exists():
            print(f"   Warning: {model_file} not found, skipping...")
            continue
        
        try:
            print(f"   Loading {model_name}...")
            
            # Load model and scaler
            model_data = joblib.load(model_file)
            model = model_data['model']
            scaler = model_data['scaler']
            
            # Get CV predictions
            y_true, y_pred = get_cv_predictions(model, scaler, X, y, n_splits)
            
            # Calculate metrics
            metrics = calculate_metrics(y_true, y_pred)
            
            # Store results
            model_results[model_name] = {
                'y_true': y_true,
                'y_pred': y_pred,
                'metrics': metrics
            }
            
            print(f"      R² = {metrics['r2']:.3f}, RMSE = {metrics['rmse']:.3f}")
            
        except Exception as e:
            print(f"   Error processing {model_name}: {str(e)}")
            continue
    
    if len(model_results) == 0:
        print("No models successfully loaded!")
        return
    
    # ========================================================================
    # CREATE INDIVIDUAL STANDALONE PLOTS IN MULTIPLE LANGUAGES
    # ========================================================================
    
    for language in languages:
        print(f"\nCreating plots in {language.upper()}...")
        
        for model_name in models_to_plot:
            if model_name not in model_results:
                continue
            
            result = model_results[model_name]
            
            # Create individual figure (Nature single column: 89mm = 3.5 inches)
            fig, ax = plt.subplots(1, 1, figsize=(3.5, 3.5), dpi=600)
            
            create_scatter_plot_density(
                ax, 
                result['y_true'], 
                result['y_pred'], 
                model_name, 
                result['metrics'], 
                response_config,
                language=language
            )
            
            # Tight layout with small padding
            plt.tight_layout(pad=0.3)
            
            # Save standalone plot (PNG)
            standalone_filename = f'{response_variable}_{model_name}_{language}.png'
            standalone_path = base_output_dir / standalone_filename
            plt.savefig(standalone_path, dpi=600, bbox_inches='tight', 
                        facecolor='white', edgecolor='none')
            print(f"   ✓ Saved: {standalone_filename}")
            
            # Save standalone plot (PDF)
            standalone_pdf = base_output_dir / f'{response_variable}_{model_name}_{language}.pdf'
            plt.savefig(standalone_pdf, format='pdf', bbox_inches='tight', 
                        facecolor='white', edgecolor='none')
            print(f"   ✓ Saved: {standalone_pdf.name}")
            
            plt.close()
    
    # ========================================================================
    # SAVE METRICS SUMMARY
    # ========================================================================
    print("\nSaving metrics summary...")
    
    summary_data = []
    for model_name, result in model_results.items():
        summary_data.append({
            'Model': model_name,
            'R2': result['metrics']['r2'],
            'RMSE': result['metrics']['rmse'],
            'MAE': result['metrics']['mae'],
            'Bias': result['metrics']['bias'],
            'RPD': result['metrics']['rpd'],
            'RPIQ': result['metrics']['rpiq'],
            'N_samples': result['metrics']['n']
        })
    
    summary_df = pd.DataFrame(summary_data)
    summary_df = summary_df.sort_values('R2', ascending=False)
    
    summary_path = base_output_dir / f'{response_variable}_metrics_summary.csv'
    summary_df.to_csv(summary_path, index=False)
    print(f"   ✓ Saved: {summary_path.name}")
    
    print("\n" + "="*80)
    print(f"STANDALONE PLOTS COMPLETED FOR: {response_variable}")
    print("="*80)
    print("\nGenerated files:")
    for language in languages:
        print(f"\n  {language.upper()} version:")
        for model_name in models_to_plot:
            if model_name in model_results:
                print(f"    - {response_variable}_{model_name}_{language}.png (600 DPI)")
                print(f"    - {response_variable}_{model_name}_{language}.pdf")
    print(f"\n  Metrics:")
    print(f"    - {response_variable}_metrics_summary.csv")
    print(f"\nAll files saved to: {base_output_dir}")
    print(f"  • {font_family} font family")
    print(f"  • Colormap: {response_config['colormap']}")
    print(f"  • Bilingual support: {', '.join(languages)}")
    print("="*80)
    
    return summary_df

# ============================================================================
# BATCH PROCESSING FOR MULTIPLE RESPONSE VARIABLES
# ============================================================================

def create_plots_for_multiple_responses(base_models_dir, data_path, response_variables,
                                       custom_predictors=None, categorical_vars=None,
                                       models_to_plot=None, n_splits=5, base_output_dir=None,
                                       languages=['en', 'fr']):
    """
    Create standalone bilingual Nature-style plots for multiple response variables
    """
    
    if models_to_plot is None:
        models_to_plot = SELECTED_MODELS
    
    print("\n" + "="*80)
    print("BATCH PLOTTING - BILINGUAL STANDALONE PLOTS")
    print("="*80)
    print(f"Response variables: {response_variables}")
    print(f"Models: {models_to_plot}")
    print(f"Languages: {', '.join(languages)}")
    print(f"Font: {font_family}")
    print("="*80)
    
    all_summaries = {}
    
    for response_var in response_variables:
        try:
            summary_df = create_standalone_plots(
                base_models_dir=base_models_dir,
                data_path=data_path,
                response_variable=response_var,
                custom_predictors=custom_predictors,
                categorical_vars=categorical_vars,
                models_to_plot=models_to_plot,
                n_splits=n_splits,
                base_output_dir=base_output_dir,
                languages=languages
            )
            all_summaries[response_var] = summary_df
            
        except Exception as e:
            print(f"\nERROR processing {response_var}: {str(e)}")
            import traceback
            traceback.print_exc()
            continue
    
    print("\n" + "="*80)
    print("BATCH PROCESSING COMPLETE!")
    print("="*80)
    print(f"\nSuccessfully processed {len(all_summaries)}/{len(response_variables)} response variables")
    print(f"Languages generated: {', '.join(languages)}")
    
    return all_summaries

# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    
    # Configuration - UPDATE THESE PATHS AND SETTINGS
    BASE_MODELS_DIR = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis'
    DATA_PATH = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis/final_dataset_regrouped_categorical.csv'
    BASE_OUTPUT_DIR = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis/standalone_plots_nature'
    
    # Custom features (same as training)
    CUSTOM_FEATURES = [
        'ampl_mean', 'TNn', 'TNx', 'Coarse_Silt_percent_complete',
        'Total_Nitrogen_per_mille_complete', 'pH_complete', 'temp_moy_mean', 
        'Exchangeable_Calcium_meq_100g_complete', 'h_maxi_mean', 'CDD', 
        'radiation_ERA_total', 'Rx1', 'R20', 'SDII',
        'h_mini_mean', 'HD', 'Available_P2O5_mg_complete', 
        'Electrical_Conductivity_mmho_cm_complete', 
        'Exchangeable_Sodium_meq_100g_complete',
        'R99p', 'growing_season_rainfall_mm', 'soil_temp_6h_mean', 
        'CWD', 'Mg_K_Ratio_complete'
    ]
    
    # Categorical features (same as training)
    CATEGORICAL_FEATURES = ['variety', 'irrigation_type', 'harvest_mode', 'crop_cycle']
    
    # Models to plot
    MODELS_TO_PLOT = ['RandomForest', 'XGBoost', 'LinearRegression']
    
    # Response variables to process
    RESPONSE_VARIABLES = ['tc_per_ha_actual', 'brix_percent']
    
    # Process multiple response variables in batch
    print("\n" + "="*80)
    print("STARTING BILINGUAL STANDALONE SCATTER PLOT GENERATION")
    print("="*80)
    
    summaries = create_plots_for_multiple_responses(
        base_models_dir=BASE_MODELS_DIR,
        data_path=DATA_PATH,
        response_variables=RESPONSE_VARIABLES,
        custom_predictors=CUSTOM_FEATURES,
        categorical_vars=CATEGORICAL_FEATURES,
        models_to_plot=MODELS_TO_PLOT,
        n_splits=5,
        base_output_dir=BASE_OUTPUT_DIR,
        languages=['en', 'fr']  # Generate both English and French versions
    )
    
    print("\n" + "="*80)
    print("ALL OPERATIONS COMPLETE!")
    print("="*80)
    print(f"  • {font_family} font family")


Using font: DejaVu Sans


STARTING BILINGUAL STANDALONE SCATTER PLOT GENERATION

BATCH PLOTTING - BILINGUAL STANDALONE PLOTS
Nature Journal Style with Compact Black Axes
Response variables: ['tc_per_ha_actual', 'brix_percent']
Models: ['RandomForest', 'XGBoost', 'LinearRegression']
Languages: en, fr
Font: DejaVu Sans

CREATING BILINGUAL STANDALONE PLOTS FOR: tc_per_ha_actual
Languages: en, fr
Models directory: C:\Users\U078-N800\Desktop\Ml_Internship\Copy\Ml_Training\Regrouped_Analysis\kfold_models_tc_per_ha_actual_with_categorical\models

Loading data for tc_per_ha_actual...
   Encoding categorical variables: ['variety', 'irrigation_type', 'harvest_mode', 'crop_cycle']
   Using 27 features
   Data loaded: 13376 samples, 27 features
   tc_per_ha_actual range: 20.51 - 224.83
Output directory: C:\Users\U078-N800\Desktop\Ml_Internship\Copy\Ml_Training\Regrouped_Analysis\standalone_plots_nature\tc_per_ha_actual_standalone

Processing models...
   Loading RandomForest...
      R² = 0.714, R